<div dir="rtl" align="right">

# NB22 — انتشار نسخه و بازتولیدپذیری

این نوت‌بوک داده، کد، مدل، تنظیمات، گزارش‌ها و شواهد موجود فاز دوم
را به یک نسخهٔ قابل ردیابی و قابل بازسازی متصل می‌کند.

ورودی اصلی، بستهٔ نهایی NB21 و Release/Handoffهای موجود مؤلفه‌هاست.
خروجی شامل Manifest، Cardها، محدودیت‌ها، Evidence index،
آزمون Restore/Validate-only و نتیجهٔ واقعی Final Release Gate است.

در این مرحله آزمایش‌ها و داوری‌های معتبر قبلی تکرار نمی‌شوند.
تولید دوبارهٔ Golden، Embedding، پاسخ و داوری جمعیت، کار پیش‌فرض NB22 نیست.

موفقیت نهایی به کنترل نسخه‌ها و هش‌ها، بازخوانی مستقل،
بازتولیدپذیری و احراز شرایط پذیرش وابسته است.
سلامت بسته‌بندی، جای PASS کیفیت محصول را نمی‌گیرد.

## نقطهٔ شروع

ابتدا وجود نسخهٔ محلی NB22/NB23 و پنج فایل نهایی C082/C085
بررسی می‌شود. این بررسی فقط‌خواندنی است و هیچ API، اجرای مدل،
تغییر چک‌لیست یا نوشتن Artifact ندارد.

</div>

In [1]:
# NB21-C001 — Centralized standard-library imports for NB21.
# Imports only; no processing, file writes, repository changes, or API calls.

from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
from importlib import metadata
import unicodedata
import ast
import csv
import hashlib
import itertools
import json
import math
import os
import platform
import random
import re
import statistics
import subprocess
import sys
import tempfile
import time
from urllib.parse import urlsplit, urlunsplit
from dotenv import dotenv_values
import tiktoken
import jsonschema
import requests

<div dir="rtl" align="right">

### NB22-C02 — بررسی ورودی نهایی و نسخه‌های محلی

- ریشه و وضعیت Git محلی ثبت می‌شود.
- وجود NB22/NB23 و پنج فایل مشخص‌شده در تحویل‌نامه بررسی می‌شود.
- هش بایت فایل‌ها با هش گزارش‌شده مقایسه می‌شود.
- ساختار اولیهٔ JSON برای طراحی Reader متناسب با دادهٔ واقعی نمایش داده می‌شود.
- فایل مفقود یا هش متفاوت، خودکار به معنی انجام‌نشدن کار قبلی نیست.
- این سل دریافت کامل بسته یا پذیرش Release را اعلام نمی‌کند.

API تازه، اجرای مدل و نوشتن فایل: صفر.

</div>

In [2]:
# NB22-C02 — Read-only final-handoff preflight
# Scope: local Git reads and five explicit files; no network or writes.

def nb22_c02_run():
    def git(cwd, *args):
        result = subprocess.run(
            ["git", "-C", str(cwd), *args],
            capture_output=True,
            text=True,
            check=False,
        )
        if result.returncode != 0:
            raise RuntimeError(
                f"LOCAL_GIT_READ_FAILED: {' '.join(args)}"
            )
        return result.stdout.strip()

    root = Path(git(Path.cwd(), "rev-parse", "--show-toplevel"))
    roadmap = (
        root / "data/Road_Maps/"
        "RAG_finance_phase2_modular_notebooks_execution_plan.md"
    )
    if not roadmap.is_file():
        raise RuntimeError("EXPECTED_PROJECT_ROADMAP_NOT_FOUND")

    branch = git(root, "branch", "--show-current")
    if branch != "rag-phase2":
        raise RuntimeError(
            f"UNEXPECTED_BRANCH: {branch}; no branch change performed"
        )

    final_dir = (
        "reports/phase2/evaluation/"
        "NB21C.C085.action49_final_summary/"
    )
    handoff_dir = (
        "reports/phase2/evaluation/"
        "NB21C.C082.action46_handoff/"
        "9186bb5a30456486b29d521db1660b944"
        "c72bf8c4fcb9d816cf578b1937ea1d9/"
    )

    expected = [
        (
            "C085_report",
            final_dir,
            "C085.report",
            "a2e7f80edb4a8057a4f184bcc8b319bc"
            "aaac432261c0ecb8ae2cdd2817427dde",
            ".json",
        ),
        (
            "C085_actions",
            final_dir,
            "C085.actions01_49_status",
            "234e1e8326bbaf5e8a06082e47b97ef6"
            "0f60b99b7d355284ca594d9b7dce5a88",
            ".json",
        ),
        (
            "C085_checklist_proposal",
            final_dir,
            "C085.checklist_status_proposal_fa",
            "39d39dd58694ce9187d78cf4de8bc601"
            "825b39742f2c8aebdf2ff4009303c069",
            ".md",
        ),
        (
            "C082_manifest",
            handoff_dir,
            "C082.manifest",
            "212f8f411a5c39b693e108c5ba3100a2"
            "91e40355625e60ab14bae343af88e174",
            ".json",
        ),
        (
            "C082_handoff",
            handoff_dir,
            "C082.handoff_NB22",
            "aeffc73207cc137a492496eb3083d290"
            "88afeab4447bc749faafc6aea397bdec",
            ".json",
        ),
    ]

    def shape(value, depth=0):
        """Show structure without printing report text or private values."""
        if isinstance(value, dict):
            if depth >= 2:
                return {"type": "object", "keys": list(value)}
            return {
                key: shape(item, depth + 1)
                for key, item in value.items()
            }
        if isinstance(value, list):
            return {
                "type": "array",
                "count": len(value),
                "first_item_shape": (
                    shape(value[0], depth + 1) if value else None
                ),
            }
        return {"type": type(value).__name__}

    rows = []
    for role, directory, stem, expected_sha, suffix in expected:
        relative = directory + stem + "." + expected_sha + suffix
        path = root / relative
        row = {
            "role": role,
            "path": relative,
            "expected_sha256_source": "USER_SUPPLIED_HANDOFF",
            "expected_file_bytes_sha256": expected_sha,
        }

        if not path.is_file():
            row["status"] = "NOT_AVAILABLE_AT_EXPLICIT_LOCAL_PATH"
            rows.append(row)
            continue

        digest = hashlib.sha256()
        with path.open("rb") as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(block)

        actual_sha = digest.hexdigest()
        row.update({
            "bytes": path.stat().st_size,
            "actual_file_bytes_sha256": actual_sha,
            "hash_matches": actual_sha == expected_sha,
        })

        if actual_sha != expected_sha:
            row["status"] = "HASH_MISMATCH_REQUIRES_RECONCILIATION"
        elif suffix == ".json":
            try:
                with path.open("r", encoding="utf-8") as stream:
                    value = json.load(stream)
                row["status"] = "HASH_MATCH_JSON_READABLE"
                row["json_structure"] = shape(value)
            except (ValueError, UnicodeError) as exc:
                row["status"] = "HASH_MATCH_JSON_READ_FAILED"
                row["error_type"] = type(exc).__name__
        else:
            row["status"] = "HASH_MATCH_CONTENT_NOT_YET_REVIEWED"

        rows.append(row)

    notebook_candidates = sorted({
        path.relative_to(root).as_posix()
        for pattern in ("22*.ipynb", "23*.ipynb")
        for path in (root / "notebooks/phase2").glob(pattern)
        if path.is_file()
    })

    summary = {
        "status": "READONLY_PREFLIGHT_COMPLETED_NOT_ACCEPTANCE",
        "branch": branch,
        "local_head": git(root, "rev-parse", "HEAD"),
        "local_changes": git(
            root, "status", "--porcelain=v1", "--untracked-files=normal"
        ).splitlines(),
        "unmerged_paths": git(
            root, "diff", "--name-only", "--diff-filter=U"
        ).splitlines(),
        "local_nb22_nb23_candidates": notebook_candidates,
        "explicit_files": rows,
        "full_manifest_reference_validation_performed": False,
        "release_acceptance_assessed": False,
        "files_written": 0,
        "model_api_calls": 0,
        "network_calls": 0,
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary


nb22_c02_result = nb22_c02_run()

{
  "status": "READONLY_PREFLIGHT_COMPLETED_NOT_ACCEPTANCE",
  "branch": "rag-phase2",
  "local_head": "271c5a1b6115021dcc538a3a11197a3e255d383e",
  "local_changes": [
    "?? notebooks/phase2/22_release_and_reproducibility.ipynb",
    "?? notebooks/phase2/checklist/NB22_remaining_work_checklist_fa.md"
  ],
  "unmerged_paths": [],
  "local_nb22_nb23_candidates": [
    "notebooks/phase2/22_release_and_reproducibility.ipynb"
  ],
  "explicit_files": [
    {
      "role": "C085_report",
      "path": "reports/phase2/evaluation/NB21C.C085.action49_final_summary/C085.report.a2e7f80edb4a8057a4f184bcc8b319bcaaac432261c0ecb8ae2cdd2817427dde.json",
      "expected_sha256_source": "USER_SUPPLIED_HANDOFF",
      "expected_file_bytes_sha256": "a2e7f80edb4a8057a4f184bcc8b319bcaaac432261c0ecb8ae2cdd2817427dde",
      "bytes": 5047,
      "actual_file_bytes_sha256": "a2e7f80edb4a8057a4f184bcc8b319bcaaac432261c0ecb8ae2cdd2817427dde",
      "hash_matches": true,
      "status": "HASH_MATCH_JSON_READABL

<div dir="rtl" align="right">

### NB22-C03 — اقدامات ۳ و ۴: مصرف جمع‌بندی و بستهٔ تحویل NB21

این بسته هفت زیرکار دارد:

۱. خواندن گزارش نهایی C085.
۲. بررسی حضور و یکتایی وضعیت هر ۴۹ اقدام.
۳. خواندن پیشنهاد وضعیت چک‌لیست، بدون اعمال آن.
۴. خواندن قرارداد و محدودیت‌های Handoff.
۵. تطبیق ارجاع Manifest با فایل کنترل‌شده در C02.
۶. کنترل وجود، هش، اندازه و خوانایی فایل‌های مستقیم بسته.
۷. خواندن شرایط باز و نمایش وضعیت واقعی برای ادامه.

ورودی‌ها همان فایل‌های تأییدشدهٔ C02 هستند.
ارجاع‌های تاریخیِ داخل گزارش‌ها به‌صورت بازگشتی دنبال نمی‌شوند.

سلامت فنی دریافت بسته، پذیرش کیفیت یا Release نیست.
وضعیت تاریخی C082 با جمع‌بندی جدیدتر C085 جدا نمایش داده می‌شود.

API، اجرای مدل، تغییر چک‌لیست و نوشتن فایل: صفر.

</div>

In [3]:
# NB22-C03 — Consume NB21 summary and direct handoff package
# Scope: offline reads only; no regeneration, network, or file writes.

def nb22_c03_run(preflight):
    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    required_roles = {
        "C085_report", "C085_actions", "C085_checklist_proposal",
        "C082_manifest", "C082_handoff",
    }
    bindings = {}
    for row in preflight["explicit_files"]:
        role = row["role"]
        if role in bindings:
            raise RuntimeError(f"DUPLICATE_PREFLIGHT_ROLE: {role}")
        bindings[role] = row
    if not required_roles.issubset(bindings):
        raise RuntimeError("REQUIRED_PREFLIGHT_BINDINGS_MISSING")

    records, loaded = {}, {}
    conflicts = []

    def local_path(relative):
        if not isinstance(relative, str) or not relative:
            raise ValueError("INVALID_PATH")
        p = Path(relative)
        if p.is_absolute() or ".." in p.parts:
            raise ValueError("NON_REPOSITORY_RELATIVE_PATH")
        target = (root / p).resolve()
        target.relative_to(root)
        return target

    def strict_json(text):
        def pairs(items):
            result = {}
            for key, value in items:
                if key in result:
                    raise ValueError("DUPLICATE_JSON_KEY")
                result[key] = value
            return result

        def invalid_constant(value):
            raise ValueError("NONFINITE_JSON_NUMBER")

        return json.loads(
            text, object_pairs_hook=pairs,
            parse_constant=invalid_constant,
        )

    def consume(binding, source, declared_format=None):
        relative = binding.get("path")
        expected = binding.get("file_bytes_sha256")
        if not isinstance(expected, str) or not re.fullmatch(
            r"[0-9a-f]{64}", expected
        ):
            raise ValueError(f"INVALID_HASH_BINDING: {source}")

        if relative in records:
            old = records[relative]
            if old["expected_sha256"] != expected:
                conflicts.append({
                    "path": relative,
                    "source": source,
                    "reason": "CONFLICTING_HASH_BINDINGS",
                })
                return False
            if "bytes" in binding and (
                old.get("bytes") is not None
                and binding["bytes"] != old["bytes"]
            ):
                conflicts.append({
                    "path": relative,
                    "source": source,
                    "reason": "CONFLICTING_SIZE_BINDINGS",
                })
                return False
            return old["status"] == "VERIFIED_READABLE"

        row = {
            "path": relative,
            "source": source,
            "expected_sha256": expected,
        }
        records[relative] = row
        try:
            path = local_path(relative)
            if not path.is_file():
                raise FileNotFoundError("EXPLICIT_FILE_NOT_AVAILABLE")

            raw = path.read_bytes()
            row["bytes"] = len(raw)
            row["actual_sha256"] = hashlib.sha256(raw).hexdigest()
            if row["actual_sha256"] != expected:
                raise ValueError("FILE_BYTES_HASH_MISMATCH")
            if "bytes" in binding and binding["bytes"] != len(raw):
                raise ValueError("DECLARED_SIZE_MISMATCH")

            text = raw.decode("utf-8")
            fmt = str(
                declared_format or binding.get("format") or ""
            ).lower()
            suffix = path.suffix.lower()

            if suffix == ".jsonl" or fmt == "jsonl":
                value = [
                    strict_json(line)
                    for line in text.splitlines() if line.strip()
                ]
            elif suffix == ".json" or fmt == "json":
                value = strict_json(text)
            else:
                value = text

            loaded[relative] = value
            row["status"] = "VERIFIED_READABLE"
            row["parsed_type"] = type(value).__name__
            if isinstance(value, (list, dict)):
                row["item_count"] = len(value)
            return True
        except Exception as exc:
            row["status"] = "REQUIRES_RECONCILIATION"
            row["error"] = (
                str(exc) if isinstance(
                    exc, (ValueError, FileNotFoundError)
                ) else type(exc).__name__
            )
            return False

    for role in sorted(required_roles):
        source = bindings[role]
        consume({
            "path": source["path"],
            "file_bytes_sha256":
                source["expected_file_bytes_sha256"],
            "bytes": source["bytes"],
        }, role)

    root_failures = [
        role for role in sorted(required_roles)
        if bindings[role]["path"] not in loaded
    ]
    if root_failures:
        print(json.dumps({
            "status": "INPUT_RECONCILIATION_REQUIRED",
            "failed_roles": root_failures,
            "files": list(records.values()),
            "files_written": 0,
            "model_api_calls": 0,
        }, ensure_ascii=False, indent=2))
        return {"records": records, "loaded": loaded}

    report = loaded[bindings["C085_report"]["path"]]
    actions = loaded[bindings["C085_actions"]["path"]]
    proposal = loaded[bindings["C085_checklist_proposal"]["path"]]
    manifest = loaded[bindings["C082_manifest"]["path"]]
    handoff = loaded[bindings["C082_handoff"]["path"]]

    if not all(isinstance(v, dict) for v in (
        report, manifest, handoff
    )):
        raise ValueError("ROOT_OBJECT_SCHEMA_MISMATCH")
    if not isinstance(actions, list):
        raise ValueError("ACTION_LEDGER_MUST_BE_ARRAY")

    action_numbers = [
        item.get("action") if isinstance(item, dict) else None
        for item in actions
    ]
    actions_valid = (
        len(actions) == 49
        and all(type(n) is int for n in action_numbers)
        and sorted(action_numbers) == list(range(1, 50))
        and all(
            isinstance(item.get("status"), str)
            and bool(item["status"].strip())
            for item in actions
        )
    )

    manifest_link = handoff.get("manifest")
    manifest_link_matches = (
        isinstance(manifest_link, dict)
        and manifest_link.get("path")
            == bindings["C082_manifest"]["path"]
        and manifest_link.get("file_bytes_sha256")
            == bindings["C082_manifest"][
                "expected_file_bytes_sha256"
            ]
    )

    artifacts = manifest.get("artifacts")
    if not isinstance(artifacts, list):
        raise ValueError("MANIFEST_ARTIFACTS_MUST_BE_ARRAY")

    malformed_bindings = []
    direct_bindings = [
        (f"manifest.artifacts[{i}]", item)
        for i, item in enumerate(artifacts)
    ]
    for container_name, container in (
        ("manifest", manifest), ("handoff", handoff)
    ):
        for key in (
            "evidence_index", "open_conditions", "source_inventory"
        ):
            if key in container:
                direct_bindings.append((
                    f"{container_name}.{key}", container[key]
                ))

    for source, binding in direct_bindings:
        if not isinstance(binding, dict):
            malformed_bindings.append(source)
            continue
        try:
            consume(binding, source)
        except ValueError as exc:
            malformed_bindings.append({
                "source": source, "error": str(exc)
            })

    # Consume explicit C085 links to the action ledger and C082 package.
    # Other C085 outputs are retained as pointers, not recursively read.
    output_links = report.get("outputs", {})
    if not isinstance(output_links, dict):
        raise ValueError("C085_OUTPUTS_MUST_BE_OBJECT")
    for key in (
        "actions01_49_status", "checklist_status_proposal",
        "existing_NB22_handoff", "existing_evaluation_manifest"
    ):
        if key in output_links:
            binding = output_links[key]
            if isinstance(binding, dict):
                try:
                    consume(binding, f"C085.outputs.{key}")
                except ValueError as exc:
                    malformed_bindings.append({
                        "source": f"C085.outputs.{key}",
                        "error": str(exc),
                    })
            else:
                malformed_bindings.append(f"C085.outputs.{key}")

    conditions = {}
    for name, container in (
        ("manifest", manifest), ("handoff", handoff)
    ):
        binding = container.get("open_conditions")
        if isinstance(binding, dict):
            path = binding.get("path")
            if path in loaded:
                conditions[path] = loaded[path]

    def safe_view(value):
        """Redact common secret fields and personal home prefixes."""
        if isinstance(value, dict):
            return {
                key: (
                    "[REDACTED]" if any(
                        token in key.lower()
                        for token in (
                            "api_key", "authorization",
                            "password", "access_token", "secret"
                        )
                    ) else safe_view(item)
                )
                for key, item in value.items()
            }
        if isinstance(value, list):
            return [safe_view(item) for item in value]
        if isinstance(value, str):
            value = value.replace(str(root), "<PROJECT_ROOT>")
            return re.sub(
                r"/(?:Users|home)/[^/\s]+",
                "<USER_HOME>", value,
            )
        return value

    failed_files = [
        row for row in records.values()
        if row["status"] != "VERIFIED_READABLE"
    ]
    technical_ok = (
        actions_valid and manifest_link_matches
        and not failed_files and not conflicts
        and not malformed_bindings
    )

    summary = {
        "status": (
            "DIRECT_PACKAGE_READ_CHECKS_PASS_NOT_RELEASE_ACCEPTANCE"
            if technical_ok else "RECONCILIATION_REQUIRED"
        ),
        "checks": {
            "49_unique_actions_with_explicit_status": actions_valid,
            "handoff_manifest_matches_C02": manifest_link_matches,
            "direct_files_verified_readable": not failed_files,
            "no_binding_conflicts": not conflicts,
            "no_malformed_bindings": not malformed_bindings,
        },
        "manifest_artifact_entries": len(artifacts),
        "unique_files_read": len(loaded),
        "C085_current_summary": {
            key: report[key] for key in (
                "status", "schema_version", "actions_accounted",
                "action48_status", "completion",
                "overall_quality_gate_status",
                "metric_gate_status_counts",
                "distinct_open_issue_count",
                "report_package_independent_reload",
            ) if key in report
        },
        "actions01_49": [
            {
                key: item[key] for key in (
                    "action", "status", "note_fa",
                    "new_acceptance_claimed"
                ) if key in item
            }
            for item in actions if isinstance(item, dict)
        ],
        "checklist_proposal_read_not_applied": proposal,
        "C082_historical_handoff": handoff,
        "open_conditions_read": conditions,
        "file_checks": list(records.values()),
        "binding_conflicts": conflicts,
        "malformed_bindings": malformed_bindings,
        "historical_external_references_followed": False,
        "semantic_accuracy_reassessed": False,
        "release_acceptance_assessed": False,
        "checklist_modified": False,
        "files_written": 0,
        "model_api_calls": 0,
        "network_calls": 0,
    }
    print(json.dumps(
        safe_view(summary), ensure_ascii=False, indent=2
    ))
    return {
        "summary": summary,
        "loaded": loaded,
        "records": records,
    }


nb22_c03_result = nb22_c03_run(nb22_c02_result)

{
  "status": "DIRECT_PACKAGE_READ_CHECKS_PASS_NOT_RELEASE_ACCEPTANCE",
  "checks": {
    "49_unique_actions_with_explicit_status": true,
    "handoff_manifest_matches_C02": true,
    "direct_files_verified_readable": true,
    "no_binding_conflicts": true,
    "no_malformed_bindings": true
  },
  "manifest_artifact_entries": 25,
  "unique_files_read": 30,
  "C085_current_summary": {
    "status": "ACTION49_FINAL_REPORTING_COMPLETE_WITH_DEFERRED_EXECUTION_AND_QUALITY_LIMITS",
    "schema_version": "NB21C_C085_ACTION49_FINAL_SUMMARY_V1",
    "actions_accounted": 49,
    "action48_status": "DEFERRED_BY_USER_UNTIL_NOTEBOOK_CONSOLIDATION",
    "completion": {
      "NB21_clean_kernel_execution_complete": false,
      "NB21_fully_accepted": false,
      "action49_reporting_complete": true,
      "all_49_actions_given_explicit_status": true,
      "all_49_actions_passed": false,
      "checklist_proposal_written": true,
      "facts_interpretations_pending_separated": true,
      "final_scop

<div dir="rtl" align="right">

### NB22-C04 — اقدام ۵: دفتر اتصال الزامات و شواهد

این اقدام چهار زیرکار دارد:

۱. ثبت مسیر، هش و نتیجهٔ کنترل ۳۰ فایل مصرف‌شده در C03.
۲. اتصال شواهد مرتبط موجود به هر یک از ۲۸ اقدام NB22.
۳. ثبت کار باقی‌مانده و مسئول رسیدگی، بدون فرض انجام‌نشدن کار قبلی.
۴. ذخیرهٔ نسخه‌دار دفتر اتصال و بازخوانی مستقل آن از دیسک.

ارجاع یک گزارش به الزام، به معنی کفایت شاهد یا پایان آن الزام نیست.
این دفتر، برنامهٔ مصرف شواهد است؛ جای Manifest نهایی Release را نمی‌گیرد.

پذیرش‌های NB19/NB20، فایل‌های اصلی محصول و منابع تاریخیِ خارج
از بسته در این سل دوباره ممیزی نمی‌شوند.
اقدام ۶ و کنترل‌های بعدی از همین دفتر ادامه پیدا می‌کنند.

فقط خروجی جدید این اقدام نوشته می‌شود؛ ورودی‌ها و چک‌لیست تغییر نمی‌کنند.
API، اجرای مدل و درخواست شبکه: صفر.

</div>

In [4]:
# NB22-C04 — Action 05: persist the requirement-to-evidence ledger
# Scope: reuse C03 results; write and independently reload one new artifact.

import os
import tempfile


def nb22_c04_run(previous):
    if previous["summary"]["status"] != (
        "DIRECT_PACKAGE_READ_CHECKS_PASS_NOT_RELEASE_ACCEPTANCE"
    ):
        raise RuntimeError("C03_DIRECT_PACKAGE_CHECKS_NOT_PASSED")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    records = previous["records"]
    loaded = previous["loaded"]
    if set(records) != set(loaded):
        raise RuntimeError("C03_RECORDS_AND_LOADED_FILES_DIFFER")
    if any(
        row["status"] != "VERIFIED_READABLE"
        or row["expected_sha256"] != row["actual_sha256"]
        for row in records.values()
    ):
        raise RuntimeError("C03_VERIFICATION_RECORD_INCONSISTENT")

    def binding(path):
        row = records[path]
        return {
            "path": path,
            "file_bytes_sha256": row["actual_sha256"],
            "bytes": row["bytes"],
            "verification_source": "NB22-C03",
        }

    def find_one(stem):
        matches = [
            path for path in loaded
            if Path(path).name.startswith(stem + ".")
        ]
        if len(matches) != 1:
            raise RuntimeError(f"EVIDENCE_NOT_UNIQUE: {stem}")
        return matches[0]

    evidence_names = {
        "summary": "C085.report",
        "actions": "C085.actions01_49_status",
        "proposal": "C085.checklist_status_proposal_fa",
        "handoff": "C082.handoff_NB22",
        "manifest": "C082.manifest",
        "benchmark": "C082.benchmark",
        "ablation": "C082.ablation",
        "contract": "C082.frozen_contract",
        "gates": "C082.metric_gates16",
        "dependencies": "C082.upstream_dependencies",
        "scenarios": "C082.regression_scenarios322",
        "catalogue": "C082.failure_catalogue322",
        "specs": "C082.regression_specs322",
        "reviews": "C082.open_reviews",
        "conditions": "C082.open_conditions",
        "index": "C082.evidence_index",
        "inventory": "C082.source_inventory",
    }
    evidence = {
        name: binding(find_one(stem))
        for name, stem in evidence_names.items()
    }

    # Existing reports are starting evidence, not automatic acceptance.
    # Missing bindings below mean "not established in this consumed scope".
    plan = [
        (1, "مأموریت و مرز NB22/NB23",
         ["handoff", "summary"], "NB22",
         "ثبت نگاشت مستقیم به نقشه و مرز مسئولیت‌ها"),
        (2, "وضعیت محلی و فایل نوت‌بوک",
         [], "NB22",
         "مصرف خروجی C02؛ خواندن دوبارهٔ Git بدون تغییر مؤثر لازم نیست"),
        (3, "مصرف جمع‌بندی C085",
         ["summary", "actions", "proposal"], "NB22",
         "شرط خواندن و کنترل در C03 محقق شده؛ پیشنهاد تیک اعمال نشده"),
        (4, "مصرف بستهٔ تحویل NB21",
         ["handoff", "manifest", "conditions", "index"], "NB22",
         "کنترل مستقیم C03 محقق شده؛ پذیرش Release از آن نتیجه نمی‌شود"),
        (5, "دفتر اتصال واحد",
         ["index", "inventory", "summary"], "NB22",
         "تولید و بازخوانی همین دفتر"),
        (6, "تطبیق پذیرش‌های NB19/NB20 و سنجه‌های BLOCKED",
         ["dependencies", "conditions", "gates"], "NB22",
         "یافتن و تطبیق تصمیم‌ها و شواهد پذیرش موجود؛ نه بازداوری عمومی"),
        (7, "Version IDهای کاندیدا",
         ["inventory", "manifest", "contract"], "NB22",
         "استخراج نسخه‌های واقعی محصول و منشأ؛ موارد نامشخص محفوظ"),
        (8, "مسیرهای Canonical",
         ["inventory", "index"], "NB22",
         "تعیین مرجع با Producer، نسخه و هش؛ نه صرف شباهت نام"),
        (9, "هش دارایی‌های اصلی",
         ["inventory", "manifest"], "NB22",
         "کنترل دارایی‌های Dataset/Chunk/Index/Config در دامنهٔ Release"),
        (10, "Schema و اتصال دارایی‌های محصول",
         ["inventory", "contract"], "NB22",
         "مصرف قرارداد و Validator معتبر مربوط به نسخهٔ محصول"),
        (11, "موجودی قابل Restore و منابع بیرونی",
         ["inventory", "index", "handoff"], "NB22",
         "تعیین فایل، Pointer، Cache و Revision لازم؛ بدون regeneration"),
        (12, "Manifest کاندیدای Release",
         ["manifest", "inventory"], "NB22",
         "ساخت Manifest کل کاندیدا؛ C082 فقط بستهٔ گزارش NB21 است"),
        (13, "Data Card",
         ["benchmark", "inventory"], "NB22",
         "مصرف Card/Manifest قبلی و تکمیل فقط شکاف نسخهٔ فعال"),
        (14, "Retrieval Card",
         ["benchmark", "ablation", "contract"], "NB22",
         "اتصال Configuration و ADR واقعی مؤلفه‌ها"),
        (15, "Evaluation Card",
         ["benchmark", "ablation", "gates", "conditions"], "NB22",
         "تجمیع نتایج موجود با دامنه و مخرج؛ بدون ارزیابی دوباره"),
        (16, "منشأ مدل و زمان/هزینه",
         ["benchmark", "summary", "conditions"], "NB22",
         "حفظ تفکیک مولد، Recovery و داور و محدودیت telemetry"),
        (17, "Known limitations",
         ["conditions", "reviews", "catalogue", "summary"], "NB22",
         "انتقال مسائل و تعویق‌ها با ارجاع؛ ثبت محدودیت به معنی حل آن نیست"),
        (18, "Evidence index و Release notes",
         ["index", "inventory", "handoff"], "NB22",
         "تکمیل اتصال در سطح Release و ثبت حدود rollback compatibility"),
        (19, "محیط و برنامهٔ اجرای امن",
         ["summary", "handoff"], "NB22",
         "بررسی محیط و Cache-safe بودن اجرای نسخهٔ نهایی"),
        (20, "Restore و Validate-only",
         ["manifest", "inventory"], "NB22",
         "Restore واقعی در مقصد جدا؛ Reload بسته جای آن نیست"),
        (21, "دامنهٔ تجمیع و اجرای پاک",
         ["summary", "actions"], "NB22",
         "مصرف تصمیم تعویق ۴۸ و تعیین دامنهٔ نسخه‌های نهایی"),
        (22, "اجرای پاک نسخه‌های نهایی",
         ["summary", "actions"], "NB22",
         "تا رسید اجرای واقعی وجود ندارد PASS اعلام نشود"),
        (23, "سلامت نوت‌بوک‌های Release",
         [], "NB22",
         "فهرست نسخه‌های فعال و رسید اجرای ذخیره‌شده بررسی شود"),
        (24, "خروجی بدون Secret و مسیر شخصی",
         [], "NB22",
         "بررسی نسخهٔ قابل تحویل؛ بدون دست‌کاری پنهانی شواهد خام"),
        (25, "انتشار Atomic و Reload خروجی نهایی",
         ["manifest"], "NB22",
         "کنترل خروجی‌های جدید NB22؛ کنترل C03 جای آن نیست"),
        (26, "Final Release Gate",
         ["gates", "dependencies", "conditions"], "NB22",
         "ارزیابی شروط Release پس از تکمیل وابستگی‌ها؛ FAIL/BLOCKED محفوظ"),
        (27, "Acceptance و Checkpoint",
         ["summary", "handoff", "conditions"], "NB22",
         "ثبت وضعیت واقعی؛ پذیرش کامل از سلامت فایل‌ها نتیجه نشود"),
        (28, "بستهٔ خروجی NB22 برای مرحلهٔ بعد",
         ["handoff"], "NB22",
         "آماده‌کردن ورودی NB23؛ اجرای اقدامات NB23 در این ردیف نیست"),
    ]

    rows = []
    for number, title, names, owner, remaining in plan:
        rows.append({
            "requirement_id": f"NB22-A{number:02d}",
            "action_number": number,
            "title_fa": title,
            "existing_evidence": [evidence[name] for name in names],
            "evidence_scope": "C03_VERIFIED_NB21_REPORT_PACKAGE",
            "evidence_sufficiency_assessed_for_full_requirement": False,
            "remaining_work_or_scope_fa": remaining,
            "coordination_owner": owner,
            "corrective_component_owner_assigned": False,
            "absence_projectwide_claimed": False,
            "checkmark_authorized_by_this_ledger": False,
        })

    if [row["action_number"] for row in rows] != list(range(1, 29)):
        raise RuntimeError("REQUIREMENT_COVERAGE_NOT_EXACTLY_01_28")

    artifact = {
        "artifact_type": "NB22_REQUIREMENT_EVIDENCE_LEDGER",
        "schema_version": "NB22_C004_ACTION05_LEDGER_V1",
        "scope": "REQUIREMENT_TO_EXISTING_EVIDENCE_MAPPING",
        "input_validation_source": "NB22-C03",
        "inputs": [binding(path) for path in sorted(records)],
        "existing_evidence": evidence,
        "requirements": rows,
        "upstream_state_preserved":
            previous["summary"]["C085_current_summary"],
        "limitations": [
            "Existing report linkage does not establish full requirement acceptance.",
            "Product artifact closure and historical external references are not checked here.",
            "NB19/NB20 acceptance reconciliation remains action 06.",
            "No new semantic assessment, model execution or regression execution.",
            "No projectwide absence inferred from this report-package scope.",
        ],
        "release_acceptance_granted": False,
        "phase2_acceptance_granted": False,
        "checklist_modified": False,
        "new_api_calls": 0,
    }

    raw = json.dumps(
        artifact, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    digest = hashlib.sha256(raw).hexdigest()

    directory = root / "reports/phase2/release/NB22.C004.action05"
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"C004.evidence_ledger.{digest}.json"

    if target.exists():
        if target.read_bytes() != raw:
            raise RuntimeError("EXISTING_OUTPUT_CONTENT_CONFLICT")
        write_status = "REUSED_IDENTICAL"
    else:
        fd, temporary = tempfile.mkstemp(
            prefix=".C004.", suffix=".tmp", dir=directory
        )
        temporary = Path(temporary)
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())

            temporary_raw = temporary.read_bytes()
            if (
                hashlib.sha256(temporary_raw).hexdigest() != digest
                or json.loads(temporary_raw) != artifact
            ):
                raise RuntimeError("TEMPORARY_OUTPUT_VALIDATION_FAILED")

            # Exclusive publication prevents overwriting an existing file.
            try:
                os.link(temporary, target)
                write_status = "CREATED"
            except FileExistsError:
                if target.read_bytes() != raw:
                    raise RuntimeError("CONCURRENT_OUTPUT_CONFLICT")
                write_status = "REUSED_IDENTICAL"
        finally:
            temporary.unlink(missing_ok=True)

    # Reload from disk, independently of the original object.
    saved = target.read_bytes()
    restored = json.loads(saved)
    if (
        hashlib.sha256(saved).hexdigest() != digest
        or restored != artifact
        or len(restored["requirements"]) != 28
        or len(restored["inputs"]) != len(records)
    ):
        raise RuntimeError("FINAL_RELOAD_VALIDATION_FAILED")

    summary = {
        "status": "ACTION05_EVIDENCE_MAPPING_WRITTEN_AND_RELOADED",
        "subtasks_completed": 4,
        "requirements_mapped": len(restored["requirements"]),
        "verified_input_bindings_reused": len(restored["inputs"]),
        "inputs_rehashed_or_reread": False,
        "input_validation_source": "NB22-C03",
        "output": {
            "path": target.relative_to(root).as_posix(),
            "file_bytes_sha256": digest,
            "bytes": len(saved),
            "write_status": write_status,
        },
        "reload_validation": "PASS",
        "action06_acceptance_reconciliation_complete": False,
        "release_acceptance_granted": False,
        "checklist_modified": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "next_action": 6,
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return summary


nb22_c04_result = nb22_c04_run(nb22_c03_result)

{
  "status": "ACTION05_EVIDENCE_MAPPING_WRITTEN_AND_RELOADED",
  "subtasks_completed": 4,
  "requirements_mapped": 28,
  "verified_input_bindings_reused": 30,
  "inputs_rehashed_or_reread": false,
  "input_validation_source": "NB22-C03",
  "output": {
    "path": "reports/phase2/release/NB22.C004.action05/C004.evidence_ledger.e7cb52fe8dddf3caa2a0076305f19813df4142bf66afc50030e886baed3b2beb.json",
    "file_bytes_sha256": "e7cb52fe8dddf3caa2a0076305f19813df4142bf66afc50030e886baed3b2beb",
    "bytes": 54421,
    "write_status": "CREATED"
  },
  "reload_validation": "PASS",
  "action06_acceptance_reconciliation_complete": false,
  "release_acceptance_granted": false,
  "checklist_modified": false,
  "new_api_calls": 0,
  "network_calls": 0,
  "next_action": 6
}


<div dir="rtl" align="right">

### NB22-C05 — اقدام ۶، بستهٔ اول: شواهد موجود پذیرش NB19/NB20

این اقدام پنج زیرکار دارد؛ این سل سه زیرکار نخست را انجام می‌دهد:

۱. استخراج علت و دامنهٔ ۱۲ سنجهٔ BLOCKED از بستهٔ خوانده‌شده.
۲. استخراج وضعیت دو وابستگی NB19/NB20.
۳. یافتن و خواندن هدفمند فایل‌های موجود پذیرش و تحویل این دو مؤلفه.

بستهٔ C03 دوباره از دیسک خوانده نمی‌شود.
جست‌وجوی تازه فقط نام فایل‌های مرتبط را در artifacts/phase2 و reports/phase2
بررسی می‌کند؛ نوت‌بوک‌ها، پاسخ‌ها و داوری‌های جمعیت دوباره اجرا نمی‌شوند.

فایل C455 با مسیر و هش مستند قبلی بررسی می‌شود.
سایر فایل‌های پیدا‌شده صرفاً نامزد شاهدند؛ نام فایل یا عبارت PASS
به‌تنهایی پذیرش کامل کیفیت را ثابت نمی‌کند.

نبود نامزد در این دامنه، نبود کار قبلی در کل پروژه نیست.
هیچ تیک، ورودی یا فایل خروجی تغییر نمی‌کند.

API، اجرای مدل و درخواست شبکه: صفر.

</div>

In [5]:
# NB22-C05 — Action 06, package 1: targeted acceptance evidence
# Scope: reuse C03; inspect local acceptance candidates; no writes/network.

def nb22_c05_run(previous, ledger_result):
    if ledger_result.get("reload_validation") != "PASS":
        raise RuntimeError("ACTION05_LEDGER_RELOAD_NOT_PASSED")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()
    loaded = previous["loaded"]

    matches = [
        (path, value) for path, value in loaded.items()
        if Path(path).name.startswith("C082.open_conditions.")
    ]
    if len(matches) != 1 or not isinstance(matches[0][1], dict):
        raise RuntimeError("VERIFIED_OPEN_CONDITIONS_NOT_UNIQUE")
    conditions_path, conditions = matches[0]

    blocked = conditions.get("blocked_metric_gates")
    dependencies = conditions.get("upstream_dependency_assessments")
    if not isinstance(blocked, list) or not isinstance(dependencies, list):
        raise RuntimeError("OPEN_CONDITIONS_STRUCTURE_MISMATCH")

    blocked_rows = [
        {
            "metric": row.get("metric"),
            "status": row.get("status"),
            "reason_fa": row.get("reason_fa"),
            "measurement": row.get("recorded_measurement_state"),
            "frozen_policy": row.get("frozen_policy"),
        }
        for row in blocked if isinstance(row, dict)
    ]
    if (
        len(blocked_rows) != 12
        or len({row["metric"] for row in blocked_rows}) != 12
        or any(row["status"] != "BLOCKED" for row in blocked_rows)
    ):
        raise RuntimeError("EXPECTED_12_DISTINCT_BLOCKED_METRICS_NOT_FOUND")
    if len(dependencies) != 2:
        raise RuntimeError("EXPECTED_TWO_UPSTREAM_DEPENDENCIES")

    # Exact historical binding already documented in the project.
    c455_sha = (
        "9b08c7991964ea8cc99ba2cc9e47e9bed"
        "7104d8c3231b75fe77277ef86475ae1"
    )
    c455_relative = (
        "artifacts/phase2/answerability_safety_confidence/"
        "work/outputs/nb20c_scoped_audit_handoff_C455_v1/"
        f"C455.handoff.{c455_sha}.json"
    )

    candidate_sources = {}
    def add_candidate(path, source):
        resolved = path.resolve()
        try:
            relative = resolved.relative_to(root).as_posix()
        except ValueError:
            return
        candidate_sources.setdefault(relative, set()).add(source)

    add_candidate(root / c455_relative, "EXACT_DOCUMENTED_C455_BINDING")

    component_tokens = (
        "nb19", "nb20", "generation", "citation",
        "answerability", "safety", "confidence",
    )
    decision_tokens = ("acceptance", "handoff", "release", "decision")
    scanned_roots = []

    for relative_dir in ("artifacts/phase2", "reports/phase2"):
        directory = root / relative_dir
        if not directory.is_dir():
            continue
        scanned_roots.append(relative_dir)
        for path in directory.rglob("*.json"):
            if not path.is_file():
                continue
            relative = path.relative_to(root).as_posix().lower()
            name = path.name.lower()
            component_related = any(
                token in relative for token in component_tokens
            )
            decision_named = any(
                token in name for token in decision_tokens
            )
            if component_related and decision_named:
                add_candidate(path, "TARGETED_FILENAME_INVENTORY")

    def safe(value):
        if isinstance(value, dict):
            return {
                key: (
                    "[REDACTED]" if any(
                        token in key.lower()
                        for token in (
                            "api_key", "authorization", "password",
                            "secret", "access_token"
                        )
                    ) else safe(item)
                )
                for key, item in value.items()
            }
        if isinstance(value, list):
            return [safe(item) for item in value]
        if isinstance(value, str):
            value = value.replace(str(root), "<PROJECT_ROOT>")
            return re.sub(
                r"/(?:Users|home)/[^/\s]+", "<USER_HOME>", value
            )
        return value

    def decision_view(value):
        # Retain decision context, not just strings containing PASS.
        if not isinstance(value, dict):
            return {"root_type": type(value).__name__}
        useful_tokens = (
            "status", "scope", "accept", "completion", "gate",
            "limit", "condition", "consumer", "producer",
            "schema", "version", "decision", "signoff",
            "provenance", "count", "independ", "handoff",
        )
        selected = {
            key: item for key, item in value.items()
            if any(token in key.lower() for token in useful_tokens)
        }
        return {
            "top_level_keys": list(value),
            "decision_context": safe(selected),
        }

    evidence_rows, payloads = [], {}
    # Bound fresh content reading; no arbitrary "absence" conclusion.
    candidate_limit = 40
    ordered = sorted(
        candidate_sources,
        key=lambda p: (p != c455_relative, p),
    )
    deferred_candidates = ordered[candidate_limit:]

    for relative in ordered[:candidate_limit]:
        path = root / relative
        row = {
            "path": relative,
            "sources": sorted(candidate_sources[relative]),
            "full_acceptance_inferred": False,
        }
        try:
            if not path.is_file():
                row["status"] = "NOT_AVAILABLE_AT_DOCUMENTED_PATH"
                evidence_rows.append(row)
                continue
            row["bytes"] = path.stat().st_size
            if row["bytes"] > 2 * 1024 * 1024:
                row["status"] = "CONTENT_REVIEW_PENDING_SIZE_LIMIT"
                evidence_rows.append(row)
                continue

            raw = path.read_bytes()
            digest = hashlib.sha256(raw).hexdigest()
            row["file_bytes_sha256"] = digest

            if relative == c455_relative:
                row["expected_file_bytes_sha256"] = c455_sha
                if digest != c455_sha:
                    row["status"] = "HASH_RECONCILIATION_REQUIRED"
                    evidence_rows.append(row)
                    continue
            else:
                row["expected_hash_binding"] = (
                    "NOT_YET_BOUND_TO_ACCEPTANCE_CHAIN"
                )

            value = json.loads(raw.decode("utf-8"))
            payloads[relative] = value
            view = decision_view(value)
            encoded = json.dumps(view, ensure_ascii=False)
            if len(encoded) > 14000:
                row["decision_view"] = {
                    "top_level_keys": (
                        list(value) if isinstance(value, dict) else []
                    ),
                    "full_context_in_runtime": True,
                    "display_omitted_due_to_size": True,
                }
            else:
                row["decision_view"] = view
            row["status"] = "READABLE_CANDIDATE_REQUIRES_SCOPE_REVIEW"
        except Exception as exc:
            row["status"] = "LOCAL_READ_REQUIRES_RECONCILIATION"
            row["error_type"] = type(exc).__name__
        evidence_rows.append(row)

    summary = {
        "status": "ACTION06_PACKAGE1_EVIDENCE_ASSEMBLED_NOT_ACCEPTANCE",
        "subtasks_in_action": 5,
        "current_package": "1_OF_2",
        "verified_conditions_source": conditions_path,
        "blocked_metrics": blocked_rows,
        "upstream_dependencies_as_recorded_by_NB21": dependencies,
        "fresh_search_scope": scanned_roots,
        "candidate_count": len(ordered),
        "candidate_read_results": evidence_rows,
        "unread_candidate_paths": deferred_candidates,
        "projectwide_absence_claimed": False,
        "full_upstream_acceptance_established": False,
        "action06_complete": False,
        "checklist_modified": False,
        "files_written": 0,
        "model_api_calls": 0,
        "network_calls": 0,
    }
    print(json.dumps(safe(summary), ensure_ascii=False, indent=2))
    return {
        "summary": summary,
        "candidate_payloads": payloads,
        "conditions": conditions,
    }


nb22_c05_result = nb22_c05_run(nb22_c03_result, nb22_c04_result)

{
  "status": "ACTION06_PACKAGE1_EVIDENCE_ASSEMBLED_NOT_ACCEPTANCE",
  "subtasks_in_action": 5,
  "current_package": "1_OF_2",
  "verified_conditions_source": "reports/phase2/evaluation/NB21C.C082.action46_handoff/9186bb5a30456486b29d521db1660b944c72bf8c4fcb9d816cf578b1937ea1d9/C082.open_conditions.e9ee463cb16c2e188a50fd3bf621d234d8dca6d48f251a35472d38453fe10bd9.json",
  "blocked_metrics": [
    {
      "metric": "answer_completeness",
      "status": "BLOCKED",
      "reason_fa": "اندازه‌گیری، مرجع، دامنه یا خط پایهٔ موردنیاز برای PASS این سنجه از شواهد مصرف‌شده احراز نشده است. نتیجهٔ محدود یا Proxy جای Gate کامل قرار نگرفت.",
      "measurement": {
        "API_request_created": false,
        "gate_state": "NOT_ESTABLISHED",
        "measurement_status": "RECORDED_REQUIREMENTS_AVAILABLE_NO_PER_REQUIREMENT_COVERAGE",
        "metric": "answer_completeness",
        "quality_gate_pass": null,
        "reason_fa": "نیازمندی‌های ۹۳۰ پرونده موجودند؛ رأی معتبر پوشش هر واحد برای جمعیت کامل

<div dir="rtl" align="right">

### NB22-C06 — اقدام ۶، بستهٔ دوم: تطبیق اسناد پذیرش و تحویل

این بسته زیرکارهای ۴ و ۵ اقدام ۶ را پیش می‌برد:

۴. بررسی دامنه و نسخهٔ اسناد سطح پذیرش، Release و Handoff مربوط به NB19/NB20.
۵. ثبت نتیجهٔ تطبیق و شکاف‌های باقی‌مانده در یک گزارش قابل بازخوانی.

ورودی، فهرست نامزدهای C05 و محتوای خوانده‌شدهٔ همان سل است.
فهرست مخزن دوباره ساخته نمی‌شود و تصمیم‌های تک‌پرونده‌ای خوانده نمی‌شوند.
نسخه‌های موجود NB19 کنار هم بررسی می‌شوند؛ شمارهٔ بالاتر خودکار مرجع نیست.

گزارش، عبارت‌ها و تصمیم‌های ثبت‌شده را با مسیر و هش حفظ می‌کند.
وجود PASS محدود، پذیرش کامل مؤلفه یا رفع ۱۲ BLOCKED تلقی نمی‌شود.
اگر دامنه یا زنجیرهٔ پذیرش کافی نباشد، شکاف صریح باقی می‌ماند.

خروجی جدید به‌صورت Atomic ذخیره و مستقلاً بازخوانی می‌شود.
چک‌لیست و اسناد تاریخی تغییر نمی‌کنند.
API، اجرای مدل و درخواست شبکه: صفر.

</div>

In [6]:
# NB22-C06 — Action 06: package-level acceptance reconciliation
# Scope: existing C05 inventory only; no per-case reads or new discovery.

def nb22_c06_run(previous):
    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    prior_rows = {
        row["path"]: row
        for row in previous["summary"]["candidate_read_results"]
    }
    inventory = set(prior_rows) | set(
        previous["summary"]["unread_candidate_paths"]
    )
    cached = previous["candidate_payloads"]

    def is_package_document(relative):
        name = Path(relative).name
        # Explicit package families; case-level decision files excluded.
        return (
            name.startswith((
                "nb19_development_handoff_v",
                "nb19_development_release_v",
                "nb19_development_acceptance_v",
                "nb19_test_handoff_chatgpt_recovery_v",
                "C455.handoff.",
                "C453.decision.",
                "C406.decision.",
                "C415.decision.",
                "C207.acceptance.",
                "C212.handoff.",
            ))
        )

    selected = sorted(p for p in inventory if is_package_document(p))
    if not selected:
        raise RuntimeError("NO_PACKAGE_DOCUMENTS_IN_EXISTING_INVENTORY")

    def safe_path(relative):
        p = Path(relative)
        if p.is_absolute() or ".." in p.parts:
            raise ValueError("INVALID_REPOSITORY_RELATIVE_PATH")
        target = (root / p).resolve()
        target.relative_to(root)
        return target

    def safe(value):
        if isinstance(value, dict):
            return {
                key: (
                    "[REDACTED]" if any(
                        token in key.lower()
                        for token in (
                            "api_key", "authorization", "password",
                            "secret", "access_token"
                        )
                    ) else safe(item)
                )
                for key, item in value.items()
            }
        if isinstance(value, list):
            return [safe(item) for item in value]
        if isinstance(value, str):
            value = value.replace(str(root), "<PROJECT_ROOT>")
            return re.sub(
                r"/(?:Users|home)/[^/\s]+", "<USER_HOME>", value
            )
        return value

    decision_tokens = (
        "accept", "status", "scope", "gate", "complete",
        "independen", "limit", "block", "condition",
        "origin", "provenance", "producer", "consumer",
        "schema_version", "release_id", "supersed",
        "run_all", "query_count", "candidate_count",
    )

    def explicit_decisions(value):
        """Extract contextual decision fields; never interpret PASS alone."""
        found = []

        def visit(node, trail, context=False):
            if isinstance(node, dict):
                for key, item in node.items():
                    relevant = any(
                        token in key.lower() for token in decision_tokens
                    )
                    visit(
                        item, trail + [str(key)],
                        context or relevant,
                    )
            elif isinstance(node, list):
                # Do not walk query/case populations or source inventories.
                if context:
                    if len(node) <= 30:
                        for i, item in enumerate(node):
                            visit(item, trail + [str(i)], True)
                    else:
                        found.append({
                            "field": ".".join(trail),
                            "list_count": len(node),
                            "contents_not_expanded": True,
                        })
            elif context:
                found.append({
                    "field": ".".join(trail),
                    "value": safe(node),
                })

        visit(value, [])
        return found

    results, values = [], {}
    fresh_reads = 0
    for relative in selected:
        row = {"path": relative}
        try:
            if relative in cached:
                value = cached[relative]
                source_row = prior_rows[relative]
                digest = source_row["file_bytes_sha256"]
                size = source_row["bytes"]
                row["read_mode"] = "REUSED_C05"
            else:
                path = safe_path(relative)
                if path.stat().st_size > 4 * 1024 * 1024:
                    raise ValueError("PACKAGE_FILE_REQUIRES_LARGER_READER")
                raw = path.read_bytes()
                digest = hashlib.sha256(raw).hexdigest()
                size = len(raw)
                value = json.loads(raw.decode("utf-8"))
                fresh_reads += 1
                row["read_mode"] = "READ_FROM_EXISTING_C05_INVENTORY"

            if not isinstance(value, dict):
                raise ValueError("PACKAGE_DOCUMENT_ROOT_NOT_OBJECT")

            # Only SHA256 explicitly encoded as a full filename component.
            embedded = re.findall(
                r"\.([0-9a-f]{64})\.json$", Path(relative).name
            )
            row["filename_hash_check"] = (
                "MATCH" if embedded and embedded[0] == digest
                else "MISMATCH" if embedded
                else "NO_FILE_BYTES_HASH_ASSERTED_BY_FILENAME"
            )
            if row["filename_hash_check"] == "MISMATCH":
                raise ValueError("FILENAME_FILE_BYTES_HASH_MISMATCH")

            row.update({
                "status": "DOCUMENT_READ_NOT_FULL_ACCEPTANCE",
                "file_bytes_sha256": digest,
                "bytes": size,
                "top_level_keys": list(value),
                "explicit_decisions": explicit_decisions(value),
            })
            values[relative] = value
        except Exception as exc:
            row.update({
                "status": "DOCUMENT_RECONCILIATION_REQUIRED",
                "error_type": type(exc).__name__,
            })
        results.append(row)

    covered_names = {Path(p).name for p in values}
    nb19_documents_read = any(
        name.startswith("nb19_") for name in covered_names
    )
    c455_read = any(
        name.startswith("C455.handoff.") for name in covered_names
    )
    read_checks_pass = (
        nb19_documents_read and c455_read
        and all(
            row["status"] == "DOCUMENT_READ_NOT_FULL_ACCEPTANCE"
            for row in results
        )
    )

    blocked = previous["summary"]["blocked_metrics"]
    report = {
        "artifact_type": "NB22_UPSTREAM_ACCEPTANCE_RECONCILIATION",
        "schema_version": "NB22_C006_ACTION06_RECONCILIATION_V1",
        "status": (
            "PACKAGE_EVIDENCE_RECONCILIATION_RECORDED_REVIEW_PENDING"
            if read_checks_pass else "INPUT_RECONCILIATION_REQUIRED"
        ),
        "source_inventory": "NB22-C05_EXISTING_CANDIDATE_LIST",
        "documents": results,
        "blocked_metric_mapping": [
            {
                "metric": item["metric"],
                "recorded_gate_status": item["status"],
                "existing_measurement": item["measurement"],
                "remaining_acceptance_requirement":
                    item["frozen_policy"],
                "automatically_unblocked": False,
            }
            for item in blocked
        ],
        "original_dependency_assessments":
            previous["summary"][
                "upstream_dependencies_as_recorded_by_NB21"
            ],
        "interpretation": {
            "existing_work_preserved": True,
            "package_document_read_checks_pass": read_checks_pass,
            "higher_version_automatically_selected": False,
            "all_existing_work_declared_missing": False,
            "full_acceptance_automatically_inferred": False,
            "scope_and_version_decision_requires_review": True,
        },
        "excluded_case_level_documents": sum(
            not is_package_document(p) for p in inventory
        ),
        "fresh_package_reads": fresh_reads,
        "new_discovery_performed": False,
        "checklist_modified": False,
        "new_api_calls": 0,
    }

    raw = json.dumps(
        safe(report), ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    digest = hashlib.sha256(raw).hexdigest()
    directory = root / "reports/phase2/release/NB22.C006.action06"
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"C006.reconciliation.{digest}.json"

    if target.exists():
        if target.read_bytes() != raw:
            raise RuntimeError("EXISTING_OUTPUT_CONFLICT")
        write_status = "REUSED_IDENTICAL"
    else:
        fd, tmp_name = tempfile.mkstemp(prefix=".C006.", dir=directory)
        temporary = Path(tmp_name)
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            if temporary.read_bytes() != raw:
                raise RuntimeError("TEMPORARY_WRITE_VALIDATION_FAILED")
            try:
                os.link(temporary, target)
                write_status = "CREATED"
            except FileExistsError:
                if target.read_bytes() != raw:
                    raise RuntimeError("CONCURRENT_OUTPUT_CONFLICT")
                write_status = "REUSED_IDENTICAL"
        finally:
            temporary.unlink(missing_ok=True)

    saved = target.read_bytes()
    restored = json.loads(saved)
    if (
        hashlib.sha256(saved).hexdigest() != digest
        or restored != safe(report)
    ):
        raise RuntimeError("RELOAD_VALIDATION_FAILED")

    # Print compact, contextual evidence rather than entire historical files.
    compact = []
    for row in results:
        compact.append({
            key: row[key] for key in (
                "path", "status", "read_mode", "filename_hash_check",
                "explicit_decisions", "error_type",
            ) if key in row
        })

    print(json.dumps(safe({
        "status": report["status"],
        "package_documents_selected": len(selected),
        "fresh_package_reads": fresh_reads,
        "case_level_documents_not_read":
            report["excluded_case_level_documents"],
        "documents": compact,
        "output": {
            "path": target.relative_to(root).as_posix(),
            "file_bytes_sha256": digest,
            "write_status": write_status,
        },
        "reload_validation": "PASS",
        "full_acceptance_granted": False,
        "action06_final_scope_decision_pending": True,
        "checklist_modified": False,
        "new_api_calls": 0,
        "network_calls": 0,
    }), ensure_ascii=False, indent=2))
    return {
        "report": restored,
        "package_payloads": values,
        "output_path": target.relative_to(root).as_posix(),
        "output_sha256": digest,
    }


nb22_c06_result = nb22_c06_run(nb22_c05_result)

{
  "status": "PACKAGE_EVIDENCE_RECONCILIATION_RECORDED_REVIEW_PENDING",
  "package_documents_selected": 19,
  "fresh_package_reads": 15,
  "case_level_documents_not_read": 209,
  "documents": [
    {
      "path": "artifacts/phase2/answerability_safety_confidence/work/outputs/nb20c_action23_24_scoped_completion_C406_v1/C406.decision.f33a4e4ef48e2839548b7647f1bbcb595bd8ebde4a8566d7bd9fc2981639ccdb.json",
      "status": "DOCUMENT_READ_NOT_FULL_ACCEPTANCE",
      "read_mode": "REUSED_C05",
      "filename_hash_check": "MATCH",
      "explicit_decisions": [
        {
          "field": "action_23_completed_in_stated_scope",
          "value": true
        },
        {
          "field": "action_24_completed_in_stated_scope",
          "value": true
        },
        {
          "field": "completion_scope",
          "value": "FOUR_SOURCE_REVIEWED_OFFLINE_CASES_AND_MISSING_STATE_BEHAVIOR"
        },
        {
          "field": "full_roadmap_confidence_requirement_accepted",
          "v

<div dir="rtl" align="right">

### NB22-C07 — اقدام ۶: ثبت نهایی تطبیق پذیرش‌های بالادست

این سل دو زیرکار پایانی اقدام ۶ را تکمیل می‌کند:

۴. تطبیق هش پذیرش‌های نسخه‌های NB19 با ارجاع Handoff همان نسخه،
و اتصال C455 به تصمیم C453.
۵. ثبت نتیجهٔ بررسی ۱۹ سند و وضعیت هر ۱۲ سنجهٔ BLOCKED.

پذیرش‌های Development در دامنهٔ خودشان حفظ می‌شوند.
وضعیت تاریخی Test در آن اسناد به وضعیت امروز کل پروژه تعمیم داده نمی‌شود.
جمع‌بندی جدید C085 و نتایج NB21 همچنان مرجع وضعیت جاری ارزیابی‌اند.

پایان اقدام ۶ یعنی تطبیق شواهد موجود و مشخص‌کردن شکاف‌ها؛
به معنی رفع موانع کیفیت، پذیرش کامل NB19/NB20 یا پذیرش Release نیست.
نبود شاهد کامل در اسناد مصرف‌شده، نبود کار در کل پروژه تلقی نمی‌شود.

ورودی‌ها از نتایج موجود C03/C05/C06 مصرف می‌شوند.
فقط گزارش جدید نوشته و مستقلاً بازخوانی می‌شود.
API، اجرای مدل، جست‌وجوی تازه و تغییر چک‌لیست: صفر.

</div>

In [7]:
# NB22-C07 — Finalize action 06 in the inspected evidence scope
# Scope: cached evidence only; write/reload one reconciliation decision.

def nb22_c07_run(c03, c05, c06):
    report = c06["report"]
    if not report["interpretation"]["package_document_read_checks_pass"]:
        raise RuntimeError("C06_PACKAGE_DOCUMENT_READ_CHECKS_NOT_PASSED")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    payloads = c06["package_payloads"]
    documents = {row["path"]: row for row in report["documents"]}

    def by_name(name):
        matches = [p for p in payloads if Path(p).name == name]
        if len(matches) != 1:
            raise RuntimeError(f"DOCUMENT_NOT_UNIQUE: {name}")
        path = matches[0]
        return path, payloads[path]

    links = []
    for version in range(1, 5):
        handoff_path, handoff = by_name(
            f"nb19_development_handoff_v{version}.json"
        )
        acceptance_path, acceptance = by_name(
            f"nb19_development_acceptance_v{version}.json"
        )
        if version in (1, 2):
            reference = handoff.get("acceptance")
            if not isinstance(reference, dict):
                raise RuntimeError("NB19_ACCEPTANCE_BINDING_MISSING")
            relative = reference.get("path")
            expected = reference.get("sha256")
            if not isinstance(relative, str):
                raise RuntimeError("NB19_ACCEPTANCE_PATH_MISSING")
            # Project documents express this binding relative to work/.
            work = Path(handoff_path).parent.parent
            resolved_relative = (work / relative).as_posix()
            if resolved_relative != acceptance_path:
                raise RuntimeError("NB19_ACCEPTANCE_PATH_BINDING_DIFFERS")
        else:
            expected = handoff.get(f"acceptance_v{version}_sha256")

        actual = documents[acceptance_path]["file_bytes_sha256"]
        if not isinstance(expected, str) or expected != actual:
            raise RuntimeError(
                f"NB19_V{version}_ACCEPTANCE_HASH_BINDING_DIFFERS"
            )
        links.append({
            "producer": handoff_path,
            "acceptance": acceptance_path,
            "file_bytes_sha256": actual,
            "binding_check": "PASS",
            "recorded_acceptance_status": acceptance.get("status"),
            "scope": "HISTORICAL_DEVELOPMENT_VERSION",
            "full_product_acceptance_inferred": False,
        })

    def by_prefix(prefix):
        matches = [
            p for p in payloads if Path(p).name.startswith(prefix)
        ]
        if len(matches) != 1:
            raise RuntimeError(f"DOCUMENT_PREFIX_NOT_UNIQUE: {prefix}")
        path = matches[0]
        return path, payloads[path]

    c455_path, c455 = by_prefix("C455.handoff.")
    c453_path, c453 = by_prefix("C453.decision.")
    gate = c455.get("entry_points", {}).get("gate", {})
    if (
        gate.get("project_relative_path") != c453_path
        or gate.get("sha256")
            != documents[c453_path]["file_bytes_sha256"]
    ):
        raise RuntimeError("C455_TO_C453_BINDING_DIFFERS")
    links.append({
        "producer": c455_path,
        "acceptance": c453_path,
        "file_bytes_sha256":
            documents[c453_path]["file_bytes_sha256"],
        "binding_check": "PASS",
        "scope": c455.get("handoff_scope"),
        "full_product_acceptance_inferred": False,
    })

    blocked = report["blocked_metric_mapping"]
    if (
        len(blocked) != 12
        or len({r["metric"] for r in blocked}) != 12
        or any(r["recorded_gate_status"] != "BLOCKED" for r in blocked)
    ):
        raise RuntimeError("BLOCKED_METRIC_ACCOUNTING_DIFFERS")

    reasons = {
        "answer_completeness":
            "پوشش معتبر هر الزام مرجع برای دامنهٔ لازم احراز نشده است.",
        "attribution_correctness":
            "صحت تمام گزاره‌های انتسابِ بیان‌شده به مرجع واجد شرایط متصل نیست.",
        "citation_correctness":
            "حمایت کلی ادعا با رأی معتبر تک‌تک پیوندهای استناد یکسان نیست.",
        "citation_locator_accuracy":
            "تطابق بازهٔ متن، مکان معنایی درست برای هر ادعا را ثابت نمی‌کند.",
        "claim_citation_coverage":
            "پوشش ساختاری، پوشش معنایی تمام ادعاهای factual نیست.",
        "groundedness":
            "جامعیت استخراج ادعاها و دامنهٔ صلاحیت مرجع کامل احراز نشده است.",
        "no_answer_precision_recall_f1":
            "نتیجهٔ مرجع مدل محفوظ است؛ کفایت آن برای Gate کامل احراز نشده است.",
        "safety_violation_rate":
            "دامنهٔ داوری‌ها، اختلاف محفوظ و پذیرش ایمنی زنده جدا هستند.",
        "latency":
            "مرز اندازه‌گیری مسیر زندهٔ محصول در شواهد مصرف‌شده برقرار نیست.",
        "cost":
            "هزینهٔ تاریخی داوری، هزینهٔ تولید محصول یا صورتحساب قطعی نیست.",
        "token_usage":
            "توکن Jobهای تاریخی، مصرف مولد روی همین دامنهٔ Test نیست.",
        "failure_rate":
            "رخدادهای Job تاریخی جای نرخ شکست مسیر محصول را نمی‌گیرند.",
    }
    if set(reasons) != {r["metric"] for r in blocked}:
        raise RuntimeError("METRIC_REASON_MAPPING_DIFFERS")

    rows = [
        {
            **item,
            "reconciliation_reason_fa": reasons[item["metric"]],
            "disposition": "PRESERVE_BLOCKED_NO_AUTOMATIC_REEXECUTION",
            "coordination_owner": "NB22",
            "corrective_implementation_owner": "NOT_ASSIGNED_BY_THIS_CELL",
            "new_api_required": "NOT_DETERMINED_BY_THIS_RECONCILIATION",
        }
        for item in blocked
    ]

    decision = {
        "artifact_type": "NB22_ACTION06_SCOPED_RECONCILIATION_DECISION",
        "schema_version": "NB22_C007_ACTION06_DECISION_V1",
        "status": "ACTION06_RECONCILIATION_COMPLETE_IN_INSPECTED_SCOPE",
        "completion_scope":
            "EXISTING_PACKAGE_ACCEPTANCE_RECONCILIATION_AND_EXPLICIT_GAPS",
        "subtasks_accounted": 5,
        "source_report": {
            "path": c06["output_path"],
            "file_bytes_sha256": c06["output_sha256"],
        },
        "document_bindings": [
            {
                "path": path,
                "file_bytes_sha256": row["file_bytes_sha256"],
                "bytes": row["bytes"],
            }
            for path, row in sorted(documents.items())
        ],
        "acceptance_link_checks": links,
        "blocked_metrics": rows,
        "upstream_findings": {
            "NB19": {
                "development_acceptance_exists": True,
                "acceptance_scope": "HISTORICAL_DEVELOPMENT_WITH_OPEN_GATES",
                "test_recovery_is_live_qwen_execution": False,
                "full_product_acceptance_established_in_inspected_scope": False,
            },
            "NB20": {
                "stored_audit_handoff_verified": True,
                "handoff_scope": c455.get("handoff_scope"),
                "formal_live_product_safety_accepted":
                    c455.get("formal_live_product_safety_accepted"),
                "semantic_judgments_independently_validated":
                    c455.get("semantic_judgments_independently_validated"),
                "preserved_limits": c455.get("preserved_roadmap_limits"),
            },
        },
        "current_NB21_summary_preserved":
            c03["summary"]["C085_current_summary"],
        "limitations": [
            "No projectwide absence claim.",
            "Historical open gates are not asserted unchanged projectwide.",
            "No highest-version automatic canonical selection.",
            "Scoped reconciliation completion does not resolve quality blockers.",
            "No full raw-data closure or independent semantic validation.",
        ],
        "release_acceptance_granted": False,
        "checklist_modified": False,
        "new_api_calls": 0,
        "next_action": 7,
    }

    raw = json.dumps(
        decision, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    digest = hashlib.sha256(raw).hexdigest()
    directory = root / "reports/phase2/release/NB22.C007.action06"
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"C007.decision.{digest}.json"

    if target.exists():
        if target.read_bytes() != raw:
            raise RuntimeError("EXISTING_OUTPUT_CONFLICT")
        write_status = "REUSED_IDENTICAL"
    else:
        fd, name = tempfile.mkstemp(prefix=".C007.", dir=directory)
        temporary = Path(name)
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            if json.loads(temporary.read_bytes()) != decision:
                raise RuntimeError("TEMPORARY_VALIDATION_FAILED")
            try:
                os.link(temporary, target)
                write_status = "CREATED"
            except FileExistsError:
                if target.read_bytes() != raw:
                    raise RuntimeError("CONCURRENT_OUTPUT_CONFLICT")
                write_status = "REUSED_IDENTICAL"
        finally:
            temporary.unlink(missing_ok=True)

    saved = target.read_bytes()
    if (
        hashlib.sha256(saved).hexdigest() != digest
        or json.loads(saved) != decision
    ):
        raise RuntimeError("RELOAD_VALIDATION_FAILED")

    summary = {
        "status": decision["status"],
        "subtasks_accounted": 5,
        "package_documents_reviewed": len(documents),
        "acceptance_hash_links_verified": len(links),
        "blocked_metrics_preserved": len(rows),
        "all_quality_blockers_resolved": False,
        "output": {
            "path": target.relative_to(root).as_posix(),
            "file_bytes_sha256": digest,
            "write_status": write_status,
        },
        "reload_validation": "PASS",
        "input_files_reread": 0,
        "new_api_calls": 0,
        "checklist_modified": False,
        "release_acceptance_granted": False,
        "next_action": 7,
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return {"summary": summary, "decision": decision}


nb22_c07_result = nb22_c07_run(
    nb22_c03_result, nb22_c05_result, nb22_c06_result
)

{
  "status": "ACTION06_RECONCILIATION_COMPLETE_IN_INSPECTED_SCOPE",
  "subtasks_accounted": 5,
  "package_documents_reviewed": 19,
  "acceptance_hash_links_verified": 5,
  "blocked_metrics_preserved": 12,
  "all_quality_blockers_resolved": false,
  "output": {
    "path": "reports/phase2/release/NB22.C007.action06/C007.decision.96788c034a66d6aa8347059f644e5d915ef369216502d9f80f714e306a38a0d5.json",
    "file_bytes_sha256": "96788c034a66d6aa8347059f644e5d915ef369216502d9f80f714e306a38a0d5",
    "write_status": "CREATED"
  },
  "reload_validation": "PASS",
  "input_files_reread": 0,
  "new_api_calls": 0,
  "checklist_modified": false,
  "release_acceptance_granted": false,
  "next_action": 7
}


<div dir="rtl" align="right">

### NB22-C08 — اقدام ۷: جمع‌آوری نسخه‌های مستند مؤلفه‌ها

این اقدام چهار زیرکار دارد:

۱. مصرف نسخه‌های ثبت‌شده در اسناد قبلی و Release/Manifestهای مؤلفه‌ها.
۲. تفکیک شناسهٔ Release، نسخهٔ Schema، نسخهٔ مدل و Revision.
۳. ثبت مسیر سند، هش بایت و محل دقیق هر مقدار.
۴. ثبت نسخه‌های چندگانه، موارد نامشخص و انتخاب‌های نیازمند شاهد.

نسخهٔ بالاتر یا نام فایل به‌تنهایی مبنای انتخاب Canonical نیست.
نسخهٔ کاندیدای کل محصول در این سل اختراع یا پذیرفته نمی‌شود.
فایل دسترسی‌نیافته، دلیل انجام‌نشدن کار قبلی نیست.

خروجی، رجیستری نسخه‌های مشاهده‌شده است؛
تعیین مسیر Canonical و کنترل هش دارایی‌های اصلی در اقدامات بعدی انجام می‌شود.

فقط اسناد کوچکِ مشخص‌شده خوانده می‌شوند.
API، اجرای مدل، بازسازی داده و تغییر چک‌لیست: صفر.

</div>

In [8]:
# NB22-C08 — Action 07: collect declared versions with provenance
# Scope: cached evidence plus explicit small metadata files; no API/rebuild.

def nb22_c08_run(c03, c06, c07):
    if c07["summary"]["reload_validation"] != "PASS":
        raise RuntimeError("ACTION06_DECISION_RELOAD_NOT_PASSED")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    # These are existing project metadata paths, not generated substitutes.
    explicit = [
        "artifacts/phase2/embedding/embedding_release_v1.json",
        "artifacts/phase2/dense_index/dense_index_release_v1.json",
        "artifacts/phase2/sparse_index/sparse_index_release_v1.json",
        "artifacts/phase2/hybrid_retrieval/hybrid_retrieval_release_v1.json",
        "artifacts/phase2/hybrid_retrieval/hybrid_configuration_v1.json",
        "artifacts/phase2/reranking/reranker_release_v1.json",
        "artifacts/phase2/reranking/reranker_configuration_v1.json",
        "artifacts/phase2/context_builder/context_builder_release_v1.json",
        "artifacts/phase2/context_builder/context_builder_configuration_v1.json",
        "artifacts/phase2/golden_dataset_v2/golden_dataset_v2_manifest.json",
        "artifacts/phase2/golden_dataset_v2/golden_dataset_v2_final_report.json",
        "artifacts/phase2/chunking/chunk_contract_evaluation_freeze_v1.json",
    ]

    sources, values = {}, {}
    # Reuse the original C03 verification records without rereading files.
    for path, value in c03["loaded"].items():
        rec = c03["records"][path]
        values[path] = value
        sources[path] = {
            "path": path,
            "file_bytes_sha256": rec["actual_sha256"],
            "bytes": rec["bytes"],
            "read_mode": "REUSED_C03",
        }

    # Reuse the package documents consumed and hashed in C06.
    for rec in c06["report"]["documents"]:
        path = rec["path"]
        if path in c06["package_payloads"]:
            if (
                path in sources
                and sources[path]["file_bytes_sha256"]
                    != rec["file_bytes_sha256"]
            ):
                raise RuntimeError("CACHED_SOURCE_HASH_CONFLICT")
            values[path] = c06["package_payloads"][path]
            sources[path] = {
                "path": path,
                "file_bytes_sha256": rec["file_bytes_sha256"],
                "bytes": rec["bytes"],
                "read_mode": "REUSED_C06",
            }

    read_gaps = []
    fresh_reads = 0
    for relative in explicit:
        if relative in values:
            continue
        path = (root / relative).resolve()
        path.relative_to(root)
        if not path.is_file():
            read_gaps.append({
                "path": relative,
                "status": "NOT_AVAILABLE_AT_EXPLICIT_LOCAL_PATH",
                "absence_projectwide_claimed": False,
            })
            continue
        if path.stat().st_size > 4 * 1024 * 1024:
            read_gaps.append({
                "path": relative,
                "status": "METADATA_SIZE_REQUIRES_SEPARATE_REVIEW",
            })
            continue
        try:
            raw = path.read_bytes()
            value = json.loads(raw.decode("utf-8"))
            if not isinstance(value, dict):
                raise ValueError("METADATA_ROOT_NOT_OBJECT")
            values[relative] = value
            sources[relative] = {
                "path": relative,
                "file_bytes_sha256": hashlib.sha256(raw).hexdigest(),
                "bytes": len(raw),
                "read_mode": "FRESH_EXPLICIT_METADATA_READ",
            }
            fresh_reads += 1
        except Exception as exc:
            read_gaps.append({
                "path": relative,
                "status": "READ_REQUIRES_RECONCILIATION",
                "error_type": type(exc).__name__,
            })

    def field_kind(key):
        key = key.lower()
        if key == "schema_version":
            return "SCHEMA_VERSION"
        if key in ("release_id", "release_candidate_id"):
            return "RELEASE_IDENTIFIER"
        if key.endswith("_release_id"):
            return "COMPONENT_RELEASE_IDENTIFIER"
        if key == "revision" or key.endswith("_revision"):
            return "REVISION"
        if key == "version" or key.endswith("_version"):
            return "DECLARED_VERSION"
        if key in (
            "model", "model_name", "model_id", "configured_model",
            "embedding_model", "reranker_model", "generation_model",
        ):
            return "MODEL_IDENTIFIER"
        return None

    observations = []
    def walk(node, source, trail=()):
        if isinstance(node, dict):
            for key, item in node.items():
                location = trail + (str(key),)
                kind = field_kind(key)
                if kind and (
                    item is None or isinstance(item, (str, int, float, bool))
                ):
                    observations.append({
                        "source_path": source,
                        "source_file_bytes_sha256":
                            sources[source]["file_bytes_sha256"],
                        "field_path": list(location),
                        "kind": kind,
                        "value": item,
                        "nonempty_value_present":
                            item is not None and item != "",
                        "canonical_selection_claimed": False,
                    })
                elif isinstance(item, dict):
                    walk(item, source, location)
                elif isinstance(item, list):
                    # Traverse metadata-sized lists, never case populations.
                    if len(item) <= 50:
                        for i, child in enumerate(item):
                            if isinstance(child, dict):
                                walk(
                                    child, source,
                                    location + (str(i),),
                                )

    for path, value in values.items():
        # Historical case catalogues are not version authorities.
        name = Path(path).name
        if any(token in name for token in (
            "failure_catalogue", "regression_specs",
            "regression_scenarios", "open_reviews",
            "actions01_49_status",
        )):
            continue
        if isinstance(value, dict):
            walk(value, path)

    if not observations:
        raise RuntimeError("NO_EXPLICIT_VERSION_FIELDS_EXTRACTED")

    coverage = []
    for relative in explicit:
        rows = [
            item for item in observations
            if item["source_path"] == relative
        ]
        coverage.append({
            "metadata_path": relative,
            "document_read": relative in values,
            "version_observation_count": len(rows),
            "declared_values": [
                {
                    "field_path": item["field_path"],
                    "kind": item["kind"],
                    "value": item["value"],
                }
                for item in rows
            ],
            "no_version_inferred_from_filename": True,
        })

    artifact = {
        "artifact_type": "NB22_DECLARED_VERSION_REGISTRY",
        "schema_version": "NB22_C008_ACTION07_VERSION_REGISTRY_V1",
        "status": "DECLARED_VERSIONS_COLLECTED_CANONICAL_SELECTION_PENDING",
        "scope": "OBSERVED_METADATA_FIELDS_NOT_FULL_PRODUCT_VERSION_FREEZE",
        "sources": [sources[p] for p in sorted(sources)],
        "observations": observations,
        "explicit_component_coverage": coverage,
        "read_gaps": read_gaps,
        "fresh_metadata_files_read": fresh_reads,
        "candidate_release_id": None,
        "candidate_release_id_status": "NOT_ASSIGNED_BY_THIS_CELL",
        "canonical_selection_complete": False,
        "version_completeness_established": False,
        "limitations": [
            "Schema versions are not product release identifiers.",
            "Historical versions are preserved without automatic supersession.",
            "Dictionary/list traversal extracts fields, not semantic authority.",
            "Missing metadata fields do not prove missing prior work.",
            "Model identifiers do not independently prove runtime model identity.",
            "Code, prompt and policy versions require authoritative bindings.",
        ],
        "checklist_modified": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
    }

    raw = json.dumps(
        artifact, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    digest = hashlib.sha256(raw).hexdigest()
    directory = root / "reports/phase2/release/NB22.C008.action07"
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"C008.versions.{digest}.json"

    if target.exists():
        if target.read_bytes() != raw:
            raise RuntimeError("EXISTING_OUTPUT_CONFLICT")
        write_status = "REUSED_IDENTICAL"
    else:
        fd, name = tempfile.mkstemp(prefix=".C008.", dir=directory)
        temporary = Path(name)
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            if temporary.read_bytes() != raw:
                raise RuntimeError("TEMPORARY_VALIDATION_FAILED")
            try:
                os.link(temporary, target)
                write_status = "CREATED"
            except FileExistsError:
                if target.read_bytes() != raw:
                    raise RuntimeError("CONCURRENT_OUTPUT_CONFLICT")
                write_status = "REUSED_IDENTICAL"
        finally:
            temporary.unlink(missing_ok=True)

    saved = target.read_bytes()
    if (
        hashlib.sha256(saved).hexdigest() != digest
        or json.loads(saved) != artifact
    ):
        raise RuntimeError("RELOAD_VALIDATION_FAILED")

    print(json.dumps({
        "status": artifact["status"],
        "observations_recorded": len(observations),
        "fresh_metadata_files_read": fresh_reads,
        "component_coverage": coverage,
        "read_gaps": read_gaps,
        "candidate_release_id_assigned": False,
        "version_completeness_established": False,
        "output": {
            "path": target.relative_to(root).as_posix(),
            "file_bytes_sha256": digest,
            "write_status": write_status,
        },
        "reload_validation": "PASS",
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "payloads": values}


nb22_c08_result = nb22_c08_run(
    nb22_c03_result, nb22_c06_result, nb22_c07_result
)

{
  "status": "DECLARED_VERSIONS_COLLECTED_CANONICAL_SELECTION_PENDING",
  "observations_recorded": 108,
  "fresh_metadata_files_read": 12,
  "component_coverage": [
    {
      "metadata_path": "artifacts/phase2/embedding/embedding_release_v1.json",
      "document_read": true,
      "version_observation_count": 4,
      "declared_values": [
        {
          "field_path": [
            "embedding",
            "model_id"
          ],
          "kind": "MODEL_IDENTIFIER",
          "value": "intfloat/multilingual-e5-small"
        },
        {
          "field_path": [
            "embedding",
            "revision"
          ],
          "kind": "REVISION",
          "value": "614241f622f53c4eeff9890bdc4f31cfecc418b3"
        },
        {
          "field_path": [
            "release_version"
          ],
          "kind": "DECLARED_VERSION",
          "value": "embedding_release_v1"
        },
        {
          "field_path": [
            "schema_version"
          ],
         

<div dir="rtl" align="right">

### NB22-C09 — اقدام ۷: تفکیک نسخهٔ واقعی از تعریف Schema

این سل سه کار باقی‌ماندهٔ جمع‌بندی رجیستری را انجام می‌دهد:

۱. جداسازی مشاهدات داخل تعریف Schema از مقادیر نسخه.
۲. ثبت نسخه‌های صریح مؤلفه‌ها با ارجاع به سند و هش.
۳. مشخص‌کردن اتصال‌های نسخه‌ای که هنوز احراز نشده‌اند.

مقدار keyword در payload_schema نوع فیلد است، نه نسخه.
نسخهٔ Schema، نسخهٔ محصول و Revision مدل جدا باقی می‌مانند.
نسخه‌های تاریخی و آزمایشی حذف یا خودکار Canonical نمی‌شوند.

تمام ورودی‌ها از C08 مصرف می‌شوند؛ هیچ سندی دوباره خوانده نمی‌شود.
خروجی قبلی حفظ و نسخهٔ اصلاح‌شدهٔ رجیستری جدا ذخیره می‌شود.
کامل‌بودن رجیستریِ مشاهدات، به معنی کامل‌بودن نسخه‌های Release نیست.

API، اجرای مدل، جست‌وجوی تازه و تغییر چک‌لیست: صفر.

</div>

In [9]:
# NB22-C09 — Action 07: qualify version observations
# Scope: reuse C08 only; preserve raw observations; no input rereads.

def nb22_c09_run(previous):
    original = previous["artifact"]
    payloads = previous["payloads"]

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    schema_containers = {
        "payload_schema", "properties", "$defs", "definitions",
    }
    retained, schema_definitions, empty = [], [], []
    for observation in original["observations"]:
        trail = observation["field_path"]
        if any(part in schema_containers for part in trail):
            schema_definitions.append({
                **observation,
                "classification":
                    "SCHEMA_DEFINITION_NOT_COMPONENT_VERSION",
            })
        elif not observation["nonempty_value_present"]:
            empty.append(observation)
        else:
            retained.append(observation)

    if (
        len(retained) + len(schema_definitions) + len(empty)
        != len(original["observations"])
    ):
        raise RuntimeError("OBSERVATION_ACCOUNTING_FAILED")

    source_bindings = {
        row["path"]: row for row in original["sources"]
    }

    def get_value(path, fields):
        value = payloads[path]
        for field in fields:
            if not isinstance(value, dict) or field not in value:
                raise RuntimeError(
                    f"EXPLICIT_VERSION_FIELD_NOT_FOUND: {path}: {fields}"
                )
            value = value[field]
        if value is None or value == "":
            raise RuntimeError("EXPLICIT_VERSION_FIELD_EMPTY")
        return {
            "value": value,
            "source_path": path,
            "source_file_bytes_sha256":
                source_bindings[path]["file_bytes_sha256"],
            "field_path": fields,
            "canonical_product_selection_claimed": False,
        }

    base = "artifacts/phase2/"
    # Exact fields observed in C08, not inferred from filenames.
    rules = [
        ("golden_dataset", base +
         "golden_dataset_v2/golden_dataset_v2_manifest.json",
         ["dataset_version"]),
        ("golden_freeze", base +
         "golden_dataset_v2/golden_dataset_v2_manifest.json",
         ["freeze_version"]),
        ("embedding_release", base +
         "embedding/embedding_release_v1.json",
         ["release_version"]),
        ("embedding_model", base +
         "embedding/embedding_release_v1.json",
         ["embedding", "model_id"]),
        ("embedding_revision", base +
         "embedding/embedding_release_v1.json",
         ["embedding", "revision"]),
        ("dense_index", base +
         "dense_index/dense_index_release_v1.json",
         ["release_version"]),
        ("sparse_index", base +
         "sparse_index/sparse_index_release_v1.json",
         ["release_id"]),
        ("hybrid", base +
         "hybrid_retrieval/hybrid_retrieval_release_v1.json",
         ["release_id"]),
        ("reranker", base +
         "reranking/reranker_release_v1.json",
         ["release_id"]),
        ("reranker_model", base +
         "reranking/reranker_configuration_v1.json",
         ["reranker", "model_id"]),
        ("reranker_revision", base +
         "reranking/reranker_configuration_v1.json",
         ["reranker", "revision"]),
        ("context_builder", base +
         "context_builder/context_builder_release_v1.json",
         ["release_id"]),
        ("chunk_evaluation_contract", base +
         "chunking/chunk_contract_evaluation_freeze_v1.json",
         ["chunk_contract_version"]),
    ]
    explicit_versions = {
        role: get_value(path, fields)
        for role, path, fields in rules
    }

    checks = [
        (
            "golden_manifest_report_dataset_version",
            explicit_versions["golden_dataset"]["value"],
            payloads[base +
                "golden_dataset_v2/golden_dataset_v2_final_report.json"
            ]["dataset_version"],
        ),
        (
            "embedding_dense_revision",
            explicit_versions["embedding_revision"]["value"],
            payloads[base +
                "dense_index/dense_index_release_v1.json"
            ]["embedding"]["revision"],
        ),
        (
            "embedding_hybrid_revision",
            explicit_versions["embedding_revision"]["value"],
            payloads[base +
                "hybrid_retrieval/hybrid_configuration_v1.json"
            ]["dense"]["model_revision"],
        ),
        (
            "reranker_context_input_release",
            explicit_versions["reranker"]["value"],
            payloads[base +
                "context_builder/context_builder_configuration_v1.json"
            ]["input_provenance"]["source_release_id"],
        ),
    ]
    consistency = [
        {
            "check": name,
            "left": left,
            "right": right,
            "status": "MATCH" if left == right else "RECONCILE",
        }
        for name, left, right in checks
    ]

    pending = [
        {
            "component": component,
            "status": "AUTHORITATIVE_BINDING_NOT_ESTABLISHED_BY_C09",
            "next_work": next_work,
            "existing_observations_preserved": True,
            "absence_projectwide_claimed": False,
        }
        for component, next_work in [
            ("code", "اتصال نسخهٔ کد به Producer و Artifact، نه صرف HEAD فعلی"),
            ("corpus", "اتصال نسخهٔ Corpus فعال از Manifest/Release مصرف‌کننده"),
            ("active_chunks", "تعیین Chunk فعال؛ قرارداد آزمایش جای نسخهٔ داده نیست"),
            ("split_and_gold_mapping", "اتصال نسخه و تعریف هش Split و Gold"),
            ("generation_and_prompt", "تعیین منشأ و نسخهٔ مولد و Prompt در دامنهٔ واقعی"),
            ("answerability_safety_confidence_policy",
             "اتصال نسخهٔ Policy و حدود پذیرش از شواهد موجود"),
            ("evaluation_freeze", "اتصال قرارداد و نسخهٔ ارزیابی به کاندیدای محصول"),
        ]
    ]

    artifact = {
        "artifact_type": "NB22_QUALIFIED_VERSION_REGISTRY",
        "schema_version": "NB22_C009_ACTION07_QUALIFIED_VERSIONS_V1",
        "status": "VERSION_INVENTORY_QUALIFIED_WITH_EXPLICIT_BINDING_GAPS",
        "input_sources": original["sources"],
        "raw_observation_count": len(original["observations"]),
        "retained_observations": retained,
        "schema_definitions_excluded_from_version_values":
            schema_definitions,
        "empty_observations": empty,
        "explicit_component_versions": explicit_versions,
        "cross_document_checks": consistency,
        "pending_authoritative_bindings": pending,
        "raw_input_registry_modified": False,
        "candidate_release_id": None,
        "canonical_selection_complete": False,
        "action07_full_completion_established": False,
        "release_acceptance_granted": False,
        "checklist_modified": False,
        "new_api_calls": 0,
    }

    raw = json.dumps(
        artifact, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    digest = hashlib.sha256(raw).hexdigest()
    directory = root / "reports/phase2/release/NB22.C009.action07"
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"C009.qualified_versions.{digest}.json"

    if target.exists():
        if target.read_bytes() != raw:
            raise RuntimeError("EXISTING_OUTPUT_CONFLICT")
        write_status = "REUSED_IDENTICAL"
    else:
        fd, name = tempfile.mkstemp(prefix=".C009.", dir=directory)
        temporary = Path(name)
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            if temporary.read_bytes() != raw:
                raise RuntimeError("TEMPORARY_VALIDATION_FAILED")
            try:
                os.link(temporary, target)
                write_status = "CREATED"
            except FileExistsError:
                if target.read_bytes() != raw:
                    raise RuntimeError("CONCURRENT_OUTPUT_CONFLICT")
                write_status = "REUSED_IDENTICAL"
        finally:
            temporary.unlink(missing_ok=True)

    saved = target.read_bytes()
    if (
        hashlib.sha256(saved).hexdigest() != digest
        or json.loads(saved) != artifact
    ):
        raise RuntimeError("RELOAD_VALIDATION_FAILED")

    print(json.dumps({
        "status": artifact["status"],
        "raw_observation_count": artifact["raw_observation_count"],
        "retained_observation_count": len(retained),
        "schema_definition_count": len(schema_definitions),
        "empty_observation_count": len(empty),
        "explicit_component_versions": {
            key: value["value"]
            for key, value in explicit_versions.items()
        },
        "cross_document_checks": consistency,
        "pending_authoritative_bindings": pending,
        "output": {
            "path": target.relative_to(root).as_posix(),
            "file_bytes_sha256": digest,
            "write_status": write_status,
        },
        "reload_validation": "PASS",
        "action07_full_completion_established": False,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "payloads": payloads}


nb22_c09_result = nb22_c09_run(nb22_c08_result)

{
  "status": "VERSION_INVENTORY_QUALIFIED_WITH_EXPLICIT_BINDING_GAPS",
  "raw_observation_count": 108,
  "retained_observation_count": 105,
  "schema_definition_count": 3,
  "empty_observation_count": 0,
  "explicit_component_versions": {
    "golden_dataset": "golden_dataset_v2_registry_driven_v2",
    "golden_freeze": "golden_dataset_v2_final_freeze_v1",
    "embedding_release": "embedding_release_v1",
    "embedding_model": "intfloat/multilingual-e5-small",
    "embedding_revision": "614241f622f53c4eeff9890bdc4f31cfecc418b3",
    "dense_index": "dense_index_v1",
    "sparse_index": "rag_finance_phase2_sparse_v1",
    "hybrid": "rag_finance_phase2_hybrid_v1",
    "reranker": "rag_finance_phase2_reranker_v1",
    "reranker_model": "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1",
    "reranker_revision": "1427fd652930e4ba29e8149678df786c240d8825",
    "context_builder": "rag_finance_phase2_context_builder_v1",
    "chunk_evaluation_contract": "chunk_contract_v1_draft"
  },
  "cross_docum

<div dir="rtl" align="right">

### NB22-C10 — اقدام ۷: اتصال نسخه‌ها به شواهد موجود

این بسته سه زیرکار دارد:

۱. تطبیق مسیر و هش Chunk مصرف‌شده در Embedding، Dense و Context.
۲. جمع‌آوری ارجاع‌های موجود برای کد، Corpus، Split/Gold،
مولد/Prompt، Policy و فریز ارزیابی.
۳. ثبت اتصال‌های صریح و موارد نیازمند تطبیق با منشأ دقیق.

مقادیر از اسناد خوانده‌شدهٔ C08 مصرف می‌شوند.
متن پاسخ‌ها، پرونده‌های خطا و داوری‌های جمعیت دوباره پیمایش نمی‌شوند.
هش Corpus بستهٔ Recovery با هش Corpus فریز‌شده ادغام نمی‌شود.
نام مدل در تنظیمات، اجرای واقعی آن مدل را ثابت نمی‌کند.

در این سل هش داده‌های اصلی دوباره محاسبه نمی‌شود؛
تطبیق، در سطح اظهارات اسناد موجود است.
تعریف نامعلوم هش صریح باقی می‌ماند.

خروجی جدید Atomic ذخیره و بازخوانی می‌شود.
API، مدل، جست‌وجوی مخزن، تغییر ورودی و تغییر چک‌لیست: صفر.

</div>

In [10]:
# NB22-C10 — Action 07: connect declared metadata to existing provenance
# Scope: cached metadata only; no data reads, discovery, models or API.

def nb22_c10_run(previous):
    registry = previous["artifact"]
    payloads = previous["payloads"]

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    sources = {
        item["path"]: item for item in registry["input_sources"]
    }
    base = "artifacts/phase2/"
    embedding_path = base + "embedding/embedding_release_v1.json"
    dense_path = base + "dense_index/dense_index_release_v1.json"
    context_path = (
        base + "context_builder/context_builder_configuration_v1.json"
    )

    embedding = payloads[embedding_path]
    dense = payloads[dense_path]
    context = payloads[context_path]
    provenance = context["input_provenance"]

    chunk_bindings = [
        {
            "consumer": "embedding",
            "source_document": embedding_path,
            "source_document_sha256":
                sources[embedding_path]["file_bytes_sha256"],
            "path": embedding["chunking"]["path"],
            "declared_sha256": embedding["chunking"]["sha256"],
            "strategy": embedding["chunking"]["strategy"],
            "record_count": embedding["chunking"]["record_count"],
        },
        {
            "consumer": "dense_index",
            "source_document": dense_path,
            "source_document_sha256":
                sources[dense_path]["file_bytes_sha256"],
            "path": dense["inputs"]["chunk_path"],
            "declared_sha256": dense["inputs"]["chunk_sha256"],
            "record_count": dense["inputs"]["chunk_count"],
        },
        {
            "consumer": "context_builder",
            "source_document": context_path,
            "source_document_sha256":
                sources[context_path]["file_bytes_sha256"],
            "path": provenance["input_paths"]["canonical_chunks"],
            "declared_sha256":
                provenance["validated_input_hashes"]["canonical_chunks"],
        },
    ]
    chunk_chain_matches = (
        len({r["path"] for r in chunk_bindings}) == 1
        and len({r["declared_sha256"] for r in chunk_bindings}) == 1
        and chunk_bindings[0]["record_count"]
            == chunk_bindings[1]["record_count"]
    )

    patterns = {
        "code": re.compile(
            r"(^|\.)(commit|commit_sha|git_commit|code_sha256|"
            r"code_version|producer_commit|git_head|head_sha)$",
            re.I,
        ),
        "corpus": re.compile(
            r"corpus|canonical_document|documents_dedup|dataset_release",
            re.I,
        ),
        "split_and_gold_mapping": re.compile(
            r"split|gold_mapping|gold_to_chunk|group_membership",
            re.I,
        ),
        "generation_and_prompt": re.compile(
            r"prompt|generation_origin|generation_model|"
            r"model_identity|qwen_execution|configured_model|"
            r"generator_version|generation_version",
            re.I,
        ),
        "answerability_safety_confidence_policy": re.compile(
            r"policy|rubric|answerability|confidence|calibration",
            re.I,
        ),
        "evaluation_freeze": re.compile(
            r"freeze|execution_contract|evaluation_version|"
            r"evaluation_blueprint|contract_version",
            re.I,
        ),
    }
    collected = {name: [] for name in patterns}
    omitted_lists = []

    excluded_files = (
        "failure_catalogue", "failure_cases",
        "regression_specs", "regression_scenarios",
        "open_reviews", "actions01_49_status",
        "error_taxonomy",
    )
    excluded_keys = {
        "payload_schema", "properties", "$defs", "definitions",
        "answer_text", "question", "evidence_text", "messages",
        "claims", "cases", "candidate_evidence_links",
    }

    def record(source, trail, value):
        field = ".".join(trail)
        for component, pattern in patterns.items():
            if pattern.search(field):
                collected[component].append({
                    "source_document": source,
                    "source_document_sha256":
                        sources[source]["file_bytes_sha256"],
                    "field_path": list(trail),
                    "declared_value": value,
                    "interpretation":
                        "SOURCE_DECLARATION_NOT_NEW_VALIDATION",
                })

    def walk(node, source, trail=()):
        if isinstance(node, dict):
            for key, value in node.items():
                if key in excluded_keys:
                    continue
                location = trail + (str(key),)
                if isinstance(value, (dict, list)):
                    walk(value, source, location)
                else:
                    record(source, location, value)
        elif isinstance(node, list):
            if len(node) > 50:
                omitted_lists.append({
                    "source_document": source,
                    "field_path": list(trail),
                    "count": len(node),
                })
                return
            for i, value in enumerate(node):
                location = trail + (str(i),)
                if isinstance(value, (dict, list)):
                    walk(value, source, location)
                else:
                    record(source, location, value)

    for path, value in payloads.items():
        if path not in sources or not isinstance(value, dict):
            continue
        if any(token in Path(path).name for token in excluded_files):
            continue
        walk(value, path)

    # Keep raw declarations intact; do not infer hash semantics from names.
    component_results = {}
    for component, entries in collected.items():
        component_results[component] = {
            "status": (
                "EXISTING_DECLARATIONS_LOCATED_RECONCILIATION_REQUIRED"
                if entries else
                "NOT_ESTABLISHED_IN_INSPECTED_METADATA"
            ),
            "declaration_count": len(entries),
            "declarations": entries,
            "authoritative_version_selected": False,
            "absence_projectwide_claimed": False,
        }

    artifact = {
        "artifact_type": "NB22_VERSION_PROVENANCE_BINDINGS",
        "schema_version": "NB22_C010_ACTION07_PROVENANCE_V1",
        "status": (
            "EXISTING_VERSION_BINDINGS_ASSEMBLED"
            if chunk_chain_matches else
            "CHUNK_CHAIN_RECONCILIATION_REQUIRED"
        ),
        "source_documents": registry["input_sources"],
        "explicit_component_versions":
            registry["explicit_component_versions"],
        "active_chunk_chain": {
            "declared_path_and_hash_agreement": chunk_chain_matches,
            "consumer_bindings": chunk_bindings,
            "data_bytes_rehashed": False,
            "hash_definition_revalidated": False,
            "productwide_canonical_selection_claimed": False,
        },
        "remaining_component_evidence": component_results,
        "large_lists_not_traversed": omitted_lists,
        "limitations": [
            "Field matching locates declarations; it does not establish authority.",
            "No producer code identity inferred from current Git HEAD.",
            "Recovery corpus and frozen corpus hashes remain distinct.",
            "A tokenizer/model declaration does not establish live model execution.",
            "Missing version identifiers are not synthesized from filenames.",
            "Data-byte hash verification remains a separate downstream task.",
        ],
        "action07_full_completion_established": False,
        "release_acceptance_granted": False,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "checklist_modified": False,
    }

    raw = json.dumps(
        artifact, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    digest = hashlib.sha256(raw).hexdigest()
    directory = root / "reports/phase2/release/NB22.C010.action07"
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"C010.bindings.{digest}.json"

    if target.exists():
        if target.read_bytes() != raw:
            raise RuntimeError("EXISTING_OUTPUT_CONFLICT")
        write_status = "REUSED_IDENTICAL"
    else:
        fd, name = tempfile.mkstemp(prefix=".C010.", dir=directory)
        temporary = Path(name)
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            if temporary.read_bytes() != raw:
                raise RuntimeError("TEMPORARY_VALIDATION_FAILED")
            try:
                os.link(temporary, target)
                write_status = "CREATED"
            except FileExistsError:
                if target.read_bytes() != raw:
                    raise RuntimeError("CONCURRENT_OUTPUT_CONFLICT")
                write_status = "REUSED_IDENTICAL"
        finally:
            temporary.unlink(missing_ok=True)

    saved = target.read_bytes()
    if (
        hashlib.sha256(saved).hexdigest() != digest
        or json.loads(saved) != artifact
    ):
        raise RuntimeError("RELOAD_VALIDATION_FAILED")

    # Display nonduplicated declarations; the full registry stays on disk.
    display = {}
    for component, result in component_results.items():
        unique, seen = [], set()
        for item in result["declarations"]:
            identity = (
                tuple(item["field_path"]),
                json.dumps(item["declared_value"], sort_keys=True),
            )
            if identity not in seen:
                seen.add(identity)
                unique.append(item)
        display[component] = {
            "status": result["status"],
            "total_declarations": len(result["declarations"]),
            "unique_declarations": unique,
        }

    print(json.dumps({
        "status": artifact["status"],
        "active_chunk_chain": artifact["active_chunk_chain"],
        "remaining_component_evidence": display,
        "output": {
            "path": target.relative_to(root).as_posix(),
            "file_bytes_sha256": digest,
            "write_status": write_status,
        },
        "reload_validation": "PASS",
        "action07_full_completion_established": False,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "payloads": payloads}


nb22_c10_result = nb22_c10_run(nb22_c09_result)

{
  "status": "EXISTING_VERSION_BINDINGS_ASSEMBLED",
  "active_chunk_chain": {
    "declared_path_and_hash_agreement": true,
    "consumer_bindings": [
      {
        "consumer": "embedding",
        "source_document": "artifacts/phase2/embedding/embedding_release_v1.json",
        "source_document_sha256": "8c48acc7c266dd153bce038ff6ec679b716e88679ccd865e15c333e4fbd7f429",
        "path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
        "declared_sha256": "3891c4ecad9332ba753cd605bbccd99da9d063f64934a30ee39e495349cffb50",
        "strategy": "baseline_phase1_char_safe",
        "record_count": 3488
      },
      {
        "consumer": "dense_index",
        "source_document": "artifacts/phase2/dense_index/dense_index_release_v1.json",
        "source_document_sha256": "706c020caa2bda77c767fda8cd1b57498818afdbdbbbdd2a72f898b55ca6ca50",
        "path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
        "declared_sha256":

<div dir="rtl" align="right">

### NB22-C11 — اقدام ۷: اتصال‌های مستقیم نسخه، Prompt و Policy

سه زیرکار این بسته:

۱. استخراج مستقیم اتصال Split/Gold و قرارداد ارزیابی از اسناد موجود.
۲. خواندن Handoff مشخص Context برای یافتن اتصال مستند Corpus.
۳. استخراج ارجاع‌های اصلی Releaseهای NB19 و Handoff نهایی NB20
برای ادامهٔ اتصال Prompt و Policy.

جست‌وجوی واژه‌ای وسیع و خروجی‌های صدها‌ردیفی تکرار نمی‌شوند.
نسخه‌های تاریخی NB19 کنار هم باقی می‌مانند و نسخهٔ بالاتر خودکار انتخاب نمی‌شود.
هش منطقی Split/Gold، هش بایت فایل معرفی نمی‌شود.

تمام اسناد قبلی از حافظهٔ همین نوت‌بوک مصرف می‌شوند.
فقط یک Handoff کوچک و مشخص، در صورت نبودن در ورودی‌های قبلی، خوانده می‌شود.
هیچ داده، پاسخ، داوری یا مدل بازتولید نمی‌شود.

این سل فایل خروجی جدید نمی‌نویسد.
API، تغییر چک‌لیست و درخواست شبکه: صفر.

</div>

In [11]:
# NB22-C11 — Direct metadata bindings for action 07
# Scope: cached metadata plus one explicit context handoff; no writes/API.

def nb22_c11_run(c09, c10):
    payloads = dict(c10["payloads"])
    source_rows = {
        row["path"]: row
        for row in c09["artifact"]["input_sources"]
    }

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    def one(prefix):
        matches = [
            p for p in payloads
            if Path(p).name.startswith(prefix)
        ]
        if len(matches) != 1:
            raise RuntimeError(f"SOURCE_NOT_UNIQUE: {prefix}")
        path = matches[0]
        return path, payloads[path]

    def origin(path):
        row = source_rows[path]
        return {
            "path": path,
            "file_bytes_sha256": row["file_bytes_sha256"],
        }

    def safe(value):
        if isinstance(value, dict):
            return {
                key: (
                    "[REDACTED]" if any(
                        token in key.lower()
                        for token in (
                            "secret", "api_key", "authorization",
                            "password", "access_token"
                        )
                    ) else safe(item)
                )
                for key, item in value.items()
            }
        if isinstance(value, list):
            return [safe(item) for item in value]
        if isinstance(value, str):
            value = value.replace(str(root), "<PROJECT_ROOT>")
            return re.sub(
                r"/(?:Users|home)/[^/\s]+", "<USER_HOME>", value
            )
        return value

    def compact(value, depth=0):
        """Show metadata structure without expanding populations or text."""
        if isinstance(value, dict):
            if depth >= 3:
                return {"object_keys": list(value)}
            return {
                key: compact(item, depth + 1)
                for key, item in value.items()
                if key.lower() not in {
                    "messages", "answer_text", "question",
                    "evidence_text", "claims", "cases",
                }
            }
        if isinstance(value, list):
            return {
                "list_count": len(value),
                "first_item_structure": (
                    compact(value[0], depth + 1)
                    if value and isinstance(value[0], dict)
                    else None
                ),
            }
        if isinstance(value, str) and len(value) > 700:
            return {
                "text_length": len(value),
                "full_text_not_displayed": True,
            }
        return safe(value)

    hybrid_path = (
        "artifacts/phase2/hybrid_retrieval/"
        "hybrid_configuration_v1.json"
    )
    hybrid = payloads[hybrid_path]
    canonical = hybrid.get("canonical_inputs")
    if not isinstance(canonical, dict):
        raise RuntimeError("HYBRID_CANONICAL_INPUTS_NOT_OBJECT")

    contract_path, contract = one("C082.frozen_contract.")
    if not isinstance(contract, dict):
        raise RuntimeError("EVALUATION_CONTRACT_NOT_OBJECT")

    # Existing project path; no recursive filesystem discovery.
    context_handoff_path = (
        "artifacts/phase2/context_builder/"
        "nb19_context_builder_handoff_v1.json"
    )
    fresh_reads = 0
    context_read_status = "REUSED_EXISTING_METADATA"
    if context_handoff_path not in payloads:
        target = (root / context_handoff_path).resolve()
        target.relative_to(root)
        if not target.is_file():
            context_read_status = "NOT_AVAILABLE_AT_EXPLICIT_LOCAL_PATH"
        elif target.stat().st_size > 1024 * 1024:
            context_read_status = "METADATA_SIZE_REQUIRES_REVIEW"
        else:
            raw = target.read_bytes()
            value = json.loads(raw.decode("utf-8"))
            if not isinstance(value, dict):
                raise RuntimeError("CONTEXT_HANDOFF_ROOT_NOT_OBJECT")
            payloads[context_handoff_path] = value
            source_rows[context_handoff_path] = {
                "path": context_handoff_path,
                "file_bytes_sha256": hashlib.sha256(raw).hexdigest(),
                "bytes": len(raw),
            }
            fresh_reads = 1
            context_read_status = "EXPLICIT_METADATA_READ"

    # Inspect package references directly, not every matching word.
    nb19_packages = []
    reference_fields = (
        "inputs", "input_paths", "input_hashes",
        "artifacts", "artifact_roles", "artifact_sha256",
        "configuration", "configuration_path",
        "configuration_sha256", "config",
        "prompt", "prompt_path", "prompt_sha256",
        "prompt_version", "generation", "model",
        "source_files", "provenance",
    )
    for version in range(1, 5):
        name = f"nb19_development_release_v{version}.json"
        matches = [p for p in payloads if Path(p).name == name]
        if len(matches) != 1:
            raise RuntimeError(f"NB19_RELEASE_NOT_UNIQUE: {name}")
        path = matches[0]
        value = payloads[path]
        nb19_packages.append({
            "source": origin(path),
            "recorded_status": value.get("status"),
            "top_level_keys": list(value),
            "direct_reference_sections": {
                field: compact(value[field])
                for field in reference_fields if field in value
            },
            "canonical_selection_claimed": False,
        })

    c455_path, c455 = one("C455.handoff.")
    recovery_path, recovery = one(
        "nb19_test_handoff_chatgpt_recovery_v1."
    )

    result = {
        "status": "DIRECT_METADATA_BINDINGS_READY_FOR_SCOPE_DECISION",
        "chunk_chain_agreement":
            c10["artifact"]["active_chunk_chain"][
                "declared_path_and_hash_agreement"
            ],
        "split_and_gold": {
            "source": origin(hybrid_path),
            "canonical_inputs": compact(canonical),
            "declared_hash_definition":
                "MUST_USE_PRODUCER_CONTRACT_NOT_ASSUME_FILE_BYTES",
        },
        "evaluation_contract": {
            "source": origin(contract_path),
            "top_level_keys": list(contract),
            "schema_version": contract.get("schema_version"),
            "artifact_type": contract.get("artifact_type"),
            "direct_bindings": {
                key: compact(contract[key])
                for key in (
                    "inputs", "input_bindings", "bindings",
                    "blueprint", "source_bindings",
                    "uncertainty_policy", "decision_provenance",
                ) if key in contract
            },
            "contract_is_product_quality_acceptance": False,
        },
        "context_handoff": {
            "read_status": context_read_status,
            "source": (
                origin(context_handoff_path)
                if context_handoff_path in payloads else
                {"path": context_handoff_path}
            ),
            "metadata": (
                compact(payloads[context_handoff_path])
                if context_handoff_path in payloads else None
            ),
        },
        "NB19_release_references": nb19_packages,
        "NB19_test_recovery": {
            "source": origin(recovery_path),
            "scope": compact(recovery.get("scope")),
            "top_level_keys": list(recovery),
            "artifact_roles": compact(recovery.get("artifact_roles")),
            "live_qwen_execution_inferred": False,
        },
        "NB20_final_handoff": {
            "source": origin(c455_path),
            "handoff_scope": c455.get("handoff_scope"),
            "entry_points": compact(c455.get("entry_points")),
            "input_hashes": compact(c455.get("input_hashes")),
            "policy_version_inferred_from_handoff_name": False,
        },
        "fresh_metadata_files_read": fresh_reads,
        "previous_input_files_reread": 0,
        "action07_full_completion_established": False,
        "files_written": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    print(json.dumps(safe(result), ensure_ascii=False, indent=2))
    return {
        "result": result,
        "payloads": payloads,
        "source_bindings": source_rows,
    }


nb22_c11_result = nb22_c11_run(nb22_c09_result, nb22_c10_result)

{
  "status": "DIRECT_METADATA_BINDINGS_READY_FOR_SCOPE_DECISION",
  "chunk_chain_agreement": true,
  "split_and_gold": {
    "source": {
      "path": "artifacts/phase2/hybrid_retrieval/hybrid_configuration_v1.json",
      "file_bytes_sha256": "e7dc28836b0e6b2f72cf19aa61507af41a76df4980b056f39af015ed91ba26d5"
    },
    "canonical_inputs": {
      "chunk_sha256": "3891c4ecad9332ba753cd605bbccd99da9d063f64934a30ee39e495349cffb50",
      "dense_logical_index_sha256": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
      "dense_release_sha256": "706c020caa2bda77c767fda8cd1b57498818afdbdbbbdd2a72f898b55ca6ca50",
      "development_query_embedding_metadata_sha256": "bc97f189d5cadb801f5aaaad9a86715509ad2a91a18dc4840b0918c0c8f2568e",
      "development_query_embedding_sha256": "ca2641a2b8a484433a85ae0682b9b600c02e0c7e4e670af4e07eeb61099e4bb5",
      "gold_mapping_sha256": "42216f67665cfc50697789dffe4eb550839ee22bcbfc0489accc407519dad4d1",
      "golden_dataset_sha256": "b

<div dir="rtl" align="right">

### NB22-C12 — اقدام ۷: تکمیل اتصال Manifest و Rubric

این بسته سه زیرکار دارد:

۱. مصرف اتصال مستند Corpus، Chunk، Split/Gold و قرارداد ارزیابی.
۲. خواندن Manifestهای مشخص NB19 و Rubric مشخصِ ارجاع‌شده در C455.
۳. ثبت نسخه‌ها، ارجاع‌های Prompt/Policy و موارد احرازنشده در دفتر واحد.

نسخهٔ بالاتر خودکار Canonical نیست.
Rubric ممیزی پسینی، Policy کامل محصول نامیده نمی‌شود.
هش و منشأ Corpus بستهٔ Recovery جدا از Corpus پردازش‌شده باقی می‌مانند.

فایل‌های داده، پاسخ، Index و داوری خوانده یا بازسازی نمی‌شوند.
فقط Metadata کوچک و مشخص خوانده می‌شود.
خروجی قبلی حفظ و جمع‌بندی جدید Atomic ذخیره و بازخوانی می‌شود.

API، اجرای مدل و تغییر چک‌لیست: صفر.

</div>

In [12]:
# NB22-C12 — Action 07: bind producer manifests and scoped rubric
# Scope: explicit small metadata only; no data/model/API execution.

def nb22_c12_run(c09, c10, c11):
    payloads = dict(c11["payloads"])
    sources = dict(c11["source_bindings"])

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    def named(name):
        matches = [p for p in payloads if Path(p).name == name]
        if len(matches) != 1:
            raise RuntimeError(f"DOCUMENT_NOT_UNIQUE: {name}")
        return matches[0], payloads[matches[0]]

    def prefixed(prefix):
        matches = [
            p for p in payloads if Path(p).name.startswith(prefix)
        ]
        if len(matches) != 1:
            raise RuntimeError(f"DOCUMENT_NOT_UNIQUE: {prefix}")
        return matches[0], payloads[matches[0]]

    nb19_work = Path("artifacts/phase2/generation_and_citation/work")
    requests = []

    for version in range(1, 5):
        source_path, release = named(
            f"nb19_development_release_v{version}.json"
        )
        if version <= 2:
            ref = release.get("manifest")
            if not isinstance(ref, dict):
                raise RuntimeError("NB19_MANIFEST_REFERENCE_NOT_OBJECT")
            relative = ref.get("path") or ref.get("relative_path")
            expected = ref.get("sha256") or ref.get("file_bytes_sha256")
            if not isinstance(relative, str):
                raise RuntimeError("NB19_MANIFEST_PATH_NOT_PRESENT")
            relative = (
                relative if relative.startswith(
                    ("artifacts/", "reports/", "data/")
                ) else (nb19_work / relative).as_posix()
            )
        else:
            # Exact versioned manifest family; expected digest is declared
            # by the corresponding release. No highest-version selection.
            relative = (
                nb19_work / "outputs" /
                f"nb19_development_manifest_v{version}.json"
            ).as_posix()
            expected = release.get(f"manifest_v{version}_sha256")

        requests.append({
            "role": f"NB19_DEVELOPMENT_MANIFEST_V{version}",
            "path": relative,
            "declared_sha256": expected,
            "binding_source": source_path,
            "path_resolution": (
                "EXPLICIT_RELEASE_REFERENCE" if version <= 2 else
                "VERSIONED_PRODUCER_MANIFEST_PATH_REQUIRES_HASH_MATCH"
            ),
        })

    recovery_path, recovery = named(
        "nb19_test_handoff_chatgpt_recovery_v1.json"
    )
    ref = recovery["artifact_roles"]["manifest"]
    requests.append({
        "role": "NB19_TEST_RECOVERY_MANIFEST",
        "path": ref["relative_path"],
        "declared_sha256": ref["sha256"],
        "binding_source": recovery_path,
        "path_resolution": "EXPLICIT_HANDOFF_REFERENCE",
    })

    c455_path, c455 = prefixed("C455.handoff.")
    rubric_refs = [
        (path, digest)
        for path, digest in c455["input_hashes"].items()
        if Path(path).name == "nb20_criticality_evaluation_rubric_v2.json"
    ]
    if len(rubric_refs) != 1:
        raise RuntimeError("C455_RUBRIC_BINDING_NOT_UNIQUE")
    requests.append({
        "role": "NB20_POSTHOC_CRITICALITY_RUBRIC",
        "path": rubric_refs[0][0],
        "declared_sha256": rubric_refs[0][1],
        "binding_source": c455_path,
        "path_resolution": "EXPLICIT_C455_REFERENCE",
    })

    metadata_rows = []
    for request in requests:
        row = dict(request)
        relative = request["path"]
        try:
            p = Path(relative)
            if p.is_absolute() or ".." in p.parts:
                raise ValueError("INVALID_PROJECT_RELATIVE_PATH")
            target = (root / p).resolve()
            target.relative_to(root)
            expected = request["declared_sha256"]
            if not isinstance(expected, str) or not re.fullmatch(
                r"[0-9a-f]{64}", expected
            ):
                raise ValueError("EXPLICIT_HASH_BINDING_REQUIRED")

            if relative in payloads:
                value = payloads[relative]
                actual = sources[relative]["file_bytes_sha256"]
                row["read_mode"] = "REUSED_CACHED_METADATA"
            else:
                if not target.is_file():
                    raise FileNotFoundError()
                if target.stat().st_size > 4 * 1024 * 1024:
                    raise ValueError("METADATA_SIZE_REQUIRES_REVIEW")
                raw = target.read_bytes()
                actual = hashlib.sha256(raw).hexdigest()
                if actual != expected:
                    row["actual_file_bytes_sha256"] = actual
                    raise ValueError("DECLARED_HASH_NOT_FILE_BYTES_MATCH")
                value = json.loads(raw.decode("utf-8"))
                if not isinstance(value, dict):
                    raise ValueError("METADATA_ROOT_NOT_OBJECT")
                payloads[relative] = value
                sources[relative] = {
                    "path": relative,
                    "file_bytes_sha256": actual,
                    "bytes": len(raw),
                }
                row["read_mode"] = "FRESH_REFERENCED_METADATA"

            if actual != expected:
                raise ValueError("CACHED_METADATA_BINDING_MISMATCH")
            row.update({
                "status": "HASH_BOUND_METADATA_READ",
                "file_bytes_sha256": actual,
                "top_level_keys": list(value),
                "metadata": value,
            })
        except Exception as exc:
            row.update({
                "status": "BINDING_REQUIRES_RECONCILIATION",
                "error_type": type(exc).__name__,
                "reason": (
                    str(exc) if isinstance(exc, ValueError)
                    else "FILE_NOT_AVAILABLE_AT_RESOLVED_PATH"
                    if isinstance(exc, FileNotFoundError)
                    else "LOCAL_METADATA_READ_FAILED"
                ),
                "absence_projectwide_claimed": False,
            })
        metadata_rows.append(row)

    def compact(node, depth=0):
        if isinstance(node, dict):
            if depth >= 3:
                return {"object_keys": list(node)}
            return {
                key: (
                    "[REDACTED]" if any(
                        token in key.lower()
                        for token in (
                            "secret", "api_key", "authorization",
                            "password", "access_token"
                        )
                    ) else compact(value, depth + 1)
                )
                for key, value in node.items()
                if key not in (
                    "messages", "answer_text", "question",
                    "evidence_text", "claims", "cases",
                )
            }
        if isinstance(node, list):
            return {"list_count": len(node)}
        if isinstance(node, str):
            if len(node) > 500:
                return {"text_length": len(node)}
            node = node.replace(str(root), "<PROJECT_ROOT>")
            return re.sub(
                r"/(?:Users|home)/[^/\s]+", "<USER_HOME>", node
            )
        return node

    # The stored report preserves compact metadata references, not raw prompts.
    result_rows = []
    for row in metadata_rows:
        result_rows.append({
            **{k: v for k, v in row.items() if k != "metadata"},
            **(
                {"metadata_summary": compact(row["metadata"])}
                if "metadata" in row else {}
            ),
        })

    handoff = c11["result"]["context_handoff"]["metadata"]
    artifact = {
        "artifact_type": "NB22_ACTION07_PRODUCER_BINDINGS",
        "schema_version": "NB22_C012_ACTION07_PRODUCER_BINDINGS_V1",
        "status": (
            "REFERENCED_METADATA_BOUND"
            if all(
                r["status"] == "HASH_BOUND_METADATA_READ"
                for r in metadata_rows
            ) else "EXPLICIT_BINDING_GAPS_RECORDED"
        ),
        "document_corpus": {
            "source": c11["result"]["context_handoff"]["source"],
            "path": handoff["input_paths"]["documents"],
            "declared_sha256": handoff["input_sha256"]["documents"],
            "data_bytes_verified_this_cell": False,
        },
        "active_chunk_chain": c10["artifact"]["active_chunk_chain"],
        "split_and_gold": c11["result"]["split_and_gold"],
        "evaluation_contract": c11["result"]["evaluation_contract"],
        "component_versions": c09["artifact"]["explicit_component_versions"],
        "producer_metadata": result_rows,
        "unresolved_code_provenance":
            "PRODUCER_CODE_VERSION_NOT_YET_BOUND",
        "policy_scope":
            "CRITICALITY_RUBRIC_IS_NOT_FULL_PRODUCT_POLICY",
        "canonical_selection_complete": False,
        "action07_full_completion_established": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }

    raw = json.dumps(
        artifact, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    digest = hashlib.sha256(raw).hexdigest()
    directory = root / "reports/phase2/release/NB22.C012.action07"
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"C012.producer_bindings.{digest}.json"

    if target.exists():
        if target.read_bytes() != raw:
            raise RuntimeError("EXISTING_OUTPUT_CONFLICT")
        write_status = "REUSED_IDENTICAL"
    else:
        fd, name = tempfile.mkstemp(prefix=".C012.", dir=directory)
        temporary = Path(name)
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            if temporary.read_bytes() != raw:
                raise RuntimeError("TEMPORARY_VALIDATION_FAILED")
            try:
                os.link(temporary, target)
                write_status = "CREATED"
            except FileExistsError:
                if target.read_bytes() != raw:
                    raise RuntimeError("CONCURRENT_OUTPUT_CONFLICT")
                write_status = "REUSED_IDENTICAL"
        finally:
            temporary.unlink(missing_ok=True)

    saved = target.read_bytes()
    if (
        hashlib.sha256(saved).hexdigest() != digest
        or json.loads(saved) != artifact
    ):
        raise RuntimeError("RELOAD_VALIDATION_FAILED")

    print(json.dumps({
        "status": artifact["status"],
        "document_corpus": artifact["document_corpus"],
        "producer_metadata": result_rows,
        "output": {
            "path": target.relative_to(root).as_posix(),
            "file_bytes_sha256": digest,
            "write_status": write_status,
        },
        "reload_validation": "PASS",
        "action07_full_completion_established": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {
        "artifact": artifact,
        "payloads": payloads,
        "source_bindings": sources,
    }


nb22_c12_result = nb22_c12_run(
    nb22_c09_result, nb22_c10_result, nb22_c11_result
)

{
  "status": "REFERENCED_METADATA_BOUND",
  "document_corpus": {
    "source": {
      "path": "artifacts/phase2/context_builder/nb19_context_builder_handoff_v1.json",
      "file_bytes_sha256": "f93075361f3e8d5fd7694268c1f89dae05a43b8fa403c982e99bfb66ad272788"
    },
    "path": "data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl",
    "declared_sha256": "9d72c709f81b1218c5aac064052c7ed1da34869f671e6e32762864e0ad70ea40",
    "data_bytes_verified_this_cell": false
  },
  "producer_metadata": [
    {
      "role": "NB19_DEVELOPMENT_MANIFEST_V1",
      "path": "artifacts/phase2/generation_and_citation/work/outputs/nb19_development_manifest_v1.json",
      "declared_sha256": "8ff58d901da662d66ac38606f78f3013b1f3eef2797ba079408bcf0e651387d6",
      "binding_source": "artifacts/phase2/generation_and_citation/work/outputs/nb19_development_release_v1.json",
      "path_resolution": "EXPLICIT_RELEASE_REFERENCE",
      "read_mode": "FRESH_REFERENCED_METADATA",
      "status": "HAS

<div dir="rtl" align="right">

### NB22-C13 — اقدام ۷: جمع‌بندی موجودی نسخه‌ها با شکاف‌های صریح

چهار زیرکار این بسته:

۱. کنترل اتصال هش Manifestهای NB19 از v4 تا v1.
۲. تطبیق نسخه و هش Prompt در Manifestهای v1 و v2.
۳. تجمیع نسخه‌های مؤلفه‌ها، Corpus، Chunk، Split/Gold،
قرارداد ارزیابی و Rubric با منشأ و دامنهٔ واقعی.
۴. ثبت موارد احرازنشده و ذخیره و بازخوانی جمع‌بندی.

نسخهٔ کد تولیدکننده از HEAD فعلی حدس زده نمی‌شود.
Rubric ممیزی، Policy کامل محصول نیست.
Prompt ثبت‌شدهٔ Development به اجرای واقعی Qwen روی Test تعمیم داده نمی‌شود.

پایان جمع‌آوری نسخه‌ها با کامل‌بودن نسخه‌های Release جداست.
انتخاب Canonical مربوط به اقدام ۸ و کنترل بایت دارایی‌ها مربوط به اقدام ۹ است.

همهٔ ورودی‌ها از نتایج قبلی مصرف می‌شوند.
API، خواندن دوبارهٔ ورودی، تغییر چک‌لیست و بازاجرای مدل: صفر.

</div>

In [13]:
# NB22-C13 — Close the action 07 inventory with explicit provenance gaps
# Scope: cached evidence only; no canonical selection or product acceptance.

def nb22_c13_run(c09, c11, c12):
    if c12["artifact"]["status"] != "REFERENCED_METADATA_BOUND":
        raise RuntimeError("C12_REFERENCED_METADATA_NOT_FULLY_BOUND")

    payloads = c12["payloads"]
    sources = c12["source_bindings"]
    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    def document(name):
        matches = [p for p in payloads if Path(p).name == name]
        if len(matches) != 1:
            raise RuntimeError(f"DOCUMENT_NOT_UNIQUE: {name}")
        p = matches[0]
        return p, payloads[p]

    def binding(path):
        return {
            "path": path,
            "file_bytes_sha256": sources[path]["file_bytes_sha256"],
        }

    manifests = {
        v: document(f"nb19_development_manifest_v{v}.json")
        for v in range(1, 5)
    }
    chain_checks = []
    for version in (2, 3, 4):
        child_path, child = manifests[version]
        parent_path, parent = manifests[version - 1]
        ref = child.get(f"parent_manifest_v{version - 1}")
        if not isinstance(ref, dict):
            raise RuntimeError("PARENT_MANIFEST_BINDING_MISSING")

        expected = ref.get("sha256")
        actual = sources[parent_path]["file_bytes_sha256"]
        if expected != actual:
            raise RuntimeError("PARENT_MANIFEST_HASH_MISMATCH")

        # Preserve the producer's path declaration; hash identifies the
        # already-read parent without inventing a new filesystem path.
        chain_checks.append({
            "child": binding(child_path),
            "parent": binding(parent_path),
            "producer_declared_parent_path": ref.get("path"),
            "hash_binding": "MATCH",
        })

    p1, m1 = manifests[1]
    p2, m2 = manifests[2]
    config1 = m1["configuration"]
    config2 = m2["generation_configuration"]
    prompt_fields = (
        "generation_prompt_template_version",
        "generation_prompt_template_sha256",
    )
    if any(config1[k] != config2[k] for k in prompt_fields):
        raise RuntimeError("DEVELOPMENT_PROMPT_BINDINGS_DIFFER")
    if m1["configuration_sha256"] != m2["generation_configuration_sha256"]:
        raise RuntimeError("DEVELOPMENT_CONFIGURATION_BINDINGS_DIFFER")

    rubric_path, rubric = document(
        "nb20_criticality_evaluation_rubric_v2.json"
    )
    recovery_path, recovery = document(
        "nb19_test_manifest_chatgpt_recovery_v1.json"
    )

    gaps = [
        {
            "component": "producer_code",
            "status": "AUTHORITATIVE_CODE_VERSION_NOT_ESTABLISHED",
            "handling": "Preserve as an explicit release provenance gap.",
            "current_git_head_used_as_substitute": False,
        },
        {
            "component": "complete_product_policy",
            "status": "FULL_PRODUCT_POLICY_VERSION_NOT_ESTABLISHED",
            "handling": "Keep the posthoc rubric separate from product policy.",
        },
        {
            "component": "split_and_gold_hash_definition",
            "status": "DECLARED_IDENTITIES_RECORDED_DEFINITION_NOT_REVALIDATED",
            "handling": "Use the producer contract during canonical binding.",
        },
    ]

    artifact = {
        "artifact_type": "NB22_ACTION07_VERSION_INVENTORY_SUMMARY",
        "schema_version": "NB22_C013_ACTION07_SUMMARY_V1",
        "status": "ACTION07_VERSION_INVENTORY_RECORDED_WITH_EXPLICIT_GAPS",
        "completion_scope":
            "COLLECTION_AND_DOCUMENTATION_OF_AVAILABLE_VERSION_IDENTITIES",
        "component_versions":
            c09["artifact"]["explicit_component_versions"],
        "document_corpus": c12["artifact"]["document_corpus"],
        "chunk_chain": c12["artifact"]["active_chunk_chain"],
        "split_and_gold": c12["artifact"]["split_and_gold"],
        "evaluation_contract": {
            "source": c11["result"]["evaluation_contract"]["source"],
            "schema_version":
                c11["result"]["evaluation_contract"]["schema_version"],
            "product_quality_acceptance": False,
        },
        "NB19_manifest_chain": chain_checks,
        "development_prompt": {
            "version": config1["generation_prompt_template_version"],
            "declared_template_sha256":
                config1["generation_prompt_template_sha256"],
            "declared_configuration_sha256":
                m1["configuration_sha256"],
            "configured_model": config1["model"],
            "sources": [binding(p1), binding(p2)],
            "scope": "RECORDED_DEVELOPMENT_CONFIGURATION",
            "template_bytes_revalidated": False,
            "live_model_identity_verified": False,
        },
        "test_recovery": {
            "source": binding(recovery_path),
            "schema_version": recovery["schema_version"],
            "adapter_version": recovery["adapter_version"],
            "generation_origin": recovery["scope"]["generation_origin"],
            "qwen_execution_claimed":
                recovery["scope"]["qwen_execution_claimed"],
            "historical_scope_preserved": True,
        },
        "posthoc_rubric": {
            "source": binding(rubric_path),
            "schema_version": rubric["schema_version"],
            "scope": rubric["scope"],
            "adoption_scope": rubric["adoption_scope"],
            "product_policy_effect": rubric["product_policy_effect"],
            "full_product_policy_inferred": False,
        },
        "explicit_gaps": gaps,
        "inventory_reporting_complete": True,
        "all_required_release_versions_established": False,
        "canonical_selection_complete": False,
        "quality_blockers_resolved": False,
        "release_acceptance_granted": False,
        "checklist_modified": False,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "next_action": 8,
    }

    raw = json.dumps(
        artifact, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    digest = hashlib.sha256(raw).hexdigest()
    directory = root / "reports/phase2/release/NB22.C013.action07"
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"C013.version_inventory.{digest}.json"

    if target.exists():
        if target.read_bytes() != raw:
            raise RuntimeError("EXISTING_OUTPUT_CONFLICT")
        write_status = "REUSED_IDENTICAL"
    else:
        fd, name = tempfile.mkstemp(prefix=".C013.", dir=directory)
        temporary = Path(name)
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            if temporary.read_bytes() != raw:
                raise RuntimeError("TEMPORARY_VALIDATION_FAILED")
            try:
                os.link(temporary, target)
                write_status = "CREATED"
            except FileExistsError:
                if target.read_bytes() != raw:
                    raise RuntimeError("CONCURRENT_OUTPUT_CONFLICT")
                write_status = "REUSED_IDENTICAL"
        finally:
            temporary.unlink(missing_ok=True)

    saved = target.read_bytes()
    if (
        hashlib.sha256(saved).hexdigest() != digest
        or json.loads(saved) != artifact
    ):
        raise RuntimeError("RELOAD_VALIDATION_FAILED")

    summary = {
        "status": artifact["status"],
        "manifest_parent_hash_links_verified": len(chain_checks),
        "development_prompt_version":
            artifact["development_prompt"]["version"],
        "development_prompt_bindings_match": True,
        "inventory_reporting_complete": True,
        "all_required_release_versions_established": False,
        "explicit_gaps": gaps,
        "output": {
            "path": target.relative_to(root).as_posix(),
            "file_bytes_sha256": digest,
            "write_status": write_status,
        },
        "reload_validation": "PASS",
        "input_files_reread": 0,
        "new_api_calls": 0,
        "checklist_modified": False,
        "release_acceptance_granted": False,
        "next_action": 8,
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return {
        "summary": summary,
        "artifact": artifact,
        "payloads": payloads,
        "source_bindings": sources,
    }


nb22_c13_result = nb22_c13_run(
    nb22_c09_result, nb22_c11_result, nb22_c12_result
)

{
  "status": "ACTION07_VERSION_INVENTORY_RECORDED_WITH_EXPLICIT_GAPS",
  "manifest_parent_hash_links_verified": 3,
  "development_prompt_version": "nb19_generation_prompt_v1",
  "development_prompt_bindings_match": true,
  "inventory_reporting_complete": true,
  "all_required_release_versions_established": false,
  "explicit_gaps": [
    {
      "component": "producer_code",
      "status": "AUTHORITATIVE_CODE_VERSION_NOT_ESTABLISHED",
      "handling": "Preserve as an explicit release provenance gap.",
      "current_git_head_used_as_substitute": false
    },
    {
      "component": "complete_product_policy",
      "status": "FULL_PRODUCT_POLICY_VERSION_NOT_ESTABLISHED",
      "handling": "Keep the posthoc rubric separate from product policy."
    },
    {
      "component": "split_and_gold_hash_definition",
      "status": "DECLARED_IDENTITIES_RECORDED_DEFINITION_NOT_REVALIDATED",
      "handling": "Use the producer contract during canonical binding."
    }
  ],
  "output": {
    "

<div dir="rtl" align="right">

### NB22-C14 — اقدام ۸: ثبت مسیرهای مرجع زنجیرهٔ موجود

چهار زیرکار این بسته:

۱. تعیین نقش Golden، Corpus، Chunk، نامزدهای Reranker،
Context و رجیستری Evidence.
۲. اتصال مسیر و هش اعلام‌شده به سند و مصرف‌کنندهٔ واقعی.
۳. بررسی توافق ارجاع‌ها و وجود محلی، بدون اسکن دوبارهٔ مخزن.
۴. ذخیرهٔ دفتر مسیرها و بازخوانی مستقل آن.

انتخاب‌ها فقط در دامنهٔ زنجیرهٔ مستند مصرف‌کننده هستند.
نسخه‌های تاریخی NB19 حذف یا خودکار جایگزین یکدیگر نمی‌شوند.
پاسخ منتخب NB21 و Policy کامل، بدون شاهد مستقیم Canonical اعلام نمی‌شوند.

وجود فایل با صحت بایت یا پذیرش کیفیت یکسان نیست.
در این سل فایل‌های اصلی باز، بارگذاری یا هش‌گیری نمی‌شوند.
شکاف تعریف هش Split/Gold به مرحلهٔ کنترل قرارداد منتقل می‌شود.

API، اجرای مدل، بازسازی داده و تغییر چک‌لیست: صفر.

</div>

In [14]:
# NB22-C14 — Action 08: bind reference paths to observed consumers
# Scope: cached metadata + explicit path existence checks; no data reads.

def nb22_c14_run(previous, c11):
    if previous["summary"]["reload_validation"] != "PASS":
        raise RuntimeError("ACTION07_SUMMARY_RELOAD_NOT_PASSED")

    payloads = previous["payloads"]
    sources = previous["source_bindings"]
    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    base = "artifacts/phase2/"
    context_path = (
        base + "context_builder/nb19_context_builder_handoff_v1.json"
    )
    config_path = (
        base + "context_builder/context_builder_configuration_v1.json"
    )
    golden_path = (
        base + "golden_dataset_v2/golden_dataset_v2_manifest.json"
    )
    context = payloads[context_path]
    configuration = payloads[config_path]
    golden = payloads[golden_path]

    def evidence(source, field):
        return {
            "source_path": source,
            "source_file_bytes_sha256":
                sources[source]["file_bytes_sha256"],
            "field_path": field,
        }

    candidates = []

    def add(role, path, digest, source, field, hash_definition):
        if not isinstance(path, str) or not path:
            raise ValueError(f"MISSING_EXPLICIT_PATH: {role}")
        relative = Path(path)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"NON_PROJECT_RELATIVE_PATH: {role}")
        target = (root / relative).resolve()
        target.relative_to(root)
        if not isinstance(digest, str) or not re.fullmatch(
            r"[0-9a-f]{64}", digest
        ):
            raise ValueError(f"INVALID_DECLARED_HASH: {role}")
        candidates.append({
            "role": role,
            "path": relative.as_posix(),
            "declared_sha256": digest,
            "hash_definition": hash_definition,
            "binding_evidence": evidence(source, field),
            "local_file_exists": target.is_file(),
            "data_bytes_verified": False,
        })

    # Filename explicitly declared by the Golden manifest.
    dataset_relative = (
        Path(golden_path).parent / golden["dataset_filename"]
    ).as_posix()
    add(
        "golden_dataset", dataset_relative,
        golden["dataset_file_sha256"],
        golden_path, ["dataset_filename", "dataset_file_sha256"],
        "FILE_BYTES_EXPLICITLY_DECLARED_BY_MANIFEST",
    )

    for role, key in (
        ("document_corpus", "documents"),
        ("active_chunks", "chunks"),
        ("reranked_candidates", "candidates"),
    ):
        add(
            role,
            context["input_paths"][key],
            context["input_sha256"][key],
            context_path, ["input_paths", key],
            "PRODUCER_DECLARED_SHA256_DEFINITION_NOT_REVALIDATED_HERE",
        )

    for role, key in (
        ("context_packets", "context_packets"),
        ("context_evidence_registry", "evidence_registry"),
    ):
        ref = context["context_artifact_sha256"][key]
        add(
            role, ref["path"], ref["sha256"],
            context_path, ["context_artifact_sha256", key],
            "PRODUCER_DECLARED_SHA256_DEFINITION_NOT_REVALIDATED_HERE",
        )

    # Add independent declarations already consumed in prior cells.
    for item in previous["artifact"]["chunk_chain"]["consumer_bindings"]:
        add(
            "active_chunks", item["path"], item["declared_sha256"],
            item["source_document"],
            ["PREVIOUSLY_RECORDED_CHUNK_BINDING"],
            "PRODUCER_DECLARED_SHA256_DEFINITION_NOT_REVALIDATED_HERE",
        )

    for role, key in (
        ("golden_dataset", "golden_dataset"),
        ("active_chunks", "canonical_chunks"),
        ("reranked_candidates", "reranked_candidates"),
    ):
        prov = configuration["input_provenance"]
        add(
            role, prov["input_paths"][key],
            prov["validated_input_hashes"][key],
            config_path, ["input_provenance", "input_paths", key],
            "PRODUCER_DECLARED_SHA256_DEFINITION_NOT_REVALIDATED_HERE",
        )

    # Bind Test Recovery inputs without equating Recovery with live Qwen.
    recovery_paths = [
        p for p in payloads
        if Path(p).name == "nb19_test_manifest_chatgpt_recovery_v1.json"
    ]
    if len(recovery_paths) != 1:
        raise RuntimeError("TEST_RECOVERY_MANIFEST_NOT_UNIQUE")
    recovery_path = recovery_paths[0]
    recovery = payloads[recovery_path]
    for role, key in (
        ("document_corpus", "documents"),
        ("active_chunks", "chunks"),
        ("context_packets", "context_packets"),
        ("context_evidence_registry", "evidence_registry"),
    ):
        ref = recovery["frozen_project_files"][key]
        add(
            role, ref["relative_path"], ref["sha256"],
            recovery_path, ["frozen_project_files", key],
            "PRODUCER_DECLARED_SHA256_DEFINITION_NOT_REVALIDATED_HERE",
        )

    roles = []
    for role in sorted({r["role"] for r in candidates}):
        entries = [r for r in candidates if r["role"] == role]
        identities = {
            (r["path"], r["declared_sha256"]) for r in entries
        }
        agreed = len(identities) == 1
        roles.append({
            "role": role,
            "status": (
                "REFERENCE_AGREEMENT_IN_OBSERVED_CONSUMER_CHAIN"
                if agreed else "REFERENCE_CONFLICT_REQUIRES_RECONCILIATION"
            ),
            "reference_path": entries[0]["path"] if agreed else None,
            "declared_sha256":
                entries[0]["declared_sha256"] if agreed else None,
            "local_file_exists":
                all(r["local_file_exists"] for r in entries),
            "evidence_bindings": entries,
            "productwide_canonical_acceptance": False,
        })

    conflicts = [
        r["role"] for r in roles
        if r["status"] == "REFERENCE_CONFLICT_REQUIRES_RECONCILIATION"
    ]
    unavailable = [
        r["role"] for r in roles if not r["local_file_exists"]
    ]
    pending = [
        {
            "role": "NB21_selected_answer_population",
            "status": "CONSUMER_SELECTED_OUTPUT_BINDING_REQUIRED",
            "reason":
                "Do not substitute NB19 historical output merely by version number.",
        },
        {
            "role": "complete_product_policy",
            "status": "AUTHORITATIVE_BINDING_REQUIRED",
            "reason": "Posthoc criticality rubric is a different artifact.",
        },
        {
            "role": "producer_code",
            "status": "PRODUCER_CODE_IDENTITY_NOT_ESTABLISHED",
            "reason": "Current HEAD is not a substitute for historical producer code.",
        },
        {
            "role": "split_and_gold_mapping",
            "status": "LOGICAL_IDENTITIES_RECORDED_PATH_AND_CONTRACT_PENDING",
            "declared_identities":
                c11["result"]["split_and_gold"]["canonical_inputs"],
        },
    ]

    artifact = {
        "artifact_type": "NB22_REFERENCE_PATH_REGISTRY",
        "schema_version": "NB22_C014_ACTION08_REFERENCE_PATHS_V1",
        "status": (
            "REFERENCE_PATHS_RECORDED_WITH_EXPLICIT_PENDING_BINDINGS"
            if not conflicts else "REFERENCE_CONFLICTS_RECORDED"
        ),
        "scope": "OBSERVED_CONTEXT_AND_RETRIEVAL_CONSUMER_CHAIN",
        "roles": roles,
        "conflicting_roles": conflicts,
        "unavailable_at_explicit_paths": unavailable,
        "pending_bindings": pending,
        "raw_data_files_read": 0,
        "raw_data_hashes_recomputed": 0,
        "action08_full_completion_established": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }

    raw = json.dumps(
        artifact, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    digest = hashlib.sha256(raw).hexdigest()
    directory = root / "reports/phase2/release/NB22.C014.action08"
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"C014.reference_paths.{digest}.json"

    if target.exists():
        if target.read_bytes() != raw:
            raise RuntimeError("EXISTING_OUTPUT_CONFLICT")
        write_status = "REUSED_IDENTICAL"
    else:
        fd, name = tempfile.mkstemp(prefix=".C014.", dir=directory)
        temporary = Path(name)
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            if temporary.read_bytes() != raw:
                raise RuntimeError("TEMPORARY_VALIDATION_FAILED")
            try:
                os.link(temporary, target)
                write_status = "CREATED"
            except FileExistsError:
                if target.read_bytes() != raw:
                    raise RuntimeError("CONCURRENT_OUTPUT_CONFLICT")
                write_status = "REUSED_IDENTICAL"
        finally:
            temporary.unlink(missing_ok=True)

    saved = target.read_bytes()
    if (
        hashlib.sha256(saved).hexdigest() != digest
        or json.loads(saved) != artifact
    ):
        raise RuntimeError("RELOAD_VALIDATION_FAILED")

    print(json.dumps({
        "status": artifact["status"],
        "roles": [
            {k: row[k] for k in (
                "role", "status", "reference_path",
                "declared_sha256", "local_file_exists"
            )}
            for row in roles
        ],
        "conflicting_roles": conflicts,
        "unavailable_at_explicit_paths": unavailable,
        "pending_bindings": pending,
        "output": {
            "path": target.relative_to(root).as_posix(),
            "file_bytes_sha256": digest,
            "write_status": write_status,
        },
        "reload_validation": "PASS",
        "raw_data_files_read": 0,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {
        "artifact": artifact,
        "payloads": payloads,
        "source_bindings": sources,
    }


nb22_c14_result = nb22_c14_run(nb22_c13_result, nb22_c11_result)

{
  "status": "REFERENCE_PATHS_RECORDED_WITH_EXPLICIT_PENDING_BINDINGS",
  "roles": [
    {
      "role": "active_chunks",
      "status": "REFERENCE_AGREEMENT_IN_OBSERVED_CONSUMER_CHAIN",
      "reference_path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
      "declared_sha256": "3891c4ecad9332ba753cd605bbccd99da9d063f64934a30ee39e495349cffb50",
      "local_file_exists": true
    },
    {
      "role": "context_evidence_registry",
      "status": "REFERENCE_AGREEMENT_IN_OBSERVED_CONSUMER_CHAIN",
      "reference_path": "data/phase2/context/context_builder_v1/context_evidence_registry_v1.jsonl",
      "declared_sha256": "730aad40c0dac5c94d702f0a22aa3a891045f458a445abd087f34303bebe9fe2",
      "local_file_exists": true
    },
    {
      "role": "context_packets",
      "status": "REFERENCE_AGREEMENT_IN_OBSERVED_CONSUMER_CHAIN",
      "reference_path": "data/phase2/context/context_builder_v1/context_packets_v1.jsonl",
      "declared_sha256": "7789bdb890

<div dir="rtl" align="right">

### NB22-C15 — اقدام ۹، بستهٔ اول: کنترل بایت شش دارایی متصل

چهار زیرکار این بسته:

۱. ثبت اندازهٔ واقعی شش فایلِ مشخص‌شده در C14.
۲. محاسبهٔ SHA256 بایت فایل‌ها به‌صورت جریانی.
۳. مقایسه با مقادیر ثبت‌شده و کنترل تغییرنکردن فایل حین خواندن.
۴. ذخیرهٔ گزارش و بازخوانی مستقل آن.

فقط همین شش مسیر خوانده می‌شوند؛ کشف مخزن تکرار نمی‌شود.
فایل‌ها به‌طور کامل در حافظه بارگذاری و JSONLها بازتولید نمی‌شوند.
برای هر فایل پیشرفت و زمان سپری‌شده نمایش داده می‌شود.

تطبیق بایت، صحت معنایی یا پذیرش کیفیت را ثابت نمی‌کند.
تعریف هشِ مبهم در سند تولیدکننده همچنان محفوظ است؛
نتیجهٔ این سل دقیقاً مقایسهٔ SHA256 بایت واقعی با مقدار اعلام‌شده است.

اقدام ۸ و کنترل کامل Index، Config و هش‌های منطقی هنوز بسته اعلام نمی‌شوند.
API، تغییر داده و تغییر چک‌لیست: صفر.

</div>

In [15]:
# NB22-C15 — Action 09: stream byte hashes for six explicit assets
# Scope: immutable data reads; write/reload one verification report.

import time


def nb22_save_json_artifact(root, relative_directory, stem, value):
    raw = json.dumps(
        value, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    digest = hashlib.sha256(raw).hexdigest()
    directory = root / relative_directory
    directory.mkdir(parents=True, exist_ok=True)
    target = directory / f"{stem}.{digest}.json"

    if target.exists():
        if target.read_bytes() != raw:
            raise RuntimeError("EXISTING_OUTPUT_CONFLICT")
        write_status = "REUSED_IDENTICAL"
    else:
        fd, name = tempfile.mkstemp(prefix=f".{stem}.", dir=directory)
        temporary = Path(name)
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(raw)
                stream.flush()
                os.fsync(stream.fileno())
            if temporary.read_bytes() != raw:
                raise RuntimeError("TEMPORARY_WRITE_VALIDATION_FAILED")
            try:
                os.link(temporary, target)
                write_status = "CREATED"
            except FileExistsError:
                if target.read_bytes() != raw:
                    raise RuntimeError("CONCURRENT_OUTPUT_CONFLICT")
                write_status = "REUSED_IDENTICAL"
        finally:
            temporary.unlink(missing_ok=True)

    saved = target.read_bytes()
    if (
        hashlib.sha256(saved).hexdigest() != digest
        or json.loads(saved) != value
    ):
        raise RuntimeError("INDEPENDENT_RELOAD_FAILED")
    return {
        "path": target.relative_to(root).as_posix(),
        "file_bytes_sha256": digest,
        "bytes": len(saved),
        "write_status": write_status,
        "reload_validation": "PASS",
    }


def nb22_c15_run(previous):
    registry = previous["artifact"]
    if registry["conflicting_roles"]:
        raise RuntimeError("C14_REFERENCE_CONFLICTS_REQUIRE_RECONCILIATION")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    required = {
        "active_chunks", "context_evidence_registry",
        "context_packets", "document_corpus",
        "golden_dataset", "reranked_candidates",
    }
    roles = registry["roles"]
    if (
        len(roles) != 6
        or {row["role"] for row in roles} != required
    ):
        raise RuntimeError("EXPECTED_SIX_EXPLICIT_ASSETS")

    work = []
    for row in roles:
        relative = row["reference_path"]
        p = Path(relative)
        if p.is_absolute() or ".." in p.parts:
            raise RuntimeError("INVALID_PROJECT_RELATIVE_PATH")
        target = (root / p).resolve()
        target.relative_to(root)
        if not target.is_file():
            raise RuntimeError(
                f"EXPLICIT_INPUT_NOT_AVAILABLE: {relative}"
            )
        work.append((row, target))

    total_bytes = sum(path.stat().st_size for _, path in work)
    print(json.dumps({
        "stage": "BYTE_HASHING_START",
        "file_count": len(work),
        "total_bytes": total_bytes,
        "total_mib": round(total_bytes / 1024**2, 2),
        "model_api_calls": 0,
    }, ensure_ascii=False), flush=True)

    def signature(stat):
        return (
            stat.st_dev, stat.st_ino, stat.st_size,
            stat.st_mtime_ns, stat.st_ctime_ns,
        )

    results = []
    for number, (row, path) in enumerate(work, 1):
        started = time.monotonic()
        result = {
            "role": row["role"],
            "path": row["reference_path"],
            "declared_sha256": row["declared_sha256"],
            "hash_computed_as": "SHA256_FILE_BYTES",
            "producer_hash_definition_revalidated": False,
        }
        try:
            with path.open("rb") as stream:
                before = os.fstat(stream.fileno())
                hasher = hashlib.sha256()
                read_bytes = 0
                for block in iter(
                    lambda: stream.read(4 * 1024 * 1024), b""
                ):
                    hasher.update(block)
                    read_bytes += len(block)
                after = os.fstat(stream.fileno())

            current = path.stat()
            stable = (
                signature(before) == signature(after)
                == signature(current)
                and read_bytes == before.st_size
            )
            actual = hasher.hexdigest()
            result.update({
                "bytes_read": read_bytes,
                "actual_file_bytes_sha256": actual,
                "file_stable_during_read": stable,
                "matches_declared_sha256":
                    actual == row["declared_sha256"],
                "status": (
                    "INPUT_CHANGED_DURING_READ" if not stable else
                    "FILE_BYTES_MATCH_DECLARED_VALUE"
                    if actual == row["declared_sha256"] else
                    "HASH_DEFINITION_OR_VERSION_RECONCILIATION_REQUIRED"
                ),
            })
        except Exception as exc:
            result.update({
                "status": "LOCAL_READ_FAILED",
                "error_type": type(exc).__name__,
            })
        result["elapsed_seconds"] = round(
            time.monotonic() - started, 3
        )
        results.append(result)
        print(json.dumps({
            "stage": "FILE_COMPLETED",
            "number": number,
            "of": len(work),
            "role": result["role"],
            "status": result["status"],
            "elapsed_seconds": result["elapsed_seconds"],
        }, ensure_ascii=False), flush=True)

    all_match = all(
        r["status"] == "FILE_BYTES_MATCH_DECLARED_VALUE"
        for r in results
    )
    artifact = {
        "artifact_type": "NB22_DATA_BYTE_HASH_VERIFICATION",
        "schema_version": "NB22_C015_ACTION09_BYTE_HASHES_V1",
        "status": (
            "SIX_DATA_BYTE_HASHES_MATCH_DECLARED_VALUES"
            if all_match else "BYTE_HASH_RECONCILIATION_REQUIRED"
        ),
        "scope": "SIX_EXPLICIT_C14_ASSETS_ONLY",
        "results": results,
        "total_bytes_planned": total_bytes,
        "upstream_reference_evidence": registry["roles"],
        "pending_action08_bindings": registry["pending_bindings"],
        "logical_index_hash_validated": False,
        "split_gold_logical_hash_validated": False,
        "all_release_assets_verified": False,
        "action09_fully_complete": False,
        "semantic_quality_assessed": False,
        "release_acceptance_granted": False,
        "input_files_modified": False,
        "checklist_modified": False,
        "new_api_calls": 0,
    }

    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C015.action09",
        "C015.byte_hashes",
        artifact,
    )
    summary = {
        "status": artifact["status"],
        "files_checked": len(results),
        "matching_files": sum(
            r["status"] == "FILE_BYTES_MATCH_DECLARED_VALUE"
            for r in results
        ),
        "results": results,
        "output": output,
        "action09_fully_complete": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "summary": summary}


nb22_c15_result = nb22_c15_run(nb22_c14_result)

{"stage": "BYTE_HASHING_START", "file_count": 6, "total_bytes": 257927401, "total_mib": 245.98, "model_api_calls": 0}
{"stage": "FILE_COMPLETED", "number": 1, "of": 6, "role": "active_chunks", "status": "FILE_BYTES_MATCH_DECLARED_VALUE", "elapsed_seconds": 0.125}
{"stage": "FILE_COMPLETED", "number": 2, "of": 6, "role": "context_evidence_registry", "status": "FILE_BYTES_MATCH_DECLARED_VALUE", "elapsed_seconds": 0.227}
{"stage": "FILE_COMPLETED", "number": 3, "of": 6, "role": "context_packets", "status": "FILE_BYTES_MATCH_DECLARED_VALUE", "elapsed_seconds": 0.056}
{"stage": "FILE_COMPLETED", "number": 4, "of": 6, "role": "document_corpus", "status": "FILE_BYTES_MATCH_DECLARED_VALUE", "elapsed_seconds": 0.188}
{"stage": "FILE_COMPLETED", "number": 5, "of": 6, "role": "golden_dataset", "status": "FILE_BYTES_MATCH_DECLARED_VALUE", "elapsed_seconds": 0.036}
{"stage": "FILE_COMPLETED", "number": 6, "of": 6, "role": "reranked_candidates", "status": "FILE_BYTES_MATCH_DECLARED_VALUE", "elapsed_

<div dir="rtl" align="right">

### NB22-C16 — اقدام ۹، بستهٔ دوم: Config و Manifest مؤلفه‌ها

سه زیرکار این بسته:

۱. تطبیق هش Configهای خوانده‌شده با ارجاع Release مصرف‌کننده.
۲. خواندن و کنترل Manifestهای مشخص و ارجاع‌شدهٔ مؤلفه‌ها.
۳. ثبت هش‌های منطقی Index به‌عنوان مقدار اعلام‌شده،
بدون جا زدن آن‌ها به‌عنوان هش بایت فایل.

شش فایل تأییدشدهٔ C15 دوباره خوانده نمی‌شوند.
Configهای قبلی از محتوای کنترل‌شدهٔ همین نوت‌بوک مصرف می‌شوند.
فقط Manifestهای JSON صریحاً ارجاع‌شده، در صورت نیاز، تازه خوانده می‌شوند.

اختلاف هش ابتدا اختلاف نسخه یا تعریف هش تلقی می‌شود؛
هیچ فایل برای عبور از کنترل بازسازی یا اصلاح نمی‌شود.
کنترل بایت Manifest، بازتولید هش منطقی Index نیست.

خروجی Atomic ذخیره و مستقلاً بازخوانی می‌شود.
API، بازکردن Qdrant، بازسازی Index و تغییر چک‌لیست: صفر.

</div>

In [16]:
# NB22-C16 — Action 09: configuration bindings and component manifests
# Scope: cached metadata + explicitly referenced JSON manifests only.

def nb22_c16_run(c13, c15):
    if c15["summary"]["status"] != (
        "SIX_DATA_BYTE_HASHES_MATCH_DECLARED_VALUES"
    ):
        raise RuntimeError("C15_DATA_BYTE_CHECKS_REQUIRE_RECONCILIATION")

    payloads = dict(c13["payloads"])
    sources = dict(c13["source_bindings"])
    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    releases = {
        "dense": "artifacts/phase2/dense_index/dense_index_release_v1.json",
        "sparse": "artifacts/phase2/sparse_index/sparse_index_release_v1.json",
        "hybrid":
            "artifacts/phase2/hybrid_retrieval/hybrid_retrieval_release_v1.json",
        "reranker":
            "artifacts/phase2/reranking/reranker_release_v1.json",
        "context":
            "artifacts/phase2/context_builder/context_builder_release_v1.json",
    }
    configs = {
        "hybrid":
            "artifacts/phase2/hybrid_retrieval/hybrid_configuration_v1.json",
        "reranker":
            "artifacts/phase2/reranking/reranker_configuration_v1.json",
        "context":
            "artifacts/phase2/context_builder/context_builder_configuration_v1.json",
    }

    def origin(path):
        return {
            "path": path,
            "file_bytes_sha256": sources[path]["file_bytes_sha256"],
        }

    # Find only explicit manifest objects within these release documents.
    def manifest_refs(node, trail=()):
        found = []
        if not isinstance(node, dict):
            return found
        if (
            trail and "manifest" in trail[-1].lower()
            and isinstance(node.get("path"), str)
            and isinstance(node.get("sha256"), str)
        ):
            found.append({
                "field_path": list(trail),
                "path": node["path"],
                "declared_sha256": node["sha256"],
            })
            return found
        if len(trail) < 3:
            for key, value in node.items():
                if isinstance(value, dict):
                    found.extend(
                        manifest_refs(value, trail + (str(key),))
                    )
        return found

    requests, unresolved = [], []
    logical_declarations = []
    for component, source in releases.items():
        release = payloads[source]
        refs = manifest_refs(release)
        if not refs:
            unresolved.append({
                "component": component,
                "source": origin(source),
                "reason": "EXPLICIT_MANIFEST_BINDING_NOT_RESOLVED",
                "top_level_keys": list(release),
            })
        for ref in refs:
            requests.append({
                **ref,
                "component": component,
                "binding_source": origin(source),
            })
        for key in (
            "logical_index_sha256", "storage_inventory_sha256"
        ):
            if key in release:
                logical_declarations.append({
                    "component": component,
                    "source": origin(source),
                    "field": key,
                    "declared_value": release[key],
                    "recomputed": False,
                    "file_bytes_hash_inferred": False,
                })

    config_checks = []
    for component, config_path in configs.items():
        release_path = releases[component]
        release = payloads[release_path]
        declarations = []

        ref = release.get("configuration")
        if isinstance(ref, dict) and ref.get("sha256"):
            declarations.append({
                "field": "configuration.sha256",
                "path": ref.get("path"),
                "digest": ref["sha256"],
            })
        if isinstance(release.get("configuration_sha256"), str):
            declarations.append({
                "field": "configuration_sha256",
                "path": config_path,
                "digest": release["configuration_sha256"],
            })
        # Context releases may carry their artifacts in a hash map.
        for container_name in ("artifact_sha256", "artifacts"):
            container = release.get(container_name)
            if isinstance(container, dict):
                for name, item in container.items():
                    if (
                        isinstance(item, dict)
                        and item.get("path") == config_path
                        and isinstance(item.get("sha256"), str)
                    ):
                        declarations.append({
                            "field": f"{container_name}.{name}.sha256",
                            "path": item["path"],
                            "digest": item["sha256"],
                        })

        actual = sources[config_path]["file_bytes_sha256"]
        config_checks.append({
            "component": component,
            "source_release": origin(release_path),
            "configuration": origin(config_path),
            "declarations": declarations,
            "status": (
                "EXPLICIT_CONFIG_HASH_BINDING_NOT_RESOLVED"
                if not declarations else
                "MATCH" if all(
                    item["digest"] == actual
                    and item["path"] in (None, config_path)
                    for item in declarations
                ) else "CONFIG_BINDING_RECONCILIATION_REQUIRED"
            ),
            "config_file_reread": False,
        })

    manifest_checks = []
    for request in requests:
        row = dict(request)
        relative = request["path"]
        try:
            p = Path(relative)
            if p.is_absolute() or ".." in p.parts:
                raise ValueError("INVALID_REPOSITORY_RELATIVE_PATH")
            target = (root / p).resolve()
            target.relative_to(root)
            expected = request["declared_sha256"]
            if not re.fullmatch(r"[0-9a-f]{64}", expected):
                raise ValueError("INVALID_DECLARED_SHA256")

            if relative in payloads:
                actual = sources[relative]["file_bytes_sha256"]
                value = payloads[relative]
                row["read_mode"] = "REUSED_VERIFIED_METADATA"
            else:
                if not target.is_file():
                    raise FileNotFoundError()
                if target.stat().st_size > 8 * 1024 * 1024:
                    raise ValueError("METADATA_SIZE_REQUIRES_REVIEW")
                raw = target.read_bytes()
                actual = hashlib.sha256(raw).hexdigest()
                row["actual_file_bytes_sha256"] = actual
                if actual != expected:
                    raise ValueError("DECLARED_HASH_DIFFERS_FROM_FILE_BYTES")
                value = json.loads(raw.decode("utf-8"))
                if not isinstance(value, dict):
                    raise ValueError("MANIFEST_ROOT_NOT_OBJECT")
                payloads[relative] = value
                sources[relative] = {
                    "path": relative,
                    "file_bytes_sha256": actual,
                    "bytes": len(raw),
                }
                row["read_mode"] = "FRESH_EXPLICIT_MANIFEST_READ"

            if actual != expected:
                raise ValueError("CACHED_MANIFEST_HASH_DIFFERS")
            row.update({
                "status": "MANIFEST_BYTES_MATCH_DECLARED_VALUE",
                "actual_file_bytes_sha256": actual,
                "top_level_keys": list(value),
                "hash_definition_reconstructed": False,
            })
        except Exception as exc:
            row.update({
                "status": "MANIFEST_RECONCILIATION_REQUIRED",
                "error_type": type(exc).__name__,
                "reason": (
                    str(exc) if isinstance(exc, ValueError)
                    else "EXPLICIT_MANIFEST_NOT_AVAILABLE"
                    if isinstance(exc, FileNotFoundError)
                    else "LOCAL_METADATA_READ_FAILED"
                ),
            })
        manifest_checks.append(row)

    # Show metadata contracts and pointers without expanding populations.
    def metadata_view(node, depth=0):
        if isinstance(node, dict):
            if depth >= 2:
                return {"keys": list(node)}
            return {
                key: metadata_view(value, depth + 1)
                for key, value in node.items()
                if key.lower() not in {
                    "messages", "secret", "api_key",
                    "authorization", "password",
                }
            }
        if isinstance(node, list):
            return {"list_count": len(node)}
        if isinstance(node, str):
            if len(node) > 400:
                return {"text_length": len(node)}
            node = node.replace(str(root), "<PROJECT_ROOT>")
            return re.sub(
                r"/(?:Users|home)/[^/\s]+", "<USER_HOME>", node
            )
        return node

    views = {
        row["path"]: metadata_view(payloads[row["path"]])
        for row in manifest_checks
        if row["status"] == "MANIFEST_BYTES_MATCH_DECLARED_VALUE"
    }

    artifact = {
        "artifact_type": "NB22_COMPONENT_MANIFEST_HASH_CHECKS",
        "schema_version": "NB22_C016_ACTION09_MANIFESTS_V1",
        "status": "COMPONENT_MANIFEST_AND_CONFIG_CHECKS_RECORDED",
        "configuration_checks": config_checks,
        "manifest_checks": manifest_checks,
        "unresolved_manifest_bindings": unresolved,
        "logical_hash_declarations": logical_declarations,
        "manifest_metadata_views": views,
        "prior_six_data_hash_checks_preserved":
            c15["artifact"]["results"],
        "qdrant_opened": False,
        "index_rebuilt": False,
        "logical_index_hashes_recomputed": False,
        "action09_fully_complete": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C016.action09",
        "C016.component_hashes", artifact,
    )
    print(json.dumps({
        "status": artifact["status"],
        "configuration_checks": config_checks,
        "manifest_checks": manifest_checks,
        "unresolved_manifest_bindings": unresolved,
        "logical_hash_declarations": logical_declarations,
        "manifest_metadata_views": views,
        "output": output,
        "prior_six_data_files_reread": 0,
        "logical_index_hashes_recomputed": False,
        "action09_fully_complete": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {
        "artifact": artifact,
        "payloads": payloads,
        "source_bindings": sources,
    }


nb22_c16_result = nb22_c16_run(nb22_c13_result, nb22_c15_result)

{
  "status": "COMPONENT_MANIFEST_AND_CONFIG_CHECKS_RECORDED",
  "configuration_checks": [
    {
      "component": "hybrid",
      "source_release": {
        "path": "artifacts/phase2/hybrid_retrieval/hybrid_retrieval_release_v1.json",
        "file_bytes_sha256": "8953c5d4d20a0f0cb2cf9fb35d86c421c06cf02b193aa2b19acba46640d0216e"
      },
      "configuration": {
        "path": "artifacts/phase2/hybrid_retrieval/hybrid_configuration_v1.json",
        "file_bytes_sha256": "e7dc28836b0e6b2f72cf19aa61507af41a76df4980b056f39af015ed91ba26d5"
      },
      "declarations": [
        {
          "field": "configuration.sha256",
          "path": "artifacts/phase2/hybrid_retrieval/hybrid_configuration_v1.json",
          "digest": "e7dc28836b0e6b2f72cf19aa61507af41a76df4980b056f39af015ed91ba26d5"
        }
      ],
      "status": "MATCH",
      "config_file_reread": false
    },
    {
      "component": "reranker",
      "source_release": {
        "path": "artifacts/phase2/reranking/reran

<div dir="rtl" align="right">

### NB22-C17 — اقدام ۹: رفع ابهام تعریف هش Config ریرنکر

سه زیرکار این بسته:

۱. تطبیق مقدار configuration_sha256 بین Config، Release و Manifest.
۲. محاسبهٔ تشخیصی هش JSON کانونی با قرارداد شناخته‌شدهٔ پروژه.
۳. خواندن محدود کد ذخیره‌شدهٔ NB17 برای یافتن تعریف واقعی هش.

محاسبهٔ تشخیصی، قرارداد تولیدکننده را تعیین یا جایگزین نمی‌کند.
حذف فیلد خودهش فقط یک فرضیهٔ مشخص برای بررسی است؛
تطبیق عددی به‌تنهایی پذیرش این فرضیه نیست.

Config از ورودی کنترل‌شدهٔ C16 مصرف می‌شود.
تنها متن نوت‌بوک تولیدکننده خوانده می‌شود؛ هیچ سل آن اجرا نمی‌شود.
فایل‌های داده، مدل، Index و پاسخ‌ها دوباره خوانده نمی‌شوند.

نوشتن فایل، تغییر ورودی، API و تغییر چک‌لیست: صفر.

</div>

In [17]:
# NB22-C17 — Reconcile reranker configuration hash semantics
# Scope: cached configuration + read-only producer code inspection.

import ast


def nb22_c17_run(previous):
    payloads = previous["payloads"]
    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    config_path = (
        "artifacts/phase2/reranking/reranker_configuration_v1.json"
    )
    release_path = (
        "artifacts/phase2/reranking/reranker_release_v1.json"
    )
    manifest_path = (
        "data/phase2/retrieval/reranker_v1/reranker_manifest.json"
    )
    config = payloads[config_path]
    release = payloads[release_path]
    manifest = payloads[manifest_path]

    declarations = {
        "configuration_internal_value":
            config.get("configuration_sha256"),
        "release_declared_value":
            release.get("configuration_sha256"),
        "manifest_declared_value":
            manifest.get("configuration_sha256"),
    }
    declaration_values = list(declarations.values())
    if not all(
        isinstance(value, str)
        and re.fullmatch(r"[0-9a-f]{64}", value)
        for value in declaration_values
    ):
        raise RuntimeError("CONFIGURATION_HASH_DECLARATION_MISSING")

    def canonical_digest(value):
        return hashlib.sha256(json.dumps(
            value, ensure_ascii=False, sort_keys=True,
            separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")).hexdigest()

    whole_payload = canonical_digest(config)
    without_self = canonical_digest({
        key: value for key, value in config.items()
        if key != "configuration_sha256"
    })
    expected = declarations["release_declared_value"]

    notebook_relative = "notebooks/phase2/17_reranking.ipynb"
    notebook_path = root / notebook_relative
    producer_evidence = {
        "path": notebook_relative,
        "executed": False,
        "status": "NOT_AVAILABLE_AT_EXPLICIT_PATH",
    }

    if notebook_path.is_file():
        raw = notebook_path.read_bytes()
        notebook = json.loads(raw.decode("utf-8"))
        cells = notebook.get("cells")
        if not isinstance(cells, list):
            raise RuntimeError("PRODUCER_NOTEBOOK_CELLS_INVALID")

        formula_excerpts = []
        hash_helpers = []
        parse_errors = []

        for index, cell in enumerate(cells):
            if cell.get("cell_type") != "code":
                continue
            source = cell.get("source", "")
            if isinstance(source, list):
                source = "".join(source)
            if not isinstance(source, str):
                continue

            lines = source.splitlines()
            # Keep source lines near the exact target field.
            positions = [
                i for i, line in enumerate(lines)
                if "configuration_sha256" in line
            ]
            intervals = []
            for i in positions:
                start, end = max(0, i - 7), min(len(lines), i + 8)
                if intervals and start <= intervals[-1][1]:
                    intervals[-1] = (
                        intervals[-1][0], max(intervals[-1][1], end)
                    )
                else:
                    intervals.append((start, end))
            for start, end in intervals:
                formula_excerpts.append({
                    "physical_cell_index": index,
                    "cell_title": lines[0] if lines else "",
                    "source_lines_1based": [start + 1, end],
                    "source_excerpt": "\n".join(lines[start:end]),
                })

            # Read hash/serialization helper definitions, never execute them.
            if "sha256" not in source or "json" not in source:
                continue
            try:
                tree = ast.parse(source)
            except SyntaxError:
                parse_errors.append(index)
                continue
            for node in ast.walk(tree):
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    segment = ast.get_source_segment(source, node) or ""
                    if "sha256" in segment and "json" in segment:
                        hash_helpers.append({
                            "physical_cell_index": index,
                            "function_name": node.name,
                            "source": segment,
                        })

        # Explicitly report display limits; omitted code is not reviewed.
        excerpt_limit = 16000
        helper_limit = 10000

        def bounded(items, text_key, limit):
            selected, used = [], 0
            for item in items:
                length = len(item[text_key])
                if used + length > limit:
                    continue
                selected.append(item)
                used += length
            return selected, len(items) - len(selected)

        shown_excerpts, omitted_excerpts = bounded(
            formula_excerpts, "source_excerpt", excerpt_limit
        )
        shown_helpers, omitted_helpers = bounded(
            hash_helpers, "source", helper_limit
        )
        producer_evidence = {
            "path": notebook_relative,
            "file_bytes_sha256": hashlib.sha256(raw).hexdigest(),
            "executed": False,
            "status": "TARGETED_PRODUCER_SOURCE_READ",
            "configuration_hash_excerpts": shown_excerpts,
            "hash_helper_definitions": shown_helpers,
            "display_omitted_excerpt_count": omitted_excerpts,
            "display_omitted_helper_count": omitted_helpers,
            "cells_with_unparsed_python": parse_errors,
            "saved_source_is_not_new_runtime_execution": True,
        }

    result = {
        "status": "RERANKER_HASH_DEFINITION_EVIDENCE_ASSEMBLED",
        "declared_values": declarations,
        "all_three_declared_values_agree":
            len(set(declaration_values)) == 1,
        "previously_verified_file_bytes_sha256":
            previous["source_bindings"][config_path][
                "file_bytes_sha256"
            ],
        "diagnostic_canonical_hashes": {
            "entire_JSON_object": whole_payload,
            "entire_JSON_matches_declared": whole_payload == expected,
            "object_without_configuration_sha256": without_self,
            "without_self_field_matches_declared":
                without_self == expected,
            "diagnostic_match_alone_establishes_contract": False,
        },
        "producer_source": producer_evidence,
        "hash_definition_acceptance_pending_source_review": True,
        "config_modified": False,
        "files_written": 0,
        "model_executed": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result


nb22_c17_result = nb22_c17_run(nb22_c16_result)

{
  "status": "RERANKER_HASH_DEFINITION_EVIDENCE_ASSEMBLED",
  "declared_values": {
    "configuration_internal_value": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738",
    "release_declared_value": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738",
    "manifest_declared_value": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738"
  },
  "all_three_declared_values_agree": true,
  "previously_verified_file_bytes_sha256": "dd2290188c4e99be1dada40b167b1041c69f7f077ebe6dea1fddcb58b4834a86",
  "diagnostic_canonical_hashes": {
    "entire_JSON_object": "3481015ca6ebcdb8e85ca373d1eab1d15a67e00e03df624817cf07aadf28a5ba",
    "entire_JSON_matches_declared": false,
    "object_without_configuration_sha256": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738",
    "without_self_field_matches_declared": true,
    "diagnostic_match_alone_establishes_contract": false
  },
  "producer_source": {
    "path": "notebooks/phase2/17_reranki

<div dir="rtl" align="right">

### NB22-C18 — اقدام ۹: ثبت تعریف صحیح هش Config ریرنکر

سه زیرکار این بسته:

۱. خواندن تعریف دقیق تابع canonical_json_bytes در NB17.
۲. تطبیق روش سریال‌سازی و هش Payload بدون فیلد خودهش.
۳. ثبت نتیجهٔ اصلاح‌شده با حفظ هش بایت فایل و گزارش قبلی.

طبق کد تولیدکننده، configuration_sha256 پیش از افزودن همین فیلد
محاسبه می‌شود؛ پس با SHA256 بایت فایل نهایی یک مفهوم نیست.

تنها تعریف تابع سریال‌سازی از نوت‌بوک خوانده می‌شود؛
هیچ کد منبع اجرا یا import نمی‌شود.
Config، داده‌ها و مدل دوباره خوانده یا اجرا نمی‌شوند.

اگر تعریف تابع قابل احراز نباشد، نتیجه صریحاً باز می‌ماند.
API، تغییر Config و تغییر چک‌لیست: صفر.

</div>

In [19]:
# NB22-C18 — Reconcile reranker payload hash with producer serialization
# Repair: represent literal keyword tuples as JSON arrays before persistence.
# Scope: static source inspection; no execution of producer code.

def nb22_c18_run(c16, c17):
    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    relative = c17["producer_source"]["path"]
    raw = (root / relative).read_bytes()
    notebook_sha = hashlib.sha256(raw).hexdigest()
    if notebook_sha != c17["producer_source"]["file_bytes_sha256"]:
        raise RuntimeError("PRODUCER_SOURCE_CHANGED_SINCE_C17")
    notebook = json.loads(raw.decode("utf-8"))

    definitions = []
    for index, cell in enumerate(notebook["cells"]):
        if cell.get("cell_type") != "code":
            continue
        text = cell.get("source", "")
        if isinstance(text, list):
            text = "".join(text)
        if "def canonical_json_bytes" not in text:
            continue
        tree = ast.parse(text)
        for node in tree.body:
            if (
                isinstance(node, ast.FunctionDef)
                and node.name == "canonical_json_bytes"
            ):
                definitions.append((index, text, node))

    function_evidence = []
    for index, text, node in definitions:
        dumps_calls = [
            child for child in ast.walk(node)
            if isinstance(child, ast.Call)
            and isinstance(child.func, ast.Attribute)
            and child.func.attr == "dumps"
            and isinstance(child.func.value, ast.Name)
            and child.func.value.id == "json"
        ]
        keywords = {}
        if len(dumps_calls) == 1:
            for keyword in dumps_calls[0].keywords:
                try:
                    keywords[keyword.arg] = ast.literal_eval(keyword.value)
                except (ValueError, TypeError):
                    keywords[keyword.arg] = "NON_LITERAL_VALUE"

        utf8_encode_present = any(
            isinstance(child, ast.Call)
            and isinstance(child.func, ast.Attribute)
            and child.func.attr == "encode"
            and child.args
            and isinstance(child.args[0], ast.Constant)
            and child.args[0].value in ("utf-8", "utf8")
            for child in ast.walk(node)
        )
        settings_match = (
            len(dumps_calls) == 1
            and keywords.get("ensure_ascii") is False
            and keywords.get("sort_keys") is True
            and keywords.get("separators") == (",", ":")
            and utf8_encode_present
        )

        # The source literal remains checked as a tuple above.
        # Its stored evidence is explicitly converted to JSON-native types.
        json_keywords = json.loads(json.dumps(
            keywords, ensure_ascii=False, allow_nan=False,
        ))
        function_evidence.append({
            "physical_cell_index": index,
            "function_source": ast.get_source_segment(text, node),
            "json_dumps_call_count": len(dumps_calls),
            "literal_json_keywords": json_keywords,
            "utf8_encode_present": utf8_encode_present,
            "serialization_settings_match": settings_match,
        })

    serialization_confirmed = (
        len(function_evidence) == 1
        and function_evidence[0]["serialization_settings_match"]
    )

    config_path = (
        "artifacts/phase2/reranking/reranker_configuration_v1.json"
    )
    config = c16["payloads"][config_path]
    payload = {
        key: value for key, value in config.items()
        if key != "configuration_sha256"
    }
    recomputed = hashlib.sha256(json.dumps(
        payload, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")).hexdigest()

    declarations = c17["declared_values"]
    declared_match = (
        len(set(declarations.values())) == 1
        and recomputed == declarations["release_declared_value"]
    )
    formula_evidence = [
        item for item in c17["producer_source"][
            "configuration_hash_excerpts"
        ]
        if (
            "reranker_configuration_payload" in item["source_excerpt"]
            and "canonical_json_bytes" in item["source_excerpt"]
            and "**reranker_configuration_payload" in item["source_excerpt"]
        )
    ]
    confirmed = (
        serialization_confirmed
        and declared_match
        and bool(formula_evidence)
    )

    result = {
        "artifact_type": "NB22_RERANKER_HASH_RECONCILIATION",
        "schema_version": "NB22_C018_ACTION09_HASH_DEFINITION_V1",
        "status": (
            "RERANKER_CONFIG_PAYLOAD_HASH_VERIFIED"
            if confirmed else
            "PRODUCER_SERIALIZATION_REVIEW_REQUIRED"
        ),
        "source_notebook": {
            "path": relative,
            "file_bytes_sha256": notebook_sha,
            "executed": False,
        },
        "serialization_definition": function_evidence,
        "payload_construction_evidence": formula_evidence,
        "configuration": {
            "path": config_path,
            "file_bytes_sha256":
                c17["previously_verified_file_bytes_sha256"],
            "hash_definition":
                "SHA256_UTF8_CANONICAL_JSON_EXCLUDING_TOP_LEVEL_CONFIGURATION_SHA256",
            "computed_payload_sha256": recomputed,
            "declared_values": declarations,
            "declarations_match_computation": declared_match,
        },
        "C16_reconciliation": {
            "historical_report_modified": False,
            "original_comparison":
                "PAYLOAD_HASH_COMPARED_WITH_FILE_BYTES_HASH",
            "corrected_result": (
                "MATCH_IN_PRODUCER_DEFINED_PAYLOAD_SCOPE"
                if confirmed else "PENDING"
            ),
        },
        "other_C16_checks_preserved": {
            "five_manifest_byte_checks":
                c16["artifact"]["manifest_checks"],
            "logical_index_hashes_recomputed": False,
        },
        "configuration_modified": False,
        "action09_fully_complete": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }

    # Verify JSON round-trip compatibility before publishing the report.
    preview = json.dumps(
        result, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    )
    if json.loads(preview) != result:
        raise RuntimeError("OUTPUT_NOT_JSON_NATIVE_BEFORE_WRITE")

    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C018.action09",
        "C018.reranker_hash_reconciliation", result,
    )
    print(json.dumps({
        "status": result["status"],
        "serialization_definition": function_evidence,
        "configuration": result["configuration"],
        "C16_reconciliation": result["C16_reconciliation"],
        "output": output,
        "configuration_modified": False,
        "action09_fully_complete": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": result, "output": output}


nb22_c18_result = nb22_c18_run(nb22_c16_result, nb22_c17_result)

{
  "status": "RERANKER_CONFIG_PAYLOAD_HASH_VERIFIED",
  "serialization_definition": [
    {
      "physical_cell_index": 5,
      "function_source": "def canonical_json_bytes(value):\n    return json.dumps(\n        value,\n        ensure_ascii=False,\n        sort_keys=True,\n        separators=(\",\", \":\"),\n    ).encode(\"utf-8\")",
      "json_dumps_call_count": 1,
      "literal_json_keywords": {
        "ensure_ascii": false,
        "sort_keys": true,
        "separators": [
          ",",
          ":"
        ]
      },
      "utf8_encode_present": true,
      "serialization_settings_match": true
    }
  ],
  "configuration": {
    "path": "artifacts/phase2/reranking/reranker_configuration_v1.json",
    "file_bytes_sha256": "dd2290188c4e99be1dada40b167b1041c69f7f077ebe6dea1fddcb58b4834a86",
    "hash_definition": "SHA256_UTF8_CANONICAL_JSON_EXCLUDING_TOP_LEVEL_CONFIGURATION_SHA256",
    "computed_payload_sha256": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badf

<div dir="rtl" align="right">

### NB22-C19 — اقدام ۹: کنترل فایل‌های فیزیکی Index

چهار زیرکار این بسته:

۱. استخراج فهرست فایل‌های Dense و Sparse از Manifestهای موجود.
۲. حل مسیرهای صریح؛ مسیر نسبی مبهم انتخاب نمی‌شود.
۳. هش‌گیری جریانی و مقایسه با مقدار مستند.
۴. ذخیرهٔ گزارش و بازخوانی مستقل.

ورودی Metadata از C16 مصرف می‌شود؛ مخزن دوباره اسکن نمی‌شود.
فایل‌های تأییدشدهٔ قبلی، اگر در این فهرست تکرار شوند،
با ارجاع به نتیجهٔ معتبر قبلی مصرف می‌شوند.

Qdrant و SQLite از طریق کتابخانهٔ پایگاه‌داده باز نمی‌شوند.
فایل تغییرکرده حین خواندن، PASS نمی‌گیرد.
اختلاف مسیر یا هش باعث بازسازی یا تغییر Index نمی‌شود.

تطبیق فایل‌های فیزیکی، بازتولید هش منطقی Index نیست.
API، اجرای مدل، تغییر Index و تغییر چک‌لیست: صفر.

</div>

In [20]:
# NB22-C19 — Action 09: physical index file byte verification
# Scope: explicit manifest entries; no DB client, discovery or mutation.

def nb22_c19_run(c16, c15, c18):
    if c18["artifact"]["status"] != "RERANKER_CONFIG_PAYLOAD_HASH_VERIFIED":
        raise RuntimeError("RERANKER_HASH_RECONCILIATION_NOT_COMPLETE")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()
    payloads = c16["payloads"]

    dense_manifest_path = (
        "data/phase2/index/dense_index_v1/dense_index_manifest.json"
    )
    sparse_manifest_path = (
        "data/phase2/index/sparse_index_v1/sparse_index_manifest.json"
    )
    dense = payloads[dense_manifest_path]
    sparse = payloads[sparse_manifest_path]

    requests, unresolved = [], []

    def add_entry(component, role, entry, source, base):
        if not isinstance(entry, dict):
            unresolved.append({
                "component": component, "role": role,
                "reason": "FILE_ENTRY_NOT_OBJECT",
            })
            return

        path_values = {
            entry[k] for k in ("path", "relative_path")
            if isinstance(entry.get(k), str) and entry[k]
        }
        hash_values = {
            entry[k] for k in ("sha256", "file_sha256", "file_bytes_sha256")
            if isinstance(entry.get(k), str) and entry[k]
        }
        if len(path_values) != 1 or len(hash_values) != 1:
            unresolved.append({
                "component": component, "role": role,
                "reason": "EXPLICIT_PATH_OR_HASH_NOT_UNIQUE",
                "entry_keys": list(entry),
            })
            return

        declared_path = next(iter(path_values))
        digest = next(iter(hash_values))
        if not re.fullmatch(r"[0-9a-f]{64}", digest):
            unresolved.append({
                "component": component, "role": role,
                "reason": "INVALID_DECLARED_SHA256",
            })
            return

        p = Path(declared_path)
        if p.is_absolute() or ".." in p.parts:
            unresolved.append({
                "component": component, "role": role,
                "reason": "PATH_REQUIRES_EXPLICIT_ADAPTER",
            })
            return

        # Explicit project paths need no alternative resolution.
        if p.parts and p.parts[0] in ("data", "artifacts", "reports"):
            options = [root / p]
        else:
            # A short path is not silently assigned a base.
            options = [root / p, base / p]
        unique = {}
        for option in options:
            resolved = option.resolve()
            resolved.relative_to(root)
            if resolved.is_file():
                unique[resolved] = True
        if len(unique) != 1:
            unresolved.append({
                "component": component, "role": role,
                "declared_path": declared_path,
                "reason": (
                    "PATH_NOT_AVAILABLE_IN_DECLARED_BASES"
                    if not unique else "AMBIGUOUS_RELATIVE_PATH"
                ),
            })
            return

        target = next(iter(unique))
        requests.append({
            "component": component,
            "role": role,
            "path": target.relative_to(root).as_posix(),
            "declared_path": declared_path,
            "declared_sha256": digest,
            "source_manifest": source,
            "path_resolution": (
                "PROJECT_RELATIVE" if p.parts[0] in
                ("data", "artifacts", "reports") else
                "UNIQUE_EXISTING_LOCATION_IN_EXPLICIT_BASES"
            ),
        })

    dense_files = dense["storage_inventory"]["files"]
    dense_base = root / dense["backend"]["storage_path"]
    if isinstance(dense_files, list):
        dense_entries = list(enumerate(dense_files))
    elif isinstance(dense_files, dict):
        dense_entries = list(dense_files.items())
    else:
        raise RuntimeError("DENSE_STORAGE_INVENTORY_STRUCTURE_UNSUPPORTED")

    for name, entry in dense_entries:
        add_entry(
            "dense", f"storage_file:{name}", entry,
            dense_manifest_path, dense_base,
        )

    sparse_files = sparse["files"]
    if not isinstance(sparse_files, dict):
        raise RuntimeError("SPARSE_FILES_STRUCTURE_UNSUPPORTED")
    for name in ("configuration", "documents", "postings", "inventory"):
        if name not in sparse_files:
            unresolved.append({
                "component": "sparse", "role": name,
                "reason": "FILE_ROLE_NOT_IN_MANIFEST",
            })
            continue
        add_entry(
            "sparse", name, sparse_files[name],
            sparse_manifest_path,
            root / Path(sparse_manifest_path).parent,
        )

    # Reuse only prior byte verification with the same path and digest.
    verified = {
        row["path"]: row
        for row in c15["artifact"]["results"]
        if row["status"] == "FILE_BYTES_MATCH_DECLARED_VALUE"
    }
    sources = c16["source_bindings"]
    unique_requests = {}
    for item in requests:
        key = item["path"]
        if (
            key in unique_requests
            and unique_requests[key]["declared_sha256"]
                != item["declared_sha256"]
        ):
            raise RuntimeError("CONFLICTING_INDEX_FILE_HASH_BINDINGS")
        unique_requests.setdefault(key, item)

    total = sum(
        (root / path).stat().st_size for path in unique_requests
        if path not in verified and path not in sources
    )
    print(json.dumps({
        "stage": "INDEX_FILE_CHECK_START",
        "explicit_unique_files": len(unique_requests),
        "new_bytes_planned": total,
        "new_mib_planned": round(total / 1024**2, 2),
        "unresolved_entries": len(unresolved),
    }, ensure_ascii=False), flush=True)

    def sig(stat):
        return (
            stat.st_dev, stat.st_ino, stat.st_size,
            stat.st_mtime_ns, stat.st_ctime_ns,
        )

    results = []
    for number, item in enumerate(unique_requests.values(), 1):
        row = dict(item)
        expected = item["declared_sha256"]
        path = root / item["path"]
        try:
            if item["path"] in verified:
                prior = verified[item["path"]]
                actual = prior["actual_file_bytes_sha256"]
                row["verification_mode"] = "REUSED_C15_BYTE_RECEIPT"
                stable = prior["file_stable_during_read"]
            elif item["path"] in sources:
                actual = sources[item["path"]]["file_bytes_sha256"]
                row["verification_mode"] = "REUSED_METADATA_BYTE_RECEIPT"
                stable = True
            else:
                started = time.monotonic()
                with path.open("rb") as stream:
                    before = os.fstat(stream.fileno())
                    hasher = hashlib.sha256()
                    count = 0
                    for block in iter(
                        lambda: stream.read(4 * 1024 * 1024), b""
                    ):
                        hasher.update(block)
                        count += len(block)
                    after = os.fstat(stream.fileno())
                stable = (
                    sig(before) == sig(after) == sig(path.stat())
                    and count == before.st_size
                )
                actual = hasher.hexdigest()
                row.update({
                    "verification_mode": "FRESH_STREAMED_BYTE_HASH",
                    "bytes_read": count,
                    "elapsed_seconds": round(
                        time.monotonic() - started, 3
                    ),
                })
            row.update({
                "actual_file_bytes_sha256": actual,
                "file_stable_at_verification": stable,
                "status": (
                    "INPUT_CHANGED_DURING_READ" if not stable else
                    "FILE_BYTES_MATCH_DECLARED_VALUE"
                    if actual == expected else
                    "HASH_DEFINITION_OR_VERSION_RECONCILIATION_REQUIRED"
                ),
            })
        except Exception as exc:
            row.update({
                "status": "LOCAL_FILE_CHECK_FAILED",
                "error_type": type(exc).__name__,
            })
        results.append(row)
        print(json.dumps({
            "stage": "INDEX_FILE_COMPLETED",
            "number": number,
            "path": row["path"],
            "status": row["status"],
        }, ensure_ascii=False), flush=True)

    all_match = (
        bool(results) and not unresolved
        and all(
            r["status"] == "FILE_BYTES_MATCH_DECLARED_VALUE"
            for r in results
        )
    )
    artifact = {
        "artifact_type": "NB22_PHYSICAL_INDEX_FILE_VERIFICATION",
        "schema_version": "NB22_C019_ACTION09_INDEX_FILES_V1",
        "status": (
            "DECLARED_INDEX_FILES_BYTE_CHECKS_PASS"
            if all_match else "INDEX_FILE_RECONCILIATION_REQUIRED"
        ),
        "scope": "MANIFEST_LISTED_DENSE_STORAGE_AND_SPARSE_CORE_FILES",
        "results": results,
        "unresolved_entries": unresolved,
        "logical_index_hashes_recomputed": False,
        "index_inventory_completeness_revalidated": False,
        "live_database_opened": False,
        "index_modified": False,
        "action09_fully_complete": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C019.action09",
        "C019.physical_index_files", artifact,
    )
    print(json.dumps({
        "status": artifact["status"],
        "results": results,
        "unresolved_entries": unresolved,
        "output": output,
        "logical_index_hashes_recomputed": False,
        "action09_fully_complete": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c19_result = nb22_c19_run(
    nb22_c16_result, nb22_c15_result, nb22_c18_result
)

{"stage": "INDEX_FILE_CHECK_START", "explicit_unique_files": 6, "new_bytes_planned": 51931118, "new_mib_planned": 49.53, "unresolved_entries": 0}
{"stage": "INDEX_FILE_COMPLETED", "number": 1, "path": "data/phase2/index/dense_index_v1/qdrant/collection/rag_finance_phase2_dense_v1/storage.sqlite", "status": "FILE_BYTES_MATCH_DECLARED_VALUE"}
{"stage": "INDEX_FILE_COMPLETED", "number": 2, "path": "data/phase2/index/dense_index_v1/qdrant/meta.json", "status": "FILE_BYTES_MATCH_DECLARED_VALUE"}
{"stage": "INDEX_FILE_COMPLETED", "number": 3, "path": "data/phase2/index/sparse_index_v1/sparse_configuration_v1.json", "status": "FILE_BYTES_MATCH_DECLARED_VALUE"}
{"stage": "INDEX_FILE_COMPLETED", "number": 4, "path": "data/phase2/index/sparse_index_v1/sparse_documents_v1.jsonl", "status": "FILE_BYTES_MATCH_DECLARED_VALUE"}
{"stage": "INDEX_FILE_COMPLETED", "number": 5, "path": "data/phase2/index/sparse_index_v1/sparse_postings_v1.jsonl", "status": "FILE_BYTES_MATCH_DECLARED_VALUE"}
{"stage": "IN

<div dir="rtl" align="right">

### NB22-C20 — اقدام ۹: تطبیق شناسه‌های منطقی و مصرف شواهد قبلی

سه زیرکار این بسته:

۱. تطبیق شناسهٔ منطقی Dense/Sparse میان اسناد متصل.
۲. خواندن فقط گزارش‌های اعتبارسنجیِ صریحاً ارجاع‌شده در Manifestها.
۳. تجمیع نتیجهٔ کنترل بایت، Config و شواهد تاریخی بدون بازاجرای Index.

تطبیق مقدار منطقی بین دو سند، بازتولید آن مقدار نیست.
گزارش تاریخی PASS با دامنه و نسخهٔ اصلی خودش مصرف می‌شود؛
اجرای تازهٔ آزمون یا پذیرش کل Release از آن نتیجه نمی‌شود.

فایل‌های فیزیکی تأییدشده در C19 و داده‌های C15 دوباره خوانده نمی‌شوند.
تنها گزارش‌های JSON کوچکِ دارای مسیر و هش مشخص خوانده می‌شوند.
در نبود ارجاع کافی، شکاف دقیق ثبت می‌شود.

Qdrant، مدل، API و آزمون‌های Index اجرا نمی‌شوند.
ورودی‌ها و چک‌لیست تغییر نمی‌کنند.

</div>

In [21]:
# NB22-C20 — Consume bound index-validation evidence without rerunning tests
# Scope: cached metadata and explicit small JSON validation reports only.

def nb22_c20_run(c16, c18, c19):
    if c19["artifact"]["status"] != "DECLARED_INDEX_FILES_BYTE_CHECKS_PASS":
        raise RuntimeError("PHYSICAL_INDEX_BYTE_CHECKS_NOT_PASSED")
    if c18["artifact"]["status"] != "RERANKER_CONFIG_PAYLOAD_HASH_VERIFIED":
        raise RuntimeError("RERANKER_CONFIG_RECONCILIATION_NOT_PASSED")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()
    payloads = dict(c16["payloads"])
    sources = dict(c16["source_bindings"])

    dense_release_path = (
        "artifacts/phase2/dense_index/dense_index_release_v1.json"
    )
    sparse_release_path = (
        "artifacts/phase2/sparse_index/sparse_index_release_v1.json"
    )
    dense_manifest_path = (
        "data/phase2/index/dense_index_v1/dense_index_manifest.json"
    )
    sparse_manifest_path = (
        "data/phase2/index/sparse_index_v1/sparse_index_manifest.json"
    )
    dense_release = payloads[dense_release_path]
    sparse_release = payloads[sparse_release_path]
    dense_manifest = payloads[dense_manifest_path]
    sparse_manifest = payloads[sparse_manifest_path]

    logical_checks = [
        {
            "component": "dense",
            "release_value": dense_release["logical_index_sha256"],
            "manifest_value":
                dense_manifest["index_content"]["logical_index_sha256"],
            "definition": "PRODUCER_DECLARED_LOGICAL_INDEX_IDENTITY",
        },
        {
            "component": "sparse",
            "release_value": sparse_release["logical_index_sha256"],
            "manifest_value": sparse_manifest["logical_index_sha256"],
            "definition": "PRODUCER_DECLARED_LOGICAL_INDEX_IDENTITY",
        },
        {
            "component": "dense_storage_inventory",
            "release_value": dense_release["storage_inventory_sha256"],
            "manifest_value":
                dense_manifest["storage_inventory"]["inventory_sha256"],
            "definition": "PRODUCER_DECLARED_STORAGE_INVENTORY_IDENTITY",
        },
    ]
    for row in logical_checks:
        row["status"] = (
            "DECLARATIONS_MATCH"
            if row["release_value"] == row["manifest_value"]
            else "LOGICAL_IDENTITY_RECONCILIATION_REQUIRED"
        )
        row["logical_hash_recomputed"] = False

    requests, gaps = [], []

    def add_reference(component, role, ref, source):
        if (
            not isinstance(ref, dict)
            or not isinstance(ref.get("path"), str)
            or not isinstance(ref.get("sha256"), str)
        ):
            gaps.append({
                "component": component,
                "role": role,
                "source": source,
                "reason": "EXPLICIT_REPORT_BINDING_NOT_RESOLVED",
            })
            return
        requests.append({
            "component": component,
            "role": role,
            "path": ref["path"],
            "expected_file_bytes_sha256": ref["sha256"],
            "binding_source": source,
        })

    dense_evidence = dense_manifest.get("evidence", {})
    if isinstance(dense_evidence, dict):
        for role in ("integrity", "snapshot_restore"):
            add_reference(
                "dense", role, dense_evidence.get(role),
                dense_manifest_path,
            )
    else:
        gaps.append({
            "component": "dense",
            "reason": "EVIDENCE_CONTAINER_NOT_OBJECT",
        })

    add_reference(
        "sparse", "reload_validation",
        sparse_manifest["files"].get("reload_validation"),
        sparse_manifest_path,
    )

    def compact(node, depth=0):
        if isinstance(node, dict):
            if depth >= 3:
                return {"object_keys": list(node)}
            return {
                key: compact(value, depth + 1)
                for key, value in node.items()
                if not any(
                    token in key.lower()
                    for token in (
                        "api_key", "authorization", "password", "secret"
                    )
                )
            }
        if isinstance(node, list):
            return {"list_count": len(node)}
        if isinstance(node, str):
            node = node.replace(str(root), "<PROJECT_ROOT>")
            node = re.sub(
                r"/(?:Users|home)/[^/\s]+", "<USER_HOME>", node
            )
            return node if len(node) <= 500 else {
                "text_length": len(node)
            }
        return node

    evidence_results = []
    for request in requests:
        row = dict(request)
        try:
            p = Path(row["path"])
            if (
                p.is_absolute() or ".." in p.parts
                or p.suffix.lower() != ".json"
            ):
                raise ValueError("REPORT_PATH_REQUIRES_EXPLICIT_ADAPTER")
            path = (root / p).resolve()
            path.relative_to(root)
            expected = row["expected_file_bytes_sha256"]
            if not re.fullmatch(r"[0-9a-f]{64}", expected):
                raise ValueError("INVALID_EXPECTED_SHA256")

            if row["path"] in payloads:
                actual = sources[row["path"]]["file_bytes_sha256"]
                value = payloads[row["path"]]
                row["read_mode"] = "REUSED_BOUND_METADATA"
            else:
                if not path.is_file():
                    raise FileNotFoundError()
                if path.stat().st_size > 4 * 1024 * 1024:
                    raise ValueError("REPORT_SIZE_REQUIRES_REVIEW")
                raw = path.read_bytes()
                actual = hashlib.sha256(raw).hexdigest()
                if actual != expected:
                    raise ValueError("REPORT_FILE_BYTES_HASH_MISMATCH")
                value = json.loads(raw.decode("utf-8"))
                if not isinstance(value, dict):
                    raise ValueError("REPORT_ROOT_NOT_OBJECT")
                payloads[row["path"]] = value
                sources[row["path"]] = {
                    "path": row["path"],
                    "file_bytes_sha256": actual,
                    "bytes": len(raw),
                }
                row["read_mode"] = "FRESH_REFERENCED_REPORT_READ"

            if actual != expected:
                raise ValueError("CACHED_REPORT_BINDING_MISMATCH")
            row.update({
                "status": "HISTORICAL_REPORT_HASH_BOUND_AND_READ",
                "actual_file_bytes_sha256": actual,
                "report_metadata": compact(value),
                "tests_rerun": False,
            })
        except Exception as exc:
            row.update({
                "status": "REPORT_RECONCILIATION_REQUIRED",
                "error_type": type(exc).__name__,
                "reason": (
                    str(exc) if isinstance(exc, ValueError)
                    else "REPORT_NOT_AVAILABLE_AT_EXPLICIT_PATH"
                    if isinstance(exc, FileNotFoundError)
                    else "LOCAL_REPORT_READ_FAILED"
                ),
            })
        evidence_results.append(row)

    artifact = {
        "artifact_type": "NB22_INDEX_HISTORICAL_EVIDENCE_RECONCILIATION",
        "schema_version": "NB22_C020_ACTION09_INDEX_EVIDENCE_V1",
        "status": "INDEX_IDENTITIES_AND_BOUND_REPORTS_ACCOUNTED",
        "logical_identity_checks": logical_checks,
        "historical_evidence": evidence_results,
        "unresolved_report_bindings": gaps,
        "physical_index_byte_verification": c19["output"],
        "reranker_payload_hash_verification": c18["output"],
        "historical_tests_reexecuted": False,
        "logical_hashes_recomputed": False,
        "live_database_opened": False,
        "action09_fully_complete": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C020.action09",
        "C020.index_evidence", artifact,
    )
    print(json.dumps({
        "status": artifact["status"],
        "logical_identity_checks": logical_checks,
        "historical_evidence": evidence_results,
        "unresolved_report_bindings": gaps,
        "output": output,
        "historical_tests_reexecuted": False,
        "physical_index_files_reread": 0,
        "action09_fully_complete": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {
        "artifact": artifact,
        "output": output,
        "payloads": payloads,
        "source_bindings": sources,
    }


nb22_c20_result = nb22_c20_run(
    nb22_c16_result, nb22_c18_result, nb22_c19_result
)

{
  "status": "INDEX_IDENTITIES_AND_BOUND_REPORTS_ACCOUNTED",
  "logical_identity_checks": [
    {
      "component": "dense",
      "release_value": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
      "manifest_value": "2fc5e50ebbd1e1e66d59c703eb48f1367b2d3deadfed62efb9d5a5e6959f7ef4",
      "definition": "PRODUCER_DECLARED_LOGICAL_INDEX_IDENTITY",
      "status": "DECLARATIONS_MATCH",
      "logical_hash_recomputed": false
    },
    {
      "component": "sparse",
      "release_value": "ce09bb4749da03c9dfa1a849de18fc8c490ce6b09eb34a30a96b9fedc478a8eb",
      "manifest_value": "ce09bb4749da03c9dfa1a849de18fc8c490ce6b09eb34a30a96b9fedc478a8eb",
      "definition": "PRODUCER_DECLARED_LOGICAL_INDEX_IDENTITY",
      "status": "DECLARATIONS_MATCH",
      "logical_hash_recomputed": false
    },
    {
      "component": "dense_storage_inventory",
      "release_value": "c0ff83e70501f4a8bc92fb1cfdf48636d68bce75a3abc4ceb55454c361cd3c6c",
      "manifest_value": "c0ff83e7

<div dir="rtl" align="right">

### NB22-C21 — اقدام ۱۰، بستهٔ اول: ساختار واقعی و تعداد رکوردها

سه زیرکار این بسته:

۱. خواندن جریانی شش JSONL مشخص و تشخیص JSON نامعتبر،
کلید تکراری یا مقدار عددی غیرمجاز.
۲. تطبیق تعداد رکورد با مقدار مستند در Manifest مربوط.
۳. ثبت ساختار نخستین رکورد، بدون نمایش متن سؤال، پاسخ یا منبع.

مقادیر مورد انتظار از Metadata خوانده‌شده گرفته می‌شوند.
برای Corpus، اگر انتظار شمارش متصل نداریم، فقط تعداد مشاهده‌شده
گزارش می‌شود و پذیرش شمارش ساخته نمی‌شود.

این بررسی، اعتبارسنجی کامل Schema یا اتصال شناسه‌ها نیست.
ساختار مشاهده‌شده، قرارداد مصوب جدید محسوب نمی‌شود.
هیچ رکورد نامعتبر حذف، اصلاح یا خودکار معتبر اعلام نمی‌شود.

هش‌گیری، مدل، API، تغییر داده و تغییر چک‌لیست: صفر.
گزارش جدید Atomic ذخیره و بازخوانی می‌شود.

</div>

In [22]:
# NB22-C21 — Action 10: streamed JSONL structure and manifest counts
# Scope: six explicit assets; no hashing, inference, repair or model calls.

def nb22_c21_run(c14, c15, c16):
    if c15["summary"]["status"] != (
        "SIX_DATA_BYTE_HASHES_MATCH_DECLARED_VALUES"
    ):
        raise RuntimeError("C15_INPUT_IDENTITY_CHECKS_NOT_PASSED")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()
    payloads = c16["payloads"]

    golden_source = (
        "artifacts/phase2/golden_dataset_v2/golden_dataset_v2_manifest.json"
    )
    embedding_source = (
        "artifacts/phase2/embedding/embedding_release_v1.json"
    )
    reranker_source = (
        "data/phase2/retrieval/reranker_v1/reranker_manifest.json"
    )
    context_source = (
        "data/phase2/context/context_builder_v1/context_builder_manifest.json"
    )

    expectations = {
        "golden_dataset": (
            payloads[golden_source]["record_count"],
            golden_source, ["record_count"],
        ),
        "active_chunks": (
            payloads[embedding_source]["chunking"]["record_count"],
            embedding_source, ["chunking", "record_count"],
        ),
        "reranked_candidates": (
            payloads[reranker_source]["output"]["record_count"],
            reranker_source, ["output", "record_count"],
        ),
        "context_packets": (
            payloads[context_source]["record_counts"]["candidate_count"],
            context_source, ["record_counts", "candidate_count"],
        ),
        "context_evidence_registry": (
            payloads[context_source]["record_counts"][
                "evidence_registry_count"
            ],
            context_source, ["record_counts", "evidence_registry_count"],
        ),
    }

    def no_duplicate_keys(pairs):
        obj = {}
        for key, value in pairs:
            if key in obj:
                raise ValueError("DUPLICATE_JSON_KEY")
            obj[key] = value
        return obj

    def no_nonfinite(value):
        raise ValueError("NONFINITE_JSON_NUMBER")

    def shape(value, depth=0):
        if isinstance(value, dict):
            if depth >= 2:
                return {"type": "object", "keys": list(value)}
            return {
                "type": "object",
                "fields": {
                    key: shape(item, depth + 1)
                    for key, item in value.items()
                },
            }
        if isinstance(value, list):
            return {
                "type": "array",
                "first_item_shape": (
                    shape(value[0], depth + 1) if value else None
                ),
            }
        return {"type": type(value).__name__}

    def signature(stat):
        return (
            stat.st_dev, stat.st_ino, stat.st_size,
            stat.st_mtime_ns, stat.st_ctime_ns,
        )

    results = []
    for item in c14["artifact"]["roles"]:
        role, relative = item["role"], item["reference_path"]
        path = (root / relative).resolve()
        path.relative_to(root)
        expected = expectations.get(role)
        valid_count = nonblank_count = blank_count = invalid_count = 0
        errors = []
        first_shape = None
        started = time.monotonic()

        with path.open("rb") as stream:
            before = os.fstat(stream.fileno())
            for line_number, raw_line in enumerate(stream, 1):
                if not raw_line.strip():
                    blank_count += 1
                    continue
                nonblank_count += 1
                try:
                    value = json.loads(
                        raw_line.decode("utf-8"),
                        object_pairs_hook=no_duplicate_keys,
                        parse_constant=no_nonfinite,
                    )
                    if not isinstance(value, dict):
                        raise ValueError("JSONL_RECORD_NOT_OBJECT")
                    valid_count += 1
                    if first_shape is None:
                        first_shape = shape(value)
                except (ValueError, UnicodeError, RecursionError) as exc:
                    invalid_count += 1
                    if len(errors) < 10:
                        errors.append({
                            "line_number": line_number,
                            "error_type": type(exc).__name__,
                        })
            after = os.fstat(stream.fileno())

        stable = (
            signature(before) == signature(after)
            == signature(path.stat())
        )
        count_status = (
            "NO_BOUND_EXPECTATION_OBSERVED_COUNT_ONLY"
            if expected is None else
            "MATCH" if valid_count == expected[0] else
            "COUNT_RECONCILIATION_REQUIRED"
        )
        results.append({
            "role": role,
            "path": relative,
            "file_stable_during_read": stable,
            "nonblank_lines": nonblank_count,
            "blank_lines": blank_count,
            "valid_object_records": valid_count,
            "invalid_records": invalid_count,
            "error_examples": errors,
            "expected_record_count": expected[0] if expected else None,
            "expectation_source": (
                {"path": expected[1], "field_path": expected[2]}
                if expected else None
            ),
            "count_status": count_status,
            "json_structure_status": (
                "REQUIRES_RECONCILIATION"
                if invalid_count or not stable else
                "ALL_NONBLANK_ROWS_READABLE_JSON_OBJECTS"
            ),
            "first_record_structure": first_shape,
            "observed_structure_is_approved_schema": False,
            "elapsed_seconds": round(time.monotonic() - started, 3),
        })
        print(json.dumps({
            "stage": "JSONL_COMPLETED",
            "role": role,
            "valid_records": valid_count,
            "invalid_records": invalid_count,
            "count_status": count_status,
        }, ensure_ascii=False), flush=True)

    artifact = {
        "artifact_type": "NB22_JSONL_STRUCTURE_AND_COUNT_CHECKS",
        "schema_version": "NB22_C021_ACTION10_STRUCTURE_COUNTS_V1",
        "status": (
            "STRUCTURE_AND_BOUND_COUNTS_CHECKED"
            if all(
                r["invalid_records"] == 0
                and r["file_stable_during_read"]
                and r["count_status"] != "COUNT_RECONCILIATION_REQUIRED"
                for r in results
            ) else "STRUCTURE_OR_COUNT_RECONCILIATION_REQUIRED"
        ),
        "results": results,
        "approved_schema_validation_complete": False,
        "reference_integrity_complete": False,
        "action10_fully_complete": False,
        "hashes_recomputed": False,
        "records_modified": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C021.action10",
        "C021.structure_counts", artifact,
    )
    print(json.dumps({
        "status": artifact["status"],
        "results": results,
        "output": output,
        "action10_fully_complete": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c21_result = nb22_c21_run(
    nb22_c14_result, nb22_c15_result, nb22_c16_result
)

{"stage": "JSONL_COMPLETED", "role": "active_chunks", "valid_records": 3488, "invalid_records": 0, "count_status": "MATCH"}
{"stage": "JSONL_COMPLETED", "role": "context_evidence_registry", "valid_records": 3355, "invalid_records": 0, "count_status": "MATCH"}
{"stage": "JSONL_COMPLETED", "role": "context_packets", "valid_records": 3813, "invalid_records": 0, "count_status": "MATCH"}
{"stage": "JSONL_COMPLETED", "role": "document_corpus", "valid_records": 19456, "invalid_records": 0, "count_status": "NO_BOUND_EXPECTATION_OBSERVED_COUNT_ONLY"}
{"stage": "JSONL_COMPLETED", "role": "golden_dataset", "valid_records": 3813, "invalid_records": 0, "count_status": "MATCH"}
{"stage": "JSONL_COMPLETED", "role": "reranked_candidates", "valid_records": 3813, "invalid_records": 0, "count_status": "MATCH"}
{
  "status": "STRUCTURE_AND_BOUND_COUNTS_CHECKED",
  "results": [
    {
      "role": "active_chunks",
      "path": "data/phase2/chunks/nb11_benchmark_v1/baseline_phase1_char_safe_v1.jsonl",
    

<div dir="rtl" align="right">

### NB22-C22 — اقدام ۱۰، بستهٔ دوم: کنترل اتصال شناسه‌ها

چهار زیرکار:

۱. کنترل شناسه‌های غیرخالی و یکتا در هر خانواده.
۲. کنترل ارجاع Documentها در Chunk، Golden و رجیستری Context.
۳. تطبیق جمعیت candidate_id در Golden، Reranker و Context
و سازگاری Split میان Reranker و Context.
۴. کنترل اتصال Evidenceهای Context به رجیستری و Pool همان پرسش؛
سپس ذخیره و بازخوانی گزارش.

Golden evidence_ids متعلق به مرجع همان Dataset است؛
بدون قرارداد اتصال، با Context evidence_id مقایسه نمی‌شود.

این کنترل ساختاری است؛ صحت معنایی پاسخ یا انتساب را نمی‌سنجد.
Rank، Score، Split و داده‌ها تغییر نمی‌کنند.
هیچ رکورد مشکل‌دار حذف یا ترمیم نمی‌شود.

فایل‌ها برای هدف تازهٔ اتصال شناسه‌ها جریانی خوانده می‌شوند؛
هش‌گیری فایل و آزمون‌های قبلی تکرار نمی‌شوند.
API، مدل و تغییر چک‌لیست: صفر.

</div>

In [23]:
# NB22-C22 — Action 10: structural identity and reference integrity
# Scope: six explicit JSONLs; no semantic judging or data mutation.

def nb22_c22_run(c14, c21):
    if c21["artifact"]["status"] != "STRUCTURE_AND_BOUND_COUNTS_CHECKED":
        raise RuntimeError("C21_STRUCTURE_CHECKS_REQUIRE_RECONCILIATION")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()
    paths = {
        row["role"]: row["reference_path"]
        for row in c14["artifact"]["roles"]
    }

    counts, examples, file_counts = {}, {}, {}
    def issue(kind, detail):
        counts[kind] = counts.get(kind, 0) + 1
        if len(examples.setdefault(kind, [])) < 8:
            examples[kind].append(detail)

    def signature(stat):
        return (
            stat.st_dev, stat.st_ino, stat.st_size,
            stat.st_mtime_ns, stat.st_ctime_ns,
        )

    def rows(role):
        path = (root / paths[role]).resolve()
        path.relative_to(root)
        count = 0
        with path.open("rb") as stream:
            before = os.fstat(stream.fileno())
            for line_number, line in enumerate(stream, 1):
                if not line.strip():
                    continue
                row = json.loads(line.decode("utf-8"))
                if not isinstance(row, dict):
                    raise RuntimeError("INPUT_STRUCTURE_CHANGED_SINCE_C21")
                count += 1
                yield line_number, row
            after = os.fstat(stream.fileno())
        if not (
            signature(before) == signature(after)
            == signature(path.stat())
        ):
            raise RuntimeError(f"INPUT_CHANGED_DURING_JOIN: {role}")
        file_counts[role] = count

    def identity(role, line, row, field, seen):
        value = row.get(field)
        if not isinstance(value, str) or not value.strip():
            issue("INVALID_PRIMARY_ID", {
                "role": role, "line": line, "field": field,
            })
            return None
        if value in seen:
            issue("DUPLICATE_PRIMARY_ID", {
                "role": role, "line": line, "id": value,
            })
            return None
        seen.add(value)
        return value

    def references(row, field, allowed, role, owner):
        values = row.get(field)
        if not isinstance(values, list):
            issue("REFERENCE_LIST_NOT_PRESENT", {
                "role": role, "id": owner, "field": field,
            })
            return
        for value in values:
            if not isinstance(value, str) or value not in allowed:
                issue("DOCUMENT_REFERENCE_NOT_RESOLVED", {
                    "role": role, "id": owner, "field": field,
                    "referenced_id": value,
                })

    document_ids = set()
    for line, row in rows("document_corpus"):
        identity(
            "document_corpus", line, row,
            "document_id", document_ids,
        )

    chunk_ids = set()
    for line, row in rows("active_chunks"):
        cid = identity(
            "active_chunks", line, row, "chunk_id", chunk_ids
        )
        references(
            row, "member_document_ids", document_ids,
            "active_chunks", cid,
        )

    golden_ids = set()
    for line, row in rows("golden_dataset"):
        cid = identity(
            "golden_dataset", line, row,
            "candidate_id", golden_ids,
        )
        references(
            row, "document_ids", document_ids, "golden_dataset", cid
        )

    evidence_ids, evidence_chunks = set(), {}
    for line, row in rows("context_evidence_registry"):
        eid = identity(
            "context_evidence_registry", line, row,
            "evidence_id", evidence_ids,
        )
        chunk = row.get("chunk_id")
        if not isinstance(chunk, str) or chunk not in chunk_ids:
            issue("EVIDENCE_CHUNK_REFERENCE_NOT_RESOLVED", {
                "line": line, "evidence_id": eid, "chunk_id": chunk,
            })
        references(
            row, "member_document_ids", document_ids,
            "context_evidence_registry", eid,
        )
        if eid is not None:
            evidence_chunks[eid] = chunk

    reranker_ids, pools, reranker_splits = set(), {}, {}
    for line, row in rows("reranked_candidates"):
        cid = identity(
            "reranked_candidates", line, row,
            "candidate_id", reranker_ids,
        )
        candidates = row.get("candidates")
        pool = set()
        if not isinstance(candidates, list):
            issue("RERANKER_POOL_NOT_ARRAY", {"candidate_id": cid})
        else:
            for item in candidates:
                chunk = item.get("chunk_id") if isinstance(item, dict) else None
                if not isinstance(chunk, str) or chunk not in chunk_ids:
                    issue("RERANKER_CHUNK_REFERENCE_NOT_RESOLVED", {
                        "candidate_id": cid, "chunk_id": chunk,
                    })
                    continue
                if chunk in pool:
                    issue("DUPLICATE_CHUNK_IN_RERANKER_POOL", {
                        "candidate_id": cid, "chunk_id": chunk,
                    })
                pool.add(chunk)
        if cid is not None:
            pools[cid] = pool
            reranker_splits[cid] = row.get("split")

    context_ids, split_counts = set(), {}
    evidence_reference_count = 0
    for line, row in rows("context_packets"):
        cid = identity(
            "context_packets", line, row,
            "candidate_id", context_ids,
        )
        split = row.get("split")
        split_key = split if isinstance(split, str) else "<INVALID>"
        split_counts[split_key] = split_counts.get(split_key, 0) + 1
        if not isinstance(split, str) or not split:
            issue("INVALID_CONTEXT_SPLIT", {"candidate_id": cid})
        if cid not in pools:
            issue("CONTEXT_QUERY_NOT_IN_RERANKER", {"candidate_id": cid})
        elif split != reranker_splits[cid]:
            issue("CONTEXT_RERANKER_SPLIT_DIFFERS", {"candidate_id": cid})

        refs = row.get("evidence_refs")
        if not isinstance(refs, list):
            issue("CONTEXT_EVIDENCE_REFS_NOT_ARRAY", {"candidate_id": cid})
            continue
        seen_refs = set()
        for ref in refs:
            evidence_reference_count += 1
            eid = ref.get("evidence_id") if isinstance(ref, dict) else None
            if not isinstance(eid, str) or eid not in evidence_ids:
                issue("CONTEXT_EVIDENCE_REFERENCE_NOT_RESOLVED", {
                    "candidate_id": cid, "evidence_id": eid,
                })
                continue
            if eid in seen_refs:
                issue("DUPLICATE_EVIDENCE_IN_CONTEXT", {
                    "candidate_id": cid, "evidence_id": eid,
                })
            seen_refs.add(eid)
            if cid in pools and evidence_chunks[eid] not in pools[cid]:
                issue("CONTEXT_EVIDENCE_OUTSIDE_QUERY_POOL", {
                    "candidate_id": cid, "evidence_id": eid,
                    "chunk_id": evidence_chunks[eid],
                })

    population_checks = {
        "golden_equals_reranker_ids": golden_ids == reranker_ids,
        "golden_equals_context_ids": golden_ids == context_ids,
        "reranker_missing_from_golden": len(reranker_ids - golden_ids),
        "golden_missing_from_reranker": len(golden_ids - reranker_ids),
        "context_missing_from_golden": len(context_ids - golden_ids),
        "golden_missing_from_context": len(golden_ids - context_ids),
    }
    population_ok = (
        population_checks["golden_equals_reranker_ids"]
        and population_checks["golden_equals_context_ids"]
    )
    artifact = {
        "artifact_type": "NB22_STRUCTURAL_REFERENCE_INTEGRITY",
        "schema_version": "NB22_C022_ACTION10_REFERENCES_V1",
        "status": (
            "CHECKED_IDENTITY_AND_REFERENCE_RELATIONS_PASS"
            if not counts and population_ok else
            "STRUCTURAL_REFERENCE_RECONCILIATION_REQUIRED"
        ),
        "scope": "EXPLICIT_IDS_DOCUMENT_CHUNK_QUERY_EVIDENCE_AND_POOL_MEMBERSHIP",
        "file_record_counts": file_counts,
        "unique_id_counts": {
            "documents": len(document_ids),
            "chunks": len(chunk_ids),
            "golden_queries": len(golden_ids),
            "reranker_queries": len(reranker_ids),
            "context_queries": len(context_ids),
            "context_evidence": len(evidence_ids),
        },
        "population_checks": population_checks,
        "context_split_counts": split_counts,
        "context_evidence_reference_count": evidence_reference_count,
        "issue_counts": counts,
        "issue_examples": examples,
        "golden_evidence_ids_equated_with_context_ids": False,
        "rank_score_fidelity_checked": False,
        "full_approved_schema_validation_complete": False,
        "semantic_correctness_assessed": False,
        "action10_fully_complete": False,
        "input_files_modified": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C022.action10",
        "C022.reference_integrity", artifact,
    )
    print(json.dumps({
        **{key: value for key, value in artifact.items()
           if key not in ("artifact_type", "schema_version")},
        "output": output,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c22_result = nb22_c22_run(nb22_c14_result, nb22_c21_result)

{
  "status": "CHECKED_IDENTITY_AND_REFERENCE_RELATIONS_PASS",
  "scope": "EXPLICIT_IDS_DOCUMENT_CHUNK_QUERY_EVIDENCE_AND_POOL_MEMBERSHIP",
  "file_record_counts": {
    "document_corpus": 19456,
    "active_chunks": 3488,
    "golden_dataset": 3813,
    "context_evidence_registry": 3355,
    "reranked_candidates": 3813,
    "context_packets": 3813
  },
  "unique_id_counts": {
    "documents": 19456,
    "chunks": 3488,
    "golden_queries": 3813,
    "reranker_queries": 3813,
    "context_queries": 3813,
    "context_evidence": 3355
  },
  "population_checks": {
    "golden_equals_reranker_ids": true,
    "golden_equals_context_ids": true,
    "reranker_missing_from_golden": 0,
    "golden_missing_from_reranker": 0,
    "context_missing_from_golden": 0,
    "golden_missing_from_context": 0
  },
  "context_split_counts": {
    "development": 2664,
    "test": 1149
  },
  "context_evidence_reference_count": 38096,
  "issue_counts": {},
  "issue_examples": {},
  "golden_evidence_ids_equa

<div dir="rtl" align="right">

### NB22-C23 — اقدام ۱۰: مصرف شواهد Reload و کنترل پوشش اعتبارسنجی

سه زیرکار:

۱. تطبیق شمارش‌های واقعی C22 با Manifestهای متصل.
۲. خواندن گزارش‌های Reload مشخص Reranker و Context با کنترل هش.
۳. ثبت کنترل‌های انجام‌شدهٔ قبلی و حدود پوشش Schema و Rank/Score.

گزارش قبلی فقط در دامنهٔ نسخه و ورودی خودش مصرف می‌شود.
وجود PASS در گزارش، کنترل کامل Schema یا صحت معنایی را خودکار ثابت نمی‌کند.
ساختار و نتایج واقعی گزارش برای تصمیم پایان اقدام نمایش داده می‌شود.

JSONLها و Index دوباره خوانده نمی‌شوند.
Config، Rank، Score و Split تغییر نمی‌کنند.
API، اجرای مدل و تغییر چک‌لیست: صفر.

</div>

In [24]:
# NB22-C23 — Action 10: consume existing reload-validation evidence
# Scope: cached manifests and two explicit small JSON reports only.

def nb22_c23_run(c16, c20, c22):
    if c22["artifact"]["status"] != (
        "CHECKED_IDENTITY_AND_REFERENCE_RELATIONS_PASS"
    ):
        raise RuntimeError("REFERENCE_INTEGRITY_REQUIRES_RECONCILIATION")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()
    payloads = dict(c20["payloads"])
    sources = dict(c20["source_bindings"])

    reranker_path = (
        "data/phase2/retrieval/reranker_v1/reranker_manifest.json"
    )
    context_path = (
        "data/phase2/context/context_builder_v1/context_builder_manifest.json"
    )
    reranker = payloads[reranker_path]
    context = payloads[context_path]
    counts = c22["artifact"]

    comparisons = [
        (
            "context_candidates",
            counts["file_record_counts"]["context_packets"],
            context["record_counts"]["candidate_count"],
        ),
        (
            "context_evidence",
            counts["file_record_counts"]["context_evidence_registry"],
            context["record_counts"]["evidence_registry_count"],
        ),
        (
            "context_evidence_references",
            counts["context_evidence_reference_count"],
            context["record_counts"]["evidence_reference_count"],
        ),
        (
            "context_development",
            counts["context_split_counts"]["development"],
            context["record_counts"]["development_candidate_count"],
        ),
        (
            "context_test",
            counts["context_split_counts"]["test"],
            context["record_counts"]["test_candidate_count"],
        ),
        (
            "reranker_candidates",
            counts["file_record_counts"]["reranked_candidates"],
            reranker["output"]["record_count"],
        ),
    ]
    count_checks = [
        {
            "check": name, "observed": observed, "manifest": expected,
            "status": "MATCH" if observed == expected else "RECONCILE",
        }
        for name, observed, expected in comparisons
    ]

    refs = [
        ("reranker", reranker_path, reranker["reload_validation"]),
        ("context", context_path, context["outputs"]["reload_validation"]),
    ]

    def view(value, depth=0):
        if isinstance(value, dict):
            if depth >= 4:
                return {"object_keys": list(value)}
            return {
                key: view(item, depth + 1)
                for key, item in value.items()
                if not any(
                    token in key.lower()
                    for token in (
                        "api_key", "secret", "authorization", "password"
                    )
                )
            }
        if isinstance(value, list):
            if all(
                isinstance(item, (str, bool, int, float, type(None)))
                for item in value
            ) and len(value) <= 20:
                return [view(item, depth + 1) for item in value]
            return {"list_count": len(value)}
        if isinstance(value, str):
            value = value.replace(str(root), "<PROJECT_ROOT>")
            value = re.sub(
                r"/(?:Users|home)/[^/\s]+", "<USER_HOME>", value
            )
            return value if len(value) <= 700 else {
                "text_length": len(value)
            }
        return value

    results = []
    for component, binding_source, ref in refs:
        if not isinstance(ref, dict):
            raise RuntimeError("RELOAD_REFERENCE_NOT_OBJECT")
        relative, expected = ref.get("path"), ref.get("sha256")
        row = {
            "component": component,
            "binding_source": binding_source,
            "path": relative,
            "expected_file_bytes_sha256": expected,
        }
        try:
            if not isinstance(relative, str):
                raise ValueError("EXPLICIT_PATH_MISSING")
            p = Path(relative)
            if p.is_absolute() or ".." in p.parts or p.suffix != ".json":
                raise ValueError("RELOAD_PATH_REQUIRES_ADAPTER")
            target = (root / p).resolve()
            target.relative_to(root)
            if not isinstance(expected, str) or not re.fullmatch(
                r"[0-9a-f]{64}", expected
            ):
                raise ValueError("EXPLICIT_HASH_MISSING")

            if relative in payloads:
                value = payloads[relative]
                actual = sources[relative]["file_bytes_sha256"]
                row["read_mode"] = "REUSED_BOUND_METADATA"
            else:
                if not target.is_file():
                    raise FileNotFoundError()
                if target.stat().st_size > 4 * 1024 * 1024:
                    raise ValueError("REPORT_SIZE_REQUIRES_REVIEW")
                raw = target.read_bytes()
                actual = hashlib.sha256(raw).hexdigest()
                if actual != expected:
                    raise ValueError("RELOAD_REPORT_HASH_MISMATCH")
                value = json.loads(raw.decode("utf-8"))
                if not isinstance(value, dict):
                    raise ValueError("RELOAD_REPORT_ROOT_NOT_OBJECT")
                payloads[relative] = value
                sources[relative] = {
                    "path": relative,
                    "file_bytes_sha256": actual,
                    "bytes": len(raw),
                }
                row["read_mode"] = "FRESH_EXPLICIT_RELOAD_REPORT"

            if actual != expected:
                raise ValueError("CACHED_RELOAD_REPORT_BINDING_MISMATCH")
            row.update({
                "status": "RELOAD_REPORT_HASH_BOUND_AND_READ",
                "actual_file_bytes_sha256": actual,
                "recorded_validation": view(value),
                "tests_reexecuted": False,
            })
        except Exception as exc:
            row.update({
                "status": "RELOAD_REPORT_RECONCILIATION_REQUIRED",
                "error_type": type(exc).__name__,
                "reason": (
                    str(exc) if isinstance(exc, ValueError)
                    else "REPORT_NOT_AVAILABLE_AT_EXPLICIT_PATH"
                    if isinstance(exc, FileNotFoundError)
                    else "LOCAL_METADATA_READ_FAILED"
                ),
            })
        results.append(row)

    artifact = {
        "artifact_type": "NB22_RELOAD_VALIDATION_COVERAGE",
        "schema_version": "NB22_C023_ACTION10_RELOAD_COVERAGE_V1",
        "status": "COUNT_AND_EXISTING_RELOAD_EVIDENCE_RECORDED",
        "count_checks": count_checks,
        "reload_reports": results,
        "current_reference_integrity_receipt": c22["output"],
        "historical_tests_reexecuted": False,
        "schema_and_rank_score_coverage_decision_pending": True,
        "action10_fully_complete": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C023.action10",
        "C023.reload_coverage", artifact,
    )
    print(json.dumps({
        "status": artifact["status"],
        "count_checks": count_checks,
        "reload_reports": results,
        "output": output,
        "jsonl_files_reread": 0,
        "historical_tests_reexecuted": False,
        "action10_fully_complete": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {
        "artifact": artifact,
        "output": output,
        "payloads": payloads,
        "source_bindings": sources,
    }


nb22_c23_result = nb22_c23_run(
    nb22_c16_result, nb22_c20_result, nb22_c22_result
)

{
  "status": "COUNT_AND_EXISTING_RELOAD_EVIDENCE_RECORDED",
  "count_checks": [
    {
      "check": "context_candidates",
      "observed": 3813,
      "manifest": 3813,
      "status": "MATCH"
    },
    {
      "check": "context_evidence",
      "observed": 3355,
      "manifest": 3355,
      "status": "MATCH"
    },
    {
      "check": "context_evidence_references",
      "observed": 38096,
      "manifest": 38096,
      "status": "MATCH"
    },
    {
      "check": "context_development",
      "observed": 2664,
      "manifest": 2664,
      "status": "MATCH"
    },
    {
      "check": "context_test",
      "observed": 1149,
      "manifest": 1149,
      "status": "MATCH"
    },
    {
      "check": "reranker_candidates",
      "observed": 3813,
      "manifest": 3813,
      "status": "MATCH"
    }
  ],
  "reload_reports": [
    {
      "component": "reranker",
      "binding_source": "data/phase2/retrieval/reranker_v1/reranker_manifest.json",
      "path": "data/phase2/retrieva

<div dir="rtl" align="right">

### NB22-C24 — جمع‌بندی پوشش اقدام ۱۰ و موجودی اولیهٔ اقدام ۱۱

سه زیرکار:

۱. تطبیق ورودی‌ها و خروجی‌های گزارش Reload با هش‌های کنترل‌شدهٔ فعلی.
۲. ثبت پوشش کنترل‌های ساختاری و Rank/Score همراه محدودیت Schema.
۳. ساخت موجودی اولیهٔ فایل‌های داده، Index، Metadata، Snapshot و مدل‌ها
برای برنامهٔ Restore.

تمام نتایج از شواهد موجود مصرف می‌شوند؛ داده‌ها دوباره خوانده نمی‌شوند.
برای Snapshot فقط وجود و اندازه بررسی می‌شود؛ صحت بایت یا Restore تازه ادعا نمی‌شود.
مدل‌ها فقط با شناسه، Revision و محدودیت‌های ثبت‌شده وارد موجودی می‌شوند؛
وجود Cache مدل فرض نمی‌شود.

این موجودی، بستهٔ کامل قابل Restore نیست.
منابع خارجی و اتصال‌های باز در فهرست باقی می‌مانند.
API، دانلود، اجرای مدل، بازکردن پایگاه‌داده و تغییر چک‌لیست: صفر.

</div>

In [25]:
# NB22-C24 — Bind reused validation evidence and inventory restore assets
# Scope: cached receipts + explicit snapshot stat; no data rereads/restore.

def nb22_c24_run(c13, c15, c16, c18, c19, c22, c23):
    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    if any(row["status"] != "MATCH"
           for row in c23["artifact"]["count_checks"]):
        raise RuntimeError("MANIFEST_COUNT_RECONCILIATION_REQUIRED")
    if c22["artifact"]["status"] != (
        "CHECKED_IDENTITY_AND_REFERENCE_RELATIONS_PASS"
    ):
        raise RuntimeError("CURRENT_REFERENCE_CHECKS_NOT_PASSED")

    data = {
        row["role"]: row for row in c15["artifact"]["results"]
    }
    metadata = c23["source_bindings"]
    payloads = c23["payloads"]

    reports = {
        row["component"]: row
        for row in c23["artifact"]["reload_reports"]
    }
    if set(reports) != {"reranker", "context"} or any(
        row["status"] != "RELOAD_REPORT_HASH_BOUND_AND_READ"
        for row in reports.values()
    ):
        raise RuntimeError("BOUND_RELOAD_REPORTS_NOT_COMPLETE")

    rr = payloads[reports["reranker"]["path"]]
    cr = payloads[reports["context"]["path"]]
    context_manifest = payloads[
        "data/phase2/context/context_builder_v1/context_builder_manifest.json"
    ]
    context_config_path = (
        "artifacts/phase2/context_builder/context_builder_configuration_v1.json"
    )

    bindings = []

    def compare(name, expected, actual):
        bindings.append({
            "check": name,
            "recorded": expected,
            "verified_current": actual,
            "status": "MATCH" if expected == actual else "RECONCILE",
        })

    compare(
        "reranker_output_identity",
        rr["logical_reranker_output_sha256"],
        data["reranked_candidates"]["actual_file_bytes_sha256"],
    )
    compare(
        "reranker_payload_configuration_identity",
        rr["configuration_sha256"],
        c18["artifact"]["configuration"]["computed_payload_sha256"],
    )
    compare(
        "context_configuration_identity",
        cr["configuration_sha256"],
        metadata[context_config_path]["file_bytes_sha256"],
    )

    for report_key, role in (
        ("candidates", "reranked_candidates"),
        ("chunks", "active_chunks"),
        ("documents", "document_corpus"),
    ):
        compare(
            f"context_input:{report_key}",
            cr["input_sha256"][report_key],
            data[role]["actual_file_bytes_sha256"],
        )

    for output_key, role in (
        ("context_packets", "context_packets"),
        ("evidence_registry", "context_evidence_registry"),
    ):
        compare(
            f"context_output:{output_key}",
            context_manifest["outputs"][output_key]["sha256"],
            data[role]["actual_file_bytes_sha256"],
        )

    if any(row["status"] != "MATCH" for row in bindings):
        raise RuntimeError("HISTORICAL_TO_CURRENT_IDENTITY_RECONCILIATION_REQUIRED")

    observed_validation = {
        "reranker": {
            key: rr["candidate_validation"].get(key)
            for key in (
                "id_order_parity", "ranking_parity",
                "score_parity", "maximum_absolute_score_delta"
            )
        },
        "context": {
            key: cr["checks"].get(key)
            for key in (
                "reranker_rank_fidelity", "reranker_score_fidelity",
                "evidence_hash_fidelity", "evidence_text_fidelity",
                "locator_fidelity", "metadata_fidelity",
                "evidence_refs_resolve", "configuration_unchanged",
            )
        },
    }
    reuse_checks_pass = (
        all(
            observed_validation["reranker"][key] == "PASS"
            for key in ("id_order_parity", "ranking_parity", "score_parity")
        )
        and all(value is True
                for value in observed_validation["context"].values())
    )

    assets = {}
    def add_asset(path, kind, digest, verification, source):
        if path in assets:
            if assets[path]["file_bytes_sha256"] != digest:
                raise RuntimeError("RESTORE_ASSET_IDENTITY_CONFLICT")
            assets[path]["evidence_sources"].append(source)
            return
        assets[path] = {
            "path": path,
            "kind": kind,
            "file_bytes_sha256": digest,
            "verification_scope": verification,
            "evidence_sources": [source],
            "copied_or_restored": False,
        }

    for row in data.values():
        add_asset(
            row["path"], "DATA_JSONL",
            row["actual_file_bytes_sha256"],
            "C15_FILE_BYTES_CHECK",
            "NB22-C15",
        )
    for row in c19["artifact"]["results"]:
        add_asset(
            row["path"], "PHYSICAL_INDEX_FILE",
            row["actual_file_bytes_sha256"],
            "C19_FILE_BYTES_CHECK",
            "NB22-C19",
        )
    for path, row in metadata.items():
        if path.endswith(".json"):
            add_asset(
                path, "VERIFIED_METADATA",
                row["file_bytes_sha256"],
                "PREVIOUS_BOUND_METADATA_BYTE_CHECK",
                "NB22_CACHED_METADATA_RECEIPTS",
            )

    dense_release_path = (
        "artifacts/phase2/dense_index/dense_index_release_v1.json"
    )
    dense_release = payloads[dense_release_path]
    snapshot = dense_release["snapshot"]
    snapshot_relative = Path(snapshot["path"])
    if snapshot_relative.is_absolute() or ".." in snapshot_relative.parts:
        raise RuntimeError("SNAPSHOT_PATH_REQUIRES_ADAPTER")
    snapshot_path = (root / snapshot_relative).resolve()
    snapshot_path.relative_to(root)
    snapshot_inventory = {
        "path": snapshot["path"],
        "declared_sha256": snapshot["sha256"],
        "exists_at_explicit_path": snapshot_path.is_file(),
        "observed_bytes": (
            snapshot_path.stat().st_size if snapshot_path.is_file() else None
        ),
        "snapshot_bytes_hashed_this_cell": False,
        "restore_executed_this_cell": False,
        "historical_restore_status": snapshot.get("restore_status"),
    }

    embedding = payloads[
        "artifacts/phase2/embedding/embedding_release_v1.json"
    ]
    reranker = payloads[
        "artifacts/phase2/reranking/reranker_configuration_v1.json"
    ]
    model_inventory = {
        "embedding": {
            "model_id": embedding["embedding"]["model_id"],
            "revision": embedding["embedding"]["revision"],
            "cache_presence_verified": False,
        },
        "reranker": {
            "model_id": reranker["reranker"]["model_id"],
            "revision": reranker["reranker"]["revision"],
            "runtime": reranker["runtime"],
            "cache_presence_verified": False,
        },
        "generation": {
            "recorded_development_prompt":
                c13["artifact"]["development_prompt"],
            "live_runtime_or_model_cache_verified": False,
            "test_recovery_kept_separate": True,
        },
    }

    artifact = {
        "artifact_type": "NB22_VALIDATION_COVERAGE_AND_RESTORE_INVENTORY",
        "schema_version": "NB22_C024_ACTION10_11_INVENTORY_V1",
        "status": (
            "REUSED_VALIDATION_BOUND_AND_INITIAL_RESTORE_ASSETS_LISTED"
            if reuse_checks_pass else
            "RECORDED_VALIDATION_SCOPE_REQUIRES_REVIEW"
        ),
        "historical_to_current_identity_checks": bindings,
        "recorded_validation_results": observed_validation,
        "recorded_rank_score_checks_reusable": reuse_checks_pass,
        "current_reference_integrity_receipt": c22["output"],
        "reload_evidence_receipt": c23["output"],
        "full_release_schema_validation_established": False,
        "action10_full_release_completion_established": False,
        "restore_file_inventory": list(assets.values()),
        "snapshot_inventory": snapshot_inventory,
        "model_inventory": model_inventory,
        "remaining_restore_requirements": [
            "Bind selected NB21 response artifacts and complete product policy.",
            "Bind producer code and a reproducible environment.",
            "Verify model/cache locations and external dependencies.",
            "Determine required embedding-cache closure from producer manifests.",
            "Verify snapshot bytes before any restore.",
            "Run actual isolated restore/validate-only in its designated action.",
        ],
        "action11_complete": False,
        "release_acceptance_granted": False,
        "tests_reexecuted": False,
        "data_files_reread": 0,
        "new_api_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C024.action10_11",
        "C024.validation_restore_inventory", artifact,
    )
    print(json.dumps({
        "status": artifact["status"],
        "historical_to_current_identity_checks": bindings,
        "recorded_validation_results": observed_validation,
        "recorded_rank_score_checks_reusable": reuse_checks_pass,
        "restore_unique_file_count": len(assets),
        "snapshot_inventory": snapshot_inventory,
        "model_inventory": model_inventory,
        "remaining_restore_requirements":
            artifact["remaining_restore_requirements"],
        "output": output,
        "full_release_schema_validation_established": False,
        "action11_complete": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {
        "artifact": artifact,
        "output": output,
        "payloads": payloads,
        "source_bindings": metadata,
    }


nb22_c24_result = nb22_c24_run(
    nb22_c13_result, nb22_c15_result, nb22_c16_result,
    nb22_c18_result, nb22_c19_result, nb22_c22_result,
    nb22_c23_result
)

{
  "status": "REUSED_VALIDATION_BOUND_AND_INITIAL_RESTORE_ASSETS_LISTED",
  "historical_to_current_identity_checks": [
    {
      "check": "reranker_output_identity",
      "recorded": "d00285bbf3865fbd51041db1612b9fdc17e9778273ea6d136978906a07268b7a",
      "verified_current": "d00285bbf3865fbd51041db1612b9fdc17e9778273ea6d136978906a07268b7a",
      "status": "MATCH"
    },
    {
      "check": "reranker_payload_configuration_identity",
      "recorded": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738",
      "verified_current": "8924c47c96bc52080ac0306daf272185a0c7c8eeb07d15a963058530badfa738",
      "status": "MATCH"
    },
    {
      "check": "context_configuration_identity",
      "recorded": "f600583bba6646022c3eaf455e204a38a6c6f5bd3fe8aae8621be8d5d35a5835",
      "verified_current": "f600583bba6646022c3eaf455e204a38a6c6f5bd3fe8aae8621be8d5d35a5835",
      "status": "MATCH"
    },
    {
      "check": "context_input:candidates",
      "recorded": "d00285bbf38

<div dir="rtl" align="right">

### NB22-C25 — اقدام ۱۱: Snapshot و Cacheهای موردنیاز

چهار زیرکار:

۱. کنترل SHA256 بایت Snapshot موجود، بدون استخراج یا Restore.
۲. موجودی فایل‌های NPZ در ریشهٔ Cache مستند Embedding.
۳. بررسی Snapshot محلی مدل‌های مشخص در Revision ثبت‌شده
و کنترل بایت ONNX ریرنکر، در صورت وجود.
۴. ذخیرهٔ وضعیت وابستگی‌ها با تفکیک وجود، تطبیق و احرازنشدن.

فقط Cache همان مدل‌ها و مسیر مستند پروژه بررسی می‌شود.
مکان پیش‌فرض Hugging Face صرفاً یکی از محل‌های بررسی است؛
نبود فایل در آن، نبود Cache در کل سیستم را ثابت نمی‌کند.

وجود پوشهٔ Revision، صحت تمام فایل‌های مدل را ثابت نمی‌کند.
شمارش NPZ نیز جای کنترل نگاشت بردارها یا Inventory hash را نمی‌گیرد.
مدل مولد زنده از نام مدل ثبت‌شده یا خروجی Recovery نتیجه نمی‌شود.

Snapshot فقط هش‌گیری می‌شود؛ آزمون Restore در اقدام مربوط انجام خواهد شد.
شبکه، دانلود، اجرای مدل، ساخت Cache و تغییر چک‌لیست: صفر.

</div>

In [26]:
# NB22-C25 — Action 11: offline snapshot and targeted cache inventory
# Scope: explicit project cache + exact HF model/revision locations only.

def nb22_c25_run(previous):
    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()
    payloads = previous["payloads"]
    inventory = previous["artifact"]

    def signature(stat):
        return (
            stat.st_dev, stat.st_ino, stat.st_size,
            stat.st_mtime_ns, stat.st_ctime_ns,
        )

    def stream_hash(path):
        with path.open("rb") as stream:
            before = os.fstat(stream.fileno())
            digest = hashlib.sha256()
            count = 0
            for block in iter(
                lambda: stream.read(4 * 1024 * 1024), b""
            ):
                digest.update(block)
                count += len(block)
            after = os.fstat(stream.fileno())
        stable = (
            signature(before) == signature(after)
            == signature(path.stat())
            and count == before.st_size
        )
        return digest.hexdigest(), count, stable

    snapshot = inventory["snapshot_inventory"]
    snapshot_path = (root / snapshot["path"]).resolve()
    snapshot_path.relative_to(root)
    snapshot_result = {
        "path": snapshot["path"],
        "expected_file_bytes_sha256": snapshot["declared_sha256"],
        "restore_executed": False,
    }
    if snapshot_path.is_file():
        actual, size, stable = stream_hash(snapshot_path)
        snapshot_result.update({
            "actual_file_bytes_sha256": actual,
            "bytes": size,
            "file_stable_during_read": stable,
            "status": (
                "SNAPSHOT_FILE_BYTES_MATCH"
                if stable and actual == snapshot["declared_sha256"]
                else "SNAPSHOT_RECONCILIATION_REQUIRED"
            ),
        })
    else:
        snapshot_result["status"] = "NOT_AVAILABLE_AT_EXPLICIT_PATH"

    embedding_release = payloads[
        "artifacts/phase2/embedding/embedding_release_v1.json"
    ]
    cache = embedding_release["cache"]
    cache_relative = Path(cache["root"])
    if cache_relative.is_absolute() or ".." in cache_relative.parts:
        raise RuntimeError("EMBEDDING_CACHE_PATH_REQUIRES_ADAPTER")
    cache_path = (root / cache_relative).resolve()
    cache_path.relative_to(root)

    npz_files = []
    if cache_path.is_dir():
        for path in sorted(cache_path.rglob("*.npz")):
            if not path.is_file():
                continue
            resolved = path.resolve()
            resolved.relative_to(cache_path)
            npz_files.append({
                "path": path.relative_to(root).as_posix(),
                "bytes": path.stat().st_size,
            })

    embedding_cache_result = {
        "root": cache["root"],
        "root_exists": cache_path.is_dir(),
        "expected_npz_count": cache["npz_file_count"],
        "observed_npz_count": len(npz_files),
        "count_matches": (
            cache_path.is_dir()
            and len(npz_files) == cache["npz_file_count"]
        ),
        "declared_inventory_sha256": cache["inventory_sha256"],
        "inventory_hash_recomputed": False,
        "npz_contents_validated": False,
        "npz_files": npz_files,
        "embedding_recomputed": False,
    }

    # Resolve standard HF cache locations without importing HF/model code.
    hf_locations = []
    explicit_hub = os.environ.get("HF_HUB_CACHE")
    if explicit_hub:
        hf_locations.append(("HF_HUB_CACHE", Path(explicit_hub)))
    hf_home = os.environ.get("HF_HOME")
    if hf_home:
        hf_locations.append(("HF_HOME_HUB", Path(hf_home) / "hub"))
    xdg_home = os.environ.get("XDG_CACHE_HOME")
    if xdg_home:
        hf_locations.append((
            "XDG_HF_HUB", Path(xdg_home) / "huggingface/hub"
        ))
    hf_locations.append((
        "DEFAULT_HF_HUB", Path.home() / ".cache/huggingface/hub"
    ))
    unique_locations = {}
    for label, location in hf_locations:
        unique_locations.setdefault(location.resolve(), label)

    model_results = []
    for role in ("embedding", "reranker"):
        model = inventory["model_inventory"][role]
        model_id, revision = model["model_id"], model["revision"]
        if not re.fullmatch(r"[0-9a-f]{40}", revision):
            raise RuntimeError("EXACT_MODEL_REVISION_NOT_SPECIFIED")
        repo_directory = "models--" + model_id.replace("/", "--")

        locations = []
        for hub_root, label in unique_locations.items():
            snapshot_relative = (
                Path(repo_directory) / "snapshots" / revision
            )
            model_snapshot = hub_root / snapshot_relative
            entry = {
                "cache_location_kind": label,
                "snapshot_relative_path": snapshot_relative.as_posix(),
                "snapshot_directory_exists": model_snapshot.is_dir(),
                "full_model_integrity_verified": False,
            }
            if model_snapshot.is_dir():
                entry["top_level_entries"] = sorted(
                    path.name for path in model_snapshot.iterdir()
                )
                if role == "reranker":
                    runtime = model["runtime"]
                    filename = Path(runtime["onnx_filename"])
                    if filename.is_absolute() or ".." in filename.parts:
                        raise RuntimeError("INVALID_ONNX_RELATIVE_PATH")
                    onnx_path = model_snapshot / filename
                    entry["onnx_relative_path"] = filename.as_posix()
                    entry["onnx_exists"] = onnx_path.is_file()
                    if onnx_path.is_file():
                        actual, size, stable = stream_hash(onnx_path)
                        expected = runtime["onnx_model_sha256"]
                        entry["onnx_byte_check"] = {
                            "expected_sha256": expected,
                            "actual_file_bytes_sha256": actual,
                            "bytes": size,
                            "file_stable_during_read": stable,
                            "status": (
                                "MATCH" if stable and actual == expected
                                else "RECONCILIATION_REQUIRED"
                            ),
                        }
            locations.append(entry)
        model_results.append({
            "role": role,
            "model_id": model_id,
            "revision": revision,
            "checked_locations": locations,
            "revision_directory_found": any(
                row["snapshot_directory_exists"] for row in locations
            ),
            "absence_projectwide_claimed": False,
            "model_loaded_or_downloaded": False,
        })

    artifact = {
        "artifact_type": "NB22_OFFLINE_RESTORE_DEPENDENCY_INVENTORY",
        "schema_version": "NB22_C025_ACTION11_CACHES_V1",
        "status": "SNAPSHOT_AND_TARGETED_CACHES_ACCOUNTED",
        "snapshot": snapshot_result,
        "embedding_cache": embedding_cache_result,
        "model_caches": model_results,
        "remaining_noncache_requirements":
            inventory["remaining_restore_requirements"],
        "complete_restore_closure_established": False,
        "action11_complete": False,
        "restore_executed": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C025.action11",
        "C025.offline_dependencies", artifact,
    )
    print(json.dumps({
        "status": artifact["status"],
        "snapshot": snapshot_result,
        "embedding_cache": {
            key: value for key, value in embedding_cache_result.items()
            if key != "npz_files"
        },
        "model_caches": model_results,
        "output": output,
        "complete_restore_closure_established": False,
        "restore_executed": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c25_result = nb22_c25_run(nb22_c24_result)

{
  "status": "SNAPSHOT_AND_TARGETED_CACHES_ACCOUNTED",
  "snapshot": {
    "path": "data/phase2/index/dense_index_v1/snapshots/dense_index_v1_qdrant_snapshot.tar.gz",
    "expected_file_bytes_sha256": "1b07eb4e2c8f02c30235e179456e01c026316ae985f1346fe96f4ae764e587a9",
    "restore_executed": false,
    "actual_file_bytes_sha256": "1b07eb4e2c8f02c30235e179456e01c026316ae985f1346fe96f4ae764e587a9",
    "bytes": 10311767,
    "file_stable_during_read": true,
    "status": "SNAPSHOT_FILE_BYTES_MATCH"
  },
  "embedding_cache": {
    "root": "artifacts/phase2/embedding_benchmark/b9ca18e710686ddca48a25c637840b716adb306717cafd8ec7c96ce9e182e6f2/multilingual_e5_small",
    "root_exists": true,
    "expected_npz_count": 90,
    "observed_npz_count": 90,
    "count_matches": true,
    "declared_inventory_sha256": "a3d29c5eadadca3a55c622a11271b6b0ffd492a1782e80a5704b0bdb3c5ea5f2",
    "inventory_hash_recomputed": false,
    "npz_contents_validated": false,
    "embedding_recomputed": false
  },
 

<div dir="rtl" align="right">

### NB22-C26 — اقدام ۱۲، بستهٔ اول: Manifest اولیهٔ کاندیدا

چهار زیرکار:

۱. مصرف موجودی ۸۴ فایل و شواهد هش قبلی، بدون خواندن دوبارهٔ آن‌ها.
۲. محاسبهٔ جریانی SHA256 بایت ۹۰ NPZ موجود برای ثبت هویت بسته.
۳. افزودن Snapshot کنترل‌شده و Locator مدل‌های محلی، با حدود اعتبارشان.
۴. ذخیرهٔ Manifest اولیه و بازخوانی مستقل.

NPZها فقط به‌عنوان فایل باینری خوانده می‌شوند؛
آرایه‌ها بارگذاری و Embeddingها تولید نمی‌شوند.
هش Inventory جدید NB22 با Inventory hash تولیدکننده یکی فرض نمی‌شود.

فایل‌های مدل فعلاً وابستگی خارجی با Locator نسخه‌دار هستند؛
بسته‌بندی کامل تمام وزن‌ها یا صحت کامل Cache آن‌ها ادعا نمی‌شود.

Release ID کلی اختراع نمی‌شود و وضعیت Manifest پیش‌نویس باقی می‌ماند.
موانع کیفیت و اتصال‌های باز حذف نمی‌شوند.
API، دانلود، مدل و تغییر چک‌لیست: صفر.

</div>

In [27]:
# NB22-C26 — Action 12: initial candidate manifest from existing receipts
# Scope: reuse verified assets; hash existing NPZ bytes only; no recomputation.

def nb22_c26_run(c13, c24, c25):
    dependencies = c25["artifact"]
    if dependencies["snapshot"]["status"] != "SNAPSHOT_FILE_BYTES_MATCH":
        raise RuntimeError("SNAPSHOT_IDENTITY_REQUIRES_RECONCILIATION")
    if not dependencies["embedding_cache"]["count_matches"]:
        raise RuntimeError("EMBEDDING_CACHE_COUNT_REQUIRES_RECONCILIATION")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    assets = {
        row["path"]: dict(row)
        for row in c24["artifact"]["restore_file_inventory"]
    }
    snapshot = dependencies["snapshot"]
    if (
        snapshot["path"] in assets
        and assets[snapshot["path"]]["file_bytes_sha256"]
            != snapshot["actual_file_bytes_sha256"]
    ):
        raise RuntimeError("SNAPSHOT_RECEIPT_CONFLICT")
    assets[snapshot["path"]] = {
        "path": snapshot["path"],
        "kind": "DENSE_INDEX_SNAPSHOT",
        "file_bytes_sha256": snapshot["actual_file_bytes_sha256"],
        "bytes": snapshot["bytes"],
        "verification_scope": "C25_SNAPSHOT_BYTE_CHECK",
        "copied_or_restored": False,
    }

    cache_files = dependencies["embedding_cache"]["npz_files"]
    expected_count = dependencies["embedding_cache"]["expected_npz_count"]
    if (
        len(cache_files) != expected_count
        or len({row["path"] for row in cache_files}) != expected_count
    ):
        raise RuntimeError("CACHE_INVENTORY_NOT_UNIQUE")

    total_bytes = sum(row["bytes"] for row in cache_files)
    print(json.dumps({
        "stage": "EXISTING_NPZ_BYTE_IDENTITY_START",
        "file_count": len(cache_files),
        "bytes_in_recorded_inventory": total_bytes,
        "embedding_recomputation": False,
    }, ensure_ascii=False), flush=True)

    def signature(stat):
        return (
            stat.st_dev, stat.st_ino, stat.st_size,
            stat.st_mtime_ns, stat.st_ctime_ns,
        )

    cache_receipts, errors = [], []
    for number, item in enumerate(cache_files, 1):
        relative = Path(item["path"])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("INVALID_CACHE_RELATIVE_PATH")
        path = (root / relative).resolve()
        path.relative_to(root)
        try:
            with path.open("rb") as stream:
                before = os.fstat(stream.fileno())
                digest = hashlib.sha256()
                count = 0
                for block in iter(
                    lambda: stream.read(4 * 1024 * 1024), b""
                ):
                    digest.update(block)
                    count += len(block)
                after = os.fstat(stream.fileno())
            if not (
                signature(before) == signature(after)
                == signature(path.stat())
                and count == before.st_size == item["bytes"]
            ):
                raise RuntimeError("CACHE_FILE_CHANGED_SINCE_INVENTORY")

            row = {
                "path": item["path"],
                "kind": "EXISTING_EMBEDDING_CACHE_NPZ",
                "file_bytes_sha256": digest.hexdigest(),
                "bytes": count,
                "verification_scope":
                    "NEW_BYTE_IDENTITY_RECORDED_NO_PRODUCER_PER_FILE_COMPARISON",
                "producer_expected_per_file_hash_bound": False,
                "array_contents_or_mapping_validated": False,
                "embedding_recomputed": False,
            }
            if (
                row["path"] in assets
                and assets[row["path"]]["file_bytes_sha256"]
                    != row["file_bytes_sha256"]
            ):
                raise RuntimeError("PRIOR_ASSET_HASH_CONFLICT")
            assets[row["path"]] = row
            cache_receipts.append(row)
        except Exception as exc:
            errors.append({
                "path": item["path"],
                "error_type": type(exc).__name__,
                "reason": (
                    str(exc) if isinstance(exc, RuntimeError)
                    else "LOCAL_CACHE_BYTE_READ_FAILED"
                ),
            })
        if number % 15 == 0 or number == len(cache_files):
            print(json.dumps({
                "stage": "NPZ_PROGRESS",
                "completed": number,
                "total": len(cache_files),
                "errors": len(errors),
            }, ensure_ascii=False), flush=True)

    # This is a new NB22 inventory identity, not the producer's inventory hash.
    inventory_payload = [
        {
            "path": row["path"],
            "file_bytes_sha256": row["file_bytes_sha256"],
            "bytes": row["bytes"],
        }
        for row in sorted(cache_receipts, key=lambda r: r["path"])
    ]
    new_inventory_sha = hashlib.sha256(json.dumps(
        inventory_payload, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")).hexdigest()

    artifact = {
        "artifact_type": "NB22_INITIAL_CANDIDATE_MANIFEST",
        "schema_version": "NB22_C026_ACTION12_INITIAL_MANIFEST_V1",
        "status": (
            "DRAFT_CANDIDATE_MANIFEST_WITH_OPEN_DEPENDENCIES"
            if not errors else "DRAFT_WITH_CACHE_READ_RECONCILIATION"
        ),
        "release_id": None,
        "release_id_assignment_status": "NOT_ASSIGNED",
        "scope": "KNOWN_LOCAL_ASSETS_AND_EXPLICIT_EXTERNAL_DEPENDENCIES",
        "component_versions": c13["artifact"]["component_versions"],
        "files": [assets[path] for path in sorted(assets)],
        "embedding_cache": {
            "file_count_with_new_byte_receipt": len(cache_receipts),
            "expected_npz_count": expected_count,
            "nb22_inventory_sha256": new_inventory_sha,
            "nb22_inventory_hash_definition":
                "SHA256_UTF8_CANONICAL_JSON_OF_SORTED_PATH_FILE_SHA256_BYTES_RECORDS",
            "producer_declared_inventory_sha256":
                dependencies["embedding_cache"]["declared_inventory_sha256"],
            "producer_inventory_hash_recomputed": False,
            "arrays_or_query_mapping_revalidated": False,
            "read_errors": errors,
        },
        "external_model_dependencies": dependencies["model_caches"],
        "generation_dependency":
            c24["artifact"]["model_inventory"]["generation"],
        "open_dependency_requirements":
            c24["artifact"]["remaining_restore_requirements"],
        "quality_acceptance": {
            "granted": False,
            "NB21_current_state":
                c13["artifact"]["component_versions"] is not None,
            "note":
                "Version inventory existence is not quality acceptance; prior FAIL/BLOCKED remain.",
        },
        "complete_restore_closure_established": False,
        "all_model_files_packaged": False,
        "action12_fully_complete": False,
        "restore_executed": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C026.action12",
        "C026.initial_candidate_manifest", artifact,
    )
    print(json.dumps({
        "status": artifact["status"],
        "manifest_file_count": len(assets),
        "prior_file_receipts_reused":
            len(c24["artifact"]["restore_file_inventory"]),
        "snapshot_receipt_reused": True,
        "npz_byte_identities_recorded": len(cache_receipts),
        "npz_read_errors": errors,
        "nb22_cache_inventory_sha256": new_inventory_sha,
        "producer_inventory_hash_recomputed": False,
        "external_model_dependency_count":
            len(dependencies["model_caches"]),
        "output": output,
        "release_id_assigned": False,
        "complete_restore_closure_established": False,
        "action12_fully_complete": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c26_result = nb22_c26_run(
    nb22_c13_result, nb22_c24_result, nb22_c25_result
)

{"stage": "EXISTING_NPZ_BYTE_IDENTITY_START", "file_count": 90, "bytes_in_recorded_inventory": 16771507, "embedding_recomputation": false}
{"stage": "NPZ_PROGRESS", "completed": 15, "total": 90, "errors": 0}
{"stage": "NPZ_PROGRESS", "completed": 30, "total": 90, "errors": 0}
{"stage": "NPZ_PROGRESS", "completed": 45, "total": 90, "errors": 0}
{"stage": "NPZ_PROGRESS", "completed": 60, "total": 90, "errors": 0}
{"stage": "NPZ_PROGRESS", "completed": 75, "total": 90, "errors": 0}
{"stage": "NPZ_PROGRESS", "completed": 90, "total": 90, "errors": 0}
{
  "status": "DRAFT_CANDIDATE_MANIFEST_WITH_OPEN_DEPENDENCIES",
  "manifest_file_count": 175,
  "prior_file_receipts_reused": 84,
  "snapshot_receipt_reused": true,
  "npz_byte_identities_recorded": 90,
  "npz_read_errors": [],
  "nb22_cache_inventory_sha256": "424b9f5d1b860fb921878a227c5a2a0959681ad0b55604292a7b76c21edfc1b6",
  "producer_inventory_hash_recomputed": false,
  "external_model_dependency_count": 2,
  "output": {
    "path": "rep

<div dir="rtl" align="right">

### NB22-C27 — اقدامات ۱۳ تا ۱۷: بستهٔ مستندات کاندیدا

پنج زیرکار این بسته:

۱. Data Card با جمعیت‌ها، دارایی‌ها و دامنهٔ اعتبار.
۲. Retrieval Card با نسخه‌های واقعی و سیاست Context.
۳. Evaluation Card با نتایج و Gate ثبت‌شدهٔ NB21.
۴. سند منشأ مدل‌ها و حدود اعتبار زمان، توکن و هزینه.
۵. Known limitations و Manifest مشتقِ متصل به این پنج سند.

داده، Metric، داوری یا مدل دوباره تولید نمی‌شود.
Cardها به شواهد و Manifest قبلی ارجاع می‌دهند.
Manifest مشتق به Cardها اشاره می‌کند؛ زنجیرهٔ هش حلقوی نمی‌شود.

این اسناد وضعیت کاندیدا را توضیح می‌دهند؛
آمادگی محصول، تکمیل همهٔ الزامات هر اقدام یا پذیرش Release اعلام نمی‌شود.
وضعیت واقعی کیفیت و وابستگی‌های باز در خروجی حفظ می‌شوند.

API، خواندن دوبارهٔ داده و تغییر چک‌لیست: صفر.

</div>

In [30]:
# NB22-C27 — Evidence-backed cards and corrected derived manifest
# Complete replacement: no multiline string literals or nested literal fences.
# Scope: cached evidence only; no model, API, metrics or input rereads.

def nb22_c27_run(c03, c13, c14, c22, c24, c26):
    if c26["output"]["reload_validation"] != "PASS":
        raise RuntimeError("C26_MANIFEST_RELOAD_NOT_PASSED")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    relative_directory = "reports/phase2/release/NB22.C027.cards"
    directory = root / relative_directory
    directory.mkdir(parents=True, exist_ok=True)

    summary = c03["summary"]["C085_current_summary"]
    required = (
        "overall_quality_gate_status", "metric_gate_status_counts",
        "distinct_open_issue_count", "action48_status",
    )
    if any(key not in summary for key in required):
        raise RuntimeError("CURRENT_NB21_SUMMARY_FIELDS_MISSING")

    versions = c13["artifact"]["component_versions"]
    counts = c22["artifact"]["file_record_counts"]
    splits = c22["artifact"]["context_split_counts"]
    payloads = c24["payloads"]
    parent = c26["output"]

    selection = payloads[
        "artifacts/phase2/context_builder/context_builder_configuration_v1.json"
    ]["selection_policy"]
    fusion = payloads[
        "artifacts/phase2/hybrid_retrieval/hybrid_retrieval_release_v1.json"
    ]["fusion"]

    summary_paths = [
        p for p in c03["records"]
        if Path(p).name.startswith("C085.report.")
    ]
    if len(summary_paths) != 1:
        raise RuntimeError("C085_REPORT_BINDING_NOT_UNIQUE")
    summary_path = summary_paths[0]
    summary_receipt = {
        "path": summary_path,
        "file_bytes_sha256":
            c03["records"][summary_path]["actual_sha256"],
    }

    def value(role):
        return versions[role]["value"]

    def json_block(obj):
        fence = chr(96) * 3
        return (
            fence + "json\n"
            + json.dumps(obj, ensure_ascii=False, indent=2)
            + "\n" + fence
        )

    def receipt_text(receipt):
        return (
            "- مسیر: " + receipt["path"] + "\n"
            "- SHA256 بایت فایل: " + receipt["file_bytes_sha256"]
        )

    def card(title, paragraphs, receipts):
        parts = [
            '<div dir="rtl" align="right">',
            "# " + title,
            "**وضعیت:** پیش‌نویس مستند به شواهد کاندیدا؛ پذیرش Release نیست.",
        ]
        parts.extend(paragraphs)
        parts.extend([
            "## شواهد مصرف‌شده",
            "\n\n".join(receipt_text(r) for r in receipts),
            "</div>",
        ])
        return "\n\n".join(parts) + "\n"

    table = [
        "| دارایی | رکورد مشاهده‌شده |",
        "| --- | ---: |",
    ]
    for role, label in (
        ("document_corpus", "Corpus"),
        ("active_chunks", "Chunk"),
        ("golden_dataset", "Golden"),
        ("reranked_candidates", "Reranked candidates"),
        ("context_packets", "Context packets"),
        ("context_evidence_registry", "Context evidence registry"),
    ):
        table.append(f"| {label} | {counts[role]} |")

    docs = {}

    docs["data_card"] = card(
        "Data Card — دارایی‌های دادهٔ کاندیدا",
        [
            "نسخهٔ Golden: " + value("golden_dataset")
            + "؛ فریز: " + value("golden_freeze") + ".",
            "\n".join(table),
            (
                f"Development برابر {splits['development']} و Test برابر "
                f"{splits['test']} است؛ این‌ها تقسیم جمعیت‌اند، نه صف تازهٔ API."
            ),
            "شناسه‌ها و اتصال‌های Document، Chunk، Query و Evidence در دامنهٔ C22 کنترل شده‌اند.",
            "شمارش Corpus مشاهده‌شده است؛ انتظار شمارش آن هنوز به Manifest مستقل متصل نشده است.",
            "شناسهٔ Evidence در Golden با رجیستری Context یکسان فرض نشده است.",
            "متن و انتساب تاریخی تغییر نکرده‌اند؛ فرادادهٔ مفقود با حدس تکمیل نمی‌شود.",
            "کنترل هش و اتصال ساختاری، صحت معنایی یا پذیرش کامل Schema را ثابت نمی‌کند.",
        ],
        [parent, c22["output"]],
    )

    docs["retrieval_card"] = card(
        "Retrieval Card — مسیر بازیابی مستند",
        [
            "Embedding: " + value("embedding_model")
            + "؛ Revision: " + value("embedding_revision") + ".",
            json_block({
                role: value(role)
                for role in (
                    "dense_index", "sparse_index", "hybrid",
                    "reranker", "context_builder",
                )
            }),
            "مدل Reranker: " + value("reranker_model")
            + "؛ Revision: " + value("reranker_revision") + ".",
            "تنظیمات Fusion ثبت‌شده:",
            json_block(fusion),
            "سیاست Context ثبت‌شده:",
            json_block(selection),
            "شواهد حفظ Rank/Score به فایل‌های فعلی متصل‌اند؛ آزمون‌های موفق دوباره اجرا نشده‌اند.",
            "محدودیت‌های parent expansion، semantic conflict و topic diversity با دامنهٔ اصلی حفظ شده‌اند.",
            "بهبود Retrieval جای پذیرش صحت پاسخ را نمی‌گیرد.",
        ],
        [parent, c24["output"]],
    )

    docs["evaluation_card"] = card(
        "Evaluation Card — نتیجهٔ ثبت‌شدهٔ NB21",
        [
            "وضعیت Gate کیفیت: " + summary["overall_quality_gate_status"],
            json_block(summary["metric_gate_status_counts"]),
            f"مسائل متمایز باز: {summary['distinct_open_issue_count']}.",
            "وضعیت اقدام ۴۸: " + summary["action48_status"],
            "دو FAIL اصلی، Recall@10 و nDCG@10 در برابر آستانه‌های فریز‌شده‌اند.",
            "کنترل عدم‌افت و آستانهٔ مطلق مستقل‌اند؛ BLOCKED خودکار PASS یا FAIL نمی‌شود.",
            "Test تاریخی قبلاً دیده شده است؛ Recovery ارزیابی محصول زندهٔ Qwen نام نمی‌گیرد.",
            "۳۲۲ مشخصات Regression، ۳۲۲ تست اجراشده یا شکست‌خورده نیست.",
            "هیچ Metric، داوری یا آستانه‌ای در تهیهٔ این Card تغییر یا دوباره محاسبه نشده است.",
        ],
        [parent, summary_receipt],
    )

    docs["model_runtime_provenance"] = card(
        "منشأ مدل، Runtime و حدود زمان و هزینه",
        [
            "Prompt ثبت‌شدهٔ Development:",
            json_block(c13["artifact"]["development_prompt"]),
            "نام مدل در Configuration، اثبات هویت Runtime نیست.",
            "منشأ Test Recovery جدا حفظ می‌شود:",
            json_block(c13["artifact"]["test_recovery"]),
            "Revision مدل‌های Embedding/Reranker در Cache محلی مشاهده شده است؛ صحت کامل و بسته‌بندی تمام فایل‌های مدل احراز نشده است.",
            "زمان Job تاریخی، latency محصول زنده نیست؛ هزینهٔ داور با هزینهٔ تولید پاسخ متفاوت است.",
            "مصرف ثبت‌نشده صفر فرض نمی‌شود؛ تعرفهٔ تاریخی، برآورد و صورتحساب جدا می‌مانند.",
            "عدد تازه‌ای برای زمان، توکن یا هزینه ساخته نشده است.",
            "Rubric ممیزی پسینی، Policy کامل محصول یا احتمال درستی پاسخ نیست.",
        ],
        [parent, c24["output"]],
    )

    docs["known_limitations"] = card(
        "Known limitations — موانع و حدود کاندیدا",
        [
            "وضعیت کیفیت: " + summary["overall_quality_gate_status"],
            f"{summary['distinct_open_issue_count']} مسئلهٔ متمایز باز حفظ شده است.",
            "### شکاف‌های نسخه",
            json_block(c13["artifact"]["explicit_gaps"]),
            "### اتصال‌های باز مسیر مرجع",
            json_block(c14["artifact"]["pending_bindings"]),
            "### الزامات باقی‌ماندهٔ Restore",
            json_block(c24["artifact"]["remaining_restore_requirements"]),
            "کنترل هش Snapshot، اجرای Restore جدید نیست؛ گزارش Restore تاریخی در دامنهٔ اصلی خودش حفظ می‌شود.",
            "اجرای پاک NB21 طبق تصمیم کاربر تا پس از تجمیع به تعویق افتاده است.",
            "نبود اتصال در ورودی بررسی، نبود کار در کل پروژه را ثابت نمی‌کند.",
            "همهٔ اصلاحات کیفیت به NB22 واگذار نشده‌اند.",
            "پذیرش کامل Schema، Policy، محصول، Release و فاز دوم احراز نشده است.",
        ],
        [parent, summary_receipt, c24["output"]],
    )

    def save_markdown(role, text):
        raw = text.encode("utf-8")
        digest = hashlib.sha256(raw).hexdigest()
        target = directory / f"C027.{role}.{digest}.md"

        if target.exists():
            if target.read_bytes() != raw:
                raise RuntimeError("EXISTING_CARD_CONFLICT")
            status = "REUSED_IDENTICAL"
        else:
            fd, name = tempfile.mkstemp(prefix=".C027.", dir=directory)
            temporary = Path(name)
            try:
                with os.fdopen(fd, "wb") as stream:
                    stream.write(raw)
                    stream.flush()
                    os.fsync(stream.fileno())
                if temporary.read_bytes() != raw:
                    raise RuntimeError("TEMPORARY_CARD_VALIDATION_FAILED")
                try:
                    os.link(temporary, target)
                    status = "CREATED"
                except FileExistsError:
                    if target.read_bytes() != raw:
                        raise RuntimeError("CONCURRENT_CARD_CONFLICT")
                    status = "REUSED_IDENTICAL"
            finally:
                temporary.unlink(missing_ok=True)

        saved = target.read_bytes()
        if saved != raw or hashlib.sha256(saved).hexdigest() != digest:
            raise RuntimeError("CARD_RELOAD_FAILED")
        return {
            "role": role,
            "path": target.relative_to(root).as_posix(),
            "kind": "DRAFT_DOCUMENTATION_CARD",
            "file_bytes_sha256": digest,
            "bytes": len(saved),
            "write_status": status,
            "reload_validation": "PASS",
        }

    cards = [save_markdown(role, text) for role, text in docs.items()]

    derived = json.loads(json.dumps(c26["artifact"]))
    derived["schema_version"] = "NB22_C027_DERIVED_CANDIDATE_MANIFEST_V1"
    derived["parent_manifest"] = {
        "path": parent["path"],
        "file_bytes_sha256": parent["file_bytes_sha256"],
    }
    derived["status"] = "DRAFT_CANDIDATE_WITH_CARDS_AND_OPEN_DEPENDENCIES"
    derived["quality_acceptance"] = {
        "granted": False,
        "overall_gate_status": summary["overall_quality_gate_status"],
        "metric_gate_status_counts": summary["metric_gate_status_counts"],
        "distinct_open_issue_count": summary["distinct_open_issue_count"],
        "action48_status": summary["action48_status"],
        "source": summary_receipt,
    }
    derived["documentation_cards"] = cards

    paths = {row["path"] for row in derived["files"]}
    if any(row["path"] in paths for row in cards):
        raise RuntimeError("CARD_PATH_ALREADY_IN_PARENT_MANIFEST")
    derived["files"].extend(cards)
    derived["documentation_is_full_release_acceptance"] = False

    output = nb22_save_json_artifact(
        root, relative_directory, "C027.derived_manifest", derived,
    )
    print(json.dumps({
        "status": derived["status"],
        "cards_written_or_reused": cards,
        "manifest_file_count": len(derived["files"]),
        "quality_acceptance": derived["quality_acceptance"],
        "parent_manifest_modified": False,
        "full_actions13_to17_acceptance_claimed": False,
        "output": output,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {
        "artifact": derived,
        "cards": cards,
        "output": output,
    }


nb22_c27_result = nb22_c27_run(
    nb22_c03_result, nb22_c13_result, nb22_c14_result,
    nb22_c22_result, nb22_c24_result, nb22_c26_result
)

{
  "status": "DRAFT_CANDIDATE_WITH_CARDS_AND_OPEN_DEPENDENCIES",
  "cards_written_or_reused": [
    {
      "role": "data_card",
      "path": "reports/phase2/release/NB22.C027.cards/C027.data_card.52dc1413c2dfc59ef0e227edac5f26f3562772351ff33d1efd15d0b6e9af98a8.md",
      "kind": "DRAFT_DOCUMENTATION_CARD",
      "file_bytes_sha256": "52dc1413c2dfc59ef0e227edac5f26f3562772351ff33d1efd15d0b6e9af98a8",
      "bytes": 1828,
      "write_status": "CREATED",
      "reload_validation": "PASS"
    },
    {
      "role": "retrieval_card",
      "path": "reports/phase2/release/NB22.C027.cards/C027.retrieval_card.40642e68a684d46a8b826651fd4901aa44804d421944750bc39c8320b960e354.md",
      "kind": "DRAFT_DOCUMENTATION_CARD",
      "file_bytes_sha256": "40642e68a684d46a8b826651fd4901aa44804d421944750bc39c8320b960e354",
      "bytes": 2113,
      "write_status": "CREATED",
      "reload_validation": "PASS"
    },
    {
      "role": "evaluation_card",
      "path": "reports/phase2/release/NB22.C02

<div dir="rtl" align="right">

### NB22-C28 — اقدام ۱۸: Evidence index و یادداشت نسخه

سه زیرکار:

۱. ساخت Evidence index برای ۱۸۰ فایلِ Manifest موجود.
۲. ثبت یادداشت نسخه، شواهد اضافه‌شده و موانع باقی‌مانده.
۳. ثبت وضعیت سازگاری و Rollback، بدون ادعای آزمون انجام‌نشده.

هش فایل‌های قبلی دوباره محاسبه نمی‌شود؛ رسیدهای موجود مصرف می‌شوند.
نوع شاهد، دامنهٔ بررسی و اعتبار آن حفظ می‌شود.
هویت تازهٔ ثبت‌شدهٔ NPZ با تطبیق مستقلِ Inventory تولیدکننده یکی نیست.

Index و یادداشت‌ها به Manifest قبلی اشاره می‌کنند.
Manifest مشتق به خروجی‌های جدید اشاره می‌کند؛ حلقهٔ هش ساخته نمی‌شود.

این اقدام مستندسازی است؛ Restore، Rollback یا پذیرش محصول اجرا نمی‌شود.
API، خواندن دوبارهٔ ورودی و تغییر چک‌لیست: صفر.

</div>

In [31]:
# NB22-C28 — Action 18: evidence index, release notes and rollback status
# Scope: existing manifest receipts; no input rereads or compatibility tests.

def nb22_c28_run(previous, c13, c24, c25):
    if previous["output"]["reload_validation"] != "PASS":
        raise RuntimeError("C27_MANIFEST_RELOAD_NOT_PASSED")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    parent = previous["output"]
    manifest = previous["artifact"]
    files = manifest["files"]
    paths = [row["path"] for row in files]
    if len(paths) != len(set(paths)):
        raise RuntimeError("DUPLICATE_MANIFEST_FILE_PATHS")

    entries = []
    kind_counts = {}
    for row in files:
        path, digest = row.get("path"), row.get("file_bytes_sha256")
        if (
            not isinstance(path, str)
            or Path(path).is_absolute()
            or ".." in Path(path).parts
            or not isinstance(digest, str)
            or not re.fullmatch(r"[0-9a-f]{64}", digest)
        ):
            raise RuntimeError("MANIFEST_ENTRY_BINDING_INVALID")
        kind = row.get("kind", "UNSPECIFIED_BY_PARENT_ENTRY")
        kind_counts[kind] = kind_counts.get(kind, 0) + 1
        entries.append({
            "path": path,
            "file_bytes_sha256": digest,
            "kind": kind,
            "verification_scope": row.get(
                "verification_scope",
                "DOCUMENT_RELOAD_VALIDATED"
                if row.get("reload_validation") == "PASS"
                else "REFER_TO_PARENT_RECEIPT",
            ),
            "producer_expected_per_file_hash_bound":
                row.get("producer_expected_per_file_hash_bound"),
            "evidence_sources": row.get("evidence_sources", []),
            "semantic_acceptance_inferred": False,
        })

    parent_binding = {
        "path": parent["path"],
        "file_bytes_sha256": parent["file_bytes_sha256"],
    }
    compatibility = [
        {
            "area": "previous_accepted_product_release",
            "status": "NOT_BOUND_IN_CURRENT_CANDIDATE",
            "rollback_target_selected": False,
        },
        {
            "area": "request_response_backward_compatibility",
            "status": "NOT_TESTED_HERE",
            "responsible_stage": "NB23_CONTRACT_WORK_AFTER_REQUIRED_INPUTS",
        },
        {
            "area": "data_index_model_version_compatibility",
            "status": "PARTIAL_PROVENANCE_BINDINGS_AVAILABLE",
            "full_runtime_compatibility_established": False,
        },
        {
            "area": "isolated_restore",
            "status": "CURRENT_NB22_RESTORE_NOT_EXECUTED",
            "historical_component_restore_evidence_preserved": True,
        },
        {
            "area": "model_and_external_dependency_portability",
            "status": "TARGETED_LOCAL_CACHES_OBSERVED_CLOSURE_INCOMPLETE",
            "full_model_packaging_verified": False,
        },
    ]

    evidence_index = {
        "artifact_type": "NB22_CANDIDATE_EVIDENCE_INDEX",
        "schema_version": "NB22_C028_ACTION18_EVIDENCE_INDEX_V1",
        "status": "DRAFT_EVIDENCE_INDEX_WITH_LIMITATIONS",
        "parent_manifest": parent_binding,
        "file_count": len(entries),
        "kind_counts": kind_counts,
        "entries": sorted(entries, key=lambda row: row["path"]),
        "quality_state": manifest["quality_acceptance"],
        "version_gaps": c13["artifact"]["explicit_gaps"],
        "restore_requirements":
            c24["artifact"]["remaining_restore_requirements"],
        "producer_embedding_inventory_hash_recomputed": False,
        "compatibility_assessments": compatibility,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }

    relative_directory = "reports/phase2/release/NB22.C028.action18"
    index_output = nb22_save_json_artifact(
        root, relative_directory, "C028.evidence_index", evidence_index,
    )
    compatibility_output = nb22_save_json_artifact(
        root, relative_directory, "C028.compatibility_status",
        {
            "artifact_type": "NB22_COMPATIBILITY_STATUS",
            "schema_version": "NB22_C028_COMPATIBILITY_V1",
            "parent_manifest": parent_binding,
            "assessments": compatibility,
            "compatibility_tests_executed": False,
            "rollback_executed": False,
            "rollback_approved": False,
        },
    )

    quality = manifest["quality_acceptance"]
    lines = [
        '<div dir="rtl" align="right">',
        "# یادداشت کاندیدا — NB22",
        "**وضعیت:** پیش‌نویس همراه وابستگی‌ها و موانع باز.",
        "Release ID کلی هنوز تعیین نشده است.",
        "## تغییر این بسته",
        f"- Evidence index برای {len(entries)} فایل موجود ساخته شد.",
        "- پنج Card قبلی و حدود شواهد آن‌ها حفظ شدند.",
        "- وضعیت سازگاری و Rollback ثبت شد؛ آزمون تازه‌ای انجام نشد.",
        "## وضعیت کیفیت",
        "- Gate: " + quality["overall_gate_status"],
        "- شمار وضعیت‌ها: " + json.dumps(
            quality["metric_gate_status_counts"], ensure_ascii=False
        ),
        f"- مسائل متمایز باز: {quality['distinct_open_issue_count']}",
        "- اقدام ۴۸: " + quality["action48_status"],
        "## محدودیت‌ها",
        "- هویت بایت NPZها ثبت شده؛ Inventory hash تولیدکننده بازتولید نشده است.",
        "- نسخهٔ کد تولیدکننده و Policy کامل هنوز اتصال نهایی ندارند.",
        "- پاسخ منتخب NB21 و تعریف هش Split/Gold هنوز نیازمند اتصال‌اند.",
        "- مشاهدهٔ Cache مدل، صحت کامل و قابلیت حمل همهٔ فایل‌های مدل نیست.",
        "- Snapshot کنترل‌شده، جای اجرای Restore فعلی NB22 را نمی‌گیرد.",
        "- Rollback و سازگاری قرارداد آزموده‌شده اعلام نمی‌شوند.",
        "## شواهد",
        "- Manifest مبنا: " + parent["path"],
        "- SHA256 بایت Manifest مبنا: " + parent["file_bytes_sha256"],
        "- Evidence index: " + index_output["path"],
        "- وضعیت سازگاری: " + compatibility_output["path"],
        "API، مدل، Restore، Rollback و تغییر چک‌لیست در این بسته: صفر.",
        "</div>",
    ]
    notes_raw = ("\n\n".join(lines) + "\n").encode("utf-8")
    notes_digest = hashlib.sha256(notes_raw).hexdigest()
    directory = root / relative_directory
    target = directory / f"C028.release_notes.{notes_digest}.md"

    if target.exists():
        if target.read_bytes() != notes_raw:
            raise RuntimeError("EXISTING_RELEASE_NOTES_CONFLICT")
        notes_status = "REUSED_IDENTICAL"
    else:
        fd, name = tempfile.mkstemp(prefix=".C028.", dir=directory)
        temporary = Path(name)
        try:
            with os.fdopen(fd, "wb") as stream:
                stream.write(notes_raw)
                stream.flush()
                os.fsync(stream.fileno())
            if temporary.read_bytes() != notes_raw:
                raise RuntimeError("TEMPORARY_NOTES_VALIDATION_FAILED")
            try:
                os.link(temporary, target)
                notes_status = "CREATED"
            except FileExistsError:
                if target.read_bytes() != notes_raw:
                    raise RuntimeError("CONCURRENT_NOTES_CONFLICT")
                notes_status = "REUSED_IDENTICAL"
        finally:
            temporary.unlink(missing_ok=True)
    saved = target.read_bytes()
    if saved != notes_raw or hashlib.sha256(saved).hexdigest() != notes_digest:
        raise RuntimeError("RELEASE_NOTES_RELOAD_FAILED")

    notes_output = {
        "path": target.relative_to(root).as_posix(),
        "file_bytes_sha256": notes_digest,
        "bytes": len(saved),
        "write_status": notes_status,
        "reload_validation": "PASS",
    }

    outputs = [
        ("evidence_index", index_output),
        ("compatibility_status", compatibility_output),
        ("release_notes", notes_output),
    ]
    derived = json.loads(json.dumps(manifest))
    derived["schema_version"] = "NB22_C028_DERIVED_MANIFEST_V1"
    derived["parent_manifest"] = parent_binding
    derived["status"] = "DRAFT_WITH_EVIDENCE_INDEX_AND_OPEN_DEPENDENCIES"
    derived["documentation_package"] = {
        role: output for role, output in outputs
    }
    for role, output in outputs:
        if output["path"] in set(paths):
            raise RuntimeError("NEW_DOCUMENT_ALREADY_IN_PARENT")
        derived["files"].append({
            "path": output["path"],
            "file_bytes_sha256": output["file_bytes_sha256"],
            "bytes": output["bytes"],
            "kind": "CANDIDATE_DOCUMENTATION",
            "role": role,
            "verification_scope": "WRITTEN_AND_INDEPENDENTLY_RELOADED",
        })
    derived["compatibility_tests_executed"] = False
    derived["rollback_approved"] = False

    manifest_output = nb22_save_json_artifact(
        root, relative_directory, "C028.derived_manifest", derived,
    )
    print(json.dumps({
        "status": derived["status"],
        "indexed_parent_files": len(entries),
        "manifest_file_count": len(derived["files"]),
        "new_document_outputs": {
            role: output for role, output in outputs
        },
        "compatibility_assessments": compatibility,
        "output": manifest_output,
        "release_acceptance_granted": False,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {
        "artifact": derived,
        "evidence_index": evidence_index,
        "output": manifest_output,
    }


nb22_c28_result = nb22_c28_run(
    nb22_c27_result, nb22_c13_result,
    nb22_c24_result, nb22_c25_result
)

{
  "status": "DRAFT_WITH_EVIDENCE_INDEX_AND_OPEN_DEPENDENCIES",
  "indexed_parent_files": 180,
  "manifest_file_count": 183,
  "new_document_outputs": {
    "evidence_index": {
      "path": "reports/phase2/release/NB22.C028.action18/C028.evidence_index.a3f1ecfcf8f6bfc1649d7e11beceefd1ba7c1c3c46bcdbc9b66a411ace39883f.json",
      "file_bytes_sha256": "a3f1ecfcf8f6bfc1649d7e11beceefd1ba7c1c3c46bcdbc9b66a411ace39883f",
      "bytes": 84049,
      "write_status": "CREATED",
      "reload_validation": "PASS"
    },
    "compatibility_status": {
      "path": "reports/phase2/release/NB22.C028.action18/C028.compatibility_status.1b611d5d918effb625ed7cfea7110a63eac19726a66d55539580e225d65fd747.json",
      "file_bytes_sha256": "1b611d5d918effb625ed7cfea7110a63eac19726a66d55539580e225d65fd747",
      "bytes": 1130,
      "write_status": "CREATED",
      "reload_validation": "PASS"
    },
    "release_notes": {
      "path": "reports/phase2/release/NB22.C028.action18/C028.release_notes.940b4b2b

<div dir="rtl" align="right">

### NB22-C29 — اقدام ۱۹: برنامهٔ اجرای امن و محیط Restore

چهار زیرکار:

۱. ثبت Python، سیستم‌عامل و نسخهٔ Packageهای مرتبط بدون بارگذاری مدل.
۲. مقایسهٔ نسخهٔ Qdrant و Runtime ثبت‌شده با محیط فعلی.
۳. بررسی وجود و اندازهٔ فایل‌های Manifest؛ بدون هش‌گیری مجدد.
۴. ذخیرهٔ برنامهٔ Restore/Validate-only مستقل با محدودیت‌های صریح.

فایل‌های dependency فقط از ریشهٔ پروژه بررسی و هش می‌شوند؛
محتوای آن‌ها، URL خصوصی، Secret یا متغیرهای محیطی نمایش داده نمی‌شوند.
هیچ نصب یا دانلودی انجام نمی‌شود.

Restore بستهٔ کاندیدا از Restore کامل محصول پذیرفته‌شده جداست.
مقصد اجرای بعدی باید پوشهٔ موقت مستقل باشد؛
Index اصلی باز یا جایگزین نمی‌شود.
در نبود ورودی، Cache یا ابزار لازم، اجرای وابسته متوقف می‌شود.

این سل فقط برنامه و وضعیت آمادگی را ثبت می‌کند.
API، شبکه، مدل، Restore و Run All تاریخی: صفر.

</div>

In [32]:
# NB22-C29 — Action 19: offline runtime facts and bounded restore plan
# Scope: package metadata, explicit asset stats and root dependency files.

import sys
import platform
import importlib.metadata


def nb22_c29_run(c24, c25, c28):
    if c28["output"]["reload_validation"] != "PASS":
        raise RuntimeError("C28_MANIFEST_RELOAD_NOT_PASSED")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()
    payloads = c24["payloads"]
    manifest = c28["artifact"]

    distributions = (
        "qdrant-client", "onnxruntime", "sentence-transformers",
        "transformers", "tokenizers", "numpy", "jsonschema",
    )
    installed = {}
    for name in distributions:
        try:
            installed[name] = {
                "status": "INSTALLED",
                "version": importlib.metadata.version(name),
            }
        except importlib.metadata.PackageNotFoundError:
            installed[name] = {
                "status": "NOT_FOUND_IN_CURRENT_PYTHON_ENVIRONMENT",
                "version": None,
            }

    dense = payloads[
        "artifacts/phase2/dense_index/dense_index_release_v1.json"
    ]
    reranker = payloads[
        "artifacts/phase2/reranking/reranker_configuration_v1.json"
    ]
    expected_versions = {
        "qdrant-client": dense["backend"]["qdrant_client_version"],
        "onnxruntime": reranker["runtime"]["onnxruntime_version"],
    }
    comparisons = []
    for name, expected in expected_versions.items():
        actual = installed[name]["version"]
        comparisons.append({
            "distribution": name,
            "producer_recorded_version": expected,
            "current_version": actual,
            "status": (
                "NOT_INSTALLED" if actual is None else
                "MATCH" if actual == expected else
                "VERSION_DIFFERENCE_REQUIRES_SCOPE_REVIEW"
            ),
            "environment_modified": False,
        })

    dependency_paths = set()
    for pattern in ("requirements*.txt", "requirements*.in"):
        dependency_paths.update(
            path for path in root.glob(pattern) if path.is_file()
        )
    for name in (
        "pyproject.toml", "uv.lock", "poetry.lock",
        "requirements.lock", "Pipfile.lock",
        "environment.yml", "environment.yaml",
    ):
        path = root / name
        if path.is_file():
            dependency_paths.add(path)

    dependencies = []
    for path in sorted(dependency_paths):
        raw = path.read_bytes()
        dependencies.append({
            "path": path.relative_to(root).as_posix(),
            "file_bytes_sha256": hashlib.sha256(raw).hexdigest(),
            "bytes": len(raw),
            "contents_not_printed": True,
            "reproducible_environment_proven_by_presence": False,
        })

    paths = [row["path"] for row in manifest["files"]]
    if len(paths) != len(set(paths)):
        raise RuntimeError("DUPLICATE_CANDIDATE_ASSET_PATHS")

    asset_issues = []
    observed_total_bytes = 0
    for row in manifest["files"]:
        relative = Path(row["path"])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("UNSAFE_MANIFEST_PATH")
        path = (root / relative).resolve()
        path.relative_to(root)
        if not path.is_file():
            asset_issues.append({
                "path": row["path"],
                "reason": "NOT_AVAILABLE_AT_EXPLICIT_PATH",
            })
            continue
        size = path.stat().st_size
        observed_total_bytes += size
        if "bytes" in row and size != row["bytes"]:
            asset_issues.append({
                "path": row["path"],
                "reason": "SIZE_CHANGED_FROM_MANIFEST",
                "recorded_bytes": row["bytes"],
                "observed_bytes": size,
            })

    snapshot = c25["artifact"]["snapshot"]
    snapshot_ready = (
        snapshot["status"] == "SNAPSHOT_FILE_BYTES_MATCH"
        and snapshot["file_stable_during_read"]
    )
    qdrant_version_matches = any(
        row["distribution"] == "qdrant-client"
        and row["status"] == "MATCH"
        for row in comparisons
    )

    readiness = {
        "candidate_filesystem_restore_inputs_present": not asset_issues,
        "snapshot_previous_byte_receipt_valid": snapshot_ready,
        "recorded_qdrant_version_available": qdrant_version_matches,
        "full_product_runtime_ready": False,
        "actual_restore_completed": False,
    }

    plan = {
        "artifact_type": "NB22_SAFE_RESTORE_EXECUTION_PLAN",
        "schema_version": "NB22_C029_ACTION19_SAFE_PLAN_V1",
        "status": (
            "BOUNDED_OFFLINE_RESTORE_PLAN_PREPARED"
            if not asset_issues and snapshot_ready else
            "RESTORE_INPUT_RECONCILIATION_REQUIRED"
        ),
        "candidate_manifest": {
            "path": c28["output"]["path"],
            "file_bytes_sha256": c28["output"]["file_bytes_sha256"],
        },
        "current_runtime": {
            "python_version": sys.version,
            "python_implementation": platform.python_implementation(),
            "operating_system": platform.system(),
            "machine": platform.machine(),
            "package_versions": installed,
            "historical_producer_environment_claimed": False,
        },
        "producer_version_comparisons": comparisons,
        "root_dependency_files": dependencies,
        "candidate_asset_count": len(paths),
        "candidate_observed_bytes": observed_total_bytes,
        "asset_stat_issues": asset_issues,
        "readiness": readiness,
        "execution_contract": {
            "destination": "NEW_ISOLATED_TEMPORARY_DIRECTORY",
            "copy_only_explicit_manifest_paths": True,
            "validate_copied_bytes_against_manifest": True,
            "source_assets_immutable": True,
            "source_index_opened": False,
            "model_loading": False,
            "download_or_api": False,
            "missing_input_behavior": "STOP_DEPENDENT_STEP",
            "cache_miss_behavior": "STOP_NO_REGENERATION",
            "historical_notebook_run_all": False,
            "snapshot_archive_path_validation_required": True,
            "snapshot_extract_to_separate_directory": True,
            "qdrant_validation_scope":
                "ISOLATED_LOCAL_COPY_ONLY_IF_VERSION_AND_INPUTS_MATCH",
            "temporary_copy_cleanup_required": True,
            "durable_result_report_required": True,
        },
        "limits": [
            "Candidate filesystem validation is not full product restoration.",
            "Current runtime facts do not establish historical producer code.",
            "Dependency file presence is not a verified lockfile closure.",
            "Model cache observations do not prove complete portable model packaging.",
            "Quality FAIL/BLOCKED and existing provenance gaps remain unchanged.",
        ],
        "action19_full_reproducibility_established": False,
        "restore_executed": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C029.action19",
        "C029.safe_restore_plan", plan,
    )
    print(json.dumps({
        "status": plan["status"],
        "current_runtime": plan["current_runtime"],
        "producer_version_comparisons": comparisons,
        "root_dependency_files": dependencies,
        "candidate_asset_count": len(paths),
        "candidate_observed_mib":
            round(observed_total_bytes / 1024**2, 2),
        "asset_stat_issues": asset_issues,
        "readiness": readiness,
        "output": output,
        "restore_executed": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": plan, "output": output}


nb22_c29_result = nb22_c29_run(
    nb22_c24_result, nb22_c25_result, nb22_c28_result
)

{
  "status": "BOUNDED_OFFLINE_RESTORE_PLAN_PREPARED",
  "current_runtime": {
    "python_version": "3.11.3 (v3.11.3:f3909b8bc8, Apr  4 2023, 20:12:10) [Clang 13.0.0 (clang-1300.0.29.30)]",
    "python_implementation": "CPython",
    "operating_system": "Darwin",
    "machine": "x86_64",
    "package_versions": {
      "qdrant-client": {
        "status": "INSTALLED",
        "version": "1.18.0"
      },
      "onnxruntime": {
        "status": "INSTALLED",
        "version": "1.16.3"
      },
      "sentence-transformers": {
        "status": "INSTALLED",
        "version": "5.0.0"
      },
      "transformers": {
        "status": "INSTALLED",
        "version": "4.53.2"
      },
      "tokenizers": {
        "status": "INSTALLED",
        "version": "0.21.2"
      },
      "numpy": {
        "status": "INSTALLED",
        "version": "1.24.3"
      },
      "jsonschema": {
        "status": "INSTALLED",
        "version": "4.26.0"
      }
    },
    "historical_producer_environment_c

<div dir="rtl" align="right">

### NB22-C30 — اقدام ۲۰: Restore مستقل و Validate-only کاندیدا

چهار بخش:

۱. کپی ۱۸۳ فایل مشخص Manifest در مقصد موقت مستقل.
۲. کنترل SHA256 تمام کپی‌ها در یک فرایند مستقل Python.
۳. استخراج امن Snapshot در مقصد موقت دوم.
۴. مقایسهٔ تعداد، شناسه‌ها و نمونهٔ Payload/Vector میان
Index بازیابی‌شده و کپی موقت Index فعلی.

فقط کپی‌های موقت توسط Qdrant باز می‌شوند.
مسیر، محتوای آرشیو و اندازهٔ استخراج پیش از نوشتن کنترل می‌شوند.
شبکه در فرایند اعتبارسنجی مسدود می‌شود.
مدل، API، دانلود و Notebook تاریخی اجرا نمی‌شود.

در پایان، پوشهٔ موقت پاک و گزارش مستقل ذخیره می‌شود.
موفقیت این آزمون، پذیرش کیفیت یا تکمیل وابستگی‌های باز محصول نیست.
اجرای پاک NB21 همچنان مطابق تصمیم قبلی به بعد از تجمیع موکول است.

</div>

In [33]:
# NB22-C30 — Action 20: isolated candidate restore and validate-only
# Scope: temporary copies only; independent process; no model/API/network.

import shutil
import subprocess


def nb22_c30_run(c28, c29, c19, c24):
    plan = c29["artifact"]
    if not (
        plan["readiness"]["candidate_filesystem_restore_inputs_present"]
        and plan["readiness"]["snapshot_previous_byte_receipt_valid"]
        and plan["readiness"]["recorded_qdrant_version_available"]
    ):
        raise RuntimeError("C29_RESTORE_PREREQUISITES_NOT_SATISFIED")

    if (
        plan["candidate_manifest"]["file_bytes_sha256"]
        != c28["output"]["file_bytes_sha256"]
    ):
        raise RuntimeError("RESTORE_PLAN_MANIFEST_BINDING_DIFFERS")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    manifest = c28["artifact"]
    files = manifest["files"]
    dense = c24["payloads"][
        "artifacts/phase2/dense_index/dense_index_release_v1.json"
    ]
    dense_bytes = sum(
        row["bytes_read"] for row in c19["artifact"]["results"]
        if row["component"] == "dense"
    )
    source_bytes = sum((root / row["path"]).stat().st_size for row in files)

    worker = r'''
import sys, json, hashlib, socket, tarfile, shutil
from pathlib import Path
import importlib.metadata

blocked_network_attempts = []
def deny_network(*args, **kwargs):
    blocked_network_attempts.append("BLOCKED_BEFORE_CONNECT")
    raise RuntimeError("NETWORK_DISABLED_FOR_RESTORE")
socket.create_connection = deny_network
socket.socket.connect = deny_network
socket.socket.connect_ex = deny_network
socket.socket.sendto = deny_network

def require(condition, message):
    if not condition:
        raise RuntimeError(message)

def digest(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

request = json.load(sys.stdin)
base = Path.cwd().resolve()
bundle = base / "bundle"
restore = base / "snapshot_restore"
result = {
    "status": "RESTORE_VALIDATION_FAILED",
    "network_guard_enabled": True,
    "source_index_opened": False,
    "model_loaded": False,
}
clients = []
try:
    for row in request["files"]:
        relative = Path(row["path"])
        require(not relative.is_absolute() and ".." not in relative.parts,
                "UNSAFE_MANIFEST_PATH")
        path = (bundle / relative).resolve()
        path.relative_to(bundle)
        require(path.is_file(), "COPIED_ASSET_MISSING")
        require(digest(path) == row["file_bytes_sha256"],
                "COPIED_ASSET_HASH_MISMATCH")
    result["copied_files_hash_verified"] = len(request["files"])

    require(
        importlib.metadata.version("qdrant-client")
        == request["qdrant_version"],
        "QDRANT_VERSION_DIFFERS",
    )
    archive = bundle / request["snapshot_path"]
    require(digest(archive) == request["snapshot_sha256"],
            "COPIED_SNAPSHOT_HASH_MISMATCH")
    restore.mkdir()

    with tarfile.open(archive, "r:gz") as tar:
        members = tar.getmembers()
        require(len(members) <= 10000, "ARCHIVE_MEMBER_LIMIT")
        seen = set()
        regular_files = 0
        expanded_bytes = 0
        for member in members:
            name = member.name
            relative = Path(name)
            require(
                not relative.is_absolute()
                and ".." not in relative.parts
                and "\\" not in name,
                "UNSAFE_ARCHIVE_MEMBER_PATH",
            )
            require(member.isdir() or member.isfile(),
                    "ARCHIVE_LINK_OR_SPECIAL_FILE_NOT_ALLOWED")
            target = (restore / relative).resolve()
            target.relative_to(restore)
            require(str(target) not in seen, "DUPLICATE_ARCHIVE_MEMBER")
            seen.add(str(target))
            if member.isfile():
                regular_files += 1
                expanded_bytes += member.size
        require(
            regular_files == request["expected_snapshot_file_count"],
            "SNAPSHOT_FILE_COUNT_DIFFERS",
        )
        require(
            expanded_bytes <= request["expanded_bytes_limit"],
            "SNAPSHOT_EXPANDED_SIZE_LIMIT",
        )
        for member in members:
            target = (restore / member.name).resolve()
            if member.isdir():
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                source = tar.extractfile(member)
                require(source is not None, "ARCHIVE_FILE_UNREADABLE")
                with source, target.open("xb") as destination:
                    shutil.copyfileobj(source, destination)

    collection = request["collection"]
    restored_roots = [
        path.parent for path in restore.rglob("meta.json")
        if (path.parent / "collection" / collection).is_dir()
    ]
    require(len(restored_roots) == 1, "RESTORED_QDRANT_ROOT_NOT_UNIQUE")
    current_copy = bundle / request["dense_storage_path"]

    from qdrant_client import QdrantClient
    restored_client = QdrantClient(path=str(restored_roots[0]))
    clients.append(restored_client)
    copied_client = QdrantClient(path=str(current_copy))
    clients.append(copied_client)

    def ids(client):
        found = []
        offset = None
        while True:
            points, offset = client.scroll(
                collection_name=collection,
                limit=1000,
                offset=offset,
                with_payload=False,
                with_vectors=False,
            )
            found.extend(str(point.id) for point in points)
            if offset is None:
                break
        require(len(found) == len(set(found)), "DUPLICATE_POINT_IDS")
        return set(found)

    restored_ids = ids(restored_client)
    copied_ids = ids(copied_client)
    require(len(restored_ids) == request["point_count"],
            "RESTORED_POINT_COUNT_DIFFERS")
    require(restored_ids == copied_ids, "RESTORED_ID_PARITY_DIFFERS")

    info = restored_client.get_collection(collection)
    vectors = info.config.params.vectors
    require(getattr(vectors, "size", None) == request["dimension"],
            "RESTORED_VECTOR_DIMENSION_DIFFERS")
    distance = getattr(vectors, "distance", None)
    distance = getattr(distance, "value", str(distance))
    require(str(distance).lower() == request["distance"].lower(),
            "RESTORED_DISTANCE_DIFFERS")

    sample_ids = sorted(restored_ids)[:7]
    def records(client):
        return {
            str(point.id): point for point in client.retrieve(
                collection_name=collection,
                ids=sample_ids,
                with_payload=True,
                with_vectors=True,
            )
        }
    left, right = records(restored_client), records(copied_client)
    require(set(left) == set(sample_ids) == set(right),
            "SAMPLE_RETRIEVAL_INCOMPLETE")
    for point_id in sample_ids:
        require(left[point_id].payload == right[point_id].payload,
                "SAMPLE_PAYLOAD_DIFFERS")
        require(left[point_id].vector == right[point_id].vector,
                "SAMPLE_VECTOR_DIFFERS")

    result.update({
        "status": "ISOLATED_CANDIDATE_AND_DENSE_SNAPSHOT_RESTORE_PASS",
        "snapshot_files_restored": regular_files,
        "snapshot_expanded_bytes": expanded_bytes,
        "restored_point_count": len(restored_ids),
        "all_point_id_parity": True,
        "vector_dimension": vectors.size,
        "distance": str(distance),
        "sample_count": len(sample_ids),
        "sample_payload_parity": True,
        "sample_vector_parity": True,
        "full_vector_parity_checked": False,
        "retrieval_benchmark_executed": False,
    })
except Exception as exc:
    result["error_type"] = type(exc).__name__
    result["reason"] = str(exc).replace(str(base), "<TEMP_RESTORE_ROOT>")
finally:
    close_errors = []
    for client in reversed(clients):
        try:
            client.close()
        except Exception as exc:
            close_errors.append(type(exc).__name__)
    result["client_close_errors"] = close_errors
    if close_errors:
        result["status"] = "RESTORE_CLIENT_CLOSE_FAILED"
    result["blocked_network_attempt_count"] = len(blocked_network_attempts)
    if blocked_network_attempts:
        result["status"] = "UNEXPECTED_NETWORK_ATTEMPT_BLOCKED"
print(json.dumps(result))
'''

    started = time.monotonic()
    temporary_root = None
    execution = None
    copy_count = 0
    copy_errors = []

    with tempfile.TemporaryDirectory(prefix="nb22_restore_") as temporary:
        temporary_root = Path(temporary)
        required_space = source_bytes + dense_bytes + 64 * 1024 * 1024
        if shutil.disk_usage(temporary_root).free < required_space:
            raise RuntimeError("INSUFFICIENT_TEMPORARY_DISK_SPACE")
        bundle = temporary_root / "bundle"
        bundle.mkdir()

        print(json.dumps({
            "stage": "ISOLATED_RESTORE_START",
            "file_count": len(files),
            "copy_mib": round(source_bytes / 1024**2, 2),
            "source_index_will_be_opened": False,
        }, ensure_ascii=False), flush=True)

        for row in files:
            relative = Path(row["path"])
            if relative.is_absolute() or ".." in relative.parts:
                raise RuntimeError("UNSAFE_MANIFEST_PATH")
            source = (root / relative).resolve()
            source.relative_to(root)
            destination = bundle / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            before = source.stat()
            shutil.copyfile(source, destination)
            after = source.stat()
            if (
                before.st_size != after.st_size
                or before.st_mtime_ns != after.st_mtime_ns
                or before.st_ctime_ns != after.st_ctime_ns
                or before.st_ino != after.st_ino
            ):
                copy_errors.append(row["path"])
            copy_count += 1

        if copy_errors:
            execution = {
                "status": "SOURCE_CHANGED_DURING_COPY",
                "paths": copy_errors,
            }
        else:
            request = {
                "files": [
                    {
                        "path": row["path"],
                        "file_bytes_sha256": row["file_bytes_sha256"],
                    }
                    for row in files
                ],
                "snapshot_path": dense["snapshot"]["path"],
                "snapshot_sha256": dense["snapshot"]["sha256"],
                "expected_snapshot_file_count": len([
                    row for row in c19["artifact"]["results"]
                    if row["component"] == "dense"
                ]),
                "expanded_bytes_limit": dense_bytes + 1024 * 1024,
                "qdrant_version": dense["backend"]["qdrant_client_version"],
                "dense_storage_path": dense["backend"]["storage_path"],
                "collection": dense["backend"]["collection_name"],
                "point_count": dense["point_count"],
                "dimension": dense["vector"]["dimension"],
                "distance": dense["vector"]["distance"],
            }
            environment = {
                key: os.environ[key]
                for key in ("PATH", "LANG", "LC_ALL", "SYSTEMROOT")
                if key in os.environ
            }
            environment.update({
                "HF_HUB_OFFLINE": "1",
                "TRANSFORMERS_OFFLINE": "1",
                "TOKENIZERS_PARALLELISM": "false",
            })
            try:
                completed = subprocess.run(
                    [sys.executable, "-I", "-c", worker],
                    input=json.dumps(request),
                    text=True,
                    capture_output=True,
                    cwd=temporary_root,
                    env=environment,
                    timeout=180,
                    check=False,
                )
                if completed.returncode != 0:
                    execution = {
                        "status": "INDEPENDENT_PROCESS_FAILED",
                        "returncode": completed.returncode,
                        "stderr_excerpt": completed.stderr[-2000:].replace(
                            str(temporary_root), "<TEMP_RESTORE_ROOT>"
                        ),
                    }
                else:
                    execution = json.loads(completed.stdout)
            except subprocess.TimeoutExpired:
                execution = {"status": "INDEPENDENT_PROCESS_TIMEOUT"}

    cleanup_complete = not temporary_root.exists()
    artifact = {
        "artifact_type": "NB22_ISOLATED_CANDIDATE_RESTORE_RESULT",
        "schema_version": "NB22_C030_ACTION20_RESTORE_V1",
        "status": execution["status"],
        "scope": "CURRENT_CANDIDATE_FILES_AND_DENSE_SNAPSHOT_ONLY",
        "candidate_manifest": plan["candidate_manifest"],
        "execution_plan": c29["output"],
        "files_copied": copy_count,
        "independent_process_result": execution,
        "temporary_directory_removed": cleanup_complete,
        "elapsed_seconds": round(time.monotonic() - started, 3),
        "full_product_restore_complete": False,
        "whole_notebook_run_all_performed": False,
        "source_index_opened": False,
        "model_executed": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C030.action20",
        "C030.isolated_restore", artifact,
    )
    print(json.dumps({
        **artifact,
        "output": output,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c30_result = nb22_c30_run(
    nb22_c28_result, nb22_c29_result,
    nb22_c19_result, nb22_c24_result
)

{"stage": "ISOLATED_RESTORE_START", "file_count": 183, "copy_mib": 324.39, "source_index_will_be_opened": false}
{
  "artifact_type": "NB22_ISOLATED_CANDIDATE_RESTORE_RESULT",
  "schema_version": "NB22_C030_ACTION20_RESTORE_V1",
  "status": "ISOLATED_CANDIDATE_AND_DENSE_SNAPSHOT_RESTORE_PASS",
  "scope": "CURRENT_CANDIDATE_FILES_AND_DENSE_SNAPSHOT_ONLY",
  "candidate_manifest": {
    "path": "reports/phase2/release/NB22.C028.action18/C028.derived_manifest.7cdbfce63956037f9645163d8c2396661ccb5627634513bfb36a6980319f8a4e.json",
    "file_bytes_sha256": "7cdbfce63956037f9645163d8c2396661ccb5627634513bfb36a6980319f8a4e"
  },
  "execution_plan": {
    "path": "reports/phase2/release/NB22.C029.action19/C029.safe_restore_plan.4d0ebb2062c2a0cb31a7e365a04bd75b79869ce7fd137e0e6c2ffd465e5105fd.json",
    "file_bytes_sha256": "4d0ebb2062c2a0cb31a7e365a04bd75b79869ce7fd137e0e6c2ffd465e5105fd",
    "bytes": 3068,
    "write_status": "CREATED",
    "reload_validation": "PASS"
  },
  "files_copied": 1

<div dir="rtl" align="right">

### NB22-C31 — اقدام ۲۱: تعیین دامنهٔ تجمیع و اجرای پاک

چهار زیرکار:

۱. بررسی فایل‌های مشخص NB20، NB21 و NB22، بدون کشف دوبارهٔ مخزن.
۲. ثبت تعداد سل، Error output و Code cellهای اجرا‌نشدهٔ نسخهٔ ذخیره‌شده.
۳. تعیین بسته‌های تجمیع و شروط اجرای امن با حفظ شواهد تاریخی.
۴. ذخیرهٔ برنامه با ارجاع به نتیجهٔ Restore موفق C30.

این بررسی فقط ساختاری است؛ سلامت Runtime یا اجرای پاک را اثبات نمی‌کند.
شماره‌ها، خروجی‌ها و کدهای تاریخی تغییر نمی‌کنند.
خطای ذخیره‌شده، موفقیت بعدی را خودکار باطل نمی‌کند و باید با نسخهٔ نهایی تطبیق یابد.

Snapshot ذخیره‌شدهٔ NB22 ممکن است آخرین سل جاری را نداشته باشد؛
بنابراین این گزارش Final Health نیست.
هیچ Notebook تازه، نسخهٔ ادغام‌شده یا مسیر Cache-only جدید ساخته نمی‌شود.

API، مدل، تجمیع، حذف خروجی و Run All: صفر.

</div>

In [34]:
# NB22-C31 — Action 21: explicit consolidation and clean-execution scope
# Scope: static saved notebook inspection only; no cell execution or edits.

def nb22_c31_run(c03, c30):
    restore = c30["artifact"]
    if restore["status"] != (
        "ISOLATED_CANDIDATE_AND_DENSE_SNAPSHOT_RESTORE_PASS"
    ):
        raise RuntimeError("C30_RESTORE_RESULT_NOT_PASSED")
    if not restore["temporary_directory_removed"]:
        raise RuntimeError("C30_TEMPORARY_CLEANUP_NOT_CONFIRMED")

    roots = [
        p for p in (Path.cwd(), *Path.cwd().parents)
        if (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUE")
    root = roots[0].resolve()

    notebook_files = {
        "NB20_A": "20_answerability_safety_confidence.ipynb",
        "NB20_B": "20b_answerability_safety_confidence_scoped_review.ipynb",
        "NB20_C": "20c_answerability_safety_confidence_audit_and_acceptance.ipynb",
        "NB21_A": "21_evaluation_and_ablation.ipynb",
        "NB21_B": "21b_evaluation_and_ablation_continued.ipynb",
        "NB21_C": "21c_evaluation_and_ablation_continued.ipynb",
        "NB22": "22_release_and_reproducibility.ipynb",
    }

    snapshots = []
    for role, filename in notebook_files.items():
        relative = "notebooks/phase2/" + filename
        path = root / relative
        row = {
            "role": role,
            "path": relative,
            "notebook_executed_or_modified": False,
        }
        if not path.is_file():
            row["status"] = "NOT_AVAILABLE_AT_EXPLICIT_PATH"
            snapshots.append(row)
            continue

        raw = path.read_bytes()
        try:
            notebook = json.loads(raw.decode("utf-8"))
            cells = notebook.get("cells")
            if not isinstance(cells, list):
                raise ValueError("NOTEBOOK_CELLS_NOT_ARRAY")

            code_indices = []
            markdown_count = 0
            unexecuted = []
            error_cells = []
            error_output_count = 0

            for index, cell in enumerate(cells):
                if cell.get("cell_type") == "markdown":
                    markdown_count += 1
                if cell.get("cell_type") != "code":
                    continue
                code_indices.append(index)
                if cell.get("execution_count") is None:
                    unexecuted.append(index)

                errors = [
                    output for output in cell.get("outputs", [])
                    if output.get("output_type") == "error"
                ]
                if errors:
                    error_cells.append(index)
                    error_output_count += len(errors)

            row.update({
                "status": "SAVED_NOTEBOOK_STRUCTURE_READ",
                "file_bytes_sha256": hashlib.sha256(raw).hexdigest(),
                "bytes": len(raw),
                "nbformat": notebook.get("nbformat"),
                "cell_count": len(cells),
                "code_cell_count": len(code_indices),
                "markdown_cell_count": markdown_count,
                "unexecuted_code_cell_count": len(unexecuted),
                "unexecuted_physical_cell_indices": unexecuted,
                "cells_with_error_output": len(error_cells),
                "error_output_count": error_output_count,
                "error_physical_cell_indices": error_cells,
                "snapshot_proves_current_runtime_health": False,
                "last_active_cell_may_be_unsaved": role == "NB22",
            })
        except (ValueError, UnicodeError, TypeError) as exc:
            row.update({
                "status": "SAVED_NOTEBOOK_REQUIRES_RECONCILIATION",
                "error_type": type(exc).__name__,
            })
        snapshots.append(row)

    consolidation_packages = [
        {
            "package": "NB20_FINAL_CONSOLIDATION",
            "inputs": ["NB20_A", "NB20_B", "NB20_C"],
            "existing_anchor": "C455_SCOPED_POSTHOC_AUDIT_HANDOFF",
            "preserve": [
                "Original raw responses, attempts and adopted decisions.",
                "Scoped confidence and calibration limitations.",
                "Original cell identities or an explicit old-to-new map.",
            ],
            "execution_prerequisites": [
                "Define final entry point and retained producer functions.",
                "Bind all required cache/checkpoint inputs.",
                "Prove no cache miss can dispatch API or regenerate outputs.",
                "Resolve effective saved errors without deleting evidence.",
            ],
            "merge_performed": False,
        },
        {
            "package": "NB21_FINAL_CONSOLIDATION",
            "inputs": ["NB21_A", "NB21_B", "NB21_C"],
            "existing_anchor": "C085_FINAL_SUMMARY_AND_C082_HANDOFF",
            "preserve": [
                "All 49 action dispositions and historical evidence.",
                "Quality FAIL/BLOCKED and ten distinct open issues.",
                "User deferral of action 48 until consolidation.",
            ],
            "execution_prerequisites": [
                "Define final entry point and accepted existing outputs.",
                "Retain producer/consumer/version mappings.",
                "Disable paid dispatch and regeneration on missing cache.",
                "Use existing metric/judgment results without retuning Test.",
            ],
            "merge_performed": False,
        },
        {
            "package": "NB22_CLEAN_EXECUTION",
            "inputs": ["NB22"],
            "preserve": [
                "Content-addressed outputs and immutable upstream inputs.",
                "Bounded scope of successful C30 restore.",
                "Open release dependencies and quality gates.",
            ],
            "execution_prerequisites": [
                "Save the current corrected notebook before structural review.",
                "Separate saved-snapshot checks from runtime Final Health.",
                "Ensure Run All safely reuses identical outputs.",
                "Bind final intended scope before clean execution.",
            ],
            "run_all_performed": False,
        },
    ]

    artifact = {
        "artifact_type": "NB22_CONSOLIDATION_AND_CLEAN_EXECUTION_SCOPE",
        "schema_version": "NB22_C031_ACTION21_SCOPE_V1",
        "status": "CONSOLIDATION_SCOPE_RECORDED_NOT_EXECUTED",
        "saved_notebook_snapshots": snapshots,
        "packages": consolidation_packages,
        "existing_restore_evidence": c30["output"],
        "NB21_action48_status_preserved":
            c03["summary"]["C085_current_summary"]["action48_status"],
        "automatic_error_output_deletion": False,
        "new_notebook_created": False,
        "notebooks_merged": False,
        "historical_notebook_run_all": False,
        "clean_kernel_execution_completed": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root, "reports/phase2/release/NB22.C031.action21",
        "C031.consolidation_scope", artifact,
    )

    print(json.dumps({
        "status": artifact["status"],
        "saved_notebook_snapshots": snapshots,
        "consolidation_package_count": len(consolidation_packages),
        "NB21_action48_status_preserved":
            artifact["NB21_action48_status_preserved"],
        "existing_restore_evidence": c30["output"],
        "output": output,
        "notebooks_merged": False,
        "clean_kernel_execution_completed": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c31_result = nb22_c31_run(nb22_c03_result, nb22_c30_result)

{
  "status": "CONSOLIDATION_SCOPE_RECORDED_NOT_EXECUTED",
  "saved_notebook_snapshots": [
    {
      "role": "NB20_A",
      "path": "notebooks/phase2/20_answerability_safety_confidence.ipynb",
      "notebook_executed_or_modified": false,
      "status": "SAVED_NOTEBOOK_STRUCTURE_READ",
      "file_bytes_sha256": "6d1d03990c8778c1d0c6b3b3cb8c18d2c6ce9062aa4888c062283b75656302f7",
      "bytes": 42183988,
      "nbformat": 4,
      "cell_count": 700,
      "code_cell_count": 354,
      "markdown_cell_count": 346,
      "unexecuted_code_cell_count": 4,
      "unexecuted_physical_cell_indices": [
        308,
        310,
        316,
        320
      ],
      "cells_with_error_output": 3,
      "error_output_count": 3,
      "error_physical_cell_indices": [
        432,
        482,
        648
      ],
      "snapshot_proves_current_runtime_health": false,
      "last_active_cell_may_be_unsaved": false
    },
    {
      "role": "NB20_B",
      "path": "notebooks/phase2/20b_answerab

<div dir="rtl" align="right">

### NB22 — C032 | بررسی نسخهٔ ذخیره‌شده، مرتبط با اقدام ۲۳

این سل سه بررسی را تجمیع می‌کند:
۱. حضور کدهای بازیابی C30 و تعیین دامنهٔ C31 در فایل ذخیره‌شده؛
۲. ساختار نوت‌بوک و شمارش خروجی‌های خطای ذخیره‌شده؛
۳. ثبت و بازخوانی گزارش بررسی.

پیش از اجرا، نوت‌بوک را ذخیره کنید. حضور کد یا نبود خروجی خطا، اثبات اجرای پاک نیست. هیچ سل قبلی، مدل یا API اجرا نمی‌شود.

</div>

In [35]:
def nb22_c32_run():
    import json
    import hashlib
    from pathlib import Path

    relative = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        parent
        for parent in [Path.cwd(), *Path.cwd().parents]
        if (parent / relative).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError(
            f"PROJECT_ROOT_NOT_UNIQUELY_RESOLVED: {len(roots)}"
        )

    root = roots[0]
    path = root / relative
    before = path.stat()
    raw = path.read_bytes()
    after = path.stat()

    if (
        before.st_size != after.st_size
        or before.st_mtime_ns != after.st_mtime_ns
    ):
        raise RuntimeError("NOTEBOOK_CHANGED_DURING_READ")

    notebook = json.loads(raw)
    cells = notebook.get("cells")
    if notebook.get("nbformat") != 4 or not isinstance(cells, list):
        raise RuntimeError("UNEXPECTED_NOTEBOOK_STRUCTURE")

    code_cells = [
        (index, cell)
        for index, cell in enumerate(cells)
        if cell.get("cell_type") == "code"
    ]

    def source_text(cell):
        source = cell.get("source", "")
        return "".join(source) if isinstance(source, list) else source

    # Construct names to avoid this checking cell matching itself.
    required_functions = {
        "C030": "nb22_" + "c30_run",
        "C031": "nb22_" + "c31_run",
    }
    bindings = {}
    for label, function_name in required_functions.items():
        matching_indices = [
            index
            for index, cell in code_cells
            if (
                f"def {function_name}(" in source_text(cell)
                or f"def {function_name} (" in source_text(cell)
            )
        ]
        bindings[label] = {
            "function": function_name,
            "physical_cell_indices": matching_indices,
            "saved_definition_present": bool(matching_indices),
        }

    error_cells = [
        index
        for index, cell in code_cells
        if any(
            output.get("output_type") == "error"
            for output in cell.get("outputs", [])
        )
    ]
    unexecuted = [
        index
        for index, cell in code_cells
        if source_text(cell).strip()
        and cell.get("execution_count") is None
    ]

    recent_code_saved = all(
        binding["saved_definition_present"]
        for binding in bindings.values()
    )

    result = {
        "artifact_type": "NB22_SAVED_NOTEBOOK_STRUCTURE_CHECK",
        "schema_version": "NB22_C032_ACTION23_V1",
        "status": (
            "RECENT_CODE_PRESENT_IN_SAVED_NOTEBOOK"
            if recent_code_saved
            else "RECENT_CODE_NOT_FULLY_PRESENT_IN_SAVED_NOTEBOOK"
        ),
        "notebook": {
            "path": relative.as_posix(),
            "file_bytes_sha256": hashlib.sha256(raw).hexdigest(),
            "bytes": len(raw),
            "cell_count": len(cells),
            "code_cell_count": len(code_cells),
            "required_saved_definitions": bindings,
            "error_physical_cell_indices": error_cells,
            "unexecuted_nonempty_code_cell_indices": unexecuted,
        },
        "current_check_cell_may_not_yet_be_saved": True,
        "clean_kernel_execution_established": False,
        "action23_fully_complete": False,
        "historical_notebooks_modified": False,
        "notebook_executed_or_modified_by_this_check": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }

    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C032.action23",
        "C032.saved_notebook_structure",
        result,
    )
    print(json.dumps(
        {**result, "output": output},
        ensure_ascii=False,
        indent=2,
    ))
    return {"artifact": result, "output": output}


nb22_c32_result = nb22_c32_run()

{
  "artifact_type": "NB22_SAVED_NOTEBOOK_STRUCTURE_CHECK",
  "schema_version": "NB22_C032_ACTION23_V1",
  "status": "RECENT_CODE_PRESENT_IN_SAVED_NOTEBOOK",
  "notebook": {
    "path": "notebooks/phase2/22_release_and_reproducibility.ipynb",
    "file_bytes_sha256": "49fe5ad604acb295195827dbe5d2de49e9630d0209ee2cc89302df40e5174f08",
    "bytes": 1861112,
    "cell_count": 64,
    "code_cell_count": 32,
    "required_saved_definitions": {
      "C030": {
        "function": "nb22_c30_run",
        "physical_cell_indices": [
          59
        ],
        "saved_definition_present": true
      },
      "C031": {
        "function": "nb22_c31_run",
        "physical_cell_indices": [
          61
        ],
        "saved_definition_present": true
      }
    },
    "error_physical_cell_indices": [],
    "unexecuted_nonempty_code_cell_indices": [
      63
    ]
  },
  "current_check_cell_may_not_yet_be_saved": true,
  "clean_kernel_execution_established": false,
  "action23_fully_complet

<div dir="rtl" align="right">

### NB22 — C033 | اقدام ۲۴: بررسی اطلاعات محرمانه و مسیرهای شخصی

سه زیرکار در یک سل:
۱. استفاده از فهرست هفت نوت‌بوک ثبت‌شده در C31؛
۲. بررسی متن سل‌ها، خروجی‌ها و متادیتا برای الگوهای مشخص؛
۳. ثبت گزارش با محل یافته‌ها، بدون ذخیره یا نمایش مقدار مشکوک.

این بررسی آفلاین است. فایل‌ها اصلاح نمی‌شوند و هیچ API یا سل قبلی اجرا نمی‌شود.
دامنه فقط همین هفت نوت‌بوک است؛ بررسی کامل بستهٔ ریلیز و رفع یافته‌ها هنوز ادعا نمی‌شود.

</div>

In [36]:
def nb22_c33_run(c31):
    import hashlib
    import json
    import re
    from pathlib import Path
    from collections import Counter

    relative_nb22 = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / relative_nb22).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0].resolve()

    snapshots = c31["artifact"]["saved_notebook_snapshots"]
    if len(snapshots) != 7:
        raise RuntimeError("EXPECTED_SEVEN_NOTEBOOK_BINDINGS")

    patterns = {
        "PRIVATE_KEY_HEADER": re.compile(
            r"-----BEGIN (?:RSA |EC |OPENSSH |DSA )?PRIVATE KEY-----"
        ),
        "GITHUB_TOKEN_SHAPE": re.compile(
            r"\b(?:gh[pousr]_[A-Za-z0-9]{20,}"
            r"|github_pat_[A-Za-z0-9_]{30,})\b"
        ),
        "HF_TOKEN_SHAPE": re.compile(r"\bhf_[A-Za-z0-9]{25,}\b"),
        "OPENAI_STYLE_TOKEN_SHAPE": re.compile(
            r"\bsk-(?:proj-|svcacct-)?[A-Za-z0-9_-]{32,}\b"
        ),
        "AWS_ACCESS_KEY_ID_SHAPE": re.compile(
            r"\b(?:AKIA|ASIA)[A-Z0-9]{16}\b"
        ),
        "CREDENTIAL_IN_URL": re.compile(
            r"https?://[^\s/:@\"'<>]+:[^\s/@\"'<>]+@"
        ),
        "PERSONAL_ABSOLUTE_PATH": re.compile(
            r"(?:/Users/|/home/)[A-Za-z0-9._-]+/"
            r"|[A-Za-z]:\\Users\\[A-Za-z0-9._ -]+\\"
        ),
    }
    # Literal assignments are review candidates, not confirmed secrets.
    patterns["CREDENTIAL_LITERAL_ASSIGNMENT"] = re.compile(
        r"""(?i)\b(?:api_key|access_token|auth_token|"""
        r"""client_secret|password)\b\s*[:=]\s*"""
        r"""["']([^"'\r\n]{12,})["']"""
    )

    findings = []
    counts = Counter()
    receipts = []
    max_locations = 2000

    def inspect_text(value, location):
        if not isinstance(value, str):
            return
        for category, pattern in patterns.items():
            for match in pattern.finditer(value):
                if category == "CREDENTIAL_LITERAL_ASSIGNMENT":
                    candidate = match.group(1).lower()
                    if (
                        any(marker in candidate for marker in (
                            "placeholder", "your_", "your-",
                            "example", "redacted", "getenv",
                            "environ", "<", ">", "${",
                        ))
                        or set(candidate) <= {"*", "x", "-"}
                    ):
                        continue
                counts[category] += 1
                if len(findings) < max_locations:
                    findings.append({
                        **location,
                        "category": category,
                        "line_in_text": value.count(
                            "\n", 0, match.start()
                        ) + 1,
                        "disposition": "REQUIRES_CONTEXT_REVIEW",
                    })

    def walk(value, location):
        if isinstance(value, str):
            inspect_text(value, location)
        elif isinstance(value, list):
            # Notebook text is frequently stored as a list of lines.
            if all(isinstance(item, str) for item in value):
                inspect_text("".join(value), location)
            else:
                for index, item in enumerate(value):
                    walk(item, {
                        **location,
                        "field": f"{location['field']}[{index}]",
                    })
        elif isinstance(value, dict):
            for key, item in value.items():
                walk(item, {
                    **location,
                    "field": f"{location['field']}.{key}",
                })

    for snapshot in snapshots:
        relative = Path(snapshot["path"])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("NON_PROJECT_NOTEBOOK_PATH")
        path = (root / relative).resolve()
        if root not in path.parents:
            raise RuntimeError("NOTEBOOK_PATH_OUTSIDE_PROJECT")

        before = path.stat()
        raw = path.read_bytes()
        after = path.stat()
        if (
            before.st_size != after.st_size
            or before.st_mtime_ns != after.st_mtime_ns
        ):
            raise RuntimeError("NOTEBOOK_CHANGED_DURING_SCAN")

        notebook = json.loads(raw)
        for index, cell in enumerate(notebook.get("cells", [])):
            walk(cell, {
                "role": snapshot["role"],
                "path": relative.as_posix(),
                "physical_cell_index": index,
                "field": "cell",
            })
        walk(notebook.get("metadata", {}), {
            "role": snapshot["role"],
            "path": relative.as_posix(),
            "physical_cell_index": None,
            "field": "notebook.metadata",
        })
        receipts.append({
            "role": snapshot["role"],
            "path": relative.as_posix(),
            "file_bytes_sha256": hashlib.sha256(raw).hexdigest(),
            "bytes": len(raw),
        })

    result = {
        "artifact_type": "NB22_NOTEBOOK_SENSITIVE_PATTERN_SCAN",
        "schema_version": "NB22_C033_ACTION24_V1",
        "status": (
            "SCOPED_SCAN_HAS_REVIEW_CANDIDATES"
            if counts else "NO_DEFINED_PATTERNS_FOUND_IN_SCANNED_SCOPE"
        ),
        "scope": "SEVEN_NOTEBOOKS_CELL_CONTENT_AND_METADATA_ONLY",
        "scanned_notebooks": receipts,
        "pattern_categories": list(patterns),
        "finding_counts": dict(counts),
        "total_pattern_matches": sum(counts.values()),
        "finding_locations": findings,
        "locations_truncated": sum(counts.values()) > len(findings),
        "matched_values_recorded": False,
        "confirmed_secret_exposure_established": False,
        "full_release_package_scanned": False,
        "action24_fully_complete": False,
        "files_modified": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C033.action24",
        "C033.notebook_sensitive_pattern_scan",
        result,
    )
    print(json.dumps({
        "status": result["status"],
        "notebooks_scanned": len(receipts),
        "finding_counts": dict(counts),
        "total_pattern_matches": sum(counts.values()),
        "first_review_locations": findings[:20],
        "matched_values_recorded": False,
        "full_release_package_scanned": False,
        "action24_fully_complete": False,
        "output": output,
        "new_api_calls": 0,
        "network_calls": 0,
        "files_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": result, "output": output}


nb22_c33_result = nb22_c33_run(nb22_c31_result)

{
  "status": "SCOPED_SCAN_HAS_REVIEW_CANDIDATES",
  "notebooks_scanned": 7,
  "finding_counts": {
    "PERSONAL_ABSOLUTE_PATH": 724
  },
  "total_pattern_matches": 724,
  "first_review_locations": [
    {
      "role": "NB20_A",
      "path": "notebooks/phase2/20_answerability_safety_confidence.ipynb",
      "physical_cell_index": 5,
      "field": "cell.outputs[0].text",
      "category": "PERSONAL_ABSOLUTE_PATH",
      "line_in_text": 218,
      "disposition": "REQUIRES_CONTEXT_REVIEW"
    },
    {
      "role": "NB20_A",
      "path": "notebooks/phase2/20_answerability_safety_confidence.ipynb",
      "physical_cell_index": 5,
      "field": "cell.outputs[0].text",
      "category": "PERSONAL_ABSOLUTE_PATH",
      "line_in_text": 225,
      "disposition": "REQUIRES_CONTEXT_REVIEW"
    },
    {
      "role": "NB20_A",
      "path": "notebooks/phase2/20_answerability_safety_confidence.ipynb",
      "physical_cell_index": 5,
      "field": "cell.outputs[0].text",
      "category": "PER

<div dir="rtl" align="right">

### NB22 — C034 | اقدام ۲۴: تفکیک یافته‌های مسیر شخصی

سه زیرکار:
۱. تفکیک یافته‌ها میان متن سل، خروجی و متادیتا؛
۲. تجمیع بر اساس نوت‌بوک و محل، با حفظ تعداد تطبیق‌ها؛
۳. ثبت برنامهٔ بررسی و اصلاح متناسب با هر محل.

این سل از گزارش موجود C033 استفاده می‌کند. هیچ مسیر شخصی نمایش داده نمی‌شود، فایل تاریخی تغییر نمی‌کند و پاک‌سازی انجام‌شده تلقی نمی‌شود.

</div>

In [37]:
def nb22_c34_run(c33):
    import json
    from pathlib import Path
    from collections import Counter

    scan = c33["artifact"]
    if scan.get("locations_truncated"):
        raise RuntimeError("COMPLETE_FINDING_LOCATIONS_REQUIRED")

    findings = scan["finding_locations"]
    if len(findings) != scan["total_pattern_matches"]:
        raise RuntimeError("FINDING_COUNT_MISMATCH")

    relative = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / relative).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0]

    def classify(field):
        if field == "cell.source":
            return "CELL_SOURCE"
        if field.startswith("cell.outputs"):
            return "SAVED_OUTPUT"
        if (
            field.startswith("cell.metadata")
            or field.startswith("notebook.metadata")
        ):
            return "METADATA"
        return "OTHER_REQUIRES_REVIEW"

    location_counts = Counter()
    category_counts = Counter()
    role_counts = {}
    affected_cells = set()

    for item in findings:
        role = item["role"]
        category = item["category"]
        surface = classify(item["field"])
        key = (
            role,
            item["path"],
            item["physical_cell_index"],
            item["field"],
            surface,
            category,
        )
        location_counts[key] += 1
        category_counts[(category, surface)] += 1
        role_counts.setdefault(role, Counter())[surface] += 1
        affected_cells.add((
            item["path"], item["physical_cell_index"]
        ))

    handling = {
        "CELL_SOURCE": (
            "Inspect path usage; replace machine-specific roots with "
            "project-relative resolution or explicit configuration while "
            "preserving producer logic."
        ),
        "SAVED_OUTPUT": (
            "Inspect a derived publication copy for targeted path "
            "redaction; preserve historical originals and provenance. "
            "Do not delete all outputs or rerun computations."
        ),
        "METADATA": (
            "Review metadata fields and remove personal path values "
            "only in the intended publication artifact."
        ),
        "OTHER_REQUIRES_REVIEW": (
            "Inspect field context before choosing any modification."
        ),
    }

    locations = []
    for key, count in sorted(
        location_counts.items(),
        key=lambda pair: tuple(str(v) for v in pair[0]),
    ):
        role, path, index, field, surface, category = key
        locations.append({
            "role": role,
            "path": path,
            "physical_cell_index": index,
            "field": field,
            "surface": surface,
            "category": category,
            "pattern_match_count": count,
            "proposed_handling": handling[surface],
            "context_review_completed": False,
            "modification_applied": False,
        })

    surface_counts = Counter()
    for (_, surface), count in category_counts.items():
        surface_counts[surface] += count

    result = {
        "artifact_type": "NB22_SENSITIVE_FINDING_TRIAGE",
        "schema_version": "NB22_C034_ACTION24_V1",
        "status": "FINDINGS_GROUPED_CONTEXT_REVIEW_PENDING",
        "source_report": c33["output"],
        "total_pattern_matches": len(findings),
        "surface_match_counts": dict(surface_counts),
        "per_notebook_match_counts": {
            role: dict(counts)
            for role, counts in sorted(role_counts.items())
        },
        "affected_cell_or_notebook_metadata_count": len(affected_cells),
        "distinct_field_location_count": len(locations),
        "distinct_personal_path_count_established": False,
        "grouped_locations": locations,
        "matched_values_recorded": False,
        "notebook_files_reread": 0,
        "cleanup_performed": False,
        "full_release_package_scanned": False,
        "action24_fully_complete": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C034.action24",
        "C034.sensitive_finding_triage",
        result,
    )
    print(json.dumps({
        "status": result["status"],
        "total_pattern_matches": len(findings),
        "surface_match_counts": result["surface_match_counts"],
        "per_notebook_match_counts": result[
            "per_notebook_match_counts"
        ],
        "distinct_field_location_count": len(locations),
        "notebook_files_reread": 0,
        "cleanup_performed": False,
        "action24_fully_complete": False,
        "output": output,
        "new_api_calls": 0,
        "network_calls": 0,
    }, ensure_ascii=False, indent=2))
    return {"artifact": result, "output": output}


nb22_c34_result = nb22_c34_run(nb22_c33_result)

{
  "status": "FINDINGS_GROUPED_CONTEXT_REVIEW_PENDING",
  "total_pattern_matches": 724,
  "surface_match_counts": {
    "SAVED_OUTPUT": 586,
    "CELL_SOURCE": 138
  },
  "per_notebook_match_counts": {
    "NB20_A": {
      "SAVED_OUTPUT": 36
    },
    "NB20_B": {
      "SAVED_OUTPUT": 8
    },
    "NB20_C": {
      "SAVED_OUTPUT": 8
    },
    "NB21_A": {
      "SAVED_OUTPUT": 15,
      "CELL_SOURCE": 17
    },
    "NB21_B": {
      "SAVED_OUTPUT": 513,
      "CELL_SOURCE": 120
    },
    "NB21_C": {
      "SAVED_OUTPUT": 6,
      "CELL_SOURCE": 1
    }
  },
  "distinct_field_location_count": 203,
  "notebook_files_reread": 0,
  "cleanup_performed": false,
  "action24_fully_complete": false,
  "output": {
    "path": "reports/phase2/release/NB22.C034.action24/C034.sensitive_finding_triage.02be46396eb33962ba371bd45da5d995aed0761d243b1c181c0697798b7c9477.json",
    "file_bytes_sha256": "02be46396eb33962ba371bd45da5d995aed0761d243b1c181c0697798b7c9477",
    "bytes": 93526,
    "write_s

<div dir="rtl" align="right">

### NB22 — C035 | اقدام ۲۴: بررسی زمینهٔ مسیرهای موجود در متن سل‌ها

سه زیرکار:
۱. انتخاب فقط محل‌های CELL_SOURCE از گزارش C034 و بررسی هویت فایل با رسید C033؛
۲. استخراج خطوط مرتبط با پوشاندن بخش شخصی مسیر؛
۳. ثبت نوع سل و زمینهٔ پوشانده‌شده برای تعیین اصلاح مناسب.

این سل کد نوت‌بوک‌ها را اجرا یا اصلاح نمی‌کند.
وجود مسیر در متن سل، به‌تنهایی وابستگی اجرایی را اثبات نمی‌کند.

</div>

In [38]:
def nb22_c35_run(c33, c34):
    import ast
    import hashlib
    import json
    import re
    from pathlib import Path
    from collections import Counter, defaultdict

    relative = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / relative).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0].resolve()

    selected = [
        row for row in c34["artifact"]["grouped_locations"]
        if row["surface"] == "CELL_SOURCE"
    ]
    receipts = {
        row["path"]: row
        for row in c33["artifact"]["scanned_notebooks"]
    }
    grouped = defaultdict(list)
    for row in selected:
        grouped[row["path"]].append(row)

    path_pattern = re.compile(
        r"(?:/Users/|/home/)[A-Za-z0-9._-]+/"
        r"|[A-Za-z]:\\Users\\[A-Za-z0-9._ -]+\\"
    )

    # Mask other recognized credential shapes before recording context.
    credential_patterns = [
        re.compile(
            r"\b(?:gh[pousr]_[A-Za-z0-9]{20,}"
            r"|github_pat_[A-Za-z0-9_]{30,})\b"
        ),
        re.compile(r"\bhf_[A-Za-z0-9]{25,}\b"),
        re.compile(
            r"\bsk-(?:proj-|svcacct-)?[A-Za-z0-9_-]{32,}\b"
        ),
        re.compile(r"\b(?:AKIA|ASIA)[A-Z0-9]{16}\b"),
        re.compile(
            r"https?://[^\s/:@\"'<>]+:[^\s/@\"'<>]+@"
        ),
    ]

    def masked(text):
        text = path_pattern.sub("<PERSONAL_HOME>/", text)
        for pattern in credential_patterns:
            text = pattern.sub("<REDACTED_CREDENTIAL_SHAPE>", text)
        return text

    evidence = []
    file_checks = []
    kinds = Counter()

    for relative_path, rows in sorted(grouped.items()):
        rel = Path(relative_path)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError("INVALID_PROJECT_RELATIVE_PATH")
        path = (root / rel).resolve()
        if root not in path.parents:
            raise RuntimeError("PATH_OUTSIDE_PROJECT")

        before = path.stat()
        raw = path.read_bytes()
        after = path.stat()
        if (
            before.st_size != after.st_size
            or before.st_mtime_ns != after.st_mtime_ns
        ):
            raise RuntimeError("NOTEBOOK_CHANGED_DURING_READ")

        actual_hash = hashlib.sha256(raw).hexdigest()
        if actual_hash != receipts[relative_path]["file_bytes_sha256"]:
            raise RuntimeError(
                "NOTEBOOK_CHANGED_SINCE_SCAN; "
                "STOPPED_BEFORE_USING_STALE_CELL_LOCATIONS"
            )

        notebook = json.loads(raw)
        file_checks.append({
            "path": relative_path,
            "file_bytes_sha256": actual_hash,
            "matches_scan_receipt": True,
        })

        for row in rows:
            index = row["physical_cell_index"]
            cell = notebook["cells"][index]
            source = cell.get("source", "")
            if isinstance(source, list):
                source = "".join(source)
            lines = source.splitlines()
            cell_type = cell.get("cell_type", "unknown")

            if cell_type == "code":
                try:
                    ast.parse(source)
                    syntax_status = "PYTHON_AST_PARSE_PASS"
                except SyntaxError:
                    syntax_status = (
                        "NOT_PARSED_AS_PLAIN_PYTHON"
                    )
            else:
                syntax_status = "NOT_APPLICABLE"

            matched_lines = []
            observed_count = 0
            for number, line in enumerate(lines, start=1):
                matches = list(path_pattern.finditer(line))
                if not matches:
                    continue
                observed_count += len(matches)
                if cell_type == "markdown":
                    kind = "MARKDOWN_TEXT"
                elif line.lstrip().startswith("#"):
                    kind = "COMMENT_LINE"
                else:
                    kind = "CODE_OR_STRING_CONTEXT_REQUIRES_REVIEW"
                kinds[kind] += len(matches)

                # Only matching lines; no surrounding unrelated content.
                safe_line = masked(line)
                marker = safe_line.find("<PERSONAL_HOME>/")
                start = max(0, marker - 160)
                excerpt = safe_line[start:start + 600]
                matched_lines.append({
                    "line_in_source": number,
                    "match_count": len(matches),
                    "context_kind": kind,
                    "masked_excerpt": excerpt,
                    "excerpt_truncated": (
                        start > 0 or start + 600 < len(safe_line)
                    ),
                })

            if observed_count != row["pattern_match_count"]:
                raise RuntimeError("SOURCE_MATCH_COUNT_CHANGED")

            evidence.append({
                "role": row["role"],
                "path": relative_path,
                "physical_cell_index": index,
                "cell_type": cell_type,
                "syntax_observation": syntax_status,
                "pattern_match_count": observed_count,
                "matched_lines": matched_lines,
                "execution_dependency_established": False,
                "modification_applied": False,
            })

    result = {
        "artifact_type": "NB22_MASKED_SOURCE_PATH_CONTEXT",
        "schema_version": "NB22_C035_ACTION24_V1",
        "status": "MASKED_SOURCE_CONTEXT_READY_FOR_REVIEW",
        "source_scan": c33["output"],
        "source_triage": c34["output"],
        "file_identity_checks": file_checks,
        "source_locations_reviewed": len(evidence),
        "source_pattern_matches": sum(
            row["pattern_match_count"] for row in evidence
        ),
        "context_kind_match_counts": dict(kinds),
        "source_contexts": evidence,
        "personal_home_values_recorded": False,
        "cleanup_performed": False,
        "action24_fully_complete": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C035.action24",
        "C035.masked_source_path_context",
        result,
    )

    examples = [
        {
            "role": row["role"],
            "physical_cell_index": row["physical_cell_index"],
            **line,
        }
        for row in evidence
        for line in row["matched_lines"]
    ]
    print(json.dumps({
        "status": result["status"],
        "notebooks_read": len(file_checks),
        "source_locations_reviewed": len(evidence),
        "source_pattern_matches": result["source_pattern_matches"],
        "context_kind_match_counts": dict(kinds),
        "masked_context_examples": examples[:24],
        "additional_context_line_count": max(0, len(examples) - 24),
        "output": output,
        "cleanup_performed": False,
        "new_api_calls": 0,
        "network_calls": 0,
    }, ensure_ascii=False, indent=2))
    return {"artifact": result, "output": output}


nb22_c35_result = nb22_c35_run(
    nb22_c33_result,
    nb22_c34_result,
)

{
  "status": "MASKED_SOURCE_CONTEXT_READY_FOR_REVIEW",
  "notebooks_read": 3,
  "source_locations_reviewed": 137,
  "source_pattern_matches": 138,
  "context_kind_match_counts": {
    "CODE_OR_STRING_CONTEXT_REQUIRES_REVIEW": 137,
    "COMMENT_LINE": 1
  },
  "masked_context_examples": [
    {
      "role": "NB21_A",
      "physical_cell_index": 661,
      "line_in_source": 12,
      "match_count": 1,
      "context_kind": "CODE_OR_STRING_CONTEXT_REQUIRES_REVIEW",
      "masked_excerpt": "        \"<PERSONAL_HOME>/AMIR_DATA/00_WORKS/01_PROJECTS/\"",
      "excerpt_truncated": false
    },
    {
      "role": "NB21_A",
      "physical_cell_index": 663,
      "line_in_source": 11,
      "match_count": 1,
      "context_kind": "CODE_OR_STRING_CONTEXT_REQUIRES_REVIEW",
      "masked_excerpt": "        \"<PERSONAL_HOME>/AMIR_DATA/00_WORKS/01_PROJECTS/\"",
      "excerpt_truncated": false
    },
    {
      "role": "NB21_A",
      "physical_cell_index": 665,
      "line_in_source": 19,
    

<div dir="rtl" align="right">

### NB22 — C036 | اقدام ۲۴: تجمیع کامل الگوهای نیازمند اصلاح

سه زیرکار:
۱. استفاده از تمام زمینه‌های پوشانده‌شدهٔ C035؛
۲. تجمیع خطوط یکسان و شمارش فراوانی آن‌ها؛
۳. نمایش تمام الگوهای متفاوت و مشخص‌کردن موارد ناقص.

این سل فقط دادهٔ موجود در حافظه را بررسی می‌کند؛ خواندن فایل، تغییر نوت‌بوک و فراخوانی API ندارد.

</div>

In [39]:
def nb22_c36_run(c35):
    import json
    from collections import Counter

    artifact = c35["artifact"]
    groups = {}
    total_matches = 0
    total_lines = 0

    for context in artifact["source_contexts"]:
        for line in context["matched_lines"]:
            total_lines += 1
            total_matches += line["match_count"]
            key = (
                context["cell_type"],
                line["context_kind"],
                line["masked_excerpt"].strip(),
                bool(line["excerpt_truncated"]),
            )
            if key not in groups:
                groups[key] = {
                    "cell_type": key[0],
                    "context_kind": key[1],
                    "masked_excerpt": key[2],
                    "excerpt_truncated": key[3],
                    "line_occurrence_count": 0,
                    "pattern_match_count": 0,
                    "roles": Counter(),
                    "locations": [],
                }
            group = groups[key]
            group["line_occurrence_count"] += 1
            group["pattern_match_count"] += line["match_count"]
            group["roles"][context["role"]] += 1
            group["locations"].append({
                "role": context["role"],
                "physical_cell_index": context["physical_cell_index"],
                "line_in_source": line["line_in_source"],
            })

    if total_matches != artifact["source_pattern_matches"]:
        raise RuntimeError("CACHED_CONTEXT_ACCOUNTING_MISMATCH")

    patterns = []
    for group in sorted(
        groups.values(),
        key=lambda item: (
            -item["pattern_match_count"],
            item["masked_excerpt"],
        ),
    ):
        patterns.append({
            **group,
            "roles": dict(group["roles"]),
        })

    result = {
        "status": "ALL_CACHED_SOURCE_CONTEXTS_GROUPED",
        "source_context_report": c35["output"],
        "source_line_count": total_lines,
        "source_pattern_matches": total_matches,
        "distinct_masked_pattern_count": len(patterns),
        "truncated_pattern_count": sum(
            item["excerpt_truncated"] for item in patterns
        ),
        "patterns": patterns,
        "notebook_files_read": 0,
        "files_modified": False,
        "cleanup_performed": False,
        "new_api_calls": 0,
        "network_calls": 0,
    }

    # Show every distinct pattern, with one example location each.
    # Full location lists remain available in nb22_c36_result.
    print(json.dumps({
        **result,
        "patterns": [
            {
                **{
                    key: value
                    for key, value in item.items()
                    if key != "locations"
                },
                "example_location": item["locations"][0],
            }
            for item in patterns
        ],
    }, ensure_ascii=False, indent=2))
    return result


nb22_c36_result = nb22_c36_run(nb22_c35_result)

{
  "status": "ALL_CACHED_SOURCE_CONTEXTS_GROUPED",
  "source_context_report": {
    "path": "reports/phase2/release/NB22.C035.action24/C035.masked_source_path_context.0d453c18b06d274c288bcd34c8194c30f8891d2cae393e8bc3486c3004849c1c.json",
    "file_bytes_sha256": "0d453c18b06d274c288bcd34c8194c30f8891d2cae393e8bc3486c3004849c1c",
    "bytes": 71255,
    "write_status": "CREATED",
    "reload_validation": "PASS"
  },
  "source_line_count": 138,
  "source_pattern_matches": 138,
  "distinct_masked_pattern_count": 4,
  "truncated_pattern_count": 0,
  "patterns": [
    {
      "cell_type": "code",
      "context_kind": "CODE_OR_STRING_CONTEXT_REQUIRES_REVIEW",
      "masked_excerpt": "Path(\"<PERSONAL_HOME>/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance\"),",
      "excerpt_truncated": false,
      "line_occurrence_count": 70,
      "pattern_match_count": 70,
      "roles": {
        "NB21_B": 70
      },
      "example_location": {
        "role": "NB21_B",
        "physical_cell

<div dir="rtl" align="right">

### NB22 — C037 | نسخهٔ اصلاح‌شدهٔ اقدام ۲۴

سه زیرکار:
۱. بررسی هویت نوت‌بوک‌های ورودی؛
۲. اصلاح مسیرهای داخل رشته‌های ساده و چندخطی، سپس پوشاندن مسیرهای خروجی و متادیتا؛
۳. ساخت نسخه‌های مشتق‌شده و بررسی بازخوانی آن‌ها.

رشته‌ها با موقعیت‌های AST اصلاح می‌شوند؛ متن پیرامون مسیر حفظ می‌شود.
هیچ کد تاریخی یا API اجرا نمی‌شود. اجرای پاک و پذیرش ریلیز همچنان تأیید نشده‌اند.

</div>

In [41]:
def nb22_c37_run(c33):
    import ast
    import copy
    import hashlib
    import io
    import json
    import os
    import re
    import tokenize
    from pathlib import Path

    nb22_path = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / nb22_path).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0].resolve()

    personal = re.compile(
        r"(?:/Users/|/home/)[A-Za-z0-9._-]+/"
        r"|[A-Za-z]:\\Users\\[A-Za-z0-9._ -]+\\"
    )

    def walk_strings(value):
        if isinstance(value, str):
            yield value
        elif isinstance(value, list):
            for item in value:
                yield from walk_strings(item)
        elif isinstance(value, dict):
            for item in value.values():
                yield from walk_strings(item)

    def redact(value):
        if isinstance(value, str):
            return personal.sub("<PERSONAL_HOME>/", value)
        if isinstance(value, list):
            return [redact(item) for item in value]
        if isinstance(value, dict):
            return {key: redact(item) for key, item in value.items()}
        return value

    def rewrite_source(source):
        if not personal.search(source):
            return source, 0, 0, 0

        lines = source.splitlines(keepends=True)
        tokens = list(tokenize.generate_tokens(
            io.StringIO(source).readline
        ))
        comment_matches = 0
        for token in reversed(tokens):
            if (
                token.type != tokenize.COMMENT
                or not personal.search(token.string)
            ):
                continue
            row, start = token.start
            _, end = token.end
            comment_matches += len(personal.findall(token.string))
            line = lines[row - 1]
            lines[row - 1] = (
                line[:start]
                + personal.sub("<PERSONAL_HOME>/", token.string)
                + line[end:]
            )
        source = "".join(lines)
        if not personal.search(source):
            return source, 0, 0, comment_matches

        tree = ast.parse(source)
        fstring_node_ids = {
            id(child)
            for node in ast.walk(tree)
            if isinstance(node, ast.JoinedStr)
            for child in ast.walk(node)
        }

        byte_lines = source.encode("utf-8").splitlines(keepends=True)
        offsets = [0]
        for line in byte_lines:
            offsets.append(offsets[-1] + len(line))

        replacements = []
        path_matches = 0
        for node in ast.walk(tree):
            if not (
                isinstance(node, ast.Constant)
                and isinstance(node.value, str)
            ):
                continue
            matches = list(personal.finditer(node.value))
            if not matches:
                continue
            if id(node) in fstring_node_ids:
                raise RuntimeError(
                    "FSTRING_PATH_REQUIRES_EXPLICIT_REVIEW"
                )
            if any("\\" in match.group() for match in matches):
                raise RuntimeError(
                    "WINDOWS_PATH_REQUIRES_EXPLICIT_REVIEW"
                )

            # Preserve every surrounding segment, including embedded
            # code text and implicitly concatenated string literals.
            parts = []
            cursor = 0
            for match in matches:
                if match.start() > cursor:
                    parts.append(repr(node.value[cursor:match.start()]))
                parts.append(
                    '(__import__("pathlib").Path.home().as_posix() + "/")'
                )
                cursor = match.end()
            if cursor < len(node.value):
                parts.append(repr(node.value[cursor:]))

            expression = "(" + " + ".join(parts) + ")"
            ast.parse(expression, mode="eval")
            start = offsets[node.lineno - 1] + node.col_offset
            end = offsets[node.end_lineno - 1] + node.end_col_offset
            replacements.append((
                start, end, expression.encode("utf-8")
            ))
            path_matches += len(matches)

        ascending = sorted(replacements)
        if any(
            left[1] > right[0]
            for left, right in zip(ascending, ascending[1:])
        ):
            raise RuntimeError("OVERLAPPING_AST_REPLACEMENTS")

        encoded = source.encode("utf-8")
        for start, end, replacement in reversed(ascending):
            encoded = encoded[:start] + replacement + encoded[end:]
        rewritten = encoded.decode("utf-8")

        ast.parse(rewritten)
        if personal.search(rewritten):
            raise RuntimeError("UNHANDLED_SOURCE_PATH_REMAINS")
        return (
            rewritten, len(replacements),
            path_matches, comment_matches,
        )

    receipts = c33["artifact"]["scanned_notebooks"]
    if len(receipts) != 7:
        raise RuntimeError("EXPECTED_SEVEN_NOTEBOOKS")

    prepared = []
    for receipt in receipts:
        relative = Path(receipt["path"])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("INVALID_NOTEBOOK_PATH")
        path = (root / relative).resolve()
        if root not in path.parents:
            raise RuntimeError("NOTEBOOK_OUTSIDE_PROJECT")

        before = path.stat()
        raw = path.read_bytes()
        after = path.stat()
        if (
            before.st_size != after.st_size
            or before.st_mtime_ns != after.st_mtime_ns
        ):
            raise RuntimeError("SOURCE_CHANGED_DURING_READ")

        digest = hashlib.sha256(raw).hexdigest()
        if (
            receipt["role"] != "NB22"
            and digest != receipt["file_bytes_sha256"]
        ):
            raise RuntimeError(
                "HISTORICAL_NOTEBOOK_CHANGED_SINCE_SCAN"
            )

        original = json.loads(raw)
        derived = copy.deepcopy(original)
        literal_nodes = path_matches = comment_matches = 0

        for index, cell in enumerate(derived.get("cells", [])):
            if cell.get("cell_type") != "code":
                continue
            source = cell.get("source", "")
            if isinstance(source, list):
                source = "".join(source)
            try:
                changed, nodes, matches, comments = rewrite_source(source)
            except Exception as exc:
                raise RuntimeError(
                    f"SOURCE_TRANSFORM_FAILED: "
                    f"role={receipt['role']}, cell={index}, "
                    f"reason={type(exc).__name__}"
                ) from None

            if nodes or comments:
                cell["source"] = changed.splitlines(keepends=True)
            literal_nodes += nodes
            path_matches += matches
            comment_matches += comments

        derived = redact(derived)
        if any(personal.search(text) for text in walk_strings(derived)):
            raise RuntimeError("PERSONAL_PATH_PATTERN_REMAINS")
        if len(derived["cells"]) != len(original["cells"]):
            raise RuntimeError("CELL_COUNT_CHANGED")

        def output_signature(notebook):
            return [
                (
                    index,
                    cell.get("execution_count"),
                    [
                        output.get("output_type")
                        for output in cell.get("outputs", [])
                    ],
                )
                for index, cell in enumerate(notebook["cells"])
                if cell.get("cell_type") == "code"
            ]

        if output_signature(original) != output_signature(derived):
            raise RuntimeError("OUTPUT_STRUCTURE_CHANGED")

        encoded = json.dumps(
            derived, ensure_ascii=False, sort_keys=True,
            separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")
        prepared.append({
            "role": receipt["role"],
            "source_path": relative.as_posix(),
            "source_file_bytes_sha256": digest,
            "source_matches_prior_scan": (
                digest == receipt["file_bytes_sha256"]
            ),
            "literal_nodes_rewritten": literal_nodes,
            "source_path_matches_rewritten": path_matches,
            "comment_matches_redacted": comment_matches,
            "remaining_personal_path_matches": 0,
            "cell_count": len(derived["cells"]),
            "output_structure_preserved": True,
            "_notebook": derived,
            "_bytes": encoded,
            "_stem": relative.stem,
        })

    # No publication until every transformation above succeeds.
    directory = (
        root / "reports/phase2/release/NB22.C037.action24/notebooks"
    )
    directory.mkdir(parents=True, exist_ok=True)
    outputs = []

    for item in prepared:
        encoded = item["_bytes"]
        digest = hashlib.sha256(encoded).hexdigest()
        target = directory / f"{item['_stem']}.{digest}.ipynb"
        try:
            with target.open("xb") as handle:
                handle.write(encoded)
                handle.flush()
                os.fsync(handle.fileno())
            write_status = "CREATED"
        except FileExistsError:
            if target.read_bytes() != encoded:
                raise RuntimeError("DERIVED_COPY_CONFLICT")
            write_status = "REUSED_IDENTICAL"

        saved = target.read_bytes()
        if (
            hashlib.sha256(saved).hexdigest() != digest
            or json.loads(saved) != item["_notebook"]
        ):
            raise RuntimeError("DERIVED_NOTEBOOK_RELOAD_FAILED")

        outputs.append({
            **{
                key: value for key, value in item.items()
                if not key.startswith("_")
            },
            "derived_path": target.relative_to(root).as_posix(),
            "file_bytes_sha256": digest,
            "bytes": len(saved),
            "write_status": write_status,
            "reload_validation": "PASS",
        })

    result = {
        "artifact_type": "NB22_DERIVED_NOTEBOOK_PATH_CLEANUP",
        "schema_version": "NB22_C037_ACTION24_V2",
        "status": "SEVEN_DERIVED_NOTEBOOK_COPIES_WRITTEN_AND_RELOADED",
        "outputs": outputs,
        "code_path_change": (
            "PERSONAL_HOME_SEGMENTS_TO_RUNTIME_PATHLIB_HOME"
        ),
        "original_notebooks_modified": False,
        "historical_error_outputs_removed": False,
        "derived_notebooks_executed": False,
        "clean_execution_established": False,
        "project_layout_independence_established": False,
        "derived_copies_added_to_candidate_manifest": False,
        "full_release_package_scanned": False,
        "action24_fully_complete": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C037.action24",
        "C037.derived_notebook_path_cleanup",
        result,
    )
    print(json.dumps(
        {**result, "output": output},
        ensure_ascii=False, indent=2,
    ))
    return {"artifact": result, "output": output}


nb22_c37_result = nb22_c37_run(nb22_c33_result)

{
  "artifact_type": "NB22_DERIVED_NOTEBOOK_PATH_CLEANUP",
  "schema_version": "NB22_C037_ACTION24_V2",
  "status": "SEVEN_DERIVED_NOTEBOOK_COPIES_WRITTEN_AND_RELOADED",
  "outputs": [
    {
      "role": "NB20_A",
      "source_path": "notebooks/phase2/20_answerability_safety_confidence.ipynb",
      "source_file_bytes_sha256": "6d1d03990c8778c1d0c6b3b3cb8c18d2c6ce9062aa4888c062283b75656302f7",
      "source_matches_prior_scan": true,
      "literal_nodes_rewritten": 0,
      "source_path_matches_rewritten": 0,
      "comment_matches_redacted": 0,
      "remaining_personal_path_matches": 0,
      "cell_count": 700,
      "output_structure_preserved": true,
      "derived_path": "reports/phase2/release/NB22.C037.action24/notebooks/20_answerability_safety_confidence.fc36c48ad264e878553922558726d405ddb61f51a8eebe7b3e666e5a4dd8d2bd.ipynb",
      "file_bytes_sha256": "fc36c48ad264e878553922558726d405ddb61f51a8eebe7b3e666e5a4dd8d2bd",
      "bytes": 40169748,
      "write_status": "CREATED"

<div dir="rtl" align="right">

### NB22 — C038 | اقدام ۲۴: بررسی دامنهٔ بستهٔ کاندیدا

سه زیرکار:
۱. استخراج فهرست ۱۸۳ فایل از مانیفست C028 و افزودن هفت نسخهٔ مشتق‌شدهٔ C037؛
۲. کنترل هش فایل‌ها و اسکن الگوهای مشخص در فایل‌های متنی؛
۳. ثبت محل یافته‌ها و وضعیت نسخهٔ ذخیره‌شدهٔ NB22.

هیچ فایل ورودی اصلاح نمی‌شود و هیچ مدل، سل تاریخی یا API اجرا نمی‌شود.
نبود تطبیق، تضمین نبود همهٔ اطلاعات محرمانه نیست.
نسخه‌های مشتق‌شده هنوز عضو مانیفست کاندیدا نیستند.

</div>

In [42]:
def nb22_c38_run(c28, c37):
    import hashlib
    import json
    import re
    from collections import Counter
    from pathlib import Path

    nb22_relative = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / nb22_relative).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0].resolve()

    def binding(row):
        if not isinstance(row, dict):
            return None
        path = row.get("path")
        digest = row.get("file_bytes_sha256") or row.get("sha256")
        if (
            isinstance(path, str)
            and isinstance(digest, str)
            and re.fullmatch(r"[0-9a-f]{64}", digest)
        ):
            return {"path": path, "expected_sha256": digest}
        return None

    # Locate the actual file inventory; do not guess artifact paths.
    inventories = []

    def find_inventory(value):
        if isinstance(value, list):
            if len(value) == 183:
                rows = [binding(item) for item in value]
                if all(row is not None for row in rows):
                    inventories.append(rows)
            for item in value:
                find_inventory(item)
        elif isinstance(value, dict):
            for item in value.values():
                find_inventory(item)

    find_inventory(c28["artifact"])
    if len(inventories) != 1:
        raise RuntimeError(
            "UNIQUE_183_FILE_INVENTORY_NOT_RESOLVED; "
            "NO_FILE_SCAN_STARTED"
        )

    targets = {}
    for row in inventories[0]:
        if row["path"] in targets:
            raise RuntimeError("DUPLICATE_CANDIDATE_FILE")
        targets[row["path"]] = {
            **row, "scope": "C028_CANDIDATE"
        }

    derived = c37["artifact"]["outputs"]
    if len(derived) != 7:
        raise RuntimeError("EXPECTED_SEVEN_DERIVED_NOTEBOOKS")
    for row in derived:
        path = row["derived_path"]
        if path in targets:
            raise RuntimeError("UNEXPECTED_SCOPE_OVERLAP")
        targets[path] = {
            "path": path,
            "expected_sha256": row["file_bytes_sha256"],
            "scope": "C037_DERIVED_NOTEBOOK",
        }

    patterns = {
        "PERSONAL_ABSOLUTE_PATH": re.compile(
            r"(?:/Users/|/home/)[A-Za-z0-9._-]+/"
            r"|[A-Za-z]:\\Users\\[A-Za-z0-9._ -]+\\"
        ),
        "PRIVATE_KEY_HEADER": re.compile(
            r"-----BEGIN (?:RSA |EC |OPENSSH |DSA )?PRIVATE KEY-----"
        ),
        "GITHUB_TOKEN_SHAPE": re.compile(
            r"\b(?:gh[pousr]_[A-Za-z0-9]{20,}"
            r"|github_pat_[A-Za-z0-9_]{30,})\b"
        ),
        "HF_TOKEN_SHAPE": re.compile(r"\bhf_[A-Za-z0-9]{25,}\b"),
        "OPENAI_STYLE_TOKEN_SHAPE": re.compile(
            r"\bsk-(?:proj-|svcacct-)?[A-Za-z0-9_-]{32,}\b"
        ),
        "AWS_ACCESS_KEY_ID_SHAPE": re.compile(
            r"\b(?:AKIA|ASIA)[A-Z0-9]{16}\b"
        ),
        "CREDENTIAL_IN_URL": re.compile(
            r"https?://[^\s/:@\"'<>]+:[^\s/@\"'<>]+@"
        ),
        "CREDENTIAL_LITERAL_ASSIGNMENT": re.compile(
            r"""(?i)\b(?:api_key|access_token|auth_token|"""
            r"""client_secret|password)\b\s*[:=]\s*"""
            r"""["']([^"'\r\n]{12,})["']"""
        ),
    }
    text_suffixes = {
        ".json", ".jsonl", ".ipynb", ".md", ".txt",
        ".py", ".yaml", ".yml", ".toml", ".csv",
        ".ini", ".cfg", ".lock",
    }
    counts = Counter()
    locations = []
    receipts = []
    location_limit = 2000

    print(json.dumps({
        "stage": "CANDIDATE_PRIVACY_SCAN_START",
        "candidate_files": 183,
        "derived_notebooks": 7,
        "model_api_calls": 0,
    }))

    for number, (relative_string, row) in enumerate(
        sorted(targets.items()), start=1
    ):
        relative = Path(relative_string)
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("INVALID_PROJECT_RELATIVE_PATH")
        path = (root / relative).resolve()
        if root not in path.parents:
            raise RuntimeError("FILE_OUTSIDE_PROJECT")

        before = path.stat()
        digest = hashlib.sha256()
        text_file = path.suffix.lower() in text_suffixes
        file_counts = Counter()

        with path.open("rb") as handle:
            if text_file:
                for line_number, raw_line in enumerate(handle, start=1):
                    digest.update(raw_line)
                    # Strict decoding: never silently skip undecodable text.
                    text = raw_line.decode("utf-8")
                    for category, pattern in patterns.items():
                        for match in pattern.finditer(text):
                            if category == "CREDENTIAL_LITERAL_ASSIGNMENT":
                                candidate = match.group(1).lower()
                                if (
                                    any(marker in candidate for marker in (
                                        "placeholder", "your_", "your-",
                                        "example", "redacted", "getenv",
                                        "environ", "<", ">", "${",
                                    ))
                                    or set(candidate) <= {"*", "x", "-"}
                                ):
                                    continue
                            counts[category] += 1
                            file_counts[category] += 1
                            if len(locations) < location_limit:
                                locations.append({
                                    "path": relative_string,
                                    "scope": row["scope"],
                                    "physical_file_line": line_number,
                                    "category": category,
                                    "disposition": "REQUIRES_CONTEXT_REVIEW",
                                })
            else:
                for block in iter(
                    lambda: handle.read(1024 * 1024), b""
                ):
                    digest.update(block)

        after = path.stat()
        if (
            before.st_size != after.st_size
            or before.st_mtime_ns != after.st_mtime_ns
        ):
            raise RuntimeError("FILE_CHANGED_DURING_SCAN")
        actual = digest.hexdigest()
        if actual != row["expected_sha256"]:
            raise RuntimeError(
                f"FILE_IDENTITY_MISMATCH: {relative_string}"
            )

        receipts.append({
            "path": relative_string,
            "scope": row["scope"],
            "file_bytes_sha256": actual,
            "bytes": after.st_size,
            "hash_match": True,
            "scan_mode": (
                "RAW_UTF8_TEXT_PATTERN_SCAN"
                if text_file else "BINARY_HASH_ONLY"
            ),
            "finding_counts": dict(file_counts),
        })
        if number % 25 == 0 or number == len(targets):
            print(json.dumps({
                "stage": "SCAN_PROGRESS",
                "completed": number,
                "total": len(targets),
                "pattern_matches": sum(counts.values()),
            }))

    # Record freshness of the saved active notebook without executing it.
    current_nb22 = root / nb22_relative
    before = current_nb22.stat()
    current_raw = current_nb22.read_bytes()
    after = current_nb22.stat()
    if (
        before.st_size != after.st_size
        or before.st_mtime_ns != after.st_mtime_ns
    ):
        raise RuntimeError("NB22_CHANGED_DURING_FRESHNESS_CHECK")

    current_hash = hashlib.sha256(current_raw).hexdigest()
    saved_notebook = json.loads(current_raw)
    nb22_copy = next(
        row for row in derived if row["role"] == "NB22"
    )

    result = {
        "artifact_type": "NB22_CANDIDATE_SENSITIVE_PATTERN_SCAN",
        "schema_version": "NB22_C038_ACTION24_V1",
        "status": (
            "PACKAGE_SCAN_HAS_REVIEW_CANDIDATES"
            if counts else "NO_DEFINED_RAW_TEXT_PATTERNS_FOUND"
        ),
        "candidate_manifest": c28["output"],
        "derived_notebook_report": c37["output"],
        "files_hash_verified": len(receipts),
        "text_files_scanned": sum(
            row["scan_mode"] == "RAW_UTF8_TEXT_PATTERN_SCAN"
            for row in receipts
        ),
        "binary_files_hash_only": sum(
            row["scan_mode"] == "BINARY_HASH_ONLY"
            for row in receipts
        ),
        "file_receipts": receipts,
        "finding_counts": dict(counts),
        "total_pattern_matches": sum(counts.values()),
        "finding_locations": locations,
        "locations_truncated": sum(counts.values()) > len(locations),
        "matched_values_recorded": False,
        "scan_limitation": (
            "RAW_UTF8_TEXT_PATTERNS_ONLY; JSON_ESCAPED_OR_ENCODED "
            "VALUES_AND_BINARY_CONTENT_NOT_SEMANTICALLY_SCANNED"
        ),
        "saved_nb22": {
            "path": nb22_relative.as_posix(),
            "file_bytes_sha256": current_hash,
            "code_cell_count": sum(
                cell.get("cell_type") == "code"
                for cell in saved_notebook.get("cells", [])
            ),
            "matches_C037_source_snapshot": (
                current_hash == nb22_copy["source_file_bytes_sha256"]
            ),
            "current_kernel_cells_all_saved_established": False,
        },
        "derived_copies_added_to_candidate_manifest": False,
        "cleanup_of_candidate_files_performed": False,
        "action24_fully_complete": False,
        "release_acceptance_granted": False,
        "input_files_modified": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C038.action24",
        "C038.candidate_sensitive_pattern_scan",
        result,
    )
    print(json.dumps({
        "status": result["status"],
        "files_hash_verified": len(receipts),
        "text_files_scanned": result["text_files_scanned"],
        "binary_files_hash_only": result["binary_files_hash_only"],
        "finding_counts": dict(counts),
        "first_review_locations": locations[:20],
        "locations_truncated": result["locations_truncated"],
        "saved_nb22": result["saved_nb22"],
        "action24_fully_complete": False,
        "output": output,
        "new_api_calls": 0,
        "network_calls": 0,
    }, ensure_ascii=False, indent=2))
    return {"artifact": result, "output": output}


nb22_c38_result = nb22_c38_run(
    nb22_c28_result,
    nb22_c37_result,
)

{"stage": "CANDIDATE_PRIVACY_SCAN_START", "candidate_files": 183, "derived_notebooks": 7, "model_api_calls": 0}
{"stage": "SCAN_PROGRESS", "completed": 25, "total": 190, "pattern_matches": 0}
{"stage": "SCAN_PROGRESS", "completed": 50, "total": 190, "pattern_matches": 0}
{"stage": "SCAN_PROGRESS", "completed": 75, "total": 190, "pattern_matches": 0}
{"stage": "SCAN_PROGRESS", "completed": 100, "total": 190, "pattern_matches": 0}
{"stage": "SCAN_PROGRESS", "completed": 125, "total": 190, "pattern_matches": 11}
{"stage": "SCAN_PROGRESS", "completed": 150, "total": 190, "pattern_matches": 11}
{"stage": "SCAN_PROGRESS", "completed": 175, "total": 190, "pattern_matches": 11}
{"stage": "SCAN_PROGRESS", "completed": 190, "total": 190, "pattern_matches": 11}
{
  "status": "PACKAGE_SCAN_HAS_REVIEW_CANDIDATES",
  "files_hash_verified": 190,
  "text_files_scanned": 98,
  "binary_files_hash_only": 92,
  "finding_counts": {
    "PERSONAL_ABSOLUTE_PATH": 11
  },
  "first_review_locations": [
    {
 

<div dir="rtl" align="right">

### NB22 — C039 | اقدام ۲۴: رسیدگی به مسیرهای دو مانیفست تاریخی

سه زیرکار:
۱. خواندن فقط دو مانیفست دارای یافته و کنترل هش با رسید C038؛
۲. شناسایی فیلدهای دارای مسیر شخصی و ساخت نسخه‌های پوشانده‌شدهٔ مستنداتی؛
۳. ثبت پیوند نسخه‌های مشتق‌شده به اصل فایل‌ها و محدودیت مصرف آن‌ها.

اصل مانیفست‌ها و ارجاعات هش تغییر نمی‌کنند.
نسخه‌های مشتق‌شده برای مستندسازی‌اند و جایگزین اجرایی یا عضو مانیفست کاندیدا محسوب نمی‌شوند.

</div>

In [43]:
def nb22_c39_run(c38):
    import hashlib
    import json
    import re
    from pathlib import Path
    from collections import Counter

    nb22_relative = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / nb22_relative).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0].resolve()

    scan = c38["artifact"]
    if scan["locations_truncated"]:
        raise RuntimeError("COMPLETE_FINDING_LOCATIONS_REQUIRED")

    expected_paths = {
        "artifacts/phase2/generation_and_citation/work/outputs/"
        "nb19_development_manifest_v3.json",
        "artifacts/phase2/generation_and_citation/work/outputs/"
        "nb19_development_manifest_v4.json",
    }
    findings = scan["finding_locations"]
    if (
        {row["path"] for row in findings} != expected_paths
        or any(
            row["category"] != "PERSONAL_ABSOLUTE_PATH"
            for row in findings
        )
    ):
        raise RuntimeError("FINDING_SCOPE_DIFFERS_FROM_REVIEWED_OUTPUT")

    receipts = {
        row["path"]: row for row in scan["file_receipts"]
    }
    expected_counts = Counter(row["path"] for row in findings)
    personal = re.compile(
        r"(?:/Users/|/home/)[A-Za-z0-9._-]+/"
        r"|[A-Za-z]:\\Users\\[A-Za-z0-9._ -]+\\"
    )

    prepared = []
    for relative_string in sorted(expected_paths):
        path = (root / relative_string).resolve()
        if root not in path.parents:
            raise RuntimeError("PATH_OUTSIDE_PROJECT")

        before = path.stat()
        raw = path.read_bytes()
        after = path.stat()
        if (
            before.st_size != after.st_size
            or before.st_mtime_ns != after.st_mtime_ns
        ):
            raise RuntimeError("MANIFEST_CHANGED_DURING_READ")
        digest = hashlib.sha256(raw).hexdigest()
        if digest != receipts[relative_string]["file_bytes_sha256"]:
            raise RuntimeError("MANIFEST_CHANGED_SINCE_C038")

        original = json.loads(raw)
        fields = []

        def transform(value, pointer=""):
            if isinstance(value, str):
                matches = list(personal.finditer(value))
                if not matches:
                    return value
                masked = personal.sub("<PERSONAL_HOME>/", value)
                fields.append({
                    "json_pointer": pointer or "/",
                    "pattern_match_count": len(matches),
                    "masked_value_excerpt": masked[:400],
                    "excerpt_truncated": len(masked) > 400,
                })
                return masked
            if isinstance(value, list):
                return [
                    transform(item, f"{pointer}/{index}")
                    for index, item in enumerate(value)
                ]
            if isinstance(value, dict):
                # Do not silently alter keys or identifiers.
                if any(personal.search(key) for key in value):
                    raise RuntimeError(
                        "PERSONAL_PATH_IN_JSON_KEY_REQUIRES_REVIEW"
                    )
                return {
                    key: transform(
                        item,
                        pointer + "/" + key.replace(
                            "~", "~0"
                        ).replace("/", "~1"),
                    )
                    for key, item in value.items()
                }
            return value

        redacted = transform(original)
        decoded_count = sum(
            field["pattern_match_count"] for field in fields
        )
        if decoded_count != expected_counts[relative_string]:
            raise RuntimeError(
                "DECODED_AND_RAW_MATCH_COUNTS_DIFFER; "
                "NO_DERIVED_VIEWS_WRITTEN"
            )

        prepared.append({
            "source": {
                "path": relative_string,
                "file_bytes_sha256": digest,
            },
            "fields": fields,
            "redacted": redacted,
            "stem": Path(relative_string).stem,
        })

    outputs = []
    for item in prepared:
        # Keep the historical payload inside an explicitly documentary
        # wrapper so it cannot be mistaken for its producer manifest.
        document = {
            "artifact_type": "NB22_REDACTED_HISTORICAL_MANIFEST_VIEW",
            "schema_version": "NB22_C039_DOCUMENTATION_VIEW_V1",
            "usage": "DOCUMENTATION_ONLY_NOT_RUNTIME_MANIFEST",
            "source": item["source"],
            "historical_hash_fields_recomputed": False,
            "historical_hash_fields_are_source_claims_only": True,
            "redacted_manifest": item["redacted"],
        }
        receipt = nb22_save_json_artifact(
            root,
            "reports/phase2/release/NB22.C039.action24",
            "C039." + item["stem"] + ".documentation_view",
            document,
        )
        outputs.append({
            "source": item["source"],
            "affected_fields": item["fields"],
            "documentation_view": receipt,
            "runtime_replacement": False,
        })

    result = {
        "artifact_type": "NB22_HISTORICAL_MANIFEST_PATH_HANDLING",
        "schema_version": "NB22_C039_ACTION24_V1",
        "status": "TWO_DOCUMENTATION_VIEWS_WRITTEN_OR_REUSED",
        "source_scan": c38["output"],
        "outputs": outputs,
        "decoded_path_matches_accounted": sum(
            field["pattern_match_count"]
            for item in outputs
            for field in item["affected_fields"]
        ),
        "original_manifests_modified": False,
        "candidate_manifest_modified": False,
        "original_candidate_path_findings_resolved": False,
        "remaining_decision": (
            "Determine whether the historical originals belong in "
            "the distributable package or remain separately bound "
            "provenance evidence; redacted views do not replace them."
        ),
        "action24_fully_complete": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C039.action24",
        "C039.historical_manifest_path_handling",
        result,
    )
    print(json.dumps(
        {**result, "output": output},
        ensure_ascii=False, indent=2,
    ))
    return {"artifact": result, "output": output}


nb22_c39_result = nb22_c39_run(nb22_c38_result)

{
  "artifact_type": "NB22_HISTORICAL_MANIFEST_PATH_HANDLING",
  "schema_version": "NB22_C039_ACTION24_V1",
  "status": "TWO_DOCUMENTATION_VIEWS_WRITTEN_OR_REUSED",
  "source_scan": {
    "path": "reports/phase2/release/NB22.C038.action24/C038.candidate_sensitive_pattern_scan.3e3c4c37d87339cb12169dbe860da3f1b2f56404acb6a49fc25d779c4cc1a6f3.json",
    "file_bytes_sha256": "3e3c4c37d87339cb12169dbe860da3f1b2f56404acb6a49fc25d779c4cc1a6f3",
    "bytes": 70142,
    "write_status": "CREATED",
    "reload_validation": "PASS"
  },
  "outputs": [
    {
      "source": {
        "path": "artifacts/phase2/generation_and_citation/work/outputs/nb19_development_manifest_v3.json",
        "file_bytes_sha256": "9f3c041917f6e7845a8d26909c745a146dec2d5c256cdeeee9ac52e350a7b20a"
      },
      "affected_fields": [
        {
          "json_pointer": "/outputs/internal_claim_gate_v2/path",
          "pattern_match_count": 1,
          "masked_value_excerpt": "<PERSONAL_HOME>/AMIR_DATA/00_WORKS/01_PROJECT

<div dir="rtl" align="right">

### NB22 — C040 | اقدام ۲۴: اتصال ارجاع‌های تاریخی به مسیرهای نسبی

سه زیرکار:
۱. استخراج دقیق ۱۱ ارجاع از دو مانیفست دارای هش تأییدشده؛
۲. بررسی مقصدها و استفادهٔ مجدد از رسیدهای C038، یا محاسبهٔ هش فقط برای مقصد فاقد رسید؛
۳. ثبت نگاشت «هش مانیفست اصلی + JSON Pointer → مسیر نسبی + هش مقصد».

این نگاشت، مانیفست تاریخی را بازنویسی نمی‌کند.
فایل‌های مقصد صرفاً برای هویت فایل بررسی می‌شوند؛ هیچ پردازش، مدل یا API اجرا نمی‌شود.

</div>

In [44]:
def nb22_c40_run(c38, c39):
    import hashlib
    import json
    from pathlib import Path, PurePosixPath

    relative_nb22 = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / relative_nb22).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0].resolve()

    receipts = {
        row["path"]: row
        for row in c38["artifact"]["file_receipts"]
    }
    verified_targets = {}
    links = []

    def load_bound_manifest(source):
        path = root / source["path"]
        before = path.stat()
        raw = path.read_bytes()
        after = path.stat()
        if (
            before.st_size != after.st_size
            or before.st_mtime_ns != after.st_mtime_ns
        ):
            raise RuntimeError("MANIFEST_CHANGED_DURING_READ")
        if hashlib.sha256(raw).hexdigest() != source[
            "file_bytes_sha256"
        ]:
            raise RuntimeError("MANIFEST_SOURCE_IDENTITY_CHANGED")
        return json.loads(raw)

    def at_pointer(value, pointer):
        if not pointer.startswith("/"):
            raise RuntimeError("INVALID_JSON_POINTER")
        for segment in pointer[1:].split("/"):
            key = segment.replace("~1", "/").replace("~0", "~")
            value = value[int(key)] if isinstance(value, list) else value[key]
        return value

    def verify_target(relative):
        if relative in verified_targets:
            return verified_targets[relative]

        path = root / relative
        resolved = path.resolve()
        if root not in resolved.parents or not resolved.is_file():
            raise RuntimeError(
                f"TARGET_NOT_AVAILABLE_INSIDE_PROJECT: {relative}"
            )

        previous = receipts.get(relative)
        if previous is not None:
            # Reuse the prior byte-identity check. A size check alone
            # is not presented as fresh hash verification.
            observed_size = resolved.stat().st_size
            if observed_size != previous["bytes"]:
                raise RuntimeError("TARGET_SIZE_CHANGED_SINCE_C038")
            result = {
                "path": relative,
                "file_bytes_sha256": previous["file_bytes_sha256"],
                "bytes": previous["bytes"],
                "verification_mode": "REUSED_C038_BYTE_HASH_RECEIPT",
                "current_existence_and_size_match": True,
                "fresh_hash_computed": False,
                "receipt_source": c38["output"],
            }
        else:
            before = resolved.stat()
            digest = hashlib.sha256()
            with resolved.open("rb") as handle:
                for block in iter(
                    lambda: handle.read(1024 * 1024), b""
                ):
                    digest.update(block)
            after = resolved.stat()
            if (
                before.st_size != after.st_size
                or before.st_mtime_ns != after.st_mtime_ns
            ):
                raise RuntimeError("TARGET_CHANGED_DURING_HASH")
            result = {
                "path": relative,
                "file_bytes_sha256": digest.hexdigest(),
                "bytes": after.st_size,
                "verification_mode": "FRESH_EXPLICIT_TARGET_BYTE_HASH",
                "fresh_hash_computed": True,
                "historical_declared_hash_comparison_claimed": False,
            }

        verified_targets[relative] = result
        return result

    for document in c39["artifact"]["outputs"]:
        source = document["source"]
        manifest = load_bound_manifest(source)

        for field in document["affected_fields"]:
            pointer = field["json_pointer"]
            original_path = at_pointer(manifest, pointer)
            if not isinstance(original_path, str):
                raise RuntimeError("REFERENCE_PATH_IS_NOT_STRING")

            # Exact repository boundary observed in C039, not a
            # filename search or a choice among guessed locations.
            marker = "/RAG-projects/RAG_finance/"
            if original_path.count(marker) != 1:
                raise RuntimeError("EXPECTED_REPOSITORY_BOUNDARY_MISSING")
            relative = original_path.split(marker, 1)[1]
            posix = PurePosixPath(relative)
            if (
                not relative
                or posix.is_absolute()
                or ".." in posix.parts
                or "\\" in relative
            ):
                raise RuntimeError("UNSAFE_RELATIVE_REFERENCE")

            relative = posix.as_posix()
            target = verify_target(relative)
            links.append({
                "source_manifest": source,
                "json_pointer": pointer,
                "project_relative_target": relative,
                "target_file_bytes_sha256": target[
                    "file_bytes_sha256"
                ],
                "target_verification_mode": target[
                    "verification_mode"
                ],
                "original_absolute_path_recorded": False,
            })

    if len(links) != 11:
        raise RuntimeError("EXPECTED_ELEVEN_REFERENCE_BINDINGS")

    result = {
        "artifact_type": "NB22_PROJECT_RELATIVE_REFERENCE_OVERLAY",
        "schema_version": "NB22_C040_ACTION24_V1",
        "status": "ELEVEN_RELATIVE_REFERENCE_BINDINGS_ESTABLISHED",
        "source_handling_report": c39["output"],
        "bindings": links,
        "unique_target_receipts": list(verified_targets.values()),
        "binding_count": len(links),
        "unique_target_count": len(verified_targets),
        "prior_target_receipts_reused": sum(
            not row["fresh_hash_computed"]
            for row in verified_targets.values()
        ),
        "new_target_hashes_computed": sum(
            row["fresh_hash_computed"]
            for row in verified_targets.values()
        ),
        "overlay_consumed_by_runtime": False,
        "original_manifests_modified": False,
        "candidate_manifest_modified": False,
        "historical_originals_distribution_decision": "OPEN",
        "original_candidate_path_findings_resolved": False,
        "action24_fully_complete": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C040.action24",
        "C040.project_relative_reference_overlay",
        result,
    )
    print(json.dumps({
        "status": result["status"],
        "binding_count": len(links),
        "unique_target_count": result["unique_target_count"],
        "prior_target_receipts_reused": result[
            "prior_target_receipts_reused"
        ],
        "new_target_hashes_computed": result[
            "new_target_hashes_computed"
        ],
        "bindings": links,
        "original_manifests_modified": False,
        "overlay_consumed_by_runtime": False,
        "historical_originals_distribution_decision": "OPEN",
        "action24_fully_complete": False,
        "output": output,
        "new_api_calls": 0,
        "network_calls": 0,
    }, ensure_ascii=False, indent=2))
    return {"artifact": result, "output": output}


nb22_c40_result = nb22_c40_run(
    nb22_c38_result,
    nb22_c39_result,
)

{
  "status": "ELEVEN_RELATIVE_REFERENCE_BINDINGS_ESTABLISHED",
  "binding_count": 11,
  "unique_target_count": 8,
  "prior_target_receipts_reused": 2,
  "new_target_hashes_computed": 6,
  "bindings": [
    {
      "source_manifest": {
        "path": "artifacts/phase2/generation_and_citation/work/outputs/nb19_development_manifest_v3.json",
        "file_bytes_sha256": "9f3c041917f6e7845a8d26909c745a146dec2d5c256cdeeee9ac52e350a7b20a"
      },
      "json_pointer": "/outputs/internal_claim_gate_v2/path",
      "project_relative_target": "artifacts/phase2/generation_and_citation/work/outputs/nb19_development_internal_claim_gate_v2.jsonl",
      "target_file_bytes_sha256": "ac9e8cfc54d9d544f7d82f982007e6fdfb9c35fafb879e75cac51703c6835f93",
      "target_verification_mode": "FRESH_EXPLICIT_TARGET_BYTE_HASH",
      "original_absolute_path_recorded": false
    },
    {
      "source_manifest": {
        "path": "artifacts/phase2/generation_and_citation/work/outputs/nb19_development_manifest

<div dir="rtl" align="right">

### NB22 — C041 | اعتبارسنجی مستقل نگاشت مسیرهای نسبی

سه زیرکار:
۱. آماده‌سازی نسخهٔ موقت از نگاشت، دو مانیفست اصلی و مقصدهای صریح آن؛
۲. مصرف نگاشت در فرایند مستقل و بررسی هش مبدأ، وجود JSON Pointer و هش مقصد برای هر ۱۱ ارجاع؛
۳. حذف پوشهٔ موقت و ثبت نتیجه.

این آزمون، اعتبارسنجی محدود نگاشت است؛ اجرای محصول، اجرای نوت‌بوک یا پذیرش ریلیز نیست.
فایل‌های اصلی تغییر نمی‌کنند و مسیرهای شخصی چاپ نمی‌شوند.

</div>

In [45]:
def nb22_c41_run(c40):
    import hashlib
    import json
    import shutil
    import subprocess
    import sys
    import tempfile
    from pathlib import Path

    nb22_relative = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / nb22_relative).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0].resolve()
    overlay = c40["artifact"]
    if len(overlay["bindings"]) != 11:
        raise RuntimeError("EXPECTED_ELEVEN_BINDINGS")

    required = {}

    def add(path, digest):
        rel = Path(path)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError("INVALID_RELATIVE_PATH")
        previous = required.get(path)
        if previous is not None and previous != digest:
            raise RuntimeError("CONFLICTING_FILE_IDENTITIES")
        required[path] = digest

    for link in overlay["bindings"]:
        source = link["source_manifest"]
        add(source["path"], source["file_bytes_sha256"])
        add(
            link["project_relative_target"],
            link["target_file_bytes_sha256"],
        )
    add(
        c40["output"]["path"],
        c40["output"]["file_bytes_sha256"],
    )

    child_code = r'''
import hashlib
import json
import socket
import sys
from pathlib import Path

attempts = []
def deny_network(*args, **kwargs):
    attempts.append(True)
    raise RuntimeError("NETWORK_DISABLED_FOR_OVERLAY_VALIDATION")

socket.socket = deny_network
socket.create_connection = deny_network
socket.getaddrinfo = deny_network

root = Path.cwd().resolve()
def checked_path(relative):
    path = Path(relative)
    if path.is_absolute() or ".." in path.parts:
        raise RuntimeError("INVALID_RELATIVE_PATH")
    resolved = (root / path).resolve()
    if root not in resolved.parents:
        raise RuntimeError("PATH_OUTSIDE_ISOLATED_ROOT")
    return resolved

def read_verified(relative, expected):
    raw = checked_path(relative).read_bytes()
    if hashlib.sha256(raw).hexdigest() != expected:
        raise RuntimeError("ISOLATED_FILE_HASH_MISMATCH")
    return raw

overlay = json.loads(read_verified(sys.argv[1], sys.argv[2]))
manifests = {}
targets = {}
count = 0

for link in overlay["bindings"]:
    source = link["source_manifest"]
    identity = (source["path"], source["file_bytes_sha256"])
    if identity not in manifests:
        manifests[identity] = json.loads(
            read_verified(*identity)
        )
    value = manifests[identity]
    pointer = link["json_pointer"]
    if not pointer.startswith("/"):
        raise RuntimeError("INVALID_JSON_POINTER")
    for segment in pointer[1:].split("/"):
        key = segment.replace("~1", "/").replace("~0", "~")
        value = value[int(key)] if isinstance(value, list) else value[key]
    if not isinstance(value, str):
        raise RuntimeError("SOURCE_REFERENCE_NOT_STRING")

    marker = "/RAG-projects/RAG_finance/"
    if (
        value.count(marker) != 1
        or value.split(marker, 1)[1]
        != link["project_relative_target"]
    ):
        raise RuntimeError("OVERLAY_DOES_NOT_MATCH_SOURCE_REFERENCE")

    target_identity = (
        link["project_relative_target"],
        link["target_file_bytes_sha256"],
    )
    if target_identity not in targets:
        read_verified(*target_identity)
        targets[target_identity] = True
    count += 1

if count != 11:
    raise RuntimeError("BINDING_COUNT_MISMATCH")

print(json.dumps({
    "status": "ISOLATED_RELATIVE_REFERENCE_VALIDATION_PASS",
    "bindings_verified": count,
    "source_manifests_verified": len(manifests),
    "unique_targets_hash_verified": len(targets),
    "network_guard_enabled": True,
    "blocked_network_attempt_count": len(attempts),
    "original_absolute_paths_used_for_file_access": False,
    "product_runtime_executed": False
}))
'''

    print(json.dumps({
        "stage": "ISOLATED_OVERLAY_VALIDATION_START",
        "explicit_files_to_copy": len(required),
        "reference_bindings": 11,
        "model_api_calls": 0,
    }))

    temporary_path = None
    child_result = None
    with tempfile.TemporaryDirectory(
        prefix="nb22_c41_overlay_"
    ) as temporary:
        temporary_path = Path(temporary)
        for relative, expected in required.items():
            source = (root / relative).resolve()
            if root not in source.parents or not source.is_file():
                raise RuntimeError("SOURCE_NOT_INSIDE_PROJECT")
            destination = temporary_path / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, destination)

            digest = hashlib.sha256()
            with destination.open("rb") as handle:
                for block in iter(
                    lambda: handle.read(1024 * 1024), b""
                ):
                    digest.update(block)
            if digest.hexdigest() != expected:
                raise RuntimeError("COPIED_FILE_IDENTITY_MISMATCH")

        process = subprocess.run(
            [
                sys.executable, "-I", "-c", child_code,
                c40["output"]["path"],
                c40["output"]["file_bytes_sha256"],
            ],
            cwd=temporary_path,
            capture_output=True,
            text=True,
            timeout=60,
            check=False,
        )
        if process.returncode != 0:
            # Avoid printing traceback values that may contain paths.
            raise RuntimeError(
                "ISOLATED_OVERLAY_VALIDATION_FAILED; "
                f"exit_code={process.returncode}"
            )
        child_result = json.loads(process.stdout)
        if child_result.get("status") != (
            "ISOLATED_RELATIVE_REFERENCE_VALIDATION_PASS"
        ):
            raise RuntimeError("UNEXPECTED_CHILD_VALIDATION_STATUS")

    removed = not temporary_path.exists()
    if not removed:
        raise RuntimeError("TEMPORARY_DIRECTORY_CLEANUP_FAILED")

    result = {
        "artifact_type": "NB22_ISOLATED_REFERENCE_OVERLAY_VALIDATION",
        "schema_version": "NB22_C041_REFERENCE_VALIDATE_V1",
        "status": child_result["status"],
        "overlay": c40["output"],
        "scope": "ELEVEN_EXPLICIT_REFERENCE_BINDINGS_ONLY",
        "files_copied_and_hash_verified": len(required),
        "independent_process_result": child_result,
        "temporary_directory_removed": removed,
        "overlay_consumed_by_validate_only_process": True,
        "overlay_integrated_into_product_runtime": False,
        "historical_originals_distribution_decision": "OPEN",
        "original_candidate_path_findings_resolved": False,
        "action24_fully_complete": False,
        "release_acceptance_granted": False,
        "input_files_modified": False,
        "new_api_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C041.action24",
        "C041.isolated_reference_overlay_validation",
        result,
    )
    print(json.dumps(
        {**result, "output": output},
        ensure_ascii=False, indent=2,
    ))
    return {"artifact": result, "output": output}


nb22_c41_result = nb22_c41_run(nb22_c40_result)

{"stage": "ISOLATED_OVERLAY_VALIDATION_START", "explicit_files_to_copy": 10, "reference_bindings": 11, "model_api_calls": 0}
{
  "artifact_type": "NB22_ISOLATED_REFERENCE_OVERLAY_VALIDATION",
  "schema_version": "NB22_C041_REFERENCE_VALIDATE_V1",
  "status": "ISOLATED_RELATIVE_REFERENCE_VALIDATION_PASS",
  "overlay": {
    "path": "reports/phase2/release/NB22.C040.action24/C040.project_relative_reference_overlay.23d3ff4f4d9dc5a1dc813b4278812cb8e87091a32796de412cd48651ea6130fa.json",
    "file_bytes_sha256": "23d3ff4f4d9dc5a1dc813b4278812cb8e87091a32796de412cd48651ea6130fa",
    "bytes": 10711,
    "write_status": "CREATED",
    "reload_validation": "PASS"
  },
  "scope": "ELEVEN_EXPLICIT_REFERENCE_BINDINGS_ONLY",
  "files_copied_and_hash_verified": 10,
  "independent_process_result": {
    "status": "ISOLATED_RELATIVE_REFERENCE_VALIDATION_PASS",
    "bindings_verified": 11,
    "source_manifests_verified": 2,
    "unique_targets_hash_verified": 8,
    "network_guard_enabled": true,
   

<div dir="rtl" align="right">

### NB22 — C042 | به‌روزرسانی کاندیدا؛ مرتبط با اقدامات ۱۲، ۱۸ و آماده‌سازی ۲۵

سه زیرکار:
۱. حفظ ۱۸۳ ورودی کاندیدای C028؛
۲. افزودن نسخه‌های مشتق‌شده، مقصدهای جدید و شواهد اعتبارسنجی با استفاده از رسیدهای موجود؛
۳. ذخیره و بازخوانی مستقل مانیفست پیش‌نویس جدید.

فایل‌های داده دوباره خوانده یا هش نمی‌شوند.
پوشش آزمون بازیابی C030 متعلق به کاندیدای قبلی است؛ آزمون C041 فقط نگاشت ارجاع‌ها را پوشش می‌دهد.
این مانیفست بستهٔ انتشار پذیرفته‌شده نیست.

</div>

In [46]:
def nb22_c42_run(c28, c30, c37, c38, c39, c40, c41):
    import copy
    import json
    import re
    from pathlib import Path

    relative = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / relative).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0]

    if c41["artifact"]["status"] != (
        "ISOLATED_RELATIVE_REFERENCE_VALIDATION_PASS"
    ):
        raise RuntimeError("REFERENCE_VALIDATION_PASS_REQUIRED")
    if not c41["artifact"]["temporary_directory_removed"]:
        raise RuntimeError("TEMPORARY_CLEANUP_REQUIRED")

    def file_binding(row):
        if not isinstance(row, dict):
            return None
        path = row.get("path")
        digest = row.get("file_bytes_sha256") or row.get("sha256")
        if (
            isinstance(path, str)
            and isinstance(digest, str)
            and re.fullmatch(r"[0-9a-f]{64}", digest)
        ):
            return path, digest
        return None

    inventories = []

    def find_inventory(value):
        if isinstance(value, list):
            if len(value) == 183 and all(
                file_binding(row) is not None for row in value
            ):
                inventories.append(value)
            for item in value:
                find_inventory(item)
        elif isinstance(value, dict):
            for item in value.values():
                find_inventory(item)

    find_inventory(c28["artifact"])
    if len(inventories) != 1:
        raise RuntimeError("UNIQUE_PARENT_INVENTORY_REQUIRED")

    files = {}
    for row in inventories[0]:
        path, digest = file_binding(row)
        if path in files:
            raise RuntimeError("DUPLICATE_PARENT_PATH")
        files[path] = {
            "path": path,
            "file_bytes_sha256": digest,
            "parent_inventory_entry": copy.deepcopy(row),
            "roles": ["INHERITED_C028_ASSET"],
            "identity_source": c28["output"],
        }

    def add(receipt, role, identity_source):
        path = receipt["path"]
        digest = receipt["file_bytes_sha256"]
        if (
            Path(path).is_absolute()
            or ".." in Path(path).parts
            or not re.fullmatch(r"[0-9a-f]{64}", digest)
        ):
            raise RuntimeError("INVALID_ADDED_FILE_BINDING")

        if path in files:
            if files[path]["file_bytes_sha256"] != digest:
                raise RuntimeError("CONFLICTING_ASSET_IDENTITY")
            if role not in files[path]["roles"]:
                files[path]["roles"].append(role)
            return

        files[path] = {
            "path": path,
            "file_bytes_sha256": digest,
            "roles": [role],
            "identity_source": identity_source,
        }
        if "bytes" in receipt:
            files[path]["bytes"] = receipt["bytes"]

    for row in c37["artifact"]["outputs"]:
        add({
            "path": row["derived_path"],
            "file_bytes_sha256": row["file_bytes_sha256"],
            "bytes": row["bytes"],
        }, "DERIVED_NOTEBOOK_NOT_EXECUTION_ACCEPTED", c37["output"])

    for row in c39["artifact"]["outputs"]:
        add(
            row["documentation_view"],
            "REDACTED_DOCUMENTATION_VIEW_NOT_RUNTIME_MANIFEST",
            c39["output"],
        )

    for row in c40["artifact"]["unique_target_receipts"]:
        add(row, "EXPLICIT_REFERENCE_TARGET", c40["output"])

    for result, role in [
        (c30, "BOUNDED_PARENT_CANDIDATE_RESTORE_EVIDENCE"),
        (c37, "DERIVED_NOTEBOOK_TRANSFORMATION_REPORT"),
        (c38, "PARENT_AND_DERIVED_ASSET_SCAN_REPORT"),
        (c39, "HISTORICAL_MANIFEST_HANDLING_REPORT"),
        (c40, "PROJECT_RELATIVE_REFERENCE_OVERLAY"),
        (c41, "ISOLATED_REFERENCE_OVERLAY_VALIDATION"),
    ]:
        add(result["output"], role, result["output"])

    historical_paths = sorted({
        row["source"]["path"]
        for row in c39["artifact"]["outputs"]
    })
    for path in historical_paths:
        if path not in files:
            raise RuntimeError("HISTORICAL_PROVENANCE_ASSET_MISSING")
        files[path]["distribution_status"] = (
            "RETAINED_LOCAL_PROVENANCE_PUBLIC_DISTRIBUTION_NOT_APPROVED"
        )

    result = {
        "artifact_type": "NB22_EXPANDED_DRAFT_CANDIDATE_MANIFEST",
        "schema_version": "NB22_C042_CANDIDATE_V1",
        "status": "EXPANDED_LOCAL_DRAFT_WITH_OPEN_RELEASE_REQUIREMENTS",
        "parent_manifest": c28["output"],
        "files": [files[path] for path in sorted(files)],
        "manifest_file_count": len(files),
        "inherited_file_count": 183,
        "added_unique_file_count": len(files) - 183,
        "parent_files_removed": 0,
        "version_and_quality_evidence_inherited_from": c28["output"],
        "evidence_index_status": (
            "PARENT_INDEX_PRESERVED_NEW_FINAL_INDEX_NOT_BUILT"
        ),
        "restore_coverage": {
            "C030": {
                "evidence": c30["output"],
                "scope": "C028_CANDIDATE_AND_DENSE_SNAPSHOT_ONLY",
            },
            "C041": {
                "evidence": c41["output"],
                "scope": "ELEVEN_REFERENCE_BINDINGS_ONLY",
            },
            "expanded_candidate_full_restore_verified": False,
        },
        "publication": {
            "status": "LOCAL_DRAFT_NOT_APPROVED_FOR_PUBLIC_DISTRIBUTION",
            "historical_originals": historical_paths,
            "restriction_is_metadata_not_access_control": True,
        },
        "open_requirements": [
            "Bind selected NB21 response population.",
            "Establish authoritative producer-code identity.",
            "Bind complete product policy.",
            "Resolve split/gold producer hash definitions.",
            "Complete reproducible environment and model dependency closure.",
            "Complete agreed consolidation and clean execution.",
            "Refresh the saved final NB22 notebook and its derived copy.",
            "Close publication privacy findings and scan newly added assets.",
            "Build final evidence index and validate final candidate.",
            "Resolve or formally disposition quality FAIL/BLOCKED issues.",
        ],
        "release_id_assigned": False,
        "release_acceptance_granted": False,
        "action24_fully_complete": False,
        "action25_fully_complete": False,
        "raw_data_files_reread": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C042.candidate",
        "C042.expanded_draft_candidate",
        result,
    )
    print(json.dumps({
        "status": result["status"],
        "inherited_file_count": 183,
        "manifest_file_count": len(files),
        "added_unique_file_count": len(files) - 183,
        "parent_files_removed": 0,
        "publication_status": result["publication"]["status"],
        "expanded_candidate_full_restore_verified": False,
        "release_acceptance_granted": False,
        "open_requirements": result["open_requirements"],
        "output": output,
        "raw_data_files_reread": 0,
        "new_api_calls": 0,
        "network_calls": 0,
    }, ensure_ascii=False, indent=2))
    return {"artifact": result, "output": output}


nb22_c42_result = nb22_c42_run(
    nb22_c28_result,
    nb22_c30_result,
    nb22_c37_result,
    nb22_c38_result,
    nb22_c39_result,
    nb22_c40_result,
    nb22_c41_result,
)

{
  "status": "EXPANDED_LOCAL_DRAFT_WITH_OPEN_RELEASE_REQUIREMENTS",
  "inherited_file_count": 183,
  "manifest_file_count": 204,
  "added_unique_file_count": 21,
  "parent_files_removed": 0,
  "publication_status": "LOCAL_DRAFT_NOT_APPROVED_FOR_PUBLIC_DISTRIBUTION",
  "expanded_candidate_full_restore_verified": false,
  "release_acceptance_granted": false,
  "open_requirements": [
    "Bind selected NB21 response population.",
    "Establish authoritative producer-code identity.",
    "Bind complete product policy.",
    "Resolve split/gold producer hash definitions.",
    "Complete reproducible environment and model dependency closure.",
    "Complete agreed consolidation and clean execution.",
    "Refresh the saved final NB22 notebook and its derived copy.",
    "Close publication privacy findings and scan newly added assets.",
    "Build final evidence index and validate final candidate.",
    "Resolve or formally disposition quality FAIL/BLOCKED issues."
  ],
  "output": {
    "p

<div dir="rtl" align="right">

### NB22 — C043 | بررسی منشأ جمعیت پاسخ‌های منتخب NB21

سه زیرکار:
۱. یافتن گزارش‌های C082 و C085 در داده‌های موجود در حافظه؛
۲. استخراج ارجاع‌های JSONL همراه با محل اشاره و هش اعلام‌شده، در صورت وجود؛
۳. نمایش شواهد برای تعیین اتصال معتبر پاسخ‌های منتخب.

هیچ فایل داده‌ای خوانده، پاسخ جدیدی تولید یا API فراخوانی نمی‌شود.
وجود یک ارجاع، به‌تنهایی انتخاب نهایی آن جمعیت را اثبات نمی‌کند.

</div>

In [47]:
def nb22_c43_run(c03, c06):
    import hashlib
    import json
    from collections import defaultdict

    caches = {
        "C03.loaded": c03["loaded"],
        "C06.package_payloads": c06["package_payloads"],
    }
    reports = {}
    seen_containers = set()

    def identify_report(value, trace):
        if not isinstance(value, (dict, list)):
            return
        identity = id(value)
        if identity in seen_containers:
            return
        seen_containers.add(identity)

        if isinstance(value, dict):
            schema = value.get("schema_version", "")
            artifact_type = value.get("artifact_type", "")
            header = f"{schema} {artifact_type}".upper()
            trace_upper = trace.upper()

            for label in ("C082", "C085"):
                header_match = label in header and "NB21" in header
                path_match = (
                    label in trace_upper
                    and "NB21" in trace_upper
                    and any(
                        key in value
                        for key in (
                            "schema_version", "artifact_type",
                            "status", "outputs",
                        )
                    )
                )
                if header_match or path_match:
                    encoded = json.dumps(
                        value, ensure_ascii=False, sort_keys=True,
                        separators=(",", ":"), allow_nan=False,
                    ).encode("utf-8")
                    digest = hashlib.sha256(encoded).hexdigest()
                    key = (label, digest)
                    if key not in reports:
                        reports[key] = {
                            "report": label,
                            "cache_location": trace,
                            "payload": value,
                        }

            for key, item in value.items():
                identify_report(item, f"{trace}/{key}")
        else:
            for index, item in enumerate(value):
                identify_report(item, f"{trace}/{index}")

    for name, cache in caches.items():
        identify_report(cache, name)

    if not reports:
        raise RuntimeError(
            "C082_C085_PAYLOADS_NOT_IDENTIFIED_IN_EXISTING_CACHES; "
            "NO_FILE_READ_OR_ARTIFACT_SELECTION_PERFORMED"
        )

    references = []

    def collect(value, trace, report, cache_location, parent=None):
        if isinstance(value, dict):
            for key, item in value.items():
                collect(
                    item, f"{trace}/{key}",
                    report, cache_location, value,
                )
        elif isinstance(value, list):
            for index, item in enumerate(value):
                collect(
                    item, f"{trace}/{index}",
                    report, cache_location, parent,
                )
        elif isinstance(value, str):
            candidate = value.strip()
            if not candidate.lower().endswith(".jsonl"):
                return

            # Report only project-relative references.
            if (
                candidate.startswith(("/", "\\"))
                or "\\" in candidate
                or ".." in candidate.split("/")
            ):
                references.append({
                    "report": report,
                    "field": trace,
                    "status": "NON_RELATIVE_REFERENCE_NOT_DISPLAYED",
                })
                return

            declarations = {}
            if isinstance(parent, dict):
                for key in (
                    "file_bytes_sha256", "sha256",
                    "output_sha256", "artifact_sha256",
                ):
                    digest = parent.get(key)
                    if (
                        isinstance(digest, str)
                        and len(digest) == 64
                        and all(c in "0123456789abcdef" for c in digest)
                    ):
                        declarations[key] = digest

            references.append({
                "report": report,
                "cache_location": cache_location,
                "field": trace,
                "reference_path": candidate,
                "same_object_hash_declarations": declarations,
                "hash_definition_established_here": False,
                "selected_answer_population_established": False,
            })

    summaries = []
    for item in reports.values():
        payload = item["payload"]
        summaries.append({
            "report": item["report"],
            "cache_location": item["cache_location"],
            "schema_version": payload.get("schema_version"),
            "artifact_type": payload.get("artifact_type"),
            "top_level_keys": sorted(payload),
        })
        collect(
            payload, "",
            item["report"], item["cache_location"],
        )

    # Deduplicate identical extracted observations.
    unique = {}
    for row in references:
        key = json.dumps(row, ensure_ascii=False, sort_keys=True)
        unique[key] = row
    references = list(unique.values())

    result = {
        "status": (
            "CACHED_NB21_JSONL_REFERENCES_EXTRACTED"
            if references
            else "NO_DIRECT_JSONL_REFERENCES_IN_IDENTIFIED_REPORTS"
        ),
        "identified_reports": summaries,
        "jsonl_references": references,
        "selected_answer_population_binding_complete": False,
        "input_files_read": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result


nb22_c43_result = nb22_c43_run(
    nb22_c03_result,
    nb22_c06_result,
)

{
  "status": "CACHED_NB21_JSONL_REFERENCES_EXTRACTED",
  "identified_reports": [
    {
      "report": "C082",
      "cache_location": "C03.loaded/reports/phase2/evaluation/NB21C.C082.action46_handoff/9186bb5a30456486b29d521db1660b944c72bf8c4fcb9d816cf578b1937ea1d9/C082.handoff_NB22.aeffc73207cc137a492496eb3083d29088afeab4447bc749faafc6aea397bdec.json",
      "schema_version": "NB21C_C082_NB22_HANDOFF_V1",
      "artifact_type": null,
      "top_level_keys": [
        "acceptance",
        "consumer",
        "consumer_protocol",
        "distinct_open_issue_count",
        "evidence_index",
        "handoff_status",
        "limits",
        "manifest",
        "metric_gate_status_counts",
        "open_conditions",
        "overall_gate_status",
        "producer",
        "schema_version"
      ]
    },
    {
      "report": "C082",
      "cache_location": "C03.loaded/reports/phase2/evaluation/NB21C.C082.action46_handoff/9186bb5a30456486b29d521db1660b944c72bf8c4fcb9d816cf578b1937ea

<div dir="rtl" align="right">

### NB22 — C044 | بررسی دقیق اسناد اتصال پاسخ‌ها

سه زیرکار:
۱. انتخاب دقیق مانیفست C082 و قرارداد فریز‌شده از حافظه؛
۲. استخراج فهرست منابع و اتصال ورودی‌های قرارداد؛
۳. نمایش زمینهٔ انتخاب پاسخ‌ها، با پوشاندن مسیرهای شخصی.

رکوردهای داخلی، گزارش مستقل محسوب نمی‌شوند.
هیچ جمعیت پاسخی انتخاب یا پذیرفته نمی‌شود و هیچ فایل یا API خوانده نمی‌شود.

</div>

In [48]:
def nb22_c44_run(c03):
    import json
    import re

    loaded = c03["loaded"]
    if not isinstance(loaded, dict):
        raise RuntimeError("EXPECTED_PATH_KEYED_C03_CACHE")

    names = {
        "handoff_manifest": (
            "C082.manifest."
            "212f8f411a5c39b693e108c5ba3100a291e40355625e60ab14bae343af88e174.json"
        ),
        "frozen_contract": (
            "C082.frozen_contract."
            "edc54bb349c632e3424aceb8da881d124d681786bf0f384be0dffdfca0ae2a6b.json"
        ),
    }
    documents = {}
    for role, filename in names.items():
        matches = [
            (path, payload)
            for path, payload in loaded.items()
            if isinstance(path, str)
            and path.rsplit("/", 1)[-1] == filename
        ]
        if len(matches) != 1:
            raise RuntimeError(
                f"EXACT_CACHED_DOCUMENT_NOT_UNIQUE: {role}"
            )
        path, payload = matches[0]
        if not isinstance(payload, dict):
            raise RuntimeError(f"EXPECTED_DOCUMENT_OBJECT: {role}")
        documents[role] = {"path": path, "payload": payload}

    selections = {
        "handoff_manifest": [
            "schema_version",
            "scope",
            "population_count",
            "original_raw_artifact_closure_claimed",
            "product_release_bundle",
            "source_inventory",
        ],
        "frozen_contract": [
            "schema_version",
            "input_bindings",
            "configuration",
            "coverage_policy",
            "decision_provenance",
        ],
    }

    personal = re.compile(
        r"(?:/Users/|/home/)[A-Za-z0-9._-]+/"
        r"|[A-Za-z]:\\Users\\[A-Za-z0-9._ -]+\\"
    )

    def mask(value):
        if isinstance(value, str):
            return personal.sub("<PERSONAL_HOME>/", value)
        if isinstance(value, list):
            return [mask(item) for item in value]
        if isinstance(value, dict):
            return {key: mask(item) for key, item in value.items()}
        return value

    extracted = {}
    for role, fields in selections.items():
        document = documents[role]
        payload = document["payload"]
        extracted[role] = {
            "cached_source_path": document["path"],
            "sections": {
                field: mask(payload[field])
                for field in fields if field in payload
            },
            "requested_fields_not_present": [
                field for field in fields if field not in payload
            ],
        }

    result = {
        "status": "EXACT_CACHED_BINDING_SECTIONS_EXTRACTED",
        "documents_inspected": 2,
        "extracted": extracted,
        "selected_answer_population_binding_complete": False,
        "input_files_read": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    print(json.dumps(result, ensure_ascii=False, indent=2))
    return result


nb22_c44_result = nb22_c44_run(nb22_c03_result)

{
  "status": "EXACT_CACHED_BINDING_SECTIONS_EXTRACTED",
  "documents_inspected": 2,
  "extracted": {
    "handoff_manifest": {
      "cached_source_path": "reports/phase2/evaluation/NB21C.C082.action46_handoff/9186bb5a30456486b29d521db1660b944c72bf8c4fcb9d816cf578b1937ea1d9/C082.manifest.212f8f411a5c39b693e108c5ba3100a291e40355625e60ab14bae343af88e174.json",
      "sections": {
        "schema_version": "NB21C_C082_EVALUATION_MANIFEST_V1",
        "scope": "REPORT_LEVEL_HISTORICAL_EVALUATION_EVIDENCE",
        "population_count": 1149,
        "original_raw_artifact_closure_claimed": false,
        "product_release_bundle": false,
        "source_inventory": {
          "bytes": 7660,
          "file_bytes_sha256": "1f8e35a920e82b81f7066fd906100d9c04ce7bbe8f5964195caeebca502bffac",
          "format": "json",
          "path": "reports/phase2/evaluation/NB21C.C082.action46_handoff/9186bb5a30456486b29d521db1660b944c72bf8c4fcb9d816cf578b1937ea1d9/C082.source_inventory.1f8e35a920e82b81f7

<div dir="rtl" align="right">

### NB22 — C045 | اقدام ۲۴: نهایی‌سازی خروجی مستنداتی قابل‌اشتراک

- هدف: ساخت خروجی مستنداتی با حفظ منشأ و بدون الگوهای شناسایی‌شدهٔ Secret و مسیر شخصی.
- سه زیرکار باقی‌مانده: تعیین فایل‌ها، تازه‌کردن نسخهٔ مشتق‌شدهٔ NB22، اعتبارسنجی و ذخیرهٔ خروجی.
- شواهد قابل‌مصرف: نسخه‌های C037، اسناد C027/C028/C039 و نگاشت و اعتبارسنجی C040/C041.
- بستهٔ فعلی: ۱ از ۱؛ اصل فایل‌ها و شواهد تاریخی تغییر نمی‌کنند.
- دامنه: هفت نوت‌بوک، پنج کارت، سه سند تاریخی و چهار سند مربوط به پاک‌سازی و نگاشت.
- دو مانیفست اصلی دارای مسیر شخصی در خروجی مستنداتی کپی نمی‌شوند؛ نسخه‌های پوشانده‌شده همراه با Lineage مصرف می‌شوند.
- شرط پایان: آماده‌سازی تمام فایل‌ها، کنترل الگوها، انتشار در پوشهٔ مستقل و بازخوانی هش تمام خروجی‌ها.
- هلث‌چک و اجرای پاک نوت‌بوک‌های قبلی مطابق تصمیم کاربر برای بهینه‌سازی باقی می‌مانند.
- API تازه، اجرای مدل، خواندن داده‌های JSONL و تغییر چک‌لیست: صفر.

</div>

In [49]:
# NB22-C045 — Action 24: consolidated documentation export
# Requires prior results from this notebook; does not execute notebooks.

def nb22_c45_run(c27, c28, c37, c39, c40, c41):
    import hashlib
    import json
    import os
    import re
    import shutil
    import tempfile
    from pathlib import Path

    nb22_relative = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / nb22_relative).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0].resolve()

    if c41["artifact"]["status"] != (
        "ISOLATED_RELATIVE_REFERENCE_VALIDATION_PASS"
    ):
        raise RuntimeError("EXISTING_REFERENCE_VALIDATION_REQUIRED")

    personal = re.compile(
        r"(?:/Users/|/home/)[A-Za-z0-9._-]+/"
        r"|[A-Za-z]:\\Users\\[A-Za-z0-9._ -]+\\"
    )
    patterns = {
        "PERSONAL_ABSOLUTE_PATH": personal,
        "PRIVATE_KEY_HEADER": re.compile(
            r"-----BEGIN (?:RSA |EC |OPENSSH |DSA )?PRIVATE KEY-----"
        ),
        "GITHUB_TOKEN_SHAPE": re.compile(
            r"\b(?:gh[pousr]_[A-Za-z0-9]{20,}"
            r"|github_pat_[A-Za-z0-9_]{30,})\b"
        ),
        "HF_TOKEN_SHAPE": re.compile(r"\bhf_[A-Za-z0-9]{25,}\b"),
        "OPENAI_STYLE_TOKEN_SHAPE": re.compile(
            r"\bsk-(?:proj-|svcacct-)?[A-Za-z0-9_-]{32,}\b"
        ),
        "AWS_ACCESS_KEY_ID_SHAPE": re.compile(
            r"\b(?:AKIA|ASIA)[A-Z0-9]{16}\b"
        ),
        "CREDENTIAL_IN_URL": re.compile(
            r"https?://[^\s/:@\"'<>]+:[^\s/@\"'<>]+@"
        ),
        "CREDENTIAL_LITERAL_ASSIGNMENT": re.compile(
            r"""(?i)\b(?:api_key|access_token|auth_token|"""
            r"""client_secret|password)\b\s*[:=]\s*"""
            r"""["']([^"'\r\n]{12,})["']"""
        ),
    }

    def strings(value):
        if isinstance(value, str):
            yield value
        elif isinstance(value, list):
            for item in value:
                yield from strings(item)
        elif isinstance(value, dict):
            for key, item in value.items():
                yield str(key)
                yield from strings(item)

    def scan(value):
        counts = {}
        for text in strings(value):
            for category, pattern in patterns.items():
                for match in pattern.finditer(text):
                    if category == "CREDENTIAL_LITERAL_ASSIGNMENT":
                        candidate = match.group(1).lower()
                        if (
                            any(marker in candidate for marker in (
                                "placeholder", "your_", "your-",
                                "example", "redacted", "getenv",
                                "environ", "<", ">", "${",
                            ))
                            or set(candidate) <= {"*", "x", "-"}
                        ):
                            continue
                    counts[category] = counts.get(category, 0) + 1
        return counts

    def redact(value):
        if isinstance(value, str):
            return personal.sub("<PERSONAL_HOME>/", value)
        if isinstance(value, list):
            return [redact(item) for item in value]
        if isinstance(value, dict):
            if any(personal.search(str(key)) for key in value):
                raise RuntimeError("PERSONAL_PATH_IN_KEY_REQUIRES_REVIEW")
            return {key: redact(item) for key, item in value.items()}
        return value

    def read_source(relative, expected=None):
        rel = Path(relative)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError("INVALID_SOURCE_PATH")
        path = (root / rel).resolve()
        if root not in path.parents:
            raise RuntimeError("SOURCE_OUTSIDE_PROJECT")
        before = path.stat()
        raw = path.read_bytes()
        after = path.stat()
        if (
            before.st_size != after.st_size
            or before.st_mtime_ns != after.st_mtime_ns
        ):
            raise RuntimeError("SOURCE_CHANGED_DURING_READ")
        digest = hashlib.sha256(raw).hexdigest()
        if expected is not None and digest != expected:
            raise RuntimeError(f"SOURCE_IDENTITY_CHANGED: {relative}")
        return raw, digest

    selected = []
    notebook_rows = c37["artifact"]["outputs"]
    if len(notebook_rows) != 7:
        raise RuntimeError("EXPECTED_SEVEN_NOTEBOOKS")

    for row in notebook_rows:
        if row["role"] == "NB22":
            raw, digest = read_source(nb22_relative.as_posix())
            notebook = json.loads(raw)
            sources = [
                "".join(cell.get("source", []))
                if isinstance(cell.get("source", []), list)
                else cell.get("source", "")
                for cell in notebook.get("cells", [])
            ]
            if not any(
                "def nb22_" + "c44_run(" in source
                for source in sources
            ):
                raise RuntimeError("SAVE_CURRENT_NB22_BEFORE_EXPORT")
            for cell, source in zip(notebook["cells"], sources):
                if (
                    cell.get("cell_type") == "code"
                    and personal.search(source)
                ):
                    raise RuntimeError(
                        "CURRENT_NB22_SOURCE_PATH_REQUIRES_TARGETED_FIX"
                    )
            notebook = redact(notebook)
            raw_export = json.dumps(
                notebook, ensure_ascii=False, sort_keys=True,
                separators=(",", ":"), allow_nan=False,
            ).encode("utf-8")
            selected.append({
                "export_path": "notebooks/NB22.ipynb",
                "source": {
                    "path": nb22_relative.as_posix(),
                    "file_bytes_sha256": digest,
                },
                "transformation": "REFRESHED_SNAPSHOT_OUTPUT_PATH_REDACTION",
                "raw": raw_export,
                "decoded": notebook,
            })
        else:
            selected.append({
                "export_path": f"notebooks/{row['role']}.ipynb",
                "receipt": {
                    "path": row["derived_path"],
                    "file_bytes_sha256": row["file_bytes_sha256"],
                },
                "transformation": "REUSED_C037_DERIVED_NOTEBOOK",
            })

    if len(c27["cards"]) != 5:
        raise RuntimeError("EXPECTED_FIVE_EXISTING_CARDS")
    for row in c27["cards"]:
        selected.append({
            "export_path": f"cards/{row['role']}.md",
            "receipt": row,
            "transformation": "UNCHANGED_HISTORICAL_DRAFT_CARD",
        })

    for role, row in c28["artifact"]["documentation_package"].items():
        suffix = Path(row["path"]).suffix
        selected.append({
            "export_path": f"historical_documents/{role}{suffix}",
            "receipt": row,
            "transformation": "UNCHANGED_HISTORICAL_DOCUMENT",
        })

    for index, row in enumerate(c39["artifact"]["outputs"], start=1):
        selected.append({
            "export_path": f"provenance/redacted_manifest_view_{index}.json",
            "receipt": row["documentation_view"],
            "transformation": "REUSED_DOCUMENTATION_ONLY_VIEW",
        })

    for role, result in [
        ("relative_reference_overlay", c40),
        ("isolated_reference_validation", c41),
    ]:
        selected.append({
            "export_path": f"provenance/{role}.json",
            "receipt": result["output"],
            "transformation": "UNCHANGED_SCOPED_EVIDENCE",
        })

    if len(selected) != 19:
        raise RuntimeError("EXPECTED_NINETEEN_DOCUMENTATION_FILES")

    # Prepare and scan everything before publishing any export.
    entries = []
    for item in selected:
        if "receipt" in item:
            receipt = item["receipt"]
            raw, digest = read_source(
                receipt["path"], receipt["file_bytes_sha256"]
            )
            item["source"] = {
                "path": receipt["path"],
                "file_bytes_sha256": digest,
            }
            item["raw"] = raw
            if Path(receipt["path"]).suffix in (".json", ".ipynb"):
                item["decoded"] = json.loads(raw)
            else:
                item["decoded"] = raw.decode("utf-8")

        findings = scan(item["decoded"])
        if findings:
            # Counts and category names only; no matched values.
            raise RuntimeError(
                "EXPORT_SCAN_REQUIRES_REVIEW: "
                + json.dumps({
                    "export_path": item["export_path"],
                    "finding_counts": findings,
                }, ensure_ascii=False)
            )

        entries.append({
            "path": item["export_path"],
            "file_bytes_sha256": hashlib.sha256(
                item["raw"]
            ).hexdigest(),
            "bytes": len(item["raw"]),
            "source": item["source"],
            "transformation": item["transformation"],
        })

    manifest = {
        "artifact_type": "NB22_SHAREABLE_DOCUMENTATION_EXPORT",
        "schema_version": "NB22_C045_ACTION24_EXPORT_V1",
        "scope": "SELECTED_DOCUMENTATION_ONLY_NOT_PRODUCT_RELEASE",
        "files": entries,
        "file_count": len(entries),
        "historical_documents_keep_their_original_time_scope": True,
        "referenced_external_artifacts_are_not_all_included": True,
        "notebooks_are_evidence_snapshots_not_clean_run_claims": True,
        "original_manifests_with_personal_paths_included": False,
        "scan_categories": list(patterns),
        "scan_scope": (
            "DECODED_JSON_KEYS_AND_VALUES_AND_MARKDOWN_TEXT; "
            "DEFINED_PATTERNS_ONLY"
        ),
        "scan_guarantees_exhaustive_secret_detection": False,
        "health_checks_deferred_to_optimization_by_user": True,
        "release_acceptance_granted": False,
    }
    manifest_raw = json.dumps(
        manifest, ensure_ascii=False, sort_keys=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")
    if scan(manifest):
        raise RuntimeError("EXPORT_MANIFEST_SCAN_FAILED")

    export_id = hashlib.sha256(manifest_raw).hexdigest()
    base = root / "reports/phase2/release/NB22.C045.action24"
    base.mkdir(parents=True, exist_ok=True)
    destination = base / ("documentation_export." + export_id)

    def verify_directory(directory):
        expected_paths = {"manifest.json"} | {
            entry["path"] for entry in entries
        }
        actual_paths = {
            path.relative_to(directory).as_posix()
            for path in directory.rglob("*") if path.is_file()
        }
        if actual_paths != expected_paths:
            raise RuntimeError("EXPORT_FILE_SET_MISMATCH")
        if (directory / "manifest.json").read_bytes() != manifest_raw:
            raise RuntimeError("EXPORT_MANIFEST_RELOAD_FAILED")
        for entry in entries:
            raw = (directory / entry["path"]).read_bytes()
            if (
                len(raw) != entry["bytes"]
                or hashlib.sha256(raw).hexdigest()
                != entry["file_bytes_sha256"]
            ):
                raise RuntimeError("EXPORT_FILE_RELOAD_FAILED")

    if destination.exists():
        verify_directory(destination)
        write_status = "REUSED_IDENTICAL"
    else:
        staging = Path(tempfile.mkdtemp(
            prefix=".C045_staging_", dir=base
        ))
        try:
            for item in selected:
                target = staging / item["export_path"]
                target.parent.mkdir(parents=True, exist_ok=True)
                with target.open("xb") as handle:
                    handle.write(item["raw"])
                    handle.flush()
                    os.fsync(handle.fileno())
            with (staging / "manifest.json").open("xb") as handle:
                handle.write(manifest_raw)
                handle.flush()
                os.fsync(handle.fileno())
            verify_directory(staging)
            if destination.exists():
                raise RuntimeError("CONCURRENT_EXPORT_PUBLICATION")
            os.rename(staging, destination)
        finally:
            if staging.exists():
                shutil.rmtree(staging)
        verify_directory(destination)
        write_status = "CREATED"

    result = {
        "artifact_type": "NB22_ACTION24_DOCUMENTATION_EXPORT_RESULT",
        "schema_version": "NB22_C045_ACTION24_RESULT_V1",
        "status": "ACTION24_DOCUMENTATION_EXPORT_COMPLETE_IN_DECLARED_SCOPE",
        "remaining_subtasks_completed": 3,
        "documentation_files": len(entries),
        "export_directory": destination.relative_to(root).as_posix(),
        "export_manifest": {
            "path": (
                destination / "manifest.json"
            ).relative_to(root).as_posix(),
            "file_bytes_sha256": export_id,
        },
        "write_status": write_status,
        "independent_disk_reload": "PASS",
        "defined_sensitive_pattern_findings": 0,
        "action24_complete_in_documentation_export_scope": True,
        "entire_204_file_candidate_privacy_cleared": False,
        "current_C045_execution_output_in_snapshot": False,
        "original_files_modified": False,
        "clean_run_or_notebook_health_acceptance_claimed": False,
        "release_acceptance_granted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
        "next_action": 25,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C045.action24",
        "C045.documentation_export_result",
        result,
    )
    print(json.dumps(
        {**result, "output": output},
        ensure_ascii=False, indent=2,
    ))
    return {
        "artifact": result,
        "output": output,
        "export_manifest": manifest,
    }


nb22_c45_result = nb22_c45_run(
    nb22_c27_result,
    nb22_c28_result,
    nb22_c37_result,
    nb22_c39_result,
    nb22_c40_result,
    nb22_c41_result,
)

{
  "artifact_type": "NB22_ACTION24_DOCUMENTATION_EXPORT_RESULT",
  "schema_version": "NB22_C045_ACTION24_RESULT_V1",
  "status": "ACTION24_DOCUMENTATION_EXPORT_COMPLETE_IN_DECLARED_SCOPE",
  "remaining_subtasks_completed": 3,
  "documentation_files": 19,
  "export_directory": "reports/phase2/release/NB22.C045.action24/documentation_export.1098832cdc45490d6611dad3b787aac510858d78b2849ddee3bad84472eaa13d",
  "export_manifest": {
    "path": "reports/phase2/release/NB22.C045.action24/documentation_export.1098832cdc45490d6611dad3b787aac510858d78b2849ddee3bad84472eaa13d/manifest.json",
    "file_bytes_sha256": "1098832cdc45490d6611dad3b787aac510858d78b2849ddee3bad84472eaa13d"
  },
  "write_status": "CREATED",
  "independent_disk_reload": "PASS",
  "defined_sensitive_pattern_findings": 0,
  "action24_complete_in_documentation_export_scope": true,
  "entire_204_file_candidate_privacy_cleared": false,
  "current_C045_execution_output_in_snapshot": false,
  "original_files_modified": false,
  

<div dir="rtl" align="right">

### NB22 — C046 | اقدام ۲۵: تثبیت مانیفست و بازخوانی مستقل

- هدف: تثبیت هویت فایل‌های checkpoint فعلی و بررسی مستقل آن‌ها.
- چهار زیرکار: تجمیع اتصال‌ها، ذخیرهٔ Atomic، کنترل مستقل فایل‌ها، ثبت نتیجه.
- بستهٔ فعلی: ۱ از ۱.
- ورودی‌ها: کاندیدای ۲۰۴فایلی C042 و خروجی مستنداتی C045.
- نتایج قبلی حفظ می‌شوند؛ داده، پاسخ، مدل، ارزیابی و بازیابی دوباره تولید نمی‌شوند.
- فرایند مستقل فقط فایل‌های فهرست‌شده را برای اندازه و هش می‌خواند.
- مانیفست خودش یا گزارش نتیجهٔ بعدی را در فهرست فایل‌هایش قرار نمی‌دهد.
- شرط پایان: تطبیق تمام فایل‌ها، ثبات هنگام خواندن و بازخوانی مستقل مانیفست.
- دامنهٔ اقدام ۲۴، تعویق هلث‌چک و موانع قبلی بدون تغییر منتقل می‌شوند.
- API و شبکه: صفر. اختلاف فایل یا هش، توقف روشن ایجاد می‌کند.

</div>

In [50]:
# NB22-C046 — Action 25: atomic checkpoint manifest and independent reload
# Uses prior results from this notebook; does not execute historical code.

def nb22_c46_run(c42, c45):
    import hashlib
    import json
    import subprocess
    import sys
    from pathlib import Path

    relative_nb22 = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / relative_nb22).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0].resolve()

    if not c45["artifact"][
        "action24_complete_in_documentation_export_scope"
    ]:
        raise RuntimeError("DOCUMENTATION_EXPORT_REQUIRED")

    def checked_path(relative):
        rel = Path(relative)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError("INVALID_RELATIVE_PATH")
        resolved = (root / rel).resolve()
        if root not in resolved.parents:
            raise RuntimeError("PATH_OUTSIDE_PROJECT")
        return resolved

    def read_bound_json(receipt):
        path = checked_path(receipt["path"])
        before = path.stat()
        raw = path.read_bytes()
        after = path.stat()
        if (
            before.st_size != after.st_size
            or before.st_mtime_ns != after.st_mtime_ns
        ):
            raise RuntimeError("MANIFEST_CHANGED_DURING_READ")
        if hashlib.sha256(raw).hexdigest() != receipt[
            "file_bytes_sha256"
        ]:
            raise RuntimeError("MANIFEST_IDENTITY_MISMATCH")
        return json.loads(raw)

    # Consume the actual persisted manifests.
    candidate = read_bound_json(c42["output"])
    export_receipt = c45["artifact"]["export_manifest"]
    export = read_bound_json(export_receipt)

    if len(candidate["files"]) != 204:
        raise RuntimeError("EXPECTED_C042_204_FILE_INVENTORY")
    if export["file_count"] != 19 or len(export["files"]) != 19:
        raise RuntimeError("EXPECTED_C045_19_FILE_EXPORT")

    files = {}

    def add(relative, digest, role, expected_bytes=None):
        if (
            not isinstance(digest, str)
            or len(digest) != 64
            or any(ch not in "0123456789abcdef" for ch in digest)
        ):
            raise RuntimeError("INVALID_FILE_SHA256")

        path = checked_path(relative)
        if not path.is_file():
            raise RuntimeError(f"EXPLICIT_FILE_UNAVAILABLE: {relative}")
        size = path.stat().st_size
        if expected_bytes is not None and size != expected_bytes:
            raise RuntimeError(f"FILE_SIZE_MISMATCH: {relative}")

        if relative in files:
            row = files[relative]
            if (
                row["file_bytes_sha256"] != digest
                or row["bytes"] != size
            ):
                raise RuntimeError("CONFLICTING_FILE_BINDINGS")
            if role not in row["roles"]:
                row["roles"].append(role)
        else:
            files[relative] = {
                "path": relative,
                "file_bytes_sha256": digest,
                "bytes": size,
                "roles": [role],
            }

    for row in candidate["files"]:
        add(
            row["path"], row["file_bytes_sha256"],
            "C042_LOCAL_CANDIDATE_ASSET",
            row.get("bytes"),
        )

    export_base = Path(export_receipt["path"]).parent
    for row in export["files"]:
        rel = Path(row["path"])
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError("INVALID_EXPORT_MEMBER")
        add(
            (export_base / rel).as_posix(),
            row["file_bytes_sha256"],
            "C045_DOCUMENTATION_EXPORT_MEMBER",
            row["bytes"],
        )

    for receipt, role in [
        (c42["output"], "PARENT_CANDIDATE_MANIFEST"),
        (export_receipt, "DOCUMENTATION_EXPORT_MANIFEST"),
        (c45["output"], "ACTION24_SCOPED_COMPLETION_REPORT"),
    ]:
        add(
            receipt["path"], receipt["file_bytes_sha256"],
            role, receipt.get("bytes"),
        )

    manifest = {
        "artifact_type": "NB22_LOCAL_CHECKPOINT_MANIFEST",
        "schema_version": "NB22_C046_ACTION25_MANIFEST_V1",
        "status": "LOCAL_CHECKPOINT_PENDING_INDEPENDENT_VALIDATION_AND_GATE",
        "parent_manifest": c42["output"],
        "documentation_export_manifest": export_receipt,
        "files": [files[path] for path in sorted(files)],
        "file_count": len(files),
        "total_bytes": sum(row["bytes"] for row in files.values()),
        "scope": "LOCAL_CANDIDATE_AND_DOCUMENTATION_EXPORT",
        "manifest_lists_itself": False,
        "future_validation_report_listed": False,
        "full_transitive_dependency_closure_claimed": False,
        "open_requirements_from_C042": candidate["open_requirements"],
        "documentation_privacy_scope": "C045_19_FILE_EXPORT_ONLY",
        "whole_checkpoint_privacy_cleared": False,
        "health_checks_deferred_to_optimization_by_user": True,
        "full_product_restore_claimed": False,
        "release_acceptance_granted": False,
    }

    manifest_output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C046.action25",
        "C046.local_checkpoint_manifest",
        manifest,
    )
    if manifest_output["path"] in files:
        raise RuntimeError("SELF_REFERENCE_IN_MANIFEST")

    child_code = r'''
import hashlib
import json
import socket
import sys
from pathlib import Path

attempts = []
def deny_network(*args, **kwargs):
    attempts.append(True)
    raise RuntimeError("NETWORK_DISABLED")

socket.socket = deny_network
socket.create_connection = deny_network
socket.getaddrinfo = deny_network

root = Path.cwd().resolve()
def checked_path(relative):
    path = Path(relative)
    if path.is_absolute() or ".." in path.parts:
        raise RuntimeError("INVALID_RELATIVE_PATH")
    resolved = (root / path).resolve()
    if root not in resolved.parents:
        raise RuntimeError("PATH_OUTSIDE_PROJECT")
    return resolved

manifest_path = checked_path(sys.argv[1])
before = manifest_path.stat()
raw = manifest_path.read_bytes()
after = manifest_path.stat()
if (
    before.st_size != after.st_size
    or before.st_mtime_ns != after.st_mtime_ns
):
    raise RuntimeError("MANIFEST_CHANGED_DURING_READ")
if hashlib.sha256(raw).hexdigest() != sys.argv[2]:
    raise RuntimeError("CHECKPOINT_MANIFEST_HASH_MISMATCH")
manifest = json.loads(raw)

seen = set()
total_bytes = 0
for number, row in enumerate(manifest["files"], start=1):
    relative = row["path"]
    if relative in seen or relative == sys.argv[1]:
        raise RuntimeError("DUPLICATE_OR_SELF_REFERENCE")
    seen.add(relative)
    path = checked_path(relative)
    before = path.stat()
    digest = hashlib.sha256()
    read_bytes = 0
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
            read_bytes += len(block)
    after = path.stat()
    if (
        before.st_size != after.st_size
        or before.st_mtime_ns != after.st_mtime_ns
        or read_bytes != before.st_size
        or read_bytes != row["bytes"]
        or digest.hexdigest() != row["file_bytes_sha256"]
    ):
        raise RuntimeError("CHECKPOINT_FILE_IDENTITY_MISMATCH")
    total_bytes += read_bytes
    if number % 50 == 0 or number == len(manifest["files"]):
        print(json.dumps({
            "stage": "CHECKPOINT_RELOAD_PROGRESS",
            "completed": number,
            "total": len(manifest["files"])
        }), flush=True)

if (
    len(seen) != manifest["file_count"]
    or total_bytes != manifest["total_bytes"]
):
    raise RuntimeError("CHECKPOINT_ACCOUNTING_MISMATCH")

print(json.dumps({
    "status": "INDEPENDENT_CHECKPOINT_RELOAD_PASS",
    "manifest_file_bytes_sha256": sys.argv[2],
    "files_verified": len(seen),
    "bytes_verified": total_bytes,
    "all_files_stable_during_read": True,
    "duplicate_paths": 0,
    "self_reference": False,
    "network_guard_enabled": True,
    "blocked_network_attempt_count": len(attempts),
    "models_or_notebooks_executed": False
}), flush=True)
'''

    print(json.dumps({
        "stage": "INDEPENDENT_CHECKPOINT_RELOAD_START",
        "file_count": manifest["file_count"],
        "read_mib": round(manifest["total_bytes"] / 1024**2, 2),
        "model_api_calls": 0,
    }))

    # Progress is streamed directly; errors do not print source values.
    process = subprocess.Popen(
        [
            sys.executable, "-I", "-c", child_code,
            manifest_output["path"],
            manifest_output["file_bytes_sha256"],
        ],
        cwd=root,
        stdout=subprocess.PIPE,
        stderr=subprocess.DEVNULL,
        text=True,
    )
    validation = None
    try:
        for line in process.stdout:
            message = json.loads(line)
            if message.get("stage"):
                print(json.dumps(message), flush=True)
            else:
                validation = message
        return_code = process.wait()
    finally:
        if process.poll() is None:
            process.kill()
            process.wait()
        process.stdout.close()

    if return_code != 0 or validation is None:
        raise RuntimeError(
            "INDEPENDENT_CHECKPOINT_RELOAD_FAILED; "
            f"exit_code={return_code}; NO_ACTION25_COMPLETION_RECORDED"
        )
    if validation["status"] != "INDEPENDENT_CHECKPOINT_RELOAD_PASS":
        raise RuntimeError("UNEXPECTED_VALIDATION_STATUS")

    result = {
        "artifact_type": "NB22_ACTION25_CHECKPOINT_RESULT",
        "schema_version": "NB22_C046_ACTION25_RESULT_V1",
        "status": "ACTION25_LOCAL_CHECKPOINT_WRITE_AND_RELOAD_COMPLETE",
        "subtasks_completed": 4,
        "manifest": manifest_output,
        "independent_validation": validation,
        "action25_complete_in_local_checkpoint_scope": True,
        "whole_checkpoint_privacy_cleared": False,
        "release_acceptance_granted": False,
        "historical_computations_reexecuted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
        "next_action": 26,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C046.action25",
        "C046.checkpoint_write_reload_result",
        result,
    )
    print(json.dumps(
        {**result, "output": output},
        ensure_ascii=False, indent=2,
    ))
    return {
        "artifact": result,
        "output": output,
        "manifest": manifest,
    }


nb22_c46_result = nb22_c46_run(
    nb22_c42_result,
    nb22_c45_result,
)

{"stage": "INDEPENDENT_CHECKPOINT_RELOAD_START", "file_count": 226, "read_mib": 776.77, "model_api_calls": 0}
{"stage": "CHECKPOINT_RELOAD_PROGRESS", "completed": 50, "total": 226}
{"stage": "CHECKPOINT_RELOAD_PROGRESS", "completed": 100, "total": 226}
{"stage": "CHECKPOINT_RELOAD_PROGRESS", "completed": 150, "total": 226}
{"stage": "CHECKPOINT_RELOAD_PROGRESS", "completed": 200, "total": 226}
{"stage": "CHECKPOINT_RELOAD_PROGRESS", "completed": 226, "total": 226}
{
  "artifact_type": "NB22_ACTION25_CHECKPOINT_RESULT",
  "schema_version": "NB22_C046_ACTION25_RESULT_V1",
  "status": "ACTION25_LOCAL_CHECKPOINT_WRITE_AND_RELOAD_COMPLETE",
  "subtasks_completed": 4,
  "manifest": {
    "path": "reports/phase2/release/NB22.C046.action25/C046.local_checkpoint_manifest.8c1c4ea794a7a028d80dbc4d7f9e23cfc519cfae7a85b966b5d341b9ffbd875e.json",
    "file_bytes_sha256": "8c1c4ea794a7a028d80dbc4d7f9e23cfc519cfae7a85b966b5d341b9ffbd875e",
    "bytes": 67066,
    "write_status": "CREATED",
    "reload

<div dir="rtl" align="right">

### NB22 — C047 | اقدام ۲۶: تصمیم واحد Gate ریلیز

- هدف: ثبت تصمیم قابل‌ردیابی بر اساس شواهد موجود.
- پنج زیرکار: دریافت شواهد، تطبیق هویت checkpoint، ساخت جدول Gate، حفظ محدودیت‌ها و تعویق‌ها، ذخیره و بازخوانی تصمیم.
- بستهٔ فعلی: ۱ از ۱.
- سلامت فنی checkpoint، بازیابی محدود، خروجی مستنداتی و کیفیت بالادست جدا گزارش می‌شوند.
- تعویق هلث‌چک و اجرای پاک برای بهینه‌سازی حفظ می‌شود؛ هیچ اجرای تازه‌ای درخواست نمی‌شود.
- محدودیت‌های قبلی از C042 منتقل می‌شوند و دوباره ممیزی یا حل‌شده اعلام نمی‌شوند.
- شرط پایان اقدام: ثبت نتیجهٔ Gate و دلایل پذیرش‌نشدن احتمالی، با ارجاع به شواهد.
- API، شبکه، بازخوانی فایل‌های داده و تغییر آستانه‌ها: صفر.

</div>

In [51]:
# NB22-C047 — Action 26: evidence-based final release gate
# Reuses validated results from this notebook; no historical reruns.

def nb22_c47_run(c27, c30, c41, c42, c45, c46):
    import json
    from pathlib import Path

    relative_nb22 = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / relative_nb22).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0]

    checkpoint = c46["artifact"]
    manifest = c46["manifest"]
    validation = checkpoint["independent_validation"]
    quality = c27["artifact"]["quality_acceptance"]
    export = c45["artifact"]

    if checkpoint["status"] != (
        "ACTION25_LOCAL_CHECKPOINT_WRITE_AND_RELOAD_COMPLETE"
    ):
        raise RuntimeError("ACTION25_COMPLETION_REQUIRED")

    linkage_ok = (
        validation["status"] == "INDEPENDENT_CHECKPOINT_RELOAD_PASS"
        and validation["manifest_file_bytes_sha256"]
        == checkpoint["manifest"]["file_bytes_sha256"]
        and validation["files_verified"] == manifest["file_count"]
        and validation["bytes_verified"] == manifest["total_bytes"]
        and validation["all_files_stable_during_read"]
        and validation["duplicate_paths"] == 0
        and validation["self_reference"] is False
    )
    if not linkage_ok:
        raise RuntimeError("CHECKPOINT_EVIDENCE_LINKAGE_MISMATCH")

    gates = [
        {
            "gate": "local_checkpoint_integrity",
            "status": "PASS",
            "scope": "C046_EXPLICIT_FILE_INVENTORY_ONLY",
            "evidence": c46["output"],
            "details": {
                "files_verified": validation["files_verified"],
                "bytes_verified": validation["bytes_verified"],
                "manifest": checkpoint["manifest"],
            },
        },
        {
            "gate": "bounded_candidate_and_dense_snapshot_restore",
            "status": (
                "PASS"
                if c30["artifact"]["status"] == (
                    "ISOLATED_CANDIDATE_AND_DENSE_SNAPSHOT_RESTORE_PASS"
                )
                else "NOT_ESTABLISHED"
            ),
            "scope": "C028_CANDIDATE_AND_DENSE_SNAPSHOT_ONLY",
            "evidence": c30["output"],
            "full_product_restore_claimed": False,
        },
        {
            "gate": "relative_reference_overlay",
            "status": (
                "PASS"
                if c41["artifact"]["status"] == (
                    "ISOLATED_RELATIVE_REFERENCE_VALIDATION_PASS"
                )
                else "NOT_ESTABLISHED"
            ),
            "scope": "ELEVEN_REFERENCE_BINDINGS_ONLY",
            "evidence": c41["output"],
            "product_runtime_integration_claimed": False,
        },
        {
            "gate": "shareable_documentation_export",
            "status": (
                "PASS"
                if (
                    export[
                        "action24_complete_in_documentation_export_scope"
                    ]
                    and export["independent_disk_reload"] == "PASS"
                    and export["defined_sensitive_pattern_findings"] == 0
                )
                else "NOT_ESTABLISHED"
            ),
            "scope": "C045_SELECTED_DOCUMENTATION_EXPORT_ONLY",
            "evidence": c45["output"],
            "entire_local_checkpoint_privacy_cleared": False,
        },
        {
            "gate": "upstream_answer_and_retrieval_quality",
            "status": quality["overall_gate_status"],
            "scope": "RECORDED_NB21_EVALUATION",
            "evidence": quality["source"],
            "metric_gate_status_counts": quality[
                "metric_gate_status_counts"
            ],
            "distinct_open_issue_count": quality[
                "distinct_open_issue_count"
            ],
            "metrics_recomputed": False,
            "thresholds_modified": False,
        },
        {
            "gate": "whole_checkpoint_public_distribution",
            "status": "NOT_CLEARED",
            "scope": "ENTIRE_C046_LOCAL_CHECKPOINT",
            "evidence": c46["output"],
            "reason": (
                "The privacy-cleared documentation export is a "
                "separate scope; the local checkpoint retains "
                "historical provenance assets."
            ),
        },
        {
            "gate": "full_product_reproducibility",
            "status": "NOT_ESTABLISHED",
            "scope": "COMPLETE_PRODUCT_RUNTIME_AND_DEPENDENCIES",
            "evidence": c42["output"],
            "reason": (
                "Bounded restore and file integrity do not establish "
                "complete product dependency closure or execution."
            ),
        },
        {
            "gate": "clean_execution_and_notebook_health",
            "status": "DEFERRED_BY_USER_TO_OPTIMIZATION",
            "scope": "FINAL_NOTEBOOK_EXECUTION_AND_HEALTH_CHECKS",
            "decision_source": (
                "CURRENT_CHAT_USER_INSTRUCTION; "
                "NO_NEW_HEALTH_CHECK_REQUESTED"
            ),
            "historical_NB21_action48_status": quality[
                "action48_status"
            ],
            "clean_execution_claimed": False,
        },
    ]

    # Preserve recorded requirements without claiming a new audit.
    recorded_requirements = c42["artifact"]["open_requirements"]
    if not isinstance(recorded_requirements, list):
        raise RuntimeError("EXPECTED_RECORDED_REQUIREMENT_LIST")

    status_counts = {}
    for gate in gates:
        status = gate["status"]
        status_counts[status] = status_counts.get(status, 0) + 1

    nonpass_gates = [
        gate["gate"] for gate in gates if gate["status"] != "PASS"
    ]
    quality_counts = quality["metric_gate_status_counts"]
    quality_not_accepted = (
        quality.get("granted") is not True
        or quality["overall_gate_status"] != "PASS"
        or quality_counts.get("FAIL", 0) > 0
        or quality_counts.get("BLOCKED", 0) > 0
    )

    # This cell records the gate. It does not authorize a release,
    # waive prerequisites, or reinterpret deferred work as PASS.
    release_granted = (
        not nonpass_gates
        and not quality_not_accepted
        and not recorded_requirements
    )

    result = {
        "artifact_type": "NB22_ACTION26_FINAL_RELEASE_GATE",
        "schema_version": "NB22_C047_ACTION26_GATE_V1",
        "status": (
            "ACTION26_GATE_RECORDED_RELEASE_ACCEPTED"
            if release_granted
            else "ACTION26_GATE_RECORDED_RELEASE_NOT_ACCEPTED"
        ),
        "subtasks_completed": 5,
        "checkpoint_manifest": checkpoint["manifest"],
        "gate_results": gates,
        "gate_status_counts": status_counts,
        "nonpass_gate_names": nonpass_gates,
        "recorded_requirements_from_C042": {
            "source": c42["output"],
            "items": recorded_requirements,
            "reassessed_or_resolved_by_this_cell": False,
            "health_related_items_deferred_by_current_user_decision": True,
        },
        "upstream_quality_status_preserved": quality[
            "overall_gate_status"
        ],
        "upstream_metric_gate_status_counts": quality_counts,
        "health_deferral_blocks_recording_this_action": False,
        "action26_gate_assessment_complete": True,
        "release_acceptance_granted": release_granted,
        "phase2_completion_claimed": False,
        "new_health_checks_or_clean_runs": 0,
        "historical_computations_reexecuted": False,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
        "next_action": 27,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C047.action26",
        "C047.final_release_gate",
        result,
    )
    print(json.dumps(
        {**result, "output": output},
        ensure_ascii=False, indent=2,
    ))
    return {"artifact": result, "output": output}


nb22_c47_result = nb22_c47_run(
    nb22_c27_result,
    nb22_c30_result,
    nb22_c41_result,
    nb22_c42_result,
    nb22_c45_result,
    nb22_c46_result,
)

{
  "artifact_type": "NB22_ACTION26_FINAL_RELEASE_GATE",
  "schema_version": "NB22_C047_ACTION26_GATE_V1",
  "status": "ACTION26_GATE_RECORDED_RELEASE_NOT_ACCEPTED",
  "subtasks_completed": 5,
  "checkpoint_manifest": {
    "path": "reports/phase2/release/NB22.C046.action25/C046.local_checkpoint_manifest.8c1c4ea794a7a028d80dbc4d7f9e23cfc519cfae7a85b966b5d341b9ffbd875e.json",
    "file_bytes_sha256": "8c1c4ea794a7a028d80dbc4d7f9e23cfc519cfae7a85b966b5d341b9ffbd875e",
    "bytes": 67066,
    "write_status": "CREATED",
    "reload_validation": "PASS"
  },
  "gate_results": [
    {
      "gate": "local_checkpoint_integrity",
      "status": "PASS",
      "scope": "C046_EXPLICIT_FILE_INVENTORY_ONLY",
      "evidence": {
        "path": "reports/phase2/release/NB22.C046.action25/C046.checkpoint_write_reload_result.c9280cd72fad5392e68c63a054a7fa82956ed911a6b56be61455ba77021871f9.json",
        "file_bytes_sha256": "c9280cd72fad5392e68c63a054a7fa82956ed911a6b56be61455ba77021871f9",
        "by

<div dir="rtl" align="right">

### NB22 — C048 | اقدام ۲۷: ثبت تصمیم و وضعیت checkpoint

- هدف: ثبت وضعیت checkpoint بر اساس تصمیم واقعی C047.
- سه زیرکار: تطبیق اتصال تصمیم و مانیفست، ثبت وضعیت و حدود مصرف، ذخیره و بازخوانی گزارش.
- بستهٔ فعلی: ۱ از ۱.
- ورودی‌ها: نتیجهٔ اقدام ۲۵ در C046 و تصمیم اقدام ۲۶ در C047.
- سلامت فنی checkpoint تأیید شده؛ پذیرش Release صادر نشده است.
- تعویق هلث‌چک برای بهینه‌سازی و وضعیت کیفیت بالادست حفظ می‌شوند.
- شرط پایان: گزارش تصمیم با ارجاع دقیق به شواهد، ذخیره و بازخوانی شود.
- API، شبکه، بازخوانی داده‌ها، تغییر آستانه و تغییر چک‌لیست: صفر.
- اقدام بعد: ۲۸، تهیهٔ بستهٔ تحویل با وضعیت واقعی ورودی.

</div>

In [52]:
# NB22-C048 — Action 27: record checkpoint disposition and actual acceptance
# Reuses C046/C047; no data reads or historical computation.

def nb22_c48_run(c46, c47):
    import copy
    import json
    from pathlib import Path

    relative_nb22 = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / relative_nb22).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0]

    checkpoint = c46["artifact"]
    gate = c47["artifact"]

    if not checkpoint["action25_complete_in_local_checkpoint_scope"]:
        raise RuntimeError("COMPLETED_LOCAL_CHECKPOINT_REQUIRED")
    if not gate["action26_gate_assessment_complete"]:
        raise RuntimeError("COMPLETED_GATE_ASSESSMENT_REQUIRED")

    checkpoint_identity = checkpoint["manifest"]
    gate_identity = gate["checkpoint_manifest"]
    for key in ("path", "file_bytes_sha256"):
        if checkpoint_identity[key] != gate_identity[key]:
            raise RuntimeError("GATE_CHECKPOINT_IDENTITY_MISMATCH")

    gate_names = [row["gate"] for row in gate["gate_results"]]
    if len(gate_names) != len(set(gate_names)):
        raise RuntimeError("DUPLICATE_GATE_NAMES")

    technical_gate = next(
        row for row in gate["gate_results"]
        if row["gate"] == "local_checkpoint_integrity"
    )
    if technical_gate["status"] != "PASS":
        raise RuntimeError("VERIFIED_LOCAL_CHECKPOINT_REQUIRED")

    granted = gate["release_acceptance_granted"]
    if not isinstance(granted, bool):
        raise RuntimeError("INVALID_ACCEPTANCE_VALUE")

    if granted and (
        gate["nonpass_gate_names"]
        or gate["upstream_quality_status_preserved"] != "PASS"
    ):
        raise RuntimeError("ACCEPTANCE_CONTRADICTS_RECORDED_GATE")

    deferral = next(
        row for row in gate["gate_results"]
        if row["gate"] == "clean_execution_and_notebook_health"
    )
    if deferral["status"] != "DEFERRED_BY_USER_TO_OPTIMIZATION":
        raise RuntimeError("EXPECTED_USER_HEALTH_DEFERRAL")

    result = {
        "artifact_type": "NB22_ACTION27_CHECKPOINT_DISPOSITION",
        "schema_version": "NB22_C048_ACTION27_V1",
        "status": (
            "ACTION27_RELEASE_ACCEPTANCE_RECORDED"
            if granted
            else "ACTION27_VERIFIED_LOCAL_CHECKPOINT_RECORDED_RELEASE_NOT_ACCEPTED"
        ),
        "subtasks_completed": 3,
        "checkpoint": copy.deepcopy(checkpoint_identity),
        "checkpoint_validation_report": copy.deepcopy(c46["output"]),
        "release_gate_report": copy.deepcopy(c47["output"]),
        "disposition": {
            "local_checkpoint_integrity": "PASS",
            "local_checkpoint_available_for_review": True,
            "release_acceptance_granted": granted,
            "product_release_id_assigned": False,
            "whole_checkpoint_public_distribution_authorized": False,
            "nonpass_gate_names": copy.deepcopy(
                gate["nonpass_gate_names"]
            ),
        },
        "preserved_quality": {
            "overall_status": gate["upstream_quality_status_preserved"],
            "metric_gate_status_counts": copy.deepcopy(
                gate["upstream_metric_gate_status_counts"]
            ),
        },
        "preserved_user_deferral": copy.deepcopy(deferral),
        "recorded_requirements": copy.deepcopy(
            gate["recorded_requirements_from_C042"]
        ),
        "NB23_handoff_disposition": {
            "prepare_evidence_and_status_handoff": True,
            "accepted_release_input_established": granted,
            "formal_NB23_start_authorized_by_this_report": False,
            "NB23_implementation_performed": False,
        },
        "action27_recording_complete": True,
        "all_NB22_requirements_satisfied": False,
        "phase2_completion_claimed": False,
        "input_files_reread": 0,
        "new_health_checks_or_clean_runs": 0,
        "historical_computations_reexecuted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
        "next_action": 28,
    }

    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C048.action27",
        "C048.checkpoint_disposition",
        result,
    )
    print(json.dumps({
        "status": result["status"],
        "subtasks_completed": 3,
        "checkpoint": result["checkpoint"],
        "local_checkpoint_integrity": "PASS",
        "release_acceptance_granted": granted,
        "health_check_status": deferral["status"],
        "upstream_quality_status": gate[
            "upstream_quality_status_preserved"
        ],
        "action27_recording_complete": True,
        "output": output,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "next_action": 28,
    }, ensure_ascii=False, indent=2))
    return {"artifact": result, "output": output}


nb22_c48_result = nb22_c48_run(
    nb22_c46_result,
    nb22_c47_result,
)

{
  "status": "ACTION27_VERIFIED_LOCAL_CHECKPOINT_RECORDED_RELEASE_NOT_ACCEPTED",
  "subtasks_completed": 3,
  "checkpoint": {
    "path": "reports/phase2/release/NB22.C046.action25/C046.local_checkpoint_manifest.8c1c4ea794a7a028d80dbc4d7f9e23cfc519cfae7a85b966b5d341b9ffbd875e.json",
    "file_bytes_sha256": "8c1c4ea794a7a028d80dbc4d7f9e23cfc519cfae7a85b966b5d341b9ffbd875e",
    "bytes": 67066,
    "write_status": "CREATED",
    "reload_validation": "PASS"
  },
  "local_checkpoint_integrity": "PASS",
  "release_acceptance_granted": false,
  "health_check_status": "DEFERRED_BY_USER_TO_OPTIMIZATION",
  "upstream_quality_status": "FAIL",
  "action27_recording_complete": true,
  "output": {
    "path": "reports/phase2/release/NB22.C048.action27/C048.checkpoint_disposition.3e008e4a6c4add1087bac30c9ca3b5744513bfe9b5f49be1169270fc4347fee9.json",
    "file_bytes_sha256": "3e008e4a6c4add1087bac30c9ca3b5744513bfe9b5f49be1169270fc4347fee9",
    "bytes": 3542,
    "write_status": "CREATED",
    "r

<div dir="rtl" align="right">

### NB22 — C049 | اقدام ۲۸: بستهٔ تحویل شواهد و وضعیت برای NB23

- هدف: تحویل مسیرها، هویت فایل‌ها، کارت‌ها، نتایج و محدودیت‌های واقعی.
- چهار زیرکار: اتصال مانیفست‌ها، اتصال مستندات و شواهد، انتقال تصمیم‌ها، ذخیره و بازخوانی تحویل‌نامه.
- بستهٔ فعلی: ۱ از ۱.
- ورودی‌ها: کارت‌های C027، اسناد C028، خروجی مستنداتی C045، checkpoint در C046، Gate در C047 و تصمیم C048.
- نتایج قبلی مصرف می‌شوند؛ فایل‌های داده و محاسبات تاریخی دوباره خوانده یا اجرا نمی‌شوند.
- تحویل‌نامه وضعیت «checkpoint محلی معتبر، Release پذیرفته‌نشده» را منتقل می‌کند.
- هلث‌چک مطابق تصمیم کاربر برای بهینه‌سازی باقی می‌ماند.
- شرط پایان بسته: تحویل‌نامهٔ نسخه‌دار با ارجاع‌های متصل و بازخوانی PASS.
- شروع رسمی NB23 با ورودی پذیرفته‌شده همچنان احراز نشده است.
- API، شبکه و تغییر مخزن یا چک‌لیست: صفر.

</div>

In [53]:
# NB22-C049 — Action 28: evidence/status handoff for NB23
# Uses existing notebook results; no NB23 implementation or historical rerun.

def nb22_c49_run(c27, c28, c45, c46, c47, c48):
    import copy
    import json
    from pathlib import Path

    relative_nb22 = Path(
        "notebooks/phase2/22_release_and_reproducibility.ipynb"
    )
    roots = [
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / relative_nb22).is_file()
    ]
    if len(roots) != 1:
        raise RuntimeError("PROJECT_ROOT_NOT_UNIQUELY_RESOLVED")
    root = roots[0]

    checkpoint = c46["artifact"]
    gate = c47["artifact"]
    disposition = c48["artifact"]
    export = c45["artifact"]

    if not disposition["action27_recording_complete"]:
        raise RuntimeError("ACTION27_RECORDING_REQUIRED")
    if not gate["action26_gate_assessment_complete"]:
        raise RuntimeError("ACTION26_ASSESSMENT_REQUIRED")

    checkpoint_identity = checkpoint["manifest"]
    for identity in (
        gate["checkpoint_manifest"],
        disposition["checkpoint"],
    ):
        for key in ("path", "file_bytes_sha256"):
            if identity[key] != checkpoint_identity[key]:
                raise RuntimeError("HANDOFF_CHECKPOINT_IDENTITY_MISMATCH")

    accepted = disposition["disposition"]["release_acceptance_granted"]
    if accepted != gate["release_acceptance_granted"]:
        raise RuntimeError("INCONSISTENT_ACCEPTANCE_DECISION")

    cards = copy.deepcopy(c27["cards"])
    if len(cards) != 5:
        raise RuntimeError("EXPECTED_FIVE_DOCUMENTATION_CARDS")
    if len({row["role"] for row in cards}) != len(cards):
        raise RuntimeError("DUPLICATE_DOCUMENTATION_CARD_ROLE")

    documentation = copy.deepcopy(
        c28["artifact"]["documentation_package"]
    )
    required_docs = {
        "evidence_index", "compatibility_status", "release_notes"
    }
    if not required_docs.issubset(documentation):
        raise RuntimeError("DOCUMENTATION_BINDINGS_INCOMPLETE")

    gates = {
        row["gate"]: row
        for row in gate["gate_results"]
    }
    required_gates = {
        "bounded_candidate_and_dense_snapshot_restore",
        "relative_reference_overlay",
        "clean_execution_and_notebook_health",
    }
    if not required_gates.issubset(gates):
        raise RuntimeError("REQUIRED_GATE_EVIDENCE_MISSING")

    handoff = {
        "artifact_type": "NB22_TO_NB23_EVIDENCE_STATUS_HANDOFF",
        "schema_version": "NB22_C049_ACTION28_HANDOFF_V1",
        "status": (
            "HANDOFF_PREPARED_ACCEPTED_RELEASE_RECORDED"
            if accepted
            else "HANDOFF_PREPARED_WITH_UNACCEPTED_RELEASE_INPUT"
        ),
        "producer": {
            "notebook": relative_nb22.as_posix(),
            "cell": "NB22-C049",
            "checklist_action": 28,
        },
        "consumer": "NB23_CONTRACT_WORK",
        "scope": "EVIDENCE_AND_STATUS_HANDOFF_ONLY",
        "inputs": {
            "checkpoint_manifest": copy.deepcopy(checkpoint_identity),
            "checkpoint_validation": copy.deepcopy(c46["output"]),
            "release_gate": copy.deepcopy(c47["output"]),
            "checkpoint_disposition": copy.deepcopy(c48["output"]),
        },
        "documentation": {
            "cards": cards,
            "historical_documents": documentation,
            "cards_and_notes_keep_their_producer_time_scope": True,
            "current_decision_authority": copy.deepcopy(c48["output"]),
            "shareable_export": {
                "directory": export["export_directory"],
                "manifest": copy.deepcopy(export["export_manifest"]),
                "result_report": copy.deepcopy(c45["output"]),
                "scope": "NINETEEN_DOCUMENTATION_FILES_ONLY",
                "checkpoint_public_distribution_cleared": False,
            },
        },
        "technical_evidence": {
            "local_checkpoint_integrity": {
                "status": "PASS",
                "evidence": copy.deepcopy(c46["output"]),
                "files_verified": checkpoint[
                    "independent_validation"
                ]["files_verified"],
            },
            "bounded_restore": copy.deepcopy(
                gates["bounded_candidate_and_dense_snapshot_restore"]
            ),
            "relative_reference_validation": copy.deepcopy(
                gates["relative_reference_overlay"]
            ),
        },
        "preserved_quality": copy.deepcopy(
            disposition["preserved_quality"]
        ),
        "preserved_user_deferral": copy.deepcopy(
            disposition["preserved_user_deferral"]
        ),
        "recorded_requirements": copy.deepcopy(
            disposition["recorded_requirements"]
        ),
        "consumer_rules": [
            "Use exact artifact identities and declared evidence scopes.",
            "Reuse existing outputs; do not regenerate producers.",
            "Do not substitute historical NB19 output for a selected "
            "NB21 response population without a consumer binding.",
            "Keep recovery outputs distinct from live model execution.",
            "Keep quality FAIL/BLOCKED results and deferred health checks.",
            "Do not infer full product restore from bounded restore.",
            "Do not treat the documentation export as a runtime release.",
            "Prepare NB23 under its separate checklist; this handoff "
            "does not implement or approve its contracts.",
        ],
        "accepted_release_input_established": accepted,
        "formal_NB23_start_authorized_by_this_handoff": False,
        "NB23_implementation_performed": False,
        "handoff_preparation_complete": True,
        "action28_accepted_release_input_condition_met": accepted,
        "all_28_actions_unconditionally_complete_claimed": False,
        "release_acceptance_granted": accepted,
        "phase2_completion_claimed": False,
        "input_files_reread": 0,
        "historical_computations_reexecuted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }

    handoff_output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C049.action28",
        "C049.NB23_evidence_status_handoff",
        handoff,
    )

    summary = {
        "artifact_type": "NB22_ACTION28_HANDOFF_RESULT",
        "schema_version": "NB22_C049_ACTION28_RESULT_V1",
        "status": handoff["status"],
        "subtasks_completed": 4,
        "handoff": handoff_output,
        "checkpoint_manifest": copy.deepcopy(checkpoint_identity),
        "documentation_card_count": len(cards),
        "local_checkpoint_integrity": "PASS",
        "upstream_quality_status": disposition[
            "preserved_quality"
        ]["overall_status"],
        "health_check_status": disposition[
            "preserved_user_deferral"
        ]["status"],
        "handoff_preparation_complete": True,
        "action28_accepted_release_input_condition_met": accepted,
        "release_acceptance_granted": accepted,
        "NB23_implementation_performed": False,
        "phase2_completion_claimed": False,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
        "next_step": (
            "Summarize NB22 checklist statuses and preserved deferrals; "
            "do not automatically reopen completed work or start NB23."
        ),
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C049.action28",
        "C049.handoff_result",
        summary,
    )
    print(json.dumps(
        {**summary, "output": output},
        ensure_ascii=False, indent=2,
    ))
    return {
        "artifact": handoff,
        "handoff_output": handoff_output,
        "summary": summary,
        "output": output,
    }


nb22_c49_result = nb22_c49_run(
    nb22_c27_result,
    nb22_c28_result,
    nb22_c45_result,
    nb22_c46_result,
    nb22_c47_result,
    nb22_c48_result,
)

{
  "artifact_type": "NB22_ACTION28_HANDOFF_RESULT",
  "schema_version": "NB22_C049_ACTION28_RESULT_V1",
  "status": "HANDOFF_PREPARED_WITH_UNACCEPTED_RELEASE_INPUT",
  "subtasks_completed": 4,
  "handoff": {
    "path": "reports/phase2/release/NB22.C049.action28/C049.NB23_evidence_status_handoff.1d5d0f87494e8c67277ce6da2d4b8811609ced91b88193f67b40ffdb03097381.json",
    "file_bytes_sha256": "1d5d0f87494e8c67277ce6da2d4b8811609ced91b88193f67b40ffdb03097381",
    "bytes": 9588,
    "write_status": "CREATED",
    "reload_validation": "PASS"
  },
  "checkpoint_manifest": {
    "path": "reports/phase2/release/NB22.C046.action25/C046.local_checkpoint_manifest.8c1c4ea794a7a028d80dbc4d7f9e23cfc519cfae7a85b966b5d341b9ffbd875e.json",
    "file_bytes_sha256": "8c1c4ea794a7a028d80dbc4d7f9e23cfc519cfae7a85b966b5d341b9ffbd875e",
    "bytes": 67066,
    "write_status": "CREATED",
    "reload_validation": "PASS"
  },
  "documentation_card_count": 5,
  "local_checkpoint_integrity": "PASS",
  "upstream

<div dir="rtl" align="right">

### NB22 — C050 | ممیزی شواهد کیفیت موجود — اصلاح تشخیص ریشه

چهار زیرکار در یک بسته:
۱. تشخیص ریشهٔ معتبر پروژه و موجودی نوت‌بوک‌های محلی فاز دو؛
۲. بررسی کد، مارک‌دان و خروجی‌های ذخیره‌شده؛
۳. مصرف گزارش‌های JSON کوچکِ صریحاً ارجاع‌شده و حافظهٔ معتبر موجود؛
۴. ثبت شواهد، مقادیر، دامنه و محدودیت بررسی.

این بررسی برای پیدا کردن نتایج قبلی است؛ هیچ سل تاریخی، مدل یا API اجرا نمی‌شود.
ذکر یک معیار، اندازه‌گیری آن نیست و پیدا شدن عدد نیز به‌تنهایی پذیرش نهایی نیست.
نبود نتیجه در دامنهٔ بررسی، اثبات انجام‌نشدن کار در کل پروژه نیست.

اصل فایل‌ها، Gate و چک‌لیست تغییر نمی‌کنند.
شمارهٔ C050 شمارهٔ سل است؛ اقدام تازه‌ای به چک‌لیست اضافه نمی‌شود.

</div>

In [55]:
# NB22-C050 — Existing quality evidence audit, corrected root resolution
# Read-only inputs; no model calls, notebook execution, or JSONL reads.

def nb22_c50_run():
    import hashlib
    import json
    import math
    import re
    import subprocess
    from collections import Counter
    from pathlib import Path

    roadmap = Path(
        "data/Road_Maps/"
        "RAG_finance_phase2_modular_notebooks_execution_plan.md"
    )
    previous_receipt = globals().get("nb22_c49_result", {}).get("output", {})
    receipt_path = previous_receipt.get("path")

    def valid_root(path):
        path = path.resolve()
        if not (
            (path / roadmap).is_file()
            and (path / "notebooks/phase2").is_dir()
            and (
                path / "notebooks/phase2/"
                "22_release_and_reproducibility.ipynb"
            ).is_file()
        ):
            return False
        if receipt_path:
            rel = Path(receipt_path)
            if rel.is_absolute() or ".." in rel.parts:
                return False
            return (path / rel).is_file()
        return True

    root = None
    root_method = None
    try:
        git = subprocess.run(
            ["git", "-C", str(Path.cwd()),
             "rev-parse", "--show-toplevel"],
            capture_output=True, text=True, timeout=10, check=False,
        )
        if git.returncode == 0:
            candidate = Path(git.stdout.strip()).resolve()
            if valid_root(candidate):
                root = candidate
                root_method = "GIT_ROOT_WITH_PROJECT_AND_PRIOR_EVIDENCE"
    except (OSError, subprocess.TimeoutExpired):
        pass

    if root is None:
        # The closest ancestor satisfying all project anchors is used.
        seen = set()
        for parent in [Path.cwd(), *Path.cwd().parents]:
            candidate = parent.resolve()
            if candidate in seen:
                continue
            seen.add(candidate)
            if valid_root(candidate):
                root = candidate
                root_method = "NEAREST_ANCESTOR_WITH_PROJECT_ANCHORS"
                break

    if root is None:
        raise RuntimeError(
            "PROJECT_ROOT_NOT_RESOLVED: "
            "required roadmap, NB22 and prior C049 receipt "
            "were not found together; no notebook scan started"
        )

    families = {
        "answer_correctness": (
            r"answer_correctness|answer_quality|model_correct|"
            r"correct_fraction|correct_rate|population_accuracy|"
            r"independent_reference_accuracy"
        ),
        "citation_correctness": (
            r"citation_correctness|citation_precision|citation_accuracy"
        ),
        "citation_coverage": (
            r"claim_citation_coverage|citation_coverage|"
            r"citation_completeness|claim_coverage"
        ),
        "citation_locator": (
            r"citation_locator_accuracy|citation_location_accuracy|"
            r"locator_accuracy|locator_fidelity|locator_validity"
        ),
        "abstention": (
            r"no_answer_f1|abstention_metrics|abstention_f1|"
            r"no_answer_metrics|abstention_precision|abstention_recall"
        ),
        "safety": (
            r"critical_safety_violation|safety_violation_rate|"
            r"unsupported_critical_claim|critical_violation|"
            r"critical_safety_test|unsupported_factual_claim_rate"
        ),
    }
    patterns = {
        name: re.compile(expression, re.I)
        for name, expression in families.items()
    }
    config_pattern = re.compile(
        r"(?:^|[/_.])(?:min|max|target|threshold|configuration|"
        r"config|metric_definitions)(?:$|[/_.])", re.I
    )
    reference_pattern = re.compile(
        r"(?:reports|artifacts)/phase2/"
        r"[A-Za-z0-9_./() -]+\.json\b"
    )
    personal = re.compile(
        r"(?:/Users/|/home/)[A-Za-z0-9._-]+/"
        r"|[A-Za-z]:\\Users\\[A-Za-z0-9._ -]+\\"
    )

    def safe_text(value):
        return personal.sub("<PERSONAL_HOME>/", str(value))[:300]

    def safe_scalar(value):
        if isinstance(value, float) and not math.isfinite(value):
            return "NON_FINITE_NUMBER"
        return safe_text(value) if isinstance(value, str) else value

    def stable_read(path):
        before = path.stat()
        raw = path.read_bytes()
        after = path.stat()
        if (
            before.st_size != after.st_size
            or before.st_mtime_ns != after.st_mtime_ns
        ):
            raise RuntimeError("INPUT_CHANGED_DURING_READ")
        return raw, hashlib.sha256(raw).hexdigest()

    observations = []
    observation_keys = set()
    observation_counts = Counter()
    mentions = Counter()
    references = set()
    notebook_inventory = []
    report_inventory = []
    issues = []
    max_observations = 15000
    max_report_bytes = 8 * 1024 * 1024
    max_total_report_bytes = 128 * 1024 * 1024
    max_report_count = 250

    def collect_references(text):
        for match in reference_pattern.finditer(text):
            relative = match.group().strip()
            if ".." not in Path(relative).parts:
                references.add(relative)

    def inspect_text(text, surface):
        for family, pattern in patterns.items():
            mentions[(family, surface)] += len(pattern.findall(text))
        collect_references(text)

    def inspect_object(value, source, field="", context=None):
        if isinstance(value, dict):
            local = dict(context or {})
            for key in (
                "schema_version", "scope", "status", "population_count",
                "evaluated_case_count", "split", "reference_type",
                "independent_reference_accuracy_established",
                "live_generator_performance_established",
                "these_are_not_independent_population_accuracy",
            ):
                item = value.get(key)
                if isinstance(item, (str, bool, int, float)):
                    local[key] = safe_scalar(item)
            for key, item in value.items():
                inspect_object(item, source, f"{field}/{key}", local)
            return

        if isinstance(value, list):
            for index, item in enumerate(value):
                inspect_object(item, source, f"{field}/{index}", context)
            return

        if isinstance(value, str):
            collect_references(value)

        eligible = (
            isinstance(value, (bool, int, float))
            or value is None
            or (
                isinstance(value, str)
                and value.upper() in {
                    "PASS", "FAIL", "BLOCKED", "NOT_ASSESSED",
                    "NOT_ESTABLISHED", "UNDEFINED", "NONCRITICAL",
                }
            )
        )
        if not eligible:
            return

        for family, pattern in patterns.items():
            if not pattern.search(field):
                continue
            kind = (
                "TARGET_OR_CONFIGURATION"
                if config_pattern.search(field)
                else "RECORDED_VALUE_REQUIRES_SCOPE_REVIEW"
            )
            key = (
                family, source["path"],
                source.get("physical_cell_index"),
                source.get("output_index"), field, repr(value),
            )
            if key in observation_keys:
                continue
            observation_keys.add(key)
            observation_counts[(family, kind)] += 1
            if len(observations) < max_observations:
                observations.append({
                    "family": family,
                    "kind": kind,
                    "source": source,
                    "field": field,
                    "value": safe_scalar(value),
                    "recorded_context": dict(context or {}),
                    "independent_accuracy_or_acceptance_inferred": False,
                })

    decoder = json.JSONDecoder()

    def inspect_output_text(text, source):
        inspect_text(text, "saved_output")
        position = 0
        while position < len(text):
            match = re.search(r"[\{\[]", text[position:])
            if match is None:
                break
            start = position + match.start()
            try:
                value, end = decoder.raw_decode(text, start)
            except (ValueError, RecursionError):
                position = start + 1
                continue
            inspect_object(value, source)
            position = end

    notebooks = sorted(
        path for path in (root / "notebooks/phase2").rglob("*.ipynb")
        if ".ipynb_checkpoints" not in path.parts
    )
    if not notebooks:
        raise RuntimeError("NO_PHASE2_NOTEBOOKS_FOUND")

    print(json.dumps({
        "stage": "EXISTING_QUALITY_EVIDENCE_AUDIT_START",
        "root_resolution": root_method,
        "prior_C049_location_checked": bool(receipt_path),
        "notebooks": len(notebooks),
        "notebook_execution": False,
        "model_api_calls": 0,
    }), flush=True)

    for number, path in enumerate(notebooks, start=1):
        relative = path.relative_to(root).as_posix()
        if root not in path.resolve().parents:
            issues.append({
                "path": relative, "stage": "notebook_read",
                "error_type": "SYMLINK_OUTSIDE_PROJECT",
            })
            continue
        try:
            raw, digest = stable_read(path)
            notebook = json.loads(raw)
            cells = notebook["cells"]
            if not isinstance(cells, list):
                raise ValueError("INVALID_CELLS")
        except Exception as exc:
            issues.append({
                "path": relative, "stage": "notebook_read",
                "error_type": type(exc).__name__,
            })
            continue

        notebook_inventory.append({
            "path": relative,
            "file_bytes_sha256": digest,
            "bytes": len(raw),
            "cell_count": len(cells),
            "code_cell_count": sum(
                cell.get("cell_type") == "code" for cell in cells
            ),
        })
        for index, cell in enumerate(cells):
            source = cell.get("source", "")
            if isinstance(source, list):
                source = "".join(source)
            if not isinstance(source, str):
                continue
            inspect_text(
                source,
                "code_source" if cell.get("cell_type") == "code"
                else "markdown_or_other_source",
            )
            for output_index, output in enumerate(cell.get("outputs", [])):
                location = {
                    "path": relative,
                    "file_bytes_sha256": digest,
                    "physical_cell_index": index,
                    "output_index": output_index,
                    "execution_count": cell.get("execution_count"),
                    "surface": "saved_notebook_output",
                }
                data = output.get("data", {})
                structured = data.get("application/json")
                if structured is not None:
                    if isinstance(structured, str):
                        inspect_output_text(structured, location)
                    else:
                        inspect_object(structured, location)
                for text in (
                    output.get("text"),
                    data.get("text/plain"),
                    data.get("text/markdown"),
                ):
                    if isinstance(text, list):
                        text = "".join(text)
                    if isinstance(text, str):
                        inspect_output_text(text, location)

        print(json.dumps({
            "stage": "NOTEBOOK_INSPECTED",
            "completed": number,
            "total": len(notebooks),
            "path": relative,
        }), flush=True)

    references.add(
        "reports/phase2/evaluation/NB21_action26_final_summary.json"
    )
    caches = {}
    for variable, key in (
        ("nb22_c03_result", "loaded"),
        ("nb22_c06_result", "package_payloads"),
        ("nb22_c12_result", "payloads"),
        ("nb22_c20_result", "payloads"),
        ("nb22_c23_result", "payloads"),
    ):
        result = globals().get(variable)
        mapping = result.get(key) if isinstance(result, dict) else None
        if isinstance(mapping, dict):
            for path, payload in mapping.items():
                if isinstance(path, str):
                    caches.setdefault(path, payload)

    # Freeze a single-hop reference list; do not crawl new references.
    report_paths = sorted(references)
    total_report_bytes = 0
    unresolved = []

    for relative in report_paths:
        rel = Path(relative)
        if rel.is_absolute() or ".." in rel.parts:
            continue
        if len(report_inventory) >= max_report_count:
            unresolved.append({
                "path": relative, "reason": "REPORT_COUNT_LIMIT"
            })
            continue

        if relative in caches:
            payload = caches[relative]
            source = {
                "path": relative,
                "surface": "previously_validated_cached_report",
                "fresh_byte_hash_computed": False,
            }
            mode = "REUSED_EXISTING_CACHE"
        else:
            path = (root / rel).resolve()
            if root not in path.parents or not path.is_file():
                unresolved.append({
                    "path": relative,
                    "reason": "UNAVAILABLE_AT_EXPLICIT_PATH",
                })
                continue
            size = path.stat().st_size
            if (
                size > max_report_bytes
                or total_report_bytes + size > max_total_report_bytes
            ):
                unresolved.append({
                    "path": relative, "reason": "BOUNDED_READ_LIMIT"
                })
                continue
            try:
                raw, digest = stable_read(path)
                payload = json.loads(raw)
            except Exception as exc:
                issues.append({
                    "path": relative, "stage": "referenced_report_read",
                    "error_type": type(exc).__name__,
                })
                continue
            total_report_bytes += len(raw)
            source = {
                "path": relative,
                "surface": "fresh_referenced_report",
                "file_bytes_sha256": digest,
                "hash_is_observation_not_consumer_binding": True,
            }
            mode = "FRESH_EXPLICIT_REFERENCE"

        inspect_object(payload, source)
        report_inventory.append({
            "path": relative,
            "read_mode": mode,
            "schema_version": (
                payload.get("schema_version")
                if isinstance(payload, dict) else None
            ),
        })

    summary = {}
    for family in families:
        recorded = observation_counts[(
            family, "RECORDED_VALUE_REQUIRES_SCOPE_REVIEW"
        )]
        summary[family] = {
            "code_mentions": mentions[(family, "code_source")],
            "markdown_mentions": mentions[
                (family, "markdown_or_other_source")
            ],
            "output_mentions": mentions[(family, "saved_output")],
            "recorded_value_observations": recorded,
            "target_or_configuration_observations": observation_counts[(
                family, "TARGET_OR_CONFIGURATION"
            )],
            "conclusion": (
                "EVIDENCE_FOUND_SCOPE_RECONCILIATION_REQUIRED"
                if recorded
                else "NO_STRUCTURED_RESULT_EXTRACTED_IN_INSPECTED_SCOPE"
            ),
            "not_done_projectwide_claimed": False,
        }

    artifact = {
        "artifact_type": "NB22_EXISTING_QUALITY_EVIDENCE_AUDIT",
        "schema_version": "NB22_C050_EVIDENCE_AUDIT_V2",
        "status": "EXISTING_EVIDENCE_AUDIT_RECORDED",
        "root_resolution": root_method,
        "scope": {
            "notebooks": "ALL_LOCAL_IPYNB_UNDER_NOTEBOOKS_PHASE2",
            "automatic_ipynb_checkpoints_excluded": True,
            "source_and_saved_text_json_outputs_inspected": True,
            "images_and_binary_outputs_inspected": False,
            "referenced_json_report_hops": 1,
            "max_report_bytes": max_report_bytes,
            "max_total_report_bytes": max_total_report_bytes,
            "max_report_count": max_report_count,
        },
        "notebook_inventory": notebook_inventory,
        "referenced_report_inventory": report_inventory,
        "family_summary": summary,
        "observations": observations,
        "observation_count": sum(observation_counts.values()),
        "observations_truncated": (
            sum(observation_counts.values()) > len(observations)
        ),
        "unresolved_explicit_references": unresolved,
        "read_issues": issues,
        "interpretation": {
            "mentions_are_not_measurements": True,
            "results_are_not_automatically_accepted": True,
            "different_scopes_and_denominators_not_merged": True,
            "absence_projectwide_not_claimed": True,
            "next_step": (
                "Reconcile found results with producer, population, "
                "reference and gate; reuse successful historical work."
            ),
        },
        "notebooks_executed": 0,
        "jsonl_files_read": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "original_files_modified": False,
        "release_gate_modified": False,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C050.evidence_audit",
        "C050.existing_quality_evidence_audit",
        artifact,
    )

    examples = {}
    for family in families:
        candidates = [
            row for row in observations
            if row["family"] == family
            and row["kind"] == "RECORDED_VALUE_REQUIRES_SCOPE_REVIEW"
        ]
        candidates.sort(key=lambda row: (
            row["source"]["surface"] == "saved_notebook_output",
            isinstance(row["value"], bool)
            or not isinstance(row["value"], (int, float)),
            row["source"]["path"],
        ))
        examples[family] = candidates[:4]

    print(json.dumps({
        "status": artifact["status"],
        "root_resolution": root_method,
        "notebooks_inspected": len(notebook_inventory),
        "reports_inspected_or_reused": len(report_inventory),
        "family_summary": summary,
        "result_examples": examples,
        "observation_count": artifact["observation_count"],
        "observations_truncated": artifact["observations_truncated"],
        "unresolved_reference_count": len(unresolved),
        "read_issue_count": len(issues),
        "output": output,
        "notebooks_executed": 0,
        "jsonl_files_read": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "release_gate_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c50_result = nb22_c50_run()

{"stage": "EXISTING_QUALITY_EVIDENCE_AUDIT_START", "root_resolution": "GIT_ROOT_WITH_PROJECT_AND_PRIOR_EVIDENCE", "prior_C049_location_checked": true, "notebooks": 40, "notebook_execution": false, "model_api_calls": 0}
{"stage": "NOTEBOOK_INSPECTED", "completed": 1, "total": 40, "path": "notebooks/phase2/00_environment_and_baseline.ipynb"}
{"stage": "NOTEBOOK_INSPECTED", "completed": 2, "total": 40, "path": "notebooks/phase2/01_website_ingestion.ipynb"}
{"stage": "NOTEBOOK_INSPECTED", "completed": 3, "total": 40, "path": "notebooks/phase2/02_website_acceptance.ipynb"}
{"stage": "NOTEBOOK_INSPECTED", "completed": 4, "total": 40, "path": "notebooks/phase2/03_book_inventory.ipynb"}
{"stage": "NOTEBOOK_INSPECTED", "completed": 5, "total": 40, "path": "notebooks/phase2/04_book_ingestion.ipynb"}
{"stage": "NOTEBOOK_INSPECTED", "completed": 6, "total": 40, "path": "notebooks/phase2/05_transcript_inventory.ipynb"}
{"stage": "NOTEBOOK_INSPECTED", "completed": 7, "total": 40, "path": "notebooks/

<div dir="rtl" align="right">

### NB22 — C051 | بررسی تکمیلی کیفیت — بستهٔ ۱ از ۳

هدف بستهٔ سه‌مرحله‌ای: تبدیل یافته‌های C050 به جمع‌بندی قابل ارجاع از نتایج واقعی کیفیت.

چک‌لیست:
۱. تفکیک نوع مشاهده‌ها؛ سل جاری.
۲. تطبیق نتایج با منشأ، جمعیت و تصمیم نهایی.
۳. ثبت جمع‌بندی و شکاف‌های واقعی.

این سل سه زیرکار دارد:
۱. کنترل شمارش و کامل‌بودن دفتر مشاهدات موجود؛
۲. تفکیک مقادیر سنجه از شمارش، تعریف، هدف و محدودیت؛
۳. ثبت تمام مشاهده‌ها همراه با علت دسته‌بندی و نمایش اعداد نیازمند تطبیق.

دسته‌بندی مکانیکی، تأیید اعتبار سنجه یا تغییر Gate نیست.
هیچ ورودی دوباره خوانده نمی‌شود و هیچ نوت‌بوک، مدل یا API اجرا نمی‌شود.
اقدام جدیدی به چک‌لیست ۲۸تایی اضافه نمی‌شود.

</div>

In [56]:
# NB22-C051 — Quality evidence review, package 1 of 3
# Reclassifies cached C050 observations; no input file reads.

def nb22_c51_run(c50):
    import json
    import re
    from collections import Counter
    from pathlib import Path

    audit = c50["artifact"]
    observations = audit["observations"]

    if audit["observations_truncated"]:
        raise RuntimeError("COMPLETE_OBSERVATION_LEDGER_REQUIRED")
    if len(observations) != audit["observation_count"]:
        raise RuntimeError("OBSERVATION_ACCOUNTING_MISMATCH")

    # C050 already resolved and wrote its report in the correct project.
    audit_relative = Path(c50["output"]["path"])
    roots = []
    for parent in [Path.cwd(), *Path.cwd().parents]:
        resolved = parent.resolve()
        if (
            (resolved / audit_relative).is_file()
            and (
                resolved / "data/Road_Maps/"
                "RAG_finance_phase2_modular_notebooks_execution_plan.md"
            ).is_file()
            and resolved not in roots
        ):
            roots.append(resolved)
    if not roots:
        raise RuntimeError("C050_PROJECT_LOCATION_NOT_RESOLVED")
    root = roots[0]

    structural = re.compile(
        r"/definitions/|/location/|/candidate_counts/|"
        r"/semantic_report_search/|questions_with_model_correction",
        re.I,
    )
    blocked = re.compile(
        r"blocked_method|existing_blocked_metric_states|"
        r"blocked_metric|not_qualified", re.I
    )
    counter = re.compile(
        r"(?:^|/)(?:tp|fp|fn|tn|model_correct|model_incorrect|"
        r"model_undetermined|candidate_records|cases_with_candidate_records)"
        r"$|(?:^|/)[^/]*(?:_counts?|_record_count|_case_count)"
        r"(?:/|$)|/confusion_matrix/|/decision_counts/",
        re.I,
    )
    metric_leaf = re.compile(
        r"precision|recall|(?:^|_)f1(?:$|_)|rate|fraction|"
        r"accuracy|correctness|coverage|groundedness|"
        r"faithfulness|completeness|score",
        re.I,
    )

    def classify(row):
        field = row["field"]
        value = row["value"]
        leaf = field.rsplit("/", 1)[-1]

        if row["kind"] == "TARGET_OR_CONFIGURATION":
            return (
                "TARGET_OR_CONFIGURATION",
                "Recorded target/configuration, not achieved performance.",
            )
        if structural.search(field):
            return (
                "DEFINITION_OR_SEARCH_METADATA",
                "Definition location or evidence-search metadata.",
            )
        if blocked.search(field):
            return (
                "METHOD_OR_BLOCKED_ACCOUNTING",
                "Method qualification or blocked-population accounting.",
            )
        if value is None:
            return (
                "EXPLICIT_UNAVAILABLE_VALUE",
                "Source explicitly recorded null; meaning needs its context.",
            )
        if isinstance(value, bool):
            return (
                "BOOLEAN_CLAIM_OR_LIMITATION",
                "Boolean assertion; not a numeric quality percentage.",
            )
        if isinstance(value, str):
            return (
                "RECORDED_STATUS",
                "Recorded status; its evidence scope remains unchanged.",
            )
        if counter.search(field):
            return (
                "COUNT_OR_CONFUSION_MATRIX",
                "Count; denominator and population must be identified.",
            )
        if (
            isinstance(value, (int, float))
            and metric_leaf.search(leaf)
        ):
            return (
                "NUMERIC_METRIC_CANDIDATE",
                "Metric-shaped scalar; validity is not established here.",
            )
        return (
            "OTHER_NUMERIC_CONTEXT",
            "Numeric observation requiring document-level interpretation.",
        )

    classified = []
    counts = Counter()
    family_counts = {}

    for number, row in enumerate(observations, start=1):
        category, reason = classify(row)
        counts[category] += 1
        family = row["family"]
        family_counts.setdefault(family, Counter())[category] += 1

        classified.append({
            "observation_id": f"C051-O{number:04d}",
            "family": family,
            "classification": category,
            "classification_reason": reason,
            "source": row["source"],
            "field": row["field"],
            "value": row["value"],
            "recorded_context": row["recorded_context"],
            "original_C050_kind": row["kind"],
            "metric_validity_established": False,
            "selected_as_current_release_metric": False,
        })

    if sum(counts.values()) != len(observations):
        raise RuntimeError("CLASSIFICATION_ACCOUNTING_MISMATCH")

    metric_candidates = [
        row for row in classified
        if row["classification"] == "NUMERIC_METRIC_CANDIDATE"
    ]

    # Group matching numerical observations for display only.
    # Different source locations remain preserved in each group.
    groups = {}
    for row in metric_candidates:
        context = row["recorded_context"]
        signature = json.dumps({
            "family": row["family"],
            "field": row["field"],
            "value": row["value"],
            "recorded_context": context,
        }, ensure_ascii=False, sort_keys=True)

        if signature not in groups:
            groups[signature] = {
                "family": row["family"],
                "field": row["field"],
                "value": row["value"],
                "recorded_context": context,
                "sources": [],
                "observation_ids": [],
                "interpretation": "NEEDS_PRODUCER_AND_SCOPE_RECONCILIATION",
            }
        groups[signature]["sources"].append(row["source"])
        groups[signature]["observation_ids"].append(
            row["observation_id"]
        )

    unresolved = audit["unresolved_explicit_references"]
    unresolved_reasons = Counter(
        row["reason"] for row in unresolved
    )

    artifact = {
        "artifact_type": "NB22_QUALITY_EVIDENCE_CLASSIFICATION",
        "schema_version": "NB22_C051_QUALITY_REVIEW_PACKAGE1_V1",
        "status": "QUALITY_REVIEW_PACKAGE1_CLASSIFICATION_COMPLETE",
        "review_package": {
            "package_number": 1,
            "planned_package_count": 3,
            "checklist": [
                {
                    "step": 1,
                    "task": "Classify existing observations",
                    "status": "COMPLETE",
                },
                {
                    "step": 2,
                    "task": "Reconcile producer, population and final decision",
                    "status": "PENDING",
                },
                {
                    "step": 3,
                    "task": "Record usable results and actual remaining gaps",
                    "status": "PENDING",
                },
            ],
        },
        "source_audit": c50["output"],
        "observations_accounted": len(classified),
        "classification_counts": dict(counts),
        "per_family_classification_counts": {
            family: dict(values)
            for family, values in family_counts.items()
        },
        "classified_observations": classified,
        "numeric_metric_groups": list(groups.values()),
        "C050_scope_limits": {
            "notebooks_inspected": len(audit["notebook_inventory"]),
            "reports_inspected_or_reused": len(
                audit["referenced_report_inventory"]
            ),
            "unresolved_reference_count": len(unresolved),
            "unresolved_reference_reasons": dict(unresolved_reasons),
            "unresolved_references_are_not_unfinished_tasks": True,
        },
        "overall_answer_accuracy_computed": False,
        "historical_values_promoted_to_live_product_accuracy": False,
        "new_checklist_action_created": False,
        "input_files_reread": 0,
        "notebooks_executed": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "release_gate_modified": False,
        "checklist_modified": False,
    }

    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C051.quality_review",
        "C051.quality_evidence_classification",
        artifact,
    )

    print(json.dumps({
        "status": artifact["status"],
        "review_package": "1_OF_3",
        "observations_accounted": len(classified),
        "classification_counts": dict(counts),
        "per_family_classification_counts": artifact[
            "per_family_classification_counts"
        ],
        "numeric_metric_groups": [
            {
                **{
                    key: value for key, value in group.items()
                    if key != "sources"
                },
                "source_count": len(group["sources"]),
                "first_source": group["sources"][0],
            }
            for group in groups.values()
        ],
        "C050_scope_limits": artifact["C050_scope_limits"],
        "output": output,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "release_gate_modified": False,
        "next_review_step": 2,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c51_result = nb22_c51_run(nb22_c50_result)

{
  "status": "QUALITY_REVIEW_PACKAGE1_CLASSIFICATION_COMPLETE",
  "review_package": "1_OF_3",
  "observations_accounted": 190,
  "classification_counts": {
    "RECORDED_STATUS": 2,
    "BOOLEAN_CLAIM_OR_LIMITATION": 107,
    "DEFINITION_OR_SEARCH_METADATA": 8,
    "TARGET_OR_CONFIGURATION": 17,
    "METHOD_OR_BLOCKED_ACCOUNTING": 2,
    "EXPLICIT_UNAVAILABLE_VALUE": 9,
    "COUNT_OR_CONFUSION_MATRIX": 35,
    "NUMERIC_METRIC_CANDIDATE": 6,
    "OTHER_NUMERIC_CONTEXT": 4
  },
  "per_family_classification_counts": {
    "citation_locator": {
      "RECORDED_STATUS": 1,
      "DEFINITION_OR_SEARCH_METADATA": 2,
      "BOOLEAN_CLAIM_OR_LIMITATION": 3,
      "COUNT_OR_CONFUSION_MATRIX": 5,
      "OTHER_NUMERIC_CONTEXT": 1,
      "EXPLICIT_UNAVAILABLE_VALUE": 1,
      "TARGET_OR_CONFIGURATION": 1
    },
    "answer_correctness": {
      "BOOLEAN_CLAIM_OR_LIMITATION": 95,
      "DEFINITION_OR_SEARCH_METADATA": 1,
      "TARGET_OR_CONFIGURATION": 1,
      "COUNT_OR_CONFUSION_MATRIX": 12,
   

<div dir="rtl" align="right">

### NB22 — C052 | بررسی تکمیلی کیفیت — بستهٔ ۲ از ۳

سه زیرکار:
۱. انتخاب شش عدد سنجه و مشاهده‌های مرتبط با وضعیت، محدودیت و شواهد استناد؛
۲. تطبیق آن‌ها با خروجی ذخیره‌شده و هویت همان نسخهٔ نوت‌بوک؛
۳. استخراج جمعیت، مخرج، نوع مرجع و محدودیت‌های گزارش اصلی.

فقط فایل‌های لازم برای بازیابی زمینهٔ خروجی‌های منتخب خوانده می‌شوند.
کد یا تست تاریخی اجرا نمی‌شود و Gate تغییر نمی‌کند.

شرط پایان: برای هر مشاهدهٔ منتخب، منشأ قابل تطبیق یا مشکل دقیق اتصال ثبت شود.
وجود null یا محدودیت در یک گزارش، نبود کار در کل پروژه محسوب نمی‌شود.
بستهٔ بعدی، جمع‌بندی نهایی همین بررسی است.

</div>

In [57]:
# NB22-C052 — Quality evidence review, package 2 of 3
# Recover context only for selected stored observations; never execute cells.

def nb22_c52_run(c50, c51):
    import hashlib
    import json
    import re
    from collections import Counter, defaultdict
    from pathlib import Path

    audit_relative = Path(c50["output"]["path"])
    roots = []
    for parent in [Path.cwd(), *Path.cwd().parents]:
        resolved = parent.resolve()
        if (
            (resolved / audit_relative).is_file()
            and (
                resolved / "data/Road_Maps/"
                "RAG_finance_phase2_modular_notebooks_execution_plan.md"
            ).is_file()
            and resolved not in roots
        ):
            roots.append(resolved)
    if not roots:
        raise RuntimeError("C050_PROJECT_LOCATION_NOT_RESOLVED")
    root = roots[0]

    rows = c51["artifact"]["classified_observations"]
    selected = [
        row for row in rows
        if row["classification"] in {
            "NUMERIC_METRIC_CANDIDATE",
            "EXPLICIT_UNAVAILABLE_VALUE",
            "METHOD_OR_BLOCKED_ACCOUNTING",
            "RECORDED_STATUS",
            "OTHER_NUMERIC_CONTEXT",
        }
        or (
            row["family"] in {
                "citation_correctness",
                "citation_coverage",
                "citation_locator",
            }
            and row["classification"] == "COUNT_OR_CONFUSION_MATRIX"
        )
    ]
    if not selected:
        raise RuntimeError("NO_SELECTED_OBSERVATIONS")

    personal = re.compile(
        r"(?:/Users/|/home/)[A-Za-z0-9._-]+/"
        r"|[A-Za-z]:\\Users\\[A-Za-z0-9._ -]+\\"
    )
    qualifier_key = re.compile(
        r"scope|limitation|denominator|population_count|"
        r"evaluated_case_count|independent|live_generator|"
        r"qualified|qualification|reference_type|"
        r"generation_origin|completion_scope|reason|status",
        re.I,
    )

    def safe(text):
        return personal.sub("<PERSONAL_HOME>/", str(text))[:700]

    def qualifiers(value, field=""):
        found = []
        if isinstance(value, dict):
            for key, item in value.items():
                pointer = f"{field}/{key}"
                if qualifier_key.search(str(key)) and (
                    isinstance(item, (str, bool, int, float))
                    or item is None
                ):
                    found.append({
                        "field": pointer,
                        "value": safe(item) if isinstance(item, str) else item,
                    })
                elif (
                    qualifier_key.search(str(key))
                    and isinstance(item, list)
                    and all(isinstance(x, str) for x in item)
                ):
                    found.append({
                        "field": pointer,
                        "value": [safe(x) for x in item],
                    })
                if isinstance(item, (dict, list)):
                    found.extend(qualifiers(item, pointer))
        elif isinstance(value, list):
            for index, item in enumerate(value):
                if isinstance(item, (dict, list)):
                    found.extend(qualifiers(item, f"{field}/{index}"))
        return found

    def stable_json(relative, expected=None):
        rel = Path(relative)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError("INVALID_SOURCE_PATH")
        path = (root / rel).resolve()
        if root not in path.parents:
            raise RuntimeError("SOURCE_OUTSIDE_PROJECT")
        before = path.stat()
        raw = path.read_bytes()
        after = path.stat()
        if (
            before.st_size != after.st_size
            or before.st_mtime_ns != after.st_mtime_ns
        ):
            raise RuntimeError("SOURCE_CHANGED_DURING_READ")
        digest = hashlib.sha256(raw).hexdigest()
        if expected is not None and digest != expected:
            raise RuntimeError("SOURCE_IDENTITY_CHANGED")
        return json.loads(raw), digest

    decoder = json.JSONDecoder()

    def json_blocks(text):
        position = 0
        while position < len(text):
            match = re.search(r"[\{\[]", text[position:])
            if match is None:
                return
            start = position + match.start()
            try:
                value, end = decoder.raw_decode(text, start)
            except (ValueError, RecursionError):
                position = start + 1
                continue
            yield value
            position = end

    def output_payloads(output):
        data = output.get("data", {})
        structured = data.get("application/json")
        if structured is not None:
            if isinstance(structured, str):
                yield from json_blocks(structured)
            else:
                yield structured
        for text in (
            output.get("text"),
            data.get("text/plain"),
            data.get("text/markdown"),
        ):
            if isinstance(text, list):
                text = "".join(text)
            if isinstance(text, str):
                yield from json_blocks(text)

    def at_field(payload, field):
        value = payload
        # Uses the same field notation produced by C050.
        for segment in field.lstrip("/").split("/"):
            value = (
                value[int(segment)]
                if isinstance(value, list)
                else value[segment]
            )
        return value

    by_file = defaultdict(list)
    for row in selected:
        by_file[row["source"]["path"]].append(row)

    caches = {}
    for variable, key in (
        ("nb22_c03_result", "loaded"),
        ("nb22_c06_result", "package_payloads"),
        ("nb22_c12_result", "payloads"),
        ("nb22_c20_result", "payloads"),
        ("nb22_c23_result", "payloads"),
    ):
        result = globals().get(variable)
        mapping = result.get(key) if isinstance(result, dict) else None
        if isinstance(mapping, dict):
            caches.update(mapping)

    contexts = {}
    reconciliation = []
    read_issues = []
    file_receipts = []
    fresh_reads = 0

    for relative, file_rows in sorted(by_file.items()):
        is_notebook = relative.endswith(".ipynb")
        expected_hashes = {
            row["source"].get("file_bytes_sha256")
            for row in file_rows
            if row["source"].get("file_bytes_sha256")
        }
        if len(expected_hashes) > 1:
            raise RuntimeError("MULTIPLE_SOURCE_VERSIONS_IN_ONE_SELECTION")
        expected = next(iter(expected_hashes), None)

        # NB22 has legitimately grown since C050. Reuse the saved
        # artifact evidence rather than substituting changed cell offsets.
        if is_notebook and relative.endswith(
            "/22_release_and_reproducibility.ipynb"
        ):
            for row in file_rows:
                reconciliation.append({
                    "observation_id": row["observation_id"],
                    "family": row["family"],
                    "field": row["field"],
                    "value": row["value"],
                    "status": "C050_SNAPSHOT_EVIDENCE_RETAINED_NOT_REREAD",
                    "source": row["source"],
                    "recorded_context": row["recorded_context"],
                })
            continue

        try:
            if not is_notebook and relative in caches:
                document = caches[relative]
                read_mode = "REUSED_EXISTING_DOCUMENT_CACHE"
                digest = expected
            else:
                document, digest = stable_json(relative, expected)
                fresh_reads += 1
                read_mode = "FRESH_CONTEXT_READ_MATCHING_PRIOR_IDENTITY"
        except Exception as exc:
            read_issues.append({
                "path": relative,
                "error_type": type(exc).__name__,
                "reason": safe(exc),
            })
            for row in file_rows:
                reconciliation.append({
                    "observation_id": row["observation_id"],
                    "family": row["family"],
                    "field": row["field"],
                    "value": row["value"],
                    "status": "CONTEXT_SOURCE_NOT_RELOADED",
                    "source": row["source"],
                    "recorded_context": row["recorded_context"],
                })
            continue

        file_receipts.append({
            "path": relative,
            "file_bytes_sha256": digest,
            "read_mode": read_mode,
        })

        block_cache = {}
        for row in file_rows:
            source = row["source"]
            try:
                if is_notebook:
                    cell_index = source["physical_cell_index"]
                    output_index = source["output_index"]
                    block_key = (cell_index, output_index)
                    if block_key not in block_cache:
                        output = document["cells"][cell_index][
                            "outputs"
                        ][output_index]
                        block_cache[block_key] = list(output_payloads(output))
                    blocks = block_cache[block_key]
                else:
                    block_key = ("report",)
                    blocks = [document]

                matching = []
                for block in blocks:
                    try:
                        actual = at_field(block, row["field"])
                    except (KeyError, IndexError, TypeError, ValueError):
                        continue
                    if actual == row["value"] and (
                        isinstance(actual, bool) == isinstance(
                            row["value"], bool
                        )
                    ):
                        matching.append(block)

                if not matching:
                    status = "FIELD_NOT_MATCHED_IN_RELOADED_CONTEXT"
                    context_ids = []
                else:
                    status = "RECORDED_VALUE_MATCHES_SOURCE"
                    context_ids = []
                    for block in matching:
                        context_bytes = json.dumps(
                            block, ensure_ascii=False, sort_keys=True,
                            separators=(",", ":"), allow_nan=False,
                        ).encode("utf-8")
                        context_id = hashlib.sha256(context_bytes).hexdigest()
                        if context_id not in contexts:
                            contexts[context_id] = {
                                "context_id": context_id,
                                "source": source,
                                "schema_version": (
                                    block.get("schema_version")
                                    if isinstance(block, dict) else None
                                ),
                                "top_level_keys": (
                                    sorted(block)
                                    if isinstance(block, dict) else None
                                ),
                                "qualifiers": qualifiers(block),
                            }
                        if context_id not in context_ids:
                            context_ids.append(context_id)

                reconciliation.append({
                    "observation_id": row["observation_id"],
                    "family": row["family"],
                    "classification": row["classification"],
                    "field": row["field"],
                    "value": row["value"],
                    "status": status,
                    "source": source,
                    "context_ids": context_ids,
                    "recorded_context": row["recorded_context"],
                    "accepted_as_current_product_accuracy": False,
                })
            except Exception as exc:
                reconciliation.append({
                    "observation_id": row["observation_id"],
                    "family": row["family"],
                    "field": row["field"],
                    "value": row["value"],
                    "status": "CONTEXT_EXTRACTION_ISSUE",
                    "error_type": type(exc).__name__,
                    "source": source,
                })

    if len(reconciliation) != len(selected):
        raise RuntimeError("SELECTED_OBSERVATION_ACCOUNTING_MISMATCH")

    counts = Counter(row["status"] for row in reconciliation)
    artifact = {
        "artifact_type": "NB22_QUALITY_EVIDENCE_CONTEXT_RECONCILIATION",
        "schema_version": "NB22_C052_QUALITY_REVIEW_PACKAGE2_V1",
        "status": "QUALITY_REVIEW_PACKAGE2_CONTEXT_CHECK_RECORDED",
        "review_package": "2_OF_3",
        "source_classification": c51["output"],
        "selected_observation_count": len(selected),
        "reconciliation_status_counts": dict(counts),
        "reconciled_observations": reconciliation,
        "source_contexts": list(contexts.values()),
        "file_receipts": file_receipts,
        "read_issues": read_issues,
        "fresh_context_file_reads": fresh_reads,
        "absence_projectwide_claimed": False,
        "metrics_recomputed": False,
        "historical_work_reexecuted": False,
        "health_checks_requested": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "release_gate_modified": False,
        "checklist_modified": False,
        "next_review_step": 3,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C052.quality_review",
        "C052.quality_evidence_context_reconciliation",
        artifact,
    )

    print(json.dumps({
        "status": artifact["status"],
        "review_package": "2_OF_3",
        "selected_observation_count": len(selected),
        "reconciliation_status_counts": dict(counts),
        "numeric_metrics": [
            row for row in reconciliation
            if row.get("classification") == "NUMERIC_METRIC_CANDIDATE"
        ],
        "source_contexts": list(contexts.values()),
        "read_issues": read_issues,
        "fresh_context_file_reads": fresh_reads,
        "output": output,
        "new_api_calls": 0,
        "network_calls": 0,
        "release_gate_modified": False,
        "next_review_step": 3,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c52_result = nb22_c52_run(
    nb22_c50_result,
    nb22_c51_result,
)

{
  "status": "QUALITY_REVIEW_PACKAGE2_CONTEXT_CHECK_RECORDED",
  "review_package": "2_OF_3",
  "selected_observation_count": 38,
  "reconciliation_status_counts": {
    "RECORDED_VALUE_MATCHES_SOURCE": 37,
    "C050_SNAPSHOT_EVIDENCE_RETAINED_NOT_REREAD": 1
  },
  "numeric_metrics": [
    {
      "observation_id": "C051-O0115",
      "family": "answer_correctness",
      "classification": "NUMERIC_METRIC_CANDIDATE",
      "field": "/rates/model_correct_fraction_of_all_cases",
      "value": 0.9512619669277633,
      "status": "RECORDED_VALUE_MATCHES_SOURCE",
      "source": {
        "path": "notebooks/phase2/21c_evaluation_and_ablation_continued.ipynb",
        "file_bytes_sha256": "a235ed497c62605ca00d2c4c0dd27744851124242d9bcc86b225190bec8148e1",
        "physical_cell_index": 34,
        "output_index": 0,
        "execution_count": 33,
        "surface": "saved_notebook_output"
      },
      "context_ids": [
        "2d4d0c304251d4ffeb993f26b7c80dbfd4963a73592b55a9af7e1755ba1897

<div dir="rtl" align="right">

### NB22 — C053 | بررسی تکمیلی کیفیت — بستهٔ ۳ از ۳

سه زیرکار:
۱. ثبت شش مقدار عددی که در C052 با خروجی اصلی تطبیق داده شدند؛
۲. تفکیک نتیجهٔ تاریخی، بررسی محدود و سنجهٔ کل‌جمعیت برای شش خانوادهٔ کیفیت؛
۳. ثبت جمع‌بندی قابل ارجاع همراه با شواهد و محدودیت ممیزی.

نتیجهٔ بررسی، «کار انجام نشده» نیست.
کارهای انجام‌شده و دامنهٔ واقعی آن‌ها ثبت می‌شوند؛ نتیجهٔ محدود به پذیرش کل محصول تبدیل نمی‌شود.

این سل فقط نتایج C050 تا C052 را مصرف می‌کند.
فایل ورودی، نوت‌بوک، مدل، ارزیابی و API دوباره خوانده یا اجرا نمی‌شوند.
Gate، آستانه‌ها و چک‌لیست تغییر نمی‌کنند.

شرط پایان بسته: جمع‌بندی نهایی ذخیره و بازخوانی شود.
این پایانِ بررسی سه‌بسته‌ای است، نه پذیرش خودکار Release.

</div>

In [58]:
# NB22-C053 — Quality evidence review, package 3 of 3
# Final scoped summary from cached evidence; no new input reads.

def nb22_c53_run(c50, c51, c52):
    import copy
    import json
    from pathlib import Path

    report_relative = Path(c52["output"]["path"])
    root = next(
        (
            parent.resolve()
            for parent in [Path.cwd(), *Path.cwd().parents]
            if (parent / report_relative).is_file()
            and (
                parent / "data/Road_Maps/"
                "RAG_finance_phase2_modular_notebooks_execution_plan.md"
            ).is_file()
        ),
        None,
    )
    if root is None:
        raise RuntimeError("C052_PROJECT_LOCATION_NOT_RESOLVED")

    reconciliation = c52["artifact"]["reconciled_observations"]
    contexts = c52["artifact"]["source_contexts"]
    numeric = [
        row for row in reconciliation
        if row.get("classification") == "NUMERIC_METRIC_CANDIDATE"
    ]
    if len(numeric) != 6:
        raise RuntimeError("EXPECTED_SIX_NUMERIC_OBSERVATIONS")
    if any(
        row["status"] != "RECORDED_VALUE_MATCHES_SOURCE"
        for row in numeric
    ):
        raise RuntimeError("NUMERIC_SOURCE_RECONCILIATION_INCOMPLETE")

    def status_contexts(status):
        matches = [
            context
            for context in contexts
            if any(
                qualifier["field"] == "/status"
                and qualifier["value"] == status
                for qualifier in context["qualifiers"]
            )
        ]
        if not matches:
            raise RuntimeError(
                f"EXPECTED_RECORDED_CONTEXT_MISSING: {status}"
            )
        return [
            {
                "context_id": context["context_id"],
                "source": copy.deepcopy(context["source"]),
                "recorded_status": status,
            }
            for context in matches
        ]

    answer_evidence = status_contexts(
        "ACTION26_OFFLINE_EVALUATION_REPORT_COMPLETE"
    )
    citation_direct = status_contexts(
        "DIRECT_CITATION_AND_SCOPED_GROUNDING_REVIEW_IMPORTED"
    )
    citation_coverage = status_contexts(
        "SCOPED_CITATION_REQUIREDNESS_AND_COVERAGE_REVIEWED"
    )
    original_review = status_contexts(
        "ORIGINAL_TEST_REVIEW_SOURCES_EXTRACTED_WITH_VERSION_ACCOUNTING"
    )
    safety_evidence = status_contexts(
        "ACTION27_EXISTING_SAFETY_EVIDENCE_CONSOLIDATED"
    )

    labels = {
        "/rates/model_correct_fraction_of_all_cases":
            "سهم رأی مدلِ درست از کل جمعیت تاریخی",
        "/rates/model_correct_rate_among_determined_judgments":
            "سهم رأی مدلِ درست میان داوری‌های تعیین‌شده",
        "/rates/model_correct_rate_among_all_valid_judgments":
            "سهم رأی مدلِ درست میان همهٔ داوری‌های معتبر",
        "/abstention_metrics/precision":
            "دقت تشخیص نبود پاسخ در زیرجمعیت قابل ارزیابی",
        "/abstention_metrics/recall":
            "بازخوانی تشخیص نبود پاسخ در زیرجمعیت قابل ارزیابی",
        "/abstention_metrics/f1":
            "امتیاز F1 تشخیص نبود پاسخ در زیرجمعیت قابل ارزیابی",
    }

    metrics = []
    for row in numeric:
        if row["field"] not in labels:
            raise RuntimeError("UNEXPECTED_METRIC_FIELD")
        context = row["recorded_context"]

        if row["family"] == "answer_correctness":
            if context.get(
                "these_are_not_independent_population_accuracy"
            ) is not True:
                raise RuntimeError("ANSWER_REFERENCE_LIMIT_REQUIRED")
            limitation = (
                "رأی تاریخی مدل؛ دقت مستقل کل جمعیت یا عملکرد "
                "مولد زنده از آن نتیجه نمی‌شود."
            )
        else:
            if context.get("scope") != (
                "CONDITIONAL_ON_EVALUABLE_MODEL_REFERENCE"
            ):
                raise RuntimeError("ABSTENTION_SCOPE_REQUIRED")
            limitation = (
                "مشروط به مرجع مدل و زیرجمعیت قابل ارزیابی؛ "
                "پذیرش مستقل عملکرد محصول زنده نیست."
            )

        metrics.append({
            "label_fa": labels[row["field"]],
            "field": row["field"],
            "value": row["value"],
            "recorded_population_count": context.get("population_count"),
            "recorded_evaluated_case_count": context.get(
                "evaluated_case_count"
            ),
            "source": copy.deepcopy(row["source"]),
            "observation_id": row["observation_id"],
            "context_ids": copy.deepcopy(row["context_ids"]),
            "source_value_match": True,
            "usable_scope_fa": limitation,
            "independent_product_accuracy_claimed": False,
        })

    families = [
        {
            "family": "answer_correctness",
            "finding_fa": (
                "سه نرخ تاریخیِ رأی مدل موجود و با خروجی اصلی "
                "تطبیق داده شده‌اند."
            ),
            "remaining_limit_fa": (
                "این نرخ‌ها به‌صراحت دقت مستقل محصول زنده نیستند."
            ),
            "evidence": answer_evidence,
        },
        {
            "family": "abstention",
            "finding_fa": (
                "Precision، Recall و F1 تاریخی موجودند؛ "
                "جمعیت قابل ارزیابی ثبت‌شده ۱۱۳۱ مورد است."
            ),
            "remaining_limit_fa": (
                "مرجع، مدل است و دامنه مشروط است؛ "
                "به کل عملکرد زنده تعمیم داده نمی‌شود."
            ),
            "evidence": answer_evidence,
        },
        {
            "family": "citation_correctness",
            "finding_fa": (
                "بررسی مستقیم و محدود اتصال استناد و پشتیبانی "
                "انجام شده و گزارش آن موجود است."
            ),
            "remaining_limit_fa": (
                "امتیاز رسمی کل‌جمعیت در منابع منتخب احراز نشده؛ "
                "بررسی محدود، نبود کار نیست."
            ),
            "evidence": citation_direct + citation_coverage,
        },
        {
            "family": "citation_coverage",
            "finding_fa": (
                "ضرورت استناد و پوشش واحدهای بررسی‌شده ارزیابی "
                "شده و شمارش‌های آن ثبت شده‌اند."
            ),
            "remaining_limit_fa": (
                "پوشش واحدهای همان بررسی، درصد رسمی کل پاسخ‌ها نیست."
            ),
            "evidence": citation_coverage + original_review,
        },
        {
            "family": "citation_locator",
            "finding_fa": (
                "رکوردهای اصلی بررسی و اتصال نسخه استخراج شده‌اند."
            ),
            "remaining_limit_fa": (
                "تعداد رکورد یا تعداد پرونده، بدون تعریف واحد و "
                "مخرج به امتیاز دقت محل استناد تبدیل نمی‌شود."
            ),
            "evidence": original_review,
        },
        {
            "family": "safety",
            "finding_fa": (
                "ممیزی بحرانی محدود، بررسی‌های هدفمند، آزمون مرز "
                "درخواست و رگرسیون محلی گزارش شده‌اند."
            ),
            "remaining_limit_fa": (
                "Rubric و مخرج این آزمون‌ها متفاوت است؛ "
                "نرخ ایمنی کل جمعیت یا مولد زنده از ترکیبشان ساخته نمی‌شود."
            ),
            "evidence": safety_evidence,
        },
    ]

    # Preserve exact assertions from the original safety summary.
    safety_statuses = []
    safety_ids = {row["context_id"] for row in safety_evidence}
    for context in contexts:
        if context["context_id"] in safety_ids:
            safety_statuses.extend(
                copy.deepcopy(qualifier)
                for qualifier in context["qualifiers"]
                if qualifier["field"].startswith(
                    "/existing_followups_and_tests/"
                )
            )

    unavailable = [
        copy.deepcopy(row)
        for row in reconciliation
        if row.get("classification") == "EXPLICIT_UNAVAILABLE_VALUE"
    ]

    artifact = {
        "artifact_type": "NB22_EXISTING_QUALITY_EVIDENCE_FINAL_REVIEW",
        "schema_version": "NB22_C053_QUALITY_REVIEW_PACKAGE3_V1",
        "status": "THREE_PACKAGE_QUALITY_EVIDENCE_REVIEW_COMPLETE_IN_SCOPE",
        "review_packages_completed": 3,
        "review_packages_planned": 3,
        "source_reports": {
            "audit": c50["output"],
            "classification": c51["output"],
            "context_reconciliation": c52["output"],
        },
        "confirmed_recorded_metrics": metrics,
        "family_findings": families,
        "recorded_scoped_safety_test_statuses": safety_statuses,
        "explicit_unavailable_values_in_selected_sources": unavailable,
        "coverage_and_limits": {
            "notebooks_inspected": len(
                c50["artifact"]["notebook_inventory"]
            ),
            "referenced_reports_inspected_or_reused": len(
                c50["artifact"]["referenced_report_inventory"]
            ),
            "observations_classified": c51["artifact"][
                "observations_accounted"
            ],
            "context_check_status_counts": c52["artifact"][
                "reconciliation_status_counts"
            ],
            "unreviewed_reference_count": len(
                c50["artifact"]["unresolved_explicit_references"]
            ),
            "report_scan_was_bounded": True,
            "all_project_reports_exhaustively_reviewed": False,
        },
        "conclusions": {
            "previous_evaluation_work_exists": True,
            "blanket_claim_no_results_exists_is_supported": False,
            "one_overall_product_accuracy_established": False,
            "all_unknown_values_are_missing_work": False,
            "automatic_rerun_of_prior_work_required": False,
            "remaining_acceptance_question": (
                "Which existing results satisfy the intended release "
                "population, reference, measurement unit and gate? "
                "Scoped results remain usable within their own scope."
            ),
        },
        "quality_review_complete_in_declared_scope": True,
        "release_gate_modified": False,
        "release_acceptance_granted_by_this_review": False,
        "health_deferral_preserved": True,
        "input_files_reread": 0,
        "historical_computations_reexecuted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
        "next_step": (
            "Use this summary to decide any targeted acceptance or "
            "improvement work. No automatic further audit or rerun."
        ),
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C053.quality_review",
        "C053.existing_quality_evidence_final_review",
        artifact,
    )

    print(json.dumps({
        "status": artifact["status"],
        "review_packages_completed": 3,
        "confirmed_recorded_metrics": metrics,
        "family_findings": families,
        "recorded_scoped_safety_test_statuses": safety_statuses,
        "coverage_and_limits": artifact["coverage_and_limits"],
        "conclusions": artifact["conclusions"],
        "quality_review_complete_in_declared_scope": True,
        "output": output,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "release_gate_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c53_result = nb22_c53_run(
    nb22_c50_result,
    nb22_c51_result,
    nb22_c52_result,
)

{
  "status": "THREE_PACKAGE_QUALITY_EVIDENCE_REVIEW_COMPLETE_IN_SCOPE",
  "review_packages_completed": 3,
  "confirmed_recorded_metrics": [
    {
      "label_fa": "سهم رأی مدلِ درست از کل جمعیت تاریخی",
      "field": "/rates/model_correct_fraction_of_all_cases",
      "value": 0.9512619669277633,
      "recorded_population_count": 1149,
      "recorded_evaluated_case_count": null,
      "source": {
        "path": "notebooks/phase2/21c_evaluation_and_ablation_continued.ipynb",
        "file_bytes_sha256": "a235ed497c62605ca00d2c4c0dd27744851124242d9bcc86b225190bec8148e1",
        "physical_cell_index": 34,
        "output_index": 0,
        "execution_count": 33,
        "surface": "saved_notebook_output"
      },
      "observation_id": "C051-O0115",
      "context_ids": [
        "2d4d0c304251d4ffeb993f26b7c80dbfd4963a73592b55a9af7e1755ba1897ad"
      ],
      "source_value_match": true,
      "usable_scope_fa": "رأی تاریخی مدل؛ دقت مستقل کل جمعیت یا عملکرد مولد زنده از آن نتیجه ن

<div dir="rtl" align="right">

### NB22 — C054 | ثبت جمع‌بندی کیفیت و مسیر ادامه

#### نتیجهٔ بررسی سه‌بسته‌ای C050 تا C053

۴۰ نوت‌بوک، شامل ۱۱ نوت‌بوک آرشیوی گلدن، و ۲۵۰ گزارش بررسی یا مصرف شدند.
۱۹۰ مشاهده دسته‌بندی شد؛ ۳۷ مشاهدهٔ منتخب با منبع تطبیق داشت و یک مشاهدهٔ Snapshot قبلی حفظ شد.

| نتیجهٔ ثبت‌شده | مقدار | دامنهٔ قابل استفاده |
| --- | ---: | --- |
| رأی مدلِ درست از کل جمعیت تاریخی | ۹۵٫۱۳٪ | ۱۱۴۹ مورد؛ دقت مستقل محصول نیست |
| رأی مدلِ درست میان داوری‌های تعیین‌شده | ۹۷٫۲۴٪ | زیرجمعیت تعیین‌شده با مخرج متفاوت |
| رأی مدلِ درست میان داوری‌های معتبر | ۹۶٫۲۱٪ | زیرجمعیت معتبر با مخرج متفاوت |
| دقت تشخیص نبود پاسخ | ۴۲٫۸۶٪ | مرجع مدل؛ زیرجمعیت قابل ارزیابی |
| بازخوانی تشخیص نبود پاسخ | ۱۰۰٪ | همان زیرجمعیت |
| امتیاز F1 تشخیص نبود پاسخ | ۰٫۶۰ | ۱۱۳۱ مورد؛ ارزیابی مشروط |

بررسی‌های محدود استناد، پوشش، محل شاهد و آزمون‌های ایمنی نیز موجودند.
گزارش‌های اصلی، محدودیت تعمیم آن‌ها به کل جمعیت یا مولد زنده را ثبت کرده‌اند.
همهٔ گزارش‌های پروژه به‌طور جامع بررسی نشده‌اند؛ نبود کار در کل پروژه ادعا نمی‌شود.

#### وضعیت پذیرش

- نتایج قبلی پیدا و دامنهٔ آن‌ها روشن‌تر شد.
- موتور و آستانه‌ها تغییر نکردند؛ بهبود تازهٔ عملکرد ادعا نمی‌شود.
- پذیرش Release همچنان صادر نشده است.
- تعویق هلث‌چک برای بهینه‌سازی حفظ می‌شود.
- بررسی عمومی اسناد در این بسته ادامه پیدا نمی‌کند.

#### کار جاری و چهار زیرکار

۱. تطبیق اتصال تصمیم و تحویل‌نامهٔ قبلی؛
۲. ثبت اعداد و محدودیت‌های تأییدشده؛
۳. ثبت مسیر رسیدگی به شکست عملکرد، کفایت شواهد و وابستگی‌های فنی؛
۴. ذخیره و بازخوانی نسخهٔ تکمیلی تحویل‌نامه.

#### مسیر اصلاح بعدی

- بازیابی: بررسی هدفمند خطاهای موجود و طراحی اصلاح در Development، در نوت‌بوک‌های ۱۶ و ۱۷.
- سنجه‌های مسدود: تعیین کفایت روش، مرجع، واحد سنجش و مخرج؛ مصرف نتایج موجود در صورت کفایت.
- نبود پاسخ: حفظ نتیجهٔ تاریخی ۰٫۶ و تعیین اینکه آیا روش آن شرط پذیرش نسخهٔ موردنظر را پوشش می‌دهد.
- بازتولیدپذیری و انتشار: رسیدگی به محدودیت‌های ثبت‌شدهٔ نسخهٔ اجرایی در دامنهٔ خودشان.

این سل مجوز اجرای مدل، تغییر مؤلفه، تغییر آستانه یا انتشار عمومی نیست.
API، شبکه، بازخوانی داده و تغییر گزارش‌های تاریخی: صفر.

</div>

In [59]:
# NB22-C054 — Consolidate quality findings into the handoff and next-work plan
# Reuses C047/C049/C053; no new evidence search or historical execution.

def nb22_c54_run(c47, c49, c53):
    import copy
    import json
    from pathlib import Path

    relative = Path(c53["output"]["path"])
    root = next(
        (
            parent.resolve()
            for parent in [Path.cwd(), *Path.cwd().parents]
            if (parent / relative).is_file()
            and (
                parent / "data/Road_Maps/"
                "RAG_finance_phase2_modular_notebooks_execution_plan.md"
            ).is_file()
        ),
        None,
    )
    if root is None:
        raise RuntimeError("QUALITY_REVIEW_PROJECT_LOCATION_NOT_RESOLVED")

    gate = c47["artifact"]
    prior_handoff = c49["artifact"]
    review = c53["artifact"]

    if not review["quality_review_complete_in_declared_scope"]:
        raise RuntimeError("COMPLETED_SCOPED_QUALITY_REVIEW_REQUIRED")
    if not prior_handoff["handoff_preparation_complete"]:
        raise RuntimeError("EXISTING_HANDOFF_REQUIRED")

    gate_checkpoint = gate["checkpoint_manifest"]
    handoff_checkpoint = prior_handoff["inputs"]["checkpoint_manifest"]
    for key in ("path", "file_bytes_sha256"):
        if gate_checkpoint[key] != handoff_checkpoint[key]:
            raise RuntimeError("GATE_HANDOFF_CHECKPOINT_MISMATCH")

    if (
        gate["release_acceptance_granted"] is not False
        or prior_handoff["release_acceptance_granted"] is not False
    ):
        raise RuntimeError("PRIOR_DECISION_DIFFERS_FROM_REVIEWED_STATE")

    metrics = copy.deepcopy(review["confirmed_recorded_metrics"])
    if len(metrics) != 6 or not all(
        row["source_value_match"] for row in metrics
    ):
        raise RuntimeError("EXPECTED_SIX_SOURCE_MATCHED_METRICS")

    workstreams = [
        {
            "id": "QUALITY-RETRIEVAL",
            "responsible_notebooks": [16, 17],
            "basis": "TWO_RECORDED_ABSOLUTE_RETRIEVAL_GATE_FAILURES",
            "next_work": (
                "Reuse the existing failure catalogue and versioned "
                "retrieval configuration. Define a bounded Development "
                "experiment before any component change."
            ),
            "constraints": [
                "Do not retune on historical Test results.",
                "Do not rebuild indexes or embeddings without demonstrated need.",
                "Do not promise threshold attainment from one experiment.",
            ],
            "implementation_started": False,
        },
        {
            "id": "QUALITY-EVIDENCE-QUALIFICATION",
            "responsible_notebooks": [19, 20, 21],
            "basis": (
                "SCOPED_REVIEWS_EXIST; FULL_POPULATION_ACCEPTANCE "
                "IS_NOT_ESTABLISHED_BY_THIS_REVIEW"
            ),
            "next_work": (
                "For each blocked release metric, identify the exact "
                "producer artifact, response version, measurement unit, "
                "eligible denominator and reference qualification. "
                "Reuse sufficient existing evidence; specify only "
                "the remaining concrete gap."
            ),
            "constraints": [
                "A null aggregate does not prove no prior review exists.",
                "Do not rerun all cases because a binding is incomplete.",
                "Do not convert scoped counts into population accuracy.",
            ],
            "implementation_started": False,
        },
        {
            "id": "QUALITY-ABSTENTION",
            "responsible_notebooks": [20, 21],
            "basis": "HISTORICAL_CONDITIONAL_MODEL_REFERENCE_F1_EXISTS",
            "next_work": (
                "Determine applicability of the historical reference "
                "and denominator to the intended release. If qualified "
                "and below target, define targeted Development correction."
            ),
            "constraints": [
                "Keep the historical conditional F1 distinct from live accuracy.",
                "No retrospective threshold changes.",
            ],
            "implementation_started": False,
        },
        {
            "id": "RELEASE-TECHNICAL-CLOSURE",
            "responsible_notebooks": [22],
            "basis": "EXISTING_RELEASE_GATE_LIMITATIONS_PRESERVED",
            "next_work": (
                "Track producer/policy/dependency and distribution "
                "requirements from the existing gate. Keep notebook "
                "health work deferred under the user's decision."
            ),
            "constraints": [
                "Documentation export clearance is not whole-package clearance.",
                "Bounded restore is not full product reproducibility.",
                "Do not reopen deferred health work in this review.",
            ],
            "implementation_started": False,
        },
    ]

    supplement = {
        "artifact_type": "NB22_QUALITY_REVIEW_DISPOSITION_SUPPLEMENT",
        "schema_version": "NB22_C054_QUALITY_DISPOSITION_V1",
        "status": "QUALITY_EVIDENCE_RECONCILED_PRIOR_NONACCEPTANCE_PRESERVED",
        "subtasks_completed": 4,
        "checkpoint_manifest": copy.deepcopy(gate_checkpoint),
        "prior_gate": copy.deepcopy(c47["output"]),
        "quality_review": copy.deepcopy(c53["output"]),
        "confirmed_recorded_metrics": metrics,
        "family_findings": copy.deepcopy(review["family_findings"]),
        "audit_limits": copy.deepcopy(review["coverage_and_limits"]),
        "interpretation": {
            "prior_evaluation_work_exists": True,
            "evidence_understanding_improved": True,
            "engine_modified": False,
            "new_performance_improvement_measured": False,
            "overall_independent_product_accuracy_established": False,
            "prior_nonacceptance_decision_overturned": False,
        },
        "existing_nonpass_gate_names": copy.deepcopy(
            gate["nonpass_gate_names"]
        ),
        "existing_requirements_record": copy.deepcopy(
            gate["recorded_requirements_from_C042"]
        ),
        "next_workstreams": workstreams,
        "broad_evidence_review_closed_in_declared_scope": True,
        "new_search_or_experiment_authorized_by_this_record": False,
        "health_deferral_preserved": True,
        "release_acceptance_granted": False,
        "phase2_completion_claimed": False,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    supplement_output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C054.quality_disposition",
        "C054.quality_disposition_supplement",
        supplement,
    )

    # Derive a successor without mutating the historical C049 handoff.
    handoff = copy.deepcopy(prior_handoff)
    handoff["schema_version"] = "NB22_C054_HANDOFF_QUALITY_SUPPLEMENT_V1"
    handoff["status"] = (
        "HANDOFF_SUPPLEMENTED_WITH_RECONCILED_QUALITY_EVIDENCE_"
        "RELEASE_NOT_ACCEPTED"
    )
    handoff["parent_handoff"] = copy.deepcopy(c49["handoff_output"])
    handoff["supplement_producer"] = {
        "notebook": "notebooks/phase2/22_release_and_reproducibility.ipynb",
        "cell": "NB22-C054",
        "scope": "POST_CHECKLIST_QUALITY_EVIDENCE_SUPPLEMENT",
    }
    handoff["quality_evidence_supplement"] = supplement_output
    handoff["quality_review_report"] = copy.deepcopy(c53["output"])
    handoff["release_acceptance_granted"] = False
    handoff["accepted_release_input_established"] = False
    handoff["formal_NB23_start_authorized_by_this_handoff"] = False

    handoff_output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C054.quality_disposition",
        "C054.supplemented_NB23_handoff",
        handoff,
    )

    summary = {
        "status": supplement["status"],
        "subtasks_completed": 4,
        "quality_review_completed_in_scope": True,
        "confirmed_recorded_metric_count": len(metrics),
        "engine_modified": False,
        "new_performance_improvement_measured": False,
        "release_acceptance_granted": False,
        "health_deferral_preserved": True,
        "supplement": supplement_output,
        "supplemented_handoff": handoff_output,
        "original_handoff_modified": False,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
        "next_step": (
            "Define a bounded improvement/acceptance work package "
            "from the recorded workstreams; do not resume a general "
            "notebook or report scan."
        ),
    }
    print(json.dumps(summary, ensure_ascii=False, indent=2))
    return {
        "artifact": supplement,
        "output": supplement_output,
        "handoff": handoff,
        "handoff_output": handoff_output,
        "summary": summary,
    }


nb22_c54_result = nb22_c54_run(
    nb22_c47_result,
    nb22_c49_result,
    nb22_c53_result,
)

{
  "status": "QUALITY_EVIDENCE_RECONCILED_PRIOR_NONACCEPTANCE_PRESERVED",
  "subtasks_completed": 4,
  "quality_review_completed_in_scope": true,
  "confirmed_recorded_metric_count": 6,
  "engine_modified": false,
  "new_performance_improvement_measured": false,
  "release_acceptance_granted": false,
  "health_deferral_preserved": true,
  "supplement": {
    "path": "reports/phase2/release/NB22.C054.quality_disposition/C054.quality_disposition_supplement.dbf1fd8e5cb2575583351d52e230d5e57f05c7ac61b8a9716e52266d1546d5c2.json",
    "file_bytes_sha256": "dbf1fd8e5cb2575583351d52e230d5e57f05c7ac61b8a9716e52266d1546d5c2",
    "bytes": 15932,
    "write_status": "CREATED",
    "reload_validation": "PASS"
  },
  "supplemented_handoff": {
    "path": "reports/phase2/release/NB22.C054.quality_disposition/C054.supplemented_NB23_handoff.f41ed8f332785201ec29a7e9ca6c7f60c122730044419d5b833745fca17c059f.json",
    "file_bytes_sha256": "f41ed8f332785201ec29a7e9ca6c7f60c122730044419d5b833745fca17c059f

<div dir="rtl" align="right">

### NB22 — C055 | کار تکمیلی ۱: صحت‌سنجی بازیابی — بستهٔ ۱ از ۲

چهار زیرکار کل کار:
۱. اتصال نسخه و فایل‌های سؤال، Split، Gold و خروجی بازیابی؛
۲. بررسی تعریف سنجه، مخرج و موارد حذف‌شده؛
۳. بازمحاسبهٔ مستقل از نتایج ذخیره‌شده؛
۴. مقایسه با گزارش و ثبت اختلاف احتمالی.

بستهٔ فعلی زیرکارهای ۱ و ۲ را آماده می‌کند:
- مصرف مانیفست‌های موجود در حافظه؛
- استخراج اتصال‌های مرتبط؛
- خواندن ایستای توابع ارزیابی تولیدکننده؛
- نمایش نتیجهٔ تاریخی برای مقایسهٔ بعدی.

هیچ تابع تاریخی، مدل، بازیابی یا API اجرا نمی‌شود.
JSONLهای پاسخ و نتایج در این سل خوانده نمی‌شوند.
کار بررسی کیفیت قبلی بسته است و تکرار نمی‌شود.
نتیجهٔ این سل آماده‌سازی ورودی بازمحاسبه است، نه تأیید صحت نهایی اعداد.

</div>

In [60]:
# NB22-C055 — Retrieval verification, package 1 of 2
# Cached bindings + static producer formulas; never execute producer code.

def nb22_c55_run():
    import ast
    import hashlib
    import json
    import re
    from pathlib import Path

    previous = globals().get("nb22_c54_result", {})
    previous_receipt = previous.get("output", {})
    previous_path = previous_receipt.get("path")
    if not previous_path:
        raise RuntimeError("C054_RESULT_REQUIRED")

    roots = [
        p.resolve()
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / previous_path).is_file()
        and (
            p / "data/Road_Maps/"
            "RAG_finance_phase2_modular_notebooks_execution_plan.md"
        ).is_file()
    ]
    if not roots:
        raise RuntimeError("PROJECT_ROOT_NOT_RESOLVED")
    root = roots[0]

    caches = {}
    for name in (
        "nb22_c08_result", "nb22_c13_result",
        "nb22_c16_result", "nb22_c20_result",
        "nb22_c23_result",
    ):
        result = globals().get(name)
        payloads = result.get("payloads") if isinstance(result, dict) else None
        if isinstance(payloads, dict):
            for path, payload in payloads.items():
                if path in caches and caches[path] != payload:
                    raise RuntimeError("CONFLICTING_CACHED_SOURCE_DOCUMENT")
                caches[path] = payload

    required = [
        "data/phase2/retrieval/reranker_v1/reranker_manifest.json",
        "artifacts/phase2/reranking/reranker_release_v1.json",
        "artifacts/phase2/golden_dataset_v2/golden_dataset_v2_manifest.json",
    ]
    documents = {
        path: caches[path] for path in required if path in caches
    }
    missing_cached_documents = [
        path for path in required if path not in caches
    ]
    if not documents:
        raise RuntimeError(
            "REQUIRED_BINDING_DOCUMENTS_NOT_FOUND_IN_CACHE; "
            "NO_FALLBACK_REPOSITORY_SCAN_PERFORMED"
        )

    relevant = re.compile(
        r"gold|split|query|evaluation|metric|rerank|candidate|"
        r"population|excluded|relevance|k_value", re.I
    )
    bindings = []

    def inspect(value, source, field=""):
        if isinstance(value, dict):
            for key, item in value.items():
                inspect(item, source, f"{field}/{key}")
        elif isinstance(value, list):
            for index, item in enumerate(value):
                inspect(item, source, f"{field}/{index}")
        elif relevant.search(field) and (
            isinstance(value, (str, bool, int, float)) or value is None
        ):
            # Do not expose user-specific absolute paths.
            if isinstance(value, str) and value.startswith(("/Users/", "/home/")):
                value = "<ABSOLUTE_PATH_REQUIRES_PROJECT_RELATIVE_BINDING>"
            bindings.append({
                "source_document": source,
                "field": field,
                "declared_value": value,
                "hash_definition_not_inferred_from_field_name": True,
            })

    for path, payload in documents.items():
        inspect(payload, path)

    producer_relative = "notebooks/phase2/17_reranking.ipynb"
    producer_path = (root / producer_relative).resolve()
    if root not in producer_path.parents:
        raise RuntimeError("PRODUCER_OUTSIDE_PROJECT")

    before = producer_path.stat()
    raw = producer_path.read_bytes()
    after = producer_path.stat()
    if (
        before.st_size != after.st_size
        or before.st_mtime_ns != after.st_mtime_ns
    ):
        raise RuntimeError("PRODUCER_CHANGED_DURING_READ")
    producer_hash = hashlib.sha256(raw).hexdigest()
    notebook = json.loads(raw)

    functions = []
    parse_issues = []
    function_name = re.compile(r"metric|evaluate|ndcg|recall", re.I)

    for index, cell in enumerate(notebook["cells"]):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", "")
        if isinstance(source, list):
            source = "".join(source)
        if not function_name.search(source):
            continue
        try:
            tree = ast.parse(source)
        except SyntaxError:
            parse_issues.append({
                "physical_cell_index": index,
                "reason": "NOT_PARSED_AS_PLAIN_PYTHON",
            })
            continue

        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                if not function_name.search(node.name):
                    continue
                text = ast.get_source_segment(source, node)
                functions.append({
                    "name": node.name,
                    "physical_cell_index": index,
                    "source": text,
                    "source_executed": False,
                })

    c23 = globals().get("nb22_c23_result")
    if not isinstance(c23, dict):
        raise RuntimeError("C023_RELOAD_EVIDENCE_REQUIRED")

    recorded_metrics = []
    for row in c23["artifact"].get("reload_reports", []):
        if row.get("component") != "reranker":
            continue
        validation = row.get("recorded_validation", {})
        metric_validation = validation.get("metric_validation", {})
        recorded_metrics.append({
            "source_report": row.get("path"),
            "source_report_file_bytes_sha256": row.get(
                "actual_file_bytes_sha256"
            ),
            "development_metrics": metric_validation.get(
                "development_metrics"
            ),
            "test_metrics": metric_validation.get("test_metrics"),
            "historical_tests_reexecuted": False,
        })

    artifact = {
        "artifact_type": "NB22_RETRIEVAL_VERIFICATION_INPUTS",
        "schema_version": "NB22_C055_RETRIEVAL_PACKAGE1_V1",
        "status": "RETRIEVAL_BINDINGS_AND_PRODUCER_FORMULAS_RECORDED",
        "work": 1,
        "package": "1_OF_2",
        "cached_documents_consumed": list(documents),
        "required_documents_not_in_selected_cache": missing_cached_documents,
        "relevant_declared_bindings": bindings,
        "producer": {
            "path": producer_relative,
            "file_bytes_sha256": producer_hash,
            "functions": functions,
            "static_parse_issues": parse_issues,
        },
        "recorded_comparison_metrics": recorded_metrics,
        "ready_for_formula_review": bool(functions),
        "independent_recomputation_complete": False,
        "prior_numbers_changed": False,
        "producer_notebooks_read": 1,
        "jsonl_files_read": 0,
        "models_or_producer_functions_executed": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "release_gate_modified": False,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C055.retrieval_verification",
        "C055.retrieval_bindings_and_formulas",
        artifact,
    )

    print(json.dumps({
        "status": artifact["status"],
        "package": "1_OF_2",
        "cached_documents_consumed": list(documents),
        "required_documents_not_in_selected_cache": missing_cached_documents,
        "relevant_declared_bindings": bindings,
        "producer_metric_functions": functions,
        "static_parse_issues": parse_issues,
        "recorded_comparison_metrics": recorded_metrics,
        "output": output,
        "jsonl_files_read": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "independent_recomputation_complete": False,
    }, ensure_ascii=False, indent=2))
    return {
        "artifact": artifact,
        "output": output,
        "payloads": documents,
    }


nb22_c55_result = nb22_c55_run()

{
  "status": "RETRIEVAL_BINDINGS_AND_PRODUCER_FORMULAS_RECORDED",
  "package": "1_OF_2",
  "cached_documents_consumed": [
    "data/phase2/retrieval/reranker_v1/reranker_manifest.json",
    "artifacts/phase2/reranking/reranker_release_v1.json",
    "artifacts/phase2/golden_dataset_v2/golden_dataset_v2_manifest.json"
  ],
  "required_documents_not_in_selected_cache": [],
  "relevant_declared_bindings": [
    {
      "source_document": "data/phase2/retrieval/reranker_v1/reranker_manifest.json",
      "field": "/input/candidate_record_count",
      "declared_value": 3813,
      "hash_definition_not_inferred_from_field_name": true
    },
    {
      "source_document": "data/phase2/retrieval/reranker_v1/reranker_manifest.json",
      "field": "/input/input_candidate_depth",
      "declared_value": 100,
      "hash_definition_not_inferred_from_field_name": true
    },
    {
      "source_document": "data/phase2/retrieval/reranker_v1/reranker_manifest.json",
      "field": "/input/nb16_hybri

<div dir="rtl" align="right">

### NB22 — C056 | صحت‌سنجی بازیابی — بستهٔ ۲ از ۲

چهار زیرکار:
۱. کنترل هش بایت Chunk، Golden، خروجی Hybrid و خروجی Reranker؛
۲. بازسازی نگاشت Gold با تطبیق دقیق Locator و کنترل هش منطقی آن؛
۳. بازمحاسبهٔ مستقل ۹ سنجه برای Development و Test؛
۴. مقایسه با Release و گزارش Reload، سپس ثبت نتیجه و اختلاف احتمالی.

تعریف سنجه مطابق تولیدکننده:
- Recall: تعداد Chunk مرتبط بازیابی‌شده تقسیم بر تعداد Chunkهای مرتبط همان سؤال؛
- nDCG: ارتباط دودویی با تنزیل لگاریتمی و رتبه‌بندی ایدئال؛
- جمع‌بندی: میانگین سنجهٔ سؤال‌ها، بدون حذف سؤال ناموفق؛
- Reciprocal Rank: اولین نتیجهٔ مرتبط در کل فهرست ذخیره‌شده.

ترتیب نتایج و Split تغییر نمی‌کند. خروجی‌های ذخیره‌شده مصرف می‌شوند.
هیچ مدل، Embedding، Index یا بازیابی اجرا نمی‌شود.
نتیجهٔ مستقل، Gate تاریخی را خودکار تغییر نمی‌دهد.
در صورت اختلاف، همان اختلاف ثبت می‌شود؛ عدد مطلوب‌تر خودکار انتخاب نمی‌شود.

</div>

In [61]:
# NB22-C056 — Independent retrieval metric verification, package 2 of 2
# Deterministic reads and calculations only; no producer code execution.

def nb22_c56_run(c55, c15, c44):
    import hashlib
    import json
    import math
    from collections import Counter, defaultdict
    from pathlib import Path

    receipt = c55["output"]
    root = next(
        (
            p.resolve()
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / receipt["path"]).is_file()
            and (
                p / "data/Road_Maps/"
                "RAG_finance_phase2_modular_notebooks_execution_plan.md"
            ).is_file()
        ),
        None,
    )
    if root is None:
        raise RuntimeError("PROJECT_ROOT_NOT_RESOLVED")

    release_path = "artifacts/phase2/reranking/reranker_release_v1.json"
    manifest_path = "data/phase2/retrieval/reranker_v1/reranker_manifest.json"
    release = c55["payloads"][release_path]
    producer_manifest = c55["payloads"][manifest_path]

    prior_bytes = {
        row["role"]: row
        for row in c15["artifact"]["results"]
    }
    specs = {
        "chunks": {
            "path": prior_bytes["active_chunks"]["path"],
            "sha256": prior_bytes["active_chunks"]["actual_file_bytes_sha256"],
        },
        "golden": {
            "path": prior_bytes["golden_dataset"]["path"],
            "sha256": prior_bytes["golden_dataset"]["actual_file_bytes_sha256"],
        },
        "hybrid": {
            "path": producer_manifest["input"]["nb16_hybrid_candidate_path"],
            "sha256": producer_manifest["input"]["nb16_hybrid_candidate_sha256"],
        },
        "reranker": {
            "path": release["candidate_artifact"]["path"],
            "sha256": release["candidate_artifact"]["sha256"],
        },
    }
    gold_identity = (
        "42216f67665cfc50697789dffe4eb550"
        "839ee22bcbfc0489accc407519dad4d1"
    )
    file_receipts = {}

    def checked_path(relative):
        rel = Path(relative)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError("INVALID_PROJECT_RELATIVE_PATH")
        path = (root / rel).resolve()
        if root not in path.parents:
            raise RuntimeError("FILE_OUTSIDE_PROJECT")
        return path

    def records(role):
        spec = specs[role]
        path = checked_path(spec["path"])
        before = path.stat()
        digest = hashlib.sha256()
        count = 0
        with path.open("rb") as handle:
            for number, raw in enumerate(handle, start=1):
                digest.update(raw)
                if not raw.strip():
                    raise RuntimeError(f"EMPTY_JSONL_LINE: {role}:{number}")
                row = json.loads(raw)
                if not isinstance(row, dict):
                    raise RuntimeError(f"NON_OBJECT_JSONL_ROW: {role}:{number}")
                count += 1
                yield row

        after = path.stat()
        if (
            before.st_size != after.st_size
            or before.st_mtime_ns != after.st_mtime_ns
        ):
            raise RuntimeError(f"FILE_CHANGED_DURING_READ: {role}")
        actual = digest.hexdigest()
        if actual != spec["sha256"]:
            raise RuntimeError(f"INPUT_BYTE_HASH_MISMATCH: {role}")
        file_receipts[role] = {
            "path": spec["path"],
            "file_bytes_sha256": actual,
            "bytes": after.st_size,
            "records": count,
            "matches_prior_binding": True,
        }
        print(json.dumps({
            "stage": "VERIFIED_INPUT_READ",
            "role": role,
            "records": count,
        }), flush=True)

    total_bytes = sum(
        checked_path(spec["path"]).stat().st_size
        for spec in specs.values()
    )
    print(json.dumps({
        "stage": "INDEPENDENT_RETRIEVAL_VERIFICATION_START",
        "file_count": 4,
        "read_mib": round(total_bytes / 1024**2, 2),
        "model_api_calls": 0,
    }), flush=True)

    def canonical(value):
        return json.dumps(
            value, ensure_ascii=False, sort_keys=True,
            separators=(",", ":"), allow_nan=False,
        ).encode("utf-8")

    locator_chunks = defaultdict(set)
    locator_types = defaultdict(set)
    chunk_ids = set()

    for row in records("chunks"):
        cid = str(row["chunk_id"])
        if cid in chunk_ids:
            raise RuntimeError("DUPLICATE_CHUNK_ID")
        chunk_ids.add(cid)
        members = row["member_locators"]
        if not isinstance(members, list) or not members:
            raise RuntimeError("INVALID_CHUNK_LOCATORS")
        for member in members:
            locator = member["locator"]
            if not isinstance(locator, dict) or not locator:
                raise RuntimeError("INVALID_CANONICAL_LOCATOR")
            key = canonical(locator)
            locator_chunks[key].add(cid)
            locator_types[key].add(str(row["source_type"]))

    gold = {}
    evidence_count = 0
    for row in records("golden"):
        qid = str(row["candidate_id"])
        if qid in gold:
            raise RuntimeError("DUPLICATE_GOLDEN_QUERY_ID")
        relevant = set()
        source_types = {str(item) for item in row["source_types"]}
        for evidence in row["evidence_locators"]:
            locator = evidence["source_locator"]
            if not isinstance(locator, dict) or not locator:
                raise RuntimeError("INVALID_GOLD_LOCATOR")
            key = canonical(locator)
            matches = locator_chunks.get(key, set())
            if not matches:
                raise RuntimeError("GOLD_LOCATOR_WITHOUT_CHUNK")
            if not (locator_types[key] & source_types):
                raise RuntimeError("GOLD_SOURCE_TYPE_MISMATCH")
            relevant.update(matches)
            evidence_count += 1
        if not relevant:
            raise RuntimeError("QUERY_WITH_EMPTY_RELEVANCE_SET")
        gold[qid] = relevant

    if len(chunk_ids) != 3488 or len(gold) != 3813:
        raise RuntimeError("PRODUCER_POPULATION_COUNT_MISMATCH")
    if evidence_count != 3924:
        raise RuntimeError("PRODUCER_EVIDENCE_LOCATOR_COUNT_MISMATCH")

    mapping_digest = hashlib.sha256()
    for qid in sorted(gold):
        encoded = canonical({
            "candidate_id": qid,
            "relevant_chunk_ids": sorted(gold[qid]),
        })
        mapping_digest.update(len(encoded).to_bytes(8, "big", signed=False))
        mapping_digest.update(encoded)
    actual_gold_identity = mapping_digest.hexdigest()
    if actual_gold_identity != gold_identity:
        raise RuntimeError("PRODUCER_LOGICAL_GOLD_MAPPING_MISMATCH")

    split_by_id = {}
    for row in records("hybrid"):
        qid = str(row["candidate_id"])
        if qid in split_by_id or qid not in gold:
            raise RuntimeError("HYBRID_QUERY_ID_MISMATCH")
        split = row["split"]
        if split not in ("development", "test"):
            raise RuntimeError("UNEXPECTED_SPLIT")
        split_by_id[qid] = split

    if set(split_by_id) != set(gold):
        raise RuntimeError("GOLD_HYBRID_POPULATION_MISMATCH")
    split_counts = Counter(split_by_id.values())
    if dict(split_counts) != {"development": 2664, "test": 1149}:
        raise RuntimeError("PRODUCER_SPLIT_COUNTS_MISMATCH")

    def independent_metrics(ranked, relevant):
        flags = [1.0 if cid in relevant else 0.0 for cid in ranked]
        result = {}
        for k in (5, 10):
            hits = math.fsum(flags[:k])
            result[f"hit_at_{k}"] = float(hits > 0)
            result[f"precision_at_{k}"] = hits / k
            result[f"recall_at_{k}"] = hits / len(relevant)
            dcg = math.fsum(
                relevance / math.log2(rank + 2)
                for rank, relevance in enumerate(flags[:k])
            )
            idcg = math.fsum(
                1.0 / math.log2(rank + 2)
                for rank in range(min(len(relevant), k))
            )
            result[f"ndcg_at_{k}"] = dcg / idcg if idcg else 0.0
        first = next(
            (index + 1 for index, flag in enumerate(flags) if flag),
            None,
        )
        result["reciprocal_rank"] = 1.0 / first if first else 0.0
        return result

    metric_rows = {"development": [], "test": []}
    query_results = []
    seen = set()

    for row in records("reranker"):
        qid = str(row["candidate_id"])
        if qid in seen or qid not in split_by_id:
            raise RuntimeError("RERANKER_QUERY_ID_MISMATCH")
        seen.add(qid)
        split = row["split"]
        if split != split_by_id[qid]:
            raise RuntimeError("RERANKER_HYBRID_SPLIT_MISMATCH")

        candidates = row["candidates"]
        if len(candidates) != 50:
            raise RuntimeError("RERANKER_DEPTH_DIFFERS_FROM_PRODUCER")
        if [item["rank"] for item in candidates] != list(range(1, 51)):
            raise RuntimeError("STORED_RANK_SEQUENCE_INVALID")

        ranked = [str(item["chunk_id"]) for item in candidates]
        if len(set(ranked)) != len(ranked):
            raise RuntimeError("DUPLICATE_RANKED_CHUNK")
        if not set(ranked).issubset(chunk_ids):
            raise RuntimeError("UNKNOWN_RANKED_CHUNK")

        scores = [float(item["reranker_score"]) for item in candidates]
        if (
            not all(math.isfinite(score) for score in scores)
            or any(left < right for left, right in zip(scores, scores[1:]))
        ):
            raise RuntimeError("INVALID_STORED_SCORE_ORDER")

        result = independent_metrics(ranked, gold[qid])
        metric_rows[split].append(result)
        query_results.append({
            "candidate_id": qid,
            "split": split,
            "relevant_chunk_count": len(gold[qid]),
            "metrics": result,
        })

    if seen != set(gold):
        raise RuntimeError("RERANKER_GOLD_POPULATION_MISMATCH")

    computed = {}
    for split, rows in metric_rows.items():
        computed[split] = {
            name: math.fsum(row[name] for row in rows) / len(rows)
            for name in rows[0]
        }

    comparisons = []
    tolerance = 1e-12
    references = [
        {
            "source": release_path,
            "development": release["development_metrics"],
            "test": release["test_metrics"],
        }
    ]
    for item in c55["artifact"]["recorded_comparison_metrics"]:
        references.append({
            "source": item["source_report"],
            "development": item["development_metrics"],
            "test": item["test_metrics"],
        })

    for reference in references:
        for split in ("development", "test"):
            expected = reference[split]
            if not isinstance(expected, dict):
                raise RuntimeError("REFERENCE_METRIC_TABLE_UNAVAILABLE")
            for name, value in computed[split].items():
                if name not in expected:
                    raise RuntimeError("REFERENCE_METRIC_MISSING")
                prior = float(expected[name])
                delta = value - prior
                comparisons.append({
                    "source": reference["source"],
                    "split": split,
                    "metric": name,
                    "recorded": prior,
                    "recomputed": value,
                    "absolute_delta": abs(delta),
                    "matches_within_tolerance": abs(delta) <= tolerance,
                })

    configuration = c44["extracted"]["frozen_contract"]["sections"][
        "configuration"
    ]
    thresholds = {
        "recall_at_10": configuration["retrieval_recall_at_10_min"],
        "ndcg_at_10": configuration["retrieval_ndcg_at_10_min"],
    }
    test_gate = {
        name: {
            "value": computed["test"][name],
            "threshold": threshold,
            "gap_to_threshold": max(0.0, threshold - computed["test"][name]),
            "status": (
                "PASS" if computed["test"][name] >= threshold else "FAIL"
            ),
        }
        for name, threshold in thresholds.items()
    }

    all_match = all(
        row["matches_within_tolerance"] for row in comparisons
    )
    artifact = {
        "artifact_type": "NB22_INDEPENDENT_RETRIEVAL_METRIC_VERIFICATION",
        "schema_version": "NB22_C056_RETRIEVAL_PACKAGE2_V1",
        "status": (
            "INDEPENDENT_RECOMPUTATION_MATCHES_RECORDED_METRICS"
            if all_match
            else "INDEPENDENT_RECOMPUTATION_HAS_RECORDED_METRIC_DIFFERENCES"
        ),
        "package": "2_OF_2",
        "source_preparation": c55["output"],
        "input_file_receipts": file_receipts,
        "gold_mapping": {
            "method": "EXACT_CANONICAL_SOURCE_LOCATOR_IDENTITY",
            "hash_definition": (
                "SHA256_OF_SORTED_CANONICAL_JSON_ROWS_WITH_"
                "8_BYTE_BIG_ENDIAN_LENGTH_PREFIX"
            ),
            "declared_sha256": gold_identity,
            "recomputed_sha256": actual_gold_identity,
            "match": True,
            "mapped_evidence_locator_count": evidence_count,
            "query_count": len(gold),
            "unique_relevant_chunk_count": len(set().union(*gold.values())),
        },
        "split_counts": dict(split_counts),
        "excluded_queries": 0,
        "aggregation": "MACRO_MEAN_OVER_ALL_QUERIES_IN_EACH_SPLIT",
        "reciprocal_rank_scope": "FULL_STORED_50_CANDIDATE_LIST",
        "computed_metrics": computed,
        "comparisons": comparisons,
        "absolute_comparison_tolerance": tolerance,
        "all_recorded_metric_comparisons_match": all_match,
        "recomputed_test_threshold_checks": test_gate,
        "per_query_results": query_results,
        "independent_recomputation_complete": True,
        "gold_label_semantic_correctness_established": False,
        "overall_answer_accuracy_established": False,
        "test_used_for_retuning": False,
        "model_or_retrieval_execution": False,
        "historical_artifacts_modified": False,
        "release_gate_modified": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C056.retrieval_verification",
        "C056.independent_retrieval_metric_verification",
        artifact,
    )
    print(json.dumps({
        "status": artifact["status"],
        "package": "2_OF_2",
        "input_files_verified": len(file_receipts),
        "gold_mapping_hash_match": True,
        "split_counts": dict(split_counts),
        "excluded_queries": 0,
        "computed_metrics": computed,
        "comparison_count": len(comparisons),
        "all_recorded_metric_comparisons_match": all_match,
        "maximum_absolute_delta": max(
            row["absolute_delta"] for row in comparisons
        ),
        "test_threshold_checks": test_gate,
        "independent_recomputation_complete": True,
        "gold_label_semantic_correctness_established": False,
        "output": output,
        "new_api_calls": 0,
        "network_calls": 0,
        "release_gate_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c56_result = nb22_c56_run(
    nb22_c55_result,
    nb22_c15_result,
    nb22_c44_result,
)

{"stage": "INDEPENDENT_RETRIEVAL_VERIFICATION_START", "file_count": 4, "read_mib": 145.72, "model_api_calls": 0}
{"stage": "VERIFIED_INPUT_READ", "role": "chunks", "records": 3488}
{"stage": "VERIFIED_INPUT_READ", "role": "golden", "records": 3813}
{"stage": "VERIFIED_INPUT_READ", "role": "hybrid", "records": 3813}
{"stage": "VERIFIED_INPUT_READ", "role": "reranker", "records": 3813}
{
  "status": "INDEPENDENT_RECOMPUTATION_MATCHES_RECORDED_METRICS",
  "package": "2_OF_2",
  "input_files_verified": 4,
  "gold_mapping_hash_match": true,
  "split_counts": {
    "development": 2664,
    "test": 1149
  },
  "excluded_queries": 0,
  "computed_metrics": {
    "development": {
      "hit_at_5": 0.7916666666666666,
      "precision_at_5": 0.20157657657657657,
      "recall_at_5": 0.5977872217455551,
      "ndcg_at_5": 0.5732765152763587,
      "hit_at_10": 0.8498498498498499,
      "precision_at_10": 0.11501501501501503,
      "recall_at_10": 0.6676743707993709,
      "ndcg_at_10": 0.598834230

<div dir="rtl" align="right">

### NB22 — C057 | کار تکمیلی ۲: تعیین علت سنجه‌های مسدود

سه زیرکار:
۱. مصرف فهرست دقیق سنجه‌های مسدود از Open Conditions نهایی C082؛
۲. اتصال به قرارداد فریز‌شده و گزارش Gate؛
۳. ثبت زمینهٔ تصمیم همراه با ارجاع به بررسی شواهد C053 و بازمحاسبهٔ C056.

این سل علت‌های اصلی را حفظ می‌کند و دربارهٔ نیاز به API یا اجرای تازه حدس نمی‌زند.
وجود نتیجهٔ محدود با پذیرش کل جمعیت برابر نیست؛ BLOCKED هم به معنی انجام‌نشدن همهٔ کارها نیست.

شرط پایان بسته: تعداد و محتوای ۱۲ وضعیت مسدود، قرارداد و مرجع تصمیم در یک خروجی قابل بررسی باشند.
فایل ورودی، مدل، بازیابی و آزمون تاریخی خوانده یا اجرا نمی‌شود.
Gate، چک‌لیست و آستانه‌ها تغییر نمی‌کنند.

</div>

In [62]:
# NB22-C057 — Exact blocked-metric decisions and acceptance contract
# Cached evidence only; no file reads, API calls, or historical reruns.

def nb22_c57_run(c03, c53, c56):
    import copy
    import json
    from pathlib import Path

    receipt_path = c56["output"]["path"]
    root = next(
        (
            p.resolve()
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / receipt_path).is_file()
            and (
                p / "data/Road_Maps/"
                "RAG_finance_phase2_modular_notebooks_execution_plan.md"
            ).is_file()
        ),
        None,
    )
    if root is None:
        raise RuntimeError("PROJECT_ROOT_NOT_RESOLVED")

    recomputation = c56["artifact"]
    if not (
        recomputation["independent_recomputation_complete"]
        and recomputation["all_recorded_metric_comparisons_match"]
    ):
        raise RuntimeError("RETRIEVAL_RECOMPUTATION_REVIEW_REQUIRED")

    loaded = c03["loaded"]
    schemas = {
        "open_conditions": "NB21C_C082_OPEN_CONDITIONS_V1",
        "frozen_contract": "nb21_action12_execution_contract_C025_v1",
        "recorded_gate": "NB21C_C081_ACTION45_REGRESSION_GATE_V1",
        "evidence_index": "NB21C_C082_EVIDENCE_INDEX_V1",
    }
    documents = {}
    sources = {}

    # Select top-level cached documents by exact schema.
    # Nested case records are not treated as separate reports.
    for role, schema in schemas.items():
        matches = [
            (path, payload)
            for path, payload in loaded.items()
            if isinstance(payload, dict)
            and payload.get("schema_version") == schema
        ]
        if len(matches) != 1:
            raise RuntimeError(
                f"EXACT_CACHED_DOCUMENT_NOT_UNIQUE: {role}"
            )
        path, payload = matches[0]
        documents[role] = payload
        sources[role] = {
            "path": path,
            "schema_version": schema,
            "consumption": "REUSED_PREVIOUSLY_VALIDATED_C03_DOCUMENT",
        }

    conditions = documents["open_conditions"]
    contract = documents["frozen_contract"]
    gate = documents["recorded_gate"]
    evidence_index = documents["evidence_index"]

    blocked = conditions["blocked_metric_gates"]
    if isinstance(blocked, list):
        entries = [
            {"source_entry_index": index, "record": copy.deepcopy(row)}
            for index, row in enumerate(blocked)
        ]
    elif isinstance(blocked, dict):
        entries = [
            {"source_entry_key": key, "record": copy.deepcopy(row)}
            for key, row in blocked.items()
        ]
    else:
        raise RuntimeError("UNEXPECTED_BLOCKED_METRIC_COLLECTION")

    expected_count = gate["metric_gate_status_counts"]["BLOCKED"]
    if len(entries) != expected_count:
        raise RuntimeError(
            "BLOCKED_COLLECTION_COUNT_DIFFERS_FROM_FINAL_GATE"
        )

    required_contract_sections = (
        "configuration",
        "coverage_policy",
        "metric_policy",
    )
    if any(key not in contract for key in required_contract_sections):
        raise RuntimeError("REQUIRED_CONTRACT_SECTION_MISSING")

    artifact = {
        "artifact_type": "NB22_BLOCKED_METRIC_ACCEPTANCE_CONTEXT",
        "schema_version": "NB22_C057_QUALITY_WORK2_V1",
        "status": "EXACT_BLOCKED_DECISIONS_AND_CONTRACT_CONNECTED",
        "subtasks_completed": 3,
        "source_documents": sources,
        "blocked_metric_count": len(entries),
        "blocked_metric_entries": entries,
        "recorded_gate_status_counts": copy.deepcopy(
            gate["metric_gate_status_counts"]
        ),
        "contract": {
            key: copy.deepcopy(contract[key])
            for key in required_contract_sections
        },
        "metric_gate_evidence_reference": copy.deepcopy(
            evidence_index.get("metric_gate_evidence")
        ),
        "upstream_dependency_assessments": copy.deepcopy(
            conditions.get("upstream_dependency_assessments")
        ),
        "existing_quality_review": {
            "report": c53["output"],
            "family_findings": copy.deepcopy(
                c53["artifact"]["family_findings"]
            ),
            "results_do_not_prove_all_blockers_resolved": True,
        },
        "retrieval_verification": {
            "report": c56["output"],
            "all_recorded_metric_comparisons_match": True,
            "test_threshold_checks": copy.deepcopy(
                recomputation["recomputed_test_threshold_checks"]
            ),
        },
        "acceptance_disposition": (
            "ORIGINAL_DECISIONS_PRESERVED; REVIEW_EXACT_REASONS "
            "BEFORE_ASSIGNING_BINDING_WORK_OR_NEW_EVALUATION"
        ),
        "all_blocked_metrics_resolved": False,
        "new_evaluation_or_api_needed_established": False,
        "input_files_reread": 0,
        "historical_computations_reexecuted": False,
        "new_api_calls": 0,
        "network_calls": 0,
        "release_gate_modified": False,
        "checklist_modified": False,
    }

    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C057.acceptance_context",
        "C057.blocked_metric_acceptance_context",
        artifact,
    )
    print(json.dumps({
        "status": artifact["status"],
        "subtasks_completed": 3,
        "blocked_metric_count": len(entries),
        "blocked_metric_entries": entries,
        "recorded_gate_status_counts": artifact[
            "recorded_gate_status_counts"
        ],
        "contract": artifact["contract"],
        "metric_gate_evidence_reference": artifact[
            "metric_gate_evidence_reference"
        ],
        "upstream_dependency_assessments": artifact[
            "upstream_dependency_assessments"
        ],
        "output": output,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "release_gate_modified": False,
    }, ensure_ascii=False, indent=2))
    return {
        "artifact": artifact,
        "output": output,
        "payloads": documents,
    }


nb22_c57_result = nb22_c57_run(
    nb22_c03_result,
    nb22_c53_result,
    nb22_c56_result,
)

{
  "status": "EXACT_BLOCKED_DECISIONS_AND_CONTRACT_CONNECTED",
  "subtasks_completed": 3,
  "blocked_metric_count": 12,
  "blocked_metric_entries": [
    {
      "source_entry_index": 0,
      "record": {
        "evaluated_components": [],
        "frozen_policy": {
          "definition": {
            "aggregation": "MACRO_PER_QUERY; ALSO_REPORT_REFERENCE_UNIT_COUNTS",
            "analysis_unit": "REQUIRED_REFERENCE_INFORMATION_UNIT",
            "definition": "Required reference information units correctly expressed in the answer / all required reference information units.",
            "eligible_population": "Queries with a qualified, versioned completeness rubric.",
            "name": "answer_completeness"
          },
          "mode": "NON_REGRESSION_AGAINST_COMPATIBLE_PRECHANGE_BASELINE"
        },
        "metric": "answer_completeness",
        "reason_fa": "اندازه‌گیری، مرجع، دامنه یا خط پایهٔ موردنیاز برای PASS این سنجه از شواهد مصرف‌شده احراز نشده است. نتیجهٔ محدود یا 

<div dir="rtl" align="right">

### NB22 — C058 | جمع‌بندی پذیرش و کارهای باقی‌مانده

چهار زیرکار در یک بسته:
۱. تفکیک دو FAIL بازیابی از هشت BLOCKED معنایی و چهار BLOCKED عملیاتی؛
۲. دسته‌بندی سؤال‌های Test بر اساس نتایج بازمحاسبه‌شدهٔ C056؛
۳. محاسبهٔ سقف نظری Recall@10 با اندازهٔ Gold هر سؤال؛
۴. ثبت بسته‌های کاری بعدی همراه با شواهد، محدودیت و مسئول.

تحلیل بازیابی:
- سؤال بدون نتیجهٔ مرتبط در Top10؛
- وجود نتیجهٔ مرتبط فقط در رتبه‌های بعدیِ فهرست ۵۰تایی؛
- نبود نتیجهٔ مرتبط در کل فهرست ۵۰تایی؛
- بازیابی بخشی یا همهٔ Gold.

این دسته‌ها نشانهٔ تشخیصی‌اند؛ علت ریشه‌ای یا راه‌حل قطعی از آن‌ها استنباط نمی‌شود.
سقف نظری Recall محدودیت ثابت k و اندازهٔ Gold را نشان می‌دهد؛ مجوز تغییر Gold یا آستانه نیست.

تمام ورودی‌ها از حافظه مصرف می‌شوند.
مدل، بازیابی، فایل داده و ارزیابی تاریخی دوباره اجرا یا خوانده نمی‌شوند.
پذیرش Release و تعویق هلث‌چک تغییر نمی‌کند.

</div>

In [63]:
# NB22-C058 — Final acceptance work inventory and cached retrieval diagnostics
# Consumes C053/C056/C057 only; no new data reads or model execution.

def nb22_c58_run(c53, c56, c57):
    import copy
    import json
    import math
    from collections import Counter
    from pathlib import Path

    receipt_path = c57["output"]["path"]
    root = next(
        (
            p.resolve()
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / receipt_path).is_file()
            and (
                p / "data/Road_Maps/"
                "RAG_finance_phase2_modular_notebooks_execution_plan.md"
            ).is_file()
        ),
        None,
    )
    if root is None:
        raise RuntimeError("PROJECT_ROOT_NOT_RESOLVED")

    verified = c56["artifact"]
    if not (
        verified["independent_recomputation_complete"]
        and verified["all_recorded_metric_comparisons_match"]
    ):
        raise RuntimeError("VERIFIED_RETRIEVAL_RESULTS_REQUIRED")

    blocked = {}
    for entry in c57["artifact"]["blocked_metric_entries"]:
        row = entry["record"]
        if not isinstance(row, dict):
            raise RuntimeError("EXPECTED_EXPLICIT_METRIC_RECORD")
        name = row["metric"]
        if name in blocked:
            raise RuntimeError("DUPLICATE_BLOCKED_METRIC")
        blocked[name] = row

    semantic_names = {
        "answer_completeness",
        "attribution_correctness",
        "citation_correctness",
        "citation_locator_accuracy",
        "claim_citation_coverage",
        "groundedness",
        "no_answer_precision_recall_f1",
        "safety_violation_rate",
    }
    operational_names = {
        "cost", "failure_rate", "latency", "token_usage"
    }
    if set(blocked) != semantic_names | operational_names:
        raise RuntimeError("BLOCKED_METRIC_SET_DIFFERS_FROM_REVIEWED_SCOPE")

    test_rows = [
        row for row in verified["per_query_results"]
        if row["split"] == "test"
    ]
    if len(test_rows) != 1149:
        raise RuntimeError("EXPECTED_1149_TEST_ROWS")
    if len({row["candidate_id"] for row in test_rows}) != len(test_rows):
        raise RuntimeError("DUPLICATE_TEST_QUERY")

    categories = Counter()
    gold_sizes = Counter()
    ceilings = []
    diagnostic_rows = []

    for row in test_rows:
        size = row["relevant_chunk_count"]
        metrics = row["metrics"]
        recall = metrics["recall_at_10"]
        reciprocal_rank = metrics["reciprocal_rank"]
        if not isinstance(size, int) or size < 1:
            raise RuntimeError("INVALID_GOLD_SET_SIZE")

        ceiling = min(10, size) / size
        if recall > ceiling + 1e-12:
            raise RuntimeError("RECALL_EXCEEDS_FIXED_GOLD_CEILING")
        ceilings.append(ceiling)
        gold_sizes[size] += 1

        if recall <= 1e-12:
            if reciprocal_rank > 0:
                category = "RELEVANT_ONLY_AFTER_TOP10_IN_STORED50"
            else:
                category = "NO_RELEVANT_CHUNK_IN_STORED50"
        elif recall >= 1.0 - 1e-12:
            category = "ALL_GOLD_RETRIEVED_IN_TOP10"
        else:
            category = "PARTIAL_GOLD_RETRIEVED_IN_TOP10"
        categories[category] += 1

        diagnostic_rows.append({
            "candidate_id": row["candidate_id"],
            "gold_chunk_count": size,
            "recall_at_10": recall,
            "ndcg_at_10": metrics["ndcg_at_10"],
            "full_stored_list_reciprocal_rank": reciprocal_rank,
            "fixed_gold_recall_at_10_ceiling": ceiling,
            "diagnostic_category": category,
        })

    if sum(categories.values()) != len(test_rows):
        raise RuntimeError("DIAGNOSTIC_POPULATION_ACCOUNTING_MISMATCH")

    macro_ceiling = math.fsum(ceilings) / len(ceilings)
    threshold = verified["recomputed_test_threshold_checks"][
        "recall_at_10"
    ]["threshold"]

    def blocked_summary(names):
        result = []
        for name in sorted(names):
            row = blocked[name]
            state = row.get("recorded_measurement_state", {})
            policy = row["frozen_policy"]
            result.append({
                "metric": name,
                "recorded_status": row["status"],
                "existing_measurement_state": state.get("measurement_status"),
                "recorded_reason_fa": state.get(
                    "reason_fa", row.get("reason_fa")
                ),
                "analysis_unit": policy["definition"]["analysis_unit"],
                "eligible_population": policy["definition"][
                    "eligible_population"
                ],
                "aggregation": policy["definition"]["aggregation"],
                "required_comparison_mode": policy.get("mode"),
                "policy": copy.deepcopy(policy),
                "new_api_requirement_established": False,
            })
        return result

    work_packages = [
        {
            "id": "FOLLOWUP-01",
            "title_fa": "اصلاح هدفمند بازیابی",
            "responsible_notebooks": [16, 17],
            "scope": (
                "Use the diagnostic cohorts and existing failure catalogue "
                "to define a bounded Development experiment."
            ),
            "prerequisite": (
                "Choose a specific hypothesis and affected cohort; "
                "do not retune on Test."
            ),
            "completion": (
                "Versioned comparison with unchanged metric definitions "
                "and an explicit evaluation population."
            ),
        },
        {
            "id": "FOLLOWUP-02",
            "title_fa": "کفایت ارزیابی ادعا، استناد، کامل‌بودن و انتساب",
            "responsible_notebooks": [19, 21],
            "metrics": sorted(
                semantic_names - {
                    "no_answer_precision_recall_f1",
                    "safety_violation_rate",
                }
            ),
            "prerequisite": (
                "Bind emitted units and existing assessments to exact "
                "response versions, qualified references and denominators."
            ),
            "completion": (
                "For each metric, either establish a contract-qualified "
                "result or identify the exact uncovered units."
            ),
        },
        {
            "id": "FOLLOWUP-03",
            "title_fa": "کفایت شواهد نبود پاسخ و ایمنی",
            "responsible_notebooks": [20, 21],
            "metrics": [
                "no_answer_precision_recall_f1", "safety_violation_rate"
            ],
            "prerequisite": (
                "Preserve conditional historical results; reconcile "
                "reference eligibility, disputed judgments and coverage."
            ),
            "completion": (
                "Qualified scoped results plus an explicit disposition "
                "of unknown or disputed critical cases."
            ),
        },
        {
            "id": "FOLLOWUP-04",
            "title_fa": "اندازه‌گیری عملیاتی نسخهٔ اجرایی",
            "responsible_stage": "VERSIONED_RUNTIME_EVALUATION",
            "metrics": sorted(operational_names),
            "prerequisite": (
                "Identify the actual generator/runtime and measurement "
                "boundary; reuse compatible telemetry where available."
            ),
            "completion": (
                "Version-bound timing, usage, failure and cost records "
                "with coverage and compatible baseline."
            ),
            "live_execution_authorized_by_this_record": False,
        },
        {
            "id": "FOLLOWUP-05",
            "title_fa": "بستن وابستگی‌های فنی و تصمیم ریلیز",
            "responsible_notebooks": [22],
            "prerequisite": (
                "Carry forward existing producer/policy/dependency/"
                "distribution requirements and the scoped quality results."
            ),
            "completion": (
                "A new evidence-based release decision for a specified "
                "candidate, without rewriting historical decisions."
            ),
            "deferred_health_work_reopened": False,
        },
    ]

    artifact = {
        "artifact_type": "NB22_ACCEPTANCE_REMAINING_WORK_SUMMARY",
        "schema_version": "NB22_C058_REMAINING_WORK_V1",
        "status": "VERIFICATION_AND_REMAINING_WORK_INVENTORY_COMPLETE",
        "subtasks_completed": 4,
        "sources": {
            "existing_quality_review": c53["output"],
            "independent_retrieval_verification": c56["output"],
            "blocked_metric_contract": c57["output"],
        },
        "retrieval_test_threshold_checks": copy.deepcopy(
            verified["recomputed_test_threshold_checks"]
        ),
        "retrieval_diagnostics": {
            "population_count": len(test_rows),
            "category_counts": dict(categories),
            "gold_chunk_count_histogram": {
                str(size): count
                for size, count in sorted(gold_sizes.items())
            },
            "fixed_gold_macro_recall_at_10_ceiling": macro_ceiling,
            "frozen_recall_threshold": threshold,
            "threshold_not_ruled_out_by_size_ceiling": (
                macro_ceiling >= threshold
            ),
            "ceiling_does_not_guarantee_achievability": True,
            "gold_semantic_correctness_established": False,
            "root_cause_not_inferred_from_category": True,
            "per_query_diagnostics": diagnostic_rows,
        },
        "semantic_blockers": blocked_summary(semantic_names),
        "operational_blockers": blocked_summary(operational_names),
        "metric_status_accounting": {
            "retrieval_absolute_failures": 2,
            "semantic_blocked_metrics": 8,
            "operational_blocked_metrics": 4,
            "recorded_gate_counts": copy.deepcopy(
                c57["artifact"]["recorded_gate_status_counts"]
            ),
            "metric_count_is_not_task_or_api_request_count": True,
        },
        "proposed_followup_packages": work_packages,
        "followup_package_count": len(work_packages),
        "effort_estimation": {
            "reliable_total_duration_established": False,
            "fresh_model_request_count_established": False,
            "reason": (
                "Affected units, reusable qualified verdict coverage, "
                "selected changes and runtime boundaries are not yet fixed."
            ),
        },
        "completed_now": [
            "Scoped discovery and reconciliation of existing quality evidence.",
            "Independent arithmetic verification of stored retrieval metrics.",
            "Exact connection of the twelve blocked decisions to their contract.",
            "A bounded remaining-work inventory and cached diagnostics.",
        ],
        "release_acceptance_granted": False,
        "engine_changed": False,
        "new_performance_improvement_claimed": False,
        "health_deferral_preserved": True,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "release_gate_modified": False,
        "checklist_modified": False,
    }
    output = nb22_save_json_artifact(
        root,
        "reports/phase2/release/NB22.C058.remaining_work",
        "C058.acceptance_remaining_work_summary",
        artifact,
    )

    print(json.dumps({
        "status": artifact["status"],
        "subtasks_completed": 4,
        "metric_status_accounting": artifact["metric_status_accounting"],
        "retrieval_diagnostic_counts": dict(categories),
        "fixed_gold_macro_recall_at_10_ceiling": macro_ceiling,
        "frozen_recall_threshold": threshold,
        "threshold_not_ruled_out_by_size_ceiling": (
            macro_ceiling >= threshold
        ),
        "proposed_followup_packages": work_packages,
        "followup_package_count": len(work_packages),
        "effort_estimation": artifact["effort_estimation"],
        "release_acceptance_granted": False,
        "output": output,
        "input_files_reread": 0,
        "new_api_calls": 0,
        "network_calls": 0,
        "release_gate_modified": False,
    }, ensure_ascii=False, indent=2))
    return {"artifact": artifact, "output": output}


nb22_c58_result = nb22_c58_run(
    nb22_c53_result,
    nb22_c56_result,
    nb22_c57_result,
)

{
  "status": "VERIFICATION_AND_REMAINING_WORK_INVENTORY_COMPLETE",
  "subtasks_completed": 4,
  "metric_status_accounting": {
    "retrieval_absolute_failures": 2,
    "semantic_blocked_metrics": 8,
    "operational_blocked_metrics": 4,
    "recorded_gate_counts": {
      "BLOCKED": 12,
      "FAIL": 2,
      "PASS": 2
    },
    "metric_count_is_not_task_or_api_request_count": true
  },
  "retrieval_diagnostic_counts": {
    "ALL_GOLD_RETRIEVED_IN_TOP10": 544,
    "RELEVANT_ONLY_AFTER_TOP10_IN_STORED50": 70,
    "PARTIAL_GOLD_RETRIEVED_IN_TOP10": 379,
    "NO_RELEVANT_CHUNK_IN_STORED50": 156
  },
  "fixed_gold_macro_recall_at_10_ceiling": 1.0,
  "frozen_recall_threshold": 0.85,
  "threshold_not_ruled_out_by_size_ceiling": true,
  "proposed_followup_packages": [
    {
      "id": "FOLLOWUP-01",
      "title_fa": "اصلاح هدفمند بازیابی",
      "responsible_notebooks": [
        16,
        17
      ],
      "scope": "Use the diagnostic cohorts and existing failure catalogue to define a 

<div dir="rtl" align="right">

# گزارش اختتام NB22 — checkpoint فعلی و مسیر ادامه

## ۱. وضعیت نهایی

کار اجرایی NB22 برای بسته‌بندی و بررسی checkpoint فعلی، ثبت تصمیم و آماده‌سازی تحویل‌نامه پایان یافت.

این اختتام به معنی پذیرش Release یا پایان فاز دوم نیست:

- سلامت فنی checkpoint محلی: PASS
- پذیرش نهایی Release: صادر نشده
- هلث‌چک و اجرای پاک نهایی: به تصمیم کاربر، موکول به بهینه‌سازی
- پیاده‌سازی NB23: انجام نشده
- وضعیت کیفیت تاریخی: دو PASS، دو FAIL و دوازده BLOCKED

نتایج تاریخی حفظ شده‌اند؛ آستانه‌ها یا وضعیت‌های ناموفق برای تکمیل ظاهری تغییر نکرده‌اند.

## ۲. دریافت و اتصال شواهد

- گزارش نهایی C085 و بستهٔ تحویل C082 نوت‌بوک ۲۱ مصرف شدند.
- دفتر اتصال الزامات و شواهد ساخته شد.
- پذیرش‌ها و وابستگی‌های NB19/NB20 در دامنهٔ بررسی تطبیق داده شدند.
- نسخه‌ها، مسیرها، منشأ و شکاف‌های اتصال ثبت شدند.
- وجود شکاف اتصال، به معنی انجام‌نشدن کار در کل پروژه تلقی نشد.

مواردی مانند اتصال پاسخ‌های منتخب، منشأ تاریخی کد، Policy کامل و برخی تعریف‌های هویت منطقی، در گزارش‌ها با محدودیت صریح باقی ماندند.

## ۳. کنترل فایل‌ها و یکپارچگی

- هش بایت Dataset، Chunk، Context، Evidence Registry و خروجی Reranker کنترل شد.
- فایل‌های فیزیکی Index و گزارش‌های تاریخیِ متصل بررسی شدند.
- تفاوت هش بایت فایل با هش Payload تنظیمات Reranker مشخص و تطبیق صحیح ثبت شد.
- شناسه‌ها، جمعیت سؤال‌ها، Splitها و ارجاع‌های Document/Chunk/Evidence بررسی شدند.
- شواهد معتبر قبلیِ Rank، Score و Reload مصرف شدند؛ آزمون‌های تاریخی صرفاً برای گزارش‌سازی تکرار نشدند.

جمعیت اصلی ثبت‌شده:

| بخش | تعداد |
| --- | ---: |
| Golden | ۳۸۱۳ |
| Development | ۲۶۶۴ |
| Test | ۱۱۴۹ |
| Chunk فعال | ۳۴۸۸ |

این کنترل‌ها، صحت معنایی تمام داده‌ها یا پاسخ‌ها را اثبات نمی‌کنند.

## ۴. Restore و وابستگی‌ها

- موجودی وابستگی‌ها، Snapshot، کش Embedding و Revision مدل‌ها بررسی شد.
- بازیابی مستقل کاندیدای مشخص و Snapshot Dense در پوشهٔ موقت PASS شد.
- تعداد Pointها ۳۴۸۸ بود؛ برابری تمام شناسه‌ها و هفت نمونهٔ Payload/Vector تأیید شد.
- مدل اجرا نشد و Index اصلی برای آن آزمون باز نشد.
- بازیابی کامل محصول و بسته‌شدن تمام وابستگی‌های اجرایی ادعا نشد.

## ۵. مانیفست و مستندات

- کاندیدا از موجودی اولیه به ۲۰۴ فایل گسترش یافت.
- پنج کارت مستنداتی ساخته شد.
- فهرست شواهد، یادداشت انتشار و وضعیت سازگاری ثبت شدند.
- checkpoint بعدی با ۲۲۶ فایل و حجم ۸۱۴٬۵۰۰٬۹۷۵ بایت ثبت شد.
- تمام ۲۲۶ فایل در فرایند مستقل از نظر اندازه و هش کنترل شدند.
- مسیر تکراری و ارجاع مانیفست به خودش وجود نداشت.
- بازخوانی مستقل PASS شد؛ مدل و شبکه استفاده نشدند.

کارت‌ها و اسناد تاریخی، دامنه و زمان تولید خود را حفظ می‌کنند؛ همهٔ آن‌ها سند پذیرش نهایی محصول نیستند.

## ۶. خروجی مستنداتی قابل‌اشتراک

- مسیرهای شخصی در نسخه‌های مشتق‌شدهٔ نوت‌بوک‌ها رسیدگی شدند.
- اصل نوت‌بوک‌ها و مانیفست‌های تاریخی تغییر نکردند.
- برای دو مانیفست NB19 نسخهٔ مستنداتی پوشانده‌شده ساخته شد.
- یازده ارجاع، به هشت مقصد یکتا با مسیر نسبی متصل شدند.
- اعتبارسنجی مستقل همین نگاشت PASS شد.
- خروجی مستنداتی ۱۹فایلی ساخته، اسکن و بازخوانی شد.

تأیید این خروجی مستنداتی به کل checkpoint محلی تعمیم داده نمی‌شود.
اسکن الگوهای تعریف‌شده، تضمین کشف همهٔ انواع Secret نیست.

## ۷. تصمیم Gate و تحویل

در C047 نتیجهٔ Gate و در C048 وضعیت checkpoint ثبت شد.
در C049 تحویل‌نامهٔ شواهد و وضعیت برای NB23 آماده شد.
در C054 تحویل‌نامه با یافته‌های بررسی کیفیت تکمیل شد.

تصمیم محفوظ:

- checkpoint محلی از نظر یکپارچگی معتبر است.
- Release پذیرفته نشده است.
- ورودی پذیرفته‌شدهٔ شروع رسمی NB23 احراز نشده است.
- محدودیت‌ها، مسائل باز و تعویق هلث‌چک منتقل شده‌اند.

## ۸. بررسی تکمیلی نتایج کیفیت موجود

در C050 تا C053:

- ۴۰ نوت‌بوک، شامل ۱۱ نوت‌بوک آرشیوی گلدن، بررسی شدند.
- ۲۵۰ گزارش بررسی یا از حافظه مصرف شدند.
- ۱۹۰ مشاهده دسته‌بندی شد.
- ۳۷ مشاهدهٔ منتخب با منبع اصلی تطبیق داشت.
- یک مشاهدهٔ Snapshot قبلی بدون جایگزینی با نسخهٔ جدید حفظ شد.
- بررسی گزارش‌ها محدود بود؛ تمام گزارش‌های پروژه به‌صورت جامع ممیزی نشدند.

نتیجه: کار ارزیابی قبلی وجود دارد؛ عبارت کلی «نتیجه‌ای نداریم» دقیق نبود.

| سنجهٔ ثبت‌شده | مقدار | محدودیت |
| --- | ---: | --- |
| رأی مدلِ درست از کل جمعیت تاریخی | ۹۵٫۱۳٪ | دقت مستقل محصول زنده نیست |
| رأی مدلِ درست میان داوری‌های تعیین‌شده | ۹۷٫۲۴٪ | مخرج محدودتر |
| رأی مدلِ درست میان داوری‌های معتبر | ۹۶٫۲۱٪ | مخرج متفاوت |
| Precision نبود پاسخ | ۴۲٫۸۶٪ | مرجع مدل؛ زیرجمعیت قابل ارزیابی |
| Recall نبود پاسخ | ۱۰۰٪ | همان دامنه |
| F1 نبود پاسخ | ۰٫۶۰ | ۱۱۳۱ مورد؛ ارزیابی مشروط |

بررسی‌های محدود استناد، کامل‌بودن، محل شاهد و ایمنی نیز موجودند.
این نتایج بدون تعیین واحد سنجش، مخرج و مرجع، به امتیاز کل محصول تبدیل نمی‌شوند.

## ۹. صحت‌سنجی مستقل بازیابی

در C055 و C056:

- هویت چهار فایل ورودی کنترل شد.
- نگاشت Gold با تطبیق دقیق Locator بازسازی شد و هش منطقی آن تطبیق داشت.
- سنجه‌ها برای Development و Test مستقلاً محاسبه شدند.
- هیچ سؤال از محاسبه حذف نشد.
- هر ۳۶ مقایسه با گزارش‌های قبلی تطبیق داشت.
- بیشترین اختلاف عددی حدود ۱٫۱۱×۱۰⁻۱۶ بود.

| سنجهٔ Test | نتیجهٔ تأییدشده | حد فریز‌شده | وضعیت |
| --- | ---: | ---: | --- |
| Recall@10 | ۰٫۶۳۴۳۱۷۶۲۶۱ | ۰٫۸۵ | FAIL |
| nDCG@10 | ۰٫۵۷۳۵۳۶۲۰۶۶ | ۰٫۷۵ | FAIL |
| Hit@10 | ۰٫۸۰۳۳۰۷۲۲۳۷ | — | نتیجهٔ ثبت‌شده |

این بررسی صحت محاسبات روی نسخهٔ مشخص را تأیید کرد؛ صحت معنایی برچسب‌های Gold را اثبات نکرد.

## ۱۰. تحلیل تشخیصی Test

| وضعیت | تعداد |
| --- | ---: |
| همهٔ Gold در Top10 | ۵۴۴ |
| بخشی از Gold در Top10 | ۳۷۹ |
| مرجع مرتبط فقط پس از Top10 در فهرست ۵۰تایی | ۷۰ |
| بدون مرجع مرتبط در فهرست ۵۰تایی | ۱۵۶ |
| جمع | ۱۱۴۹ |

سقف نظری میانگین Recall@10 با اندازهٔ Gold فعلی برابر ۱ است.
پس اندازهٔ Gold به‌تنهایی هدف ۰٫۸۵ را ناممکن نمی‌کند؛ رسیدن به هدف تضمین نشده است.

این دسته‌بندی علت ریشه‌ای را تعیین نمی‌کند.
نبود مرجع در فهرست ۵۰تایی، نبود آن در Corpus یا Index را اثبات نمی‌کند.
اصلاح صرفِ ۷۰ سؤال نیز برای رسیدن به هدف کافی نیست.

## ۱۱. وضعیت دوازده سنجهٔ مسدود

در C057 علت دقیق وضعیت‌ها از اسناد نهایی مصرف شد:

- هشت سنجهٔ معنایی: کامل‌بودن، انتساب، صحت استناد، محل استناد، پوشش استناد، Groundedness، نبود پاسخ و ایمنی.
- چهار سنجهٔ عملیاتی: هزینه، نرخ شکست، زمان و مصرف توکن.

BLOCKED به معنی اثبات کیفیت پایین یا انجام‌نشدن تمام کارها نیست.
دلایل شامل دامنهٔ محدود، مرجع واجدشرایط، پوشش واحدهای سنجش، اختلاف رأی، خط پایهٔ سازگار یا مرز اجرای زنده‌اند.




</div>